<a href="https://colab.research.google.com/github/sofiahuch/clasificacion-textos-esperanto/blob/main/clasificacion_esperanto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Asignatura: Minería de Textos
Curso: 2026-2027

*   Adriana Pérez
*   Sofía Huch
*   Victoria Pico



In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

SEMILLA = 42

# Fase 1 - Repositorio: Gestión del repositorio y construcción del dataset.

## 1.1 SELECCIÓN DE TEMÁTICAS:

Elegimos ropa, calzado y joyas porque son temas cercanos dentro de la indumentaria y los complementos, comparten vocabulario sobra marcas, materiales, etc.

| Clase | Categorías de Wikipedia en esperanto |
|---|---|
| ropa | `Vestoj` |
| calzado | `Piedvesto`, `Ŝuoj` |
| joyas | `Juveloj` |

Excluimos algunas ramas para no mezclar clases ni centrarnos en personas o geología. Consultamos hasta dos niveles de subcategorías, max 70 categorias y 220 articulos por clase.

In [ ]:
API= 'https://eo.wikipedia.org/w/api.php'
REPOSITORIO= 'https://github.com/sofiahuch/clasificacion-textos-esperanto'
URL_DATASET= 'https://raw.githubusercontent.com/sofiahuch/clasificacion-textos-esperanto/main/dataset_esperanto.csv'
URL_FUENTES= URL_DATASET.replace('dataset_esperanto.csv', 'fuentes_dataset.csv')
CAT= {'ropa': ['Kategorio:Vestoj'],
              'calzado': ['Kategorio:Piedvesto', 'Kategorio:Ŝuoj'],
              'joyas': ['Kategorio:Juveloj']}
EXCLUIR= ['piedvest', 'ŝu', 'juvel', 'akcesora', 'nud', 'tajlor',
           'kostumist', 'lavado', 'gemoj', 'diamanta industrio']
HEADERS = {'User-Agent': 'EsperantoDatasetCUNEF/1.0 (https://github.com/sofiahuch/clasificacion-textos-esperanto)'}

## 1.2 Construcción y 1.3 API Wikipedia

API: devuelve los artículos y su HTML, conservamos los parrafos de al menos 15 palabras y quitamas tablas, enlaces. Sustituimos punto y coma por comas.

Cada consulta tiene un máximo de tres intentos. Si no se obtiene una respuesta válida la generación se detiene y se utiliza la copia guardada.

In [ ]:
def consultar_api(parametros):
    parametros['format'] = 'json'
    parametros['formatversion'] = 2
    for intento in range(3):
        try:
            respuesta = requests.get(API, params=parametros, headers=HEADERS, timeout=15)
            if respuesta.status_code == 200:
                datos = respuesta.json()
                if isinstance(datos, dict) and 'error' not in datos:
                    return datos
        except (requests.RequestException, ValueError):
            pass
    return None

def buscar_articulos(clase):
    pendientes = CATEGORIAS[clase].copy()
    visitadas = []
    articulos = {}
    for nivel in range(3):
        siguientes = []
        for categoria in pendientes:
            if categoria in visitadas:
                continue
            if len(visitadas) >= 70:
                break
            visitadas.append(categoria)
            parametros = {'action': 'query', 'list': 'categorymembers',
                          'cmtitle': categoria, 'cmtype': 'page|subcat', 'cmlimit': 500}
            while True:
                datos = consultar_api(parametros)
                if datos is None or 'query' not in datos:
                    return None
                for pagina in datos['query']['categorymembers']:
                    if pagina['ns'] == 0 and len(articulos) < 220:
                        articulos[pagina['pageid']] = pagina['title']
                    elif pagina['ns'] == 14 and nivel < 2:
                        incluir = True
                        for palabra in EXCLUIR[clase]:
                            if palabra in pagina['title'].lower():
                                incluir = False
                        if incluir:
                            siguientes.append(pagina['title'])
                if 'continue' not in datos:
                    break
                parametros.update(datos['continue'])
        pendientes = siguientes
    return articulos

Se extraen los párrafos y se descartan aquellos dentro de tablas, listas, etc. Quitamos las referencias entre los corchetes, URLS y saltos de línea. Los enlaces conservan su texto visible y los puntos y comas internos se sustituyen por comas.

Se conservan párrafos de al menos 15 palabras. Los artículos que aparecen en varias clases se excluyen. Los textos repetidos se comparan en minúsculas y si reciben clases distintas eliminamos sus apariciones.

Guardamos como máximo los 400 primeros párrafos válidos de cada clase.

In [ ]:
def limpiar_texto(texto):
  while '[' in texto:
    inicio = texto.find('[')
    final = texto.find(']', inicio)
    if final == -1:
      break
    texto = texto[:inicio] + texto[final + 1:]
  palabras = []
  for palabra in texto.split():
    if 'http://' not in palabra and 'https://' not in palabra and 'www.' not in palabra:
            palabras.append(palabra)
    texto = ' '.join(palabras).replace(';', ',')
    return texto.replace('[', '').replace(']', '').replace('<', '').replace('>', '')


def extraer_parrafos(html):
  soup = BeautifulSoup(html, 'html.parser')
  parrafos = []
  for p in soup.find_all('p'):
        if p.find_parent('table') or p.find_parent('figure') or p.find_parent('li'):
            continue
        if p.find_parent(attrs={'class': 'hatnote'}):
            continue
        texto = limpiar_texto(p.text)
        if len(texto.split()) >= 15:
            parrafos.append(texto)
  return parrafos

def comprobar_dataset(dataset):
  if list(dataset.columns) != ['text', 'class'] or len(dataset) == 0:
        return False
  textos = []
  clases = []
  for texto, clase in dataset.values:
        if not isinstance(texto, str) or clase not in CATEGORIAS:
            return False
        if len(texto.split()) < 15 or texto.lower() in textos:
            return False
        if ';' in texto or '\n' in texto or '\r' in texto or '[' in texto or ']' in texto:
            return False
        if 'http://' in texto or 'https://' in texto or 'www.' in texto or '<' in texto or '>' in texto:
            return False
        textos.append(texto.lower())
        if clase not in clases:
            clases.append(clase)
  return len(clases) == len(CATEGORIAS)


In [ ]:
def crear_dataset():
    articulos_por_clase = {}
    apariciones = {}
    for clase in CATEGORIAS:
        articulos = buscar_articulos(clase)
        if articulos is None:
            return None
        articulos_por_clase[clase] = articulos
        for pageid in articulos:
            if pageid not in apariciones:
                apariciones[pageid] = 0
            apariciones[pageid] += 1
    textos = {}
    conflictos = []
    for clase in CATEGORIAS:
        print('Descargando:', clase)
        for pageid in articulos_por_clase[clase]:
            if apariciones[pageid] > 1:
                continue
            datos = consultar_api({'action': 'parse', 'pageid': pageid, 'prop': 'text|revid'})
            if datos is None or 'parse' not in datos:
                return None
            pagina = datos['parse']
            for texto in extraer_parrafos(pagina['text']):
                clave = texto.lower()
                if clave in textos:
                    if textos[clave]['class'] != clase:
                        conflictos.append(clave)
                else:
                    textos[clave] = {'text': texto, 'class': clase, 'pageid': pageid,
                        'titulo': pagina['title'], 'revision': pagina['revid'],
                        'url': 'https://eo.wikipedia.org/w/index.php?oldid=' + str(pagina['revid'])}
    filas = []
    cantidades = {'ropa': 0, 'calzado': 0, 'joyas': 0}
    for clave in textos:
        fila = textos[clave]
        clase = fila['class']
        if clave not in conflictos and cantidades[clase] < 400:
            filas.append(fila)
            cantidades[clase] += 1
    fuentes = pd.DataFrame(filas)
    if len(fuentes) == 0:
        return None
    dataset = fuentes[['text', 'class']].copy()
    if not comprobar_dataset(dataset):
        return None
    return dataset, fuentes

## 1.4 Generar los datos

In [ ]:
GENERAR_DATASET = False

In [ ]:
DATOS_GUARDADOS = r'''text;class;pageid;titulo;revision;url
Zorio ( japane 草 履 , zōri ) (ankaŭ flipflapo kiel adapto de la anglalingva nomo) estas speco de malferma piedfingra ŝuo uzata ĉefe pro la komforto porti ilin. La nomo flipflapo estas derivita onomatopee de la sono farita per la paŝoj kiam oni piediras portante ilin. La nomo zorio devenas el la japana plej konata origina tipo de tiaj sandaloj.;calzado;50174;Zorio;9316687;https://eo.wikipedia.org/w/index.php?oldid=9316687
Ekzistas pluraj specoj de zorioj, kun la plej signifa diferenco laŭ formo. Zorioj estas plejparte konsistantaj el plando kaj rimeno, kiu disigas la dikfingron al la aliaj kvar piedfingroj . La rimeno konektas la plandumon al la piedo ambaŭflanke. La fermitaj zorioj estas malsamaj laŭ dezajno - ili estas nur malfermitaj sur la dorsa flanko de la ŝuo por permesi enigi la piedon en la zorion.;calzado;50174;Zorio;9316687;https://eo.wikipedia.org/w/index.php?oldid=9316687
Hodiaŭ ekzistas diversaj uzoj por zorio, interalie kiel komfortaj ŝuoj por promeni sur la plaĝo kaj en la naĝejo . Malfermitaj pantofloj kaj fermitaj pantofloj povas esti komfortaj por promeni hejme aŭ surstrate. La komforto de la zori-ŝuoj reflektas en la fleksebla prezo, kiu komenciĝas per tre malmultekostaj prezoj, kaj la facileco piedvesti ilin.;calzado;50174;Zorio;9316687;https://eo.wikipedia.org/w/index.php?oldid=9316687
Zorioj devenas de Japanio , kie pajlaj zorioj estis portataj dum jarmiloj. La japanlingva nomo zōri konsistas el la ĉindevenaj morfemoj sō ( japane 草 ) “herbo” kaj ri ( japane 履 ) “ŝuo”, pro la tradicia konsisto el pajlo.;calzado;50174;Zorio;9316687;https://eo.wikipedia.org/w/index.php?oldid=9316687
Tradiciaj japanaj zorioj estas plektitaj el rizo - pajlo . Ofte oni uzas ilin kune kun tabioj, specialaj ŝtrumpetoj . Aliaj tipoj de japanaj ŝuoj estas getaoj kaj waraji .;calzado;50174;Zorio;9316687;https://eo.wikipedia.org/w/index.php?oldid=9316687
Post la Dua Mondmilito , dum kiu aliaj landoj batalis kontraŭ Japanio, la revenintaj Usonaj, Aŭstraliaj, kaj Nov-Zelandaj soldatoj popularigis zoriecajn sandalojn en siaj landoj.;calzado;50174;Zorio;9316687;https://eo.wikipedia.org/w/index.php?oldid=9316687
La migrado de japanoj en Brazilon popularigis la zorion en Brazilo, la plej granda produktanto estas la brazila marko Havaianas , kiu ekde 1962 — laŭ propra deklaro — produktis pli ol 2,3 miliardojn da paroj.;calzado;50174;Zorio;9316687;https://eo.wikipedia.org/w/index.php?oldid=9316687
Sandalo estas piedvesto , konsistanta nur el plandumo ligita per ŝnuroj aŭ rimenoj al la piedo. ( PIV );calzado;50175;Sandalo;9024913;https://eo.wikipedia.org/w/index.php?oldid=9024913
Ŝuisto riparas piedvestoj, kaj laŭ la propra nomo en iuj naciaj lingvoj ri estas nomata sandalisto, sed tiu profesio pli kaj pli malaperas de la moderna mondo;calzado;50175;Sandalo;9024913;https://eo.wikipedia.org/w/index.php?oldid=9024913
Ŝuo estas piedvesto kun solida plandumo , taŭga por eksterhejma uzo. Ŝuoj protektas piedojn kontraŭ malpuraĵoj, malmolaj grundoj, malvarmo kaj vundaj akraĵoj. Ili povas helpi pli facile paŝi , sed ankaŭ foje povas malhelpi la paŝadon (ekz-e altaj kalkanumoj aŭ platformaj ŝuoj ). Formoj de ŝuo ofte ŝanĝiĝas laŭmode.;calzado;51523;Ŝuo;9423495;https://eo.wikipedia.org/w/index.php?oldid=9423495
Oni faras ŝuojn kutime laŭ formo de piedo. La ekstera supro kutime estas farita el fleksebla materialo, ( ledo natura aŭ artefarita). La plandumo konsistas el pli rigida, ne foruziĝema materialo (ledo, plasto aŭ kaŭĉuko ), sed ĝi devas tamen bone adaptiĝi al la paŝado. La kalkanumo konsistas ofte el malmola materialo.;calzado;51523;Ŝuo;9423495;https://eo.wikipedia.org/w/index.php?oldid=9423495
Ŝtrumpo estas (ofte trikita ) vesto portata je la piedo kiu kovras la suron kaj genuon . Oni ne konfuzu ĝin kun ŝtrumpeto (aŭ duonŝtrumpo) kiu ne kovras la genuon, kaj kalsonŝtrumpoj kiu estas speco de subvesto .;calzado;51526;Ŝtrumpo;9039888;https://eo.wikipedia.org/w/index.php?oldid=9039888
Ŝtrumpoj malnove estis farataj el tolo , sed nun ili estas farataj el lano , silko , kotono aŭ nilono .;calzado;51526;Ŝtrumpo;9039888;https://eo.wikipedia.org/w/index.php?oldid=9039888
"La dikeco de la nilonŝtrumpo (la nombro da fadenoj) estas esprimata en denaro (den.). Ju pli malalta la denarnombro, des pli fajna la ŝtrumpo, sed ankaŭ: des pli fragila (la fifama ""eskalo"").";calzado;51526;Ŝtrumpo;9039888;https://eo.wikipedia.org/w/index.php?oldid=9039888
La ŝuformilo (ŝuista ŝtipo) estas formita peco el ligno , plasto aŭ metalo , kiu necesas por fari ŝuon .;calzado;103660;Ŝuformilo;9505432;https://eo.wikipedia.org/w/index.php?oldid=9505432
Ju pli precize la ŝuformilo kopias la piedon (kaj — ĉe botoj — ankaŭ la suron ), des pli bonan kunformiĝon la ŝuo havos. Krome la ŝuformilo varias laŭ modaj kriterioj. Preskaŭ ĉiuj ŝuoj estas produktataj per ŝuformiloj. Tradicie ili estis faritaj el ligno, en moderna fabrikado oni faras ilin el blokoj de plasto, kiujn oni skulptas per CNC -maŝinoj.;calzado;103660;Ŝuformilo;9505432;https://eo.wikipedia.org/w/index.php?oldid=9505432
Ŝuformiloj estas farataj el molaj materialoj (ligno, plasto), ĉar dum la proceso de ŝufarado necesas provizore fiksi la partojn de la ŝuo per prempingloj , kiuj estas forigataj post finfarado de la ŝuo.;calzado;103660;Ŝuformilo;9505432;https://eo.wikipedia.org/w/index.php?oldid=9505432
La ŝuo estas konstruata ĉirkaŭ la ŝuformilo: unue la interna plandumo estas fiksita sub la sf, poste la supra ŝuo metita super la ŝuformilo kaj fiksita al la interna plandumo. Fine la ekstera plandumo estas alkonstruata. Oni distingas la ŝuformilon por konfekciaj ŝuo aŭ boto de tiu por ortopediaj ŝuoj kaj de tiu por almezurita ŝuo.;calzado;103660;Ŝuformilo;9505432;https://eo.wikipedia.org/w/index.php?oldid=9505432
Antaŭ ol eniri la serian fabrikadon de nova ŝuo la ŝuformilo mem estas farata el ligno laŭ dudimensiaj paperaj formŝablonoj. La pretigita ligna ŝuformilo tiam estas la modelo por la ŝuformiloj el plasto, kiuj estas uzataj en amasproduktado , kaj ĝi servas por proporcia grandigo al aliaj ŝu-longoj kaj ŝu-larĝoj. Ortopedia ŝuo aŭ boto estas farata de ortopedia ŝuisto laŭ la kuracista recepto tute individue por la piedo de la paciento. Necesas gipsmodelo por individue modeli la ŝuformilon. Por fari almezuritan ŝuon la ŝuisto mezuras la longon, larĝon, alton kaj ĉirkaŭon de la piedo kaj mendas specifan lignan ŝuformilon ĉe ŝuformilisto. La ŝuisto povas detale korekti ĝin per forraspado respektive algluo de leda surmetaĵo.;calzado;103660;Ŝuformilo;9505432;https://eo.wikipedia.org/w/index.php?oldid=9505432
Ŝtrumpeto estas mallonga saka piedvesto kovranta nur la piedon kaj la malsupron de la kruro . La ŝtrumpeto ŝirmas la piedon kontraŭ malvarmo, sed ankaŭ kontraŭ froto kaŭzata de la ŝuo . Ĉar la ŝtrumpeto ŝirmas la piedon kontraŭ la frotado en la ŝuo iu kiu surhavas ŝtrumpetojn ankaŭ malpli estas ĝenata de haŭtveziketoj .;calzado;124193;Ŝtrumpeto;9187233;https://eo.wikipedia.org/w/index.php?oldid=9187233
Lignoŝuoj estas ligna piedvestaĵo kiu jam ekde antikvaj tempoj ekzistas kaj dum la Mezepoko estis portataj en grandaj partoj de Eŭropo antaŭ ĉio de laboristoj kaj terkultivistoj .;calzado;161229;Lignoŝuo;9104536;https://eo.wikipedia.org/w/index.php?oldid=9104536
Piedirado sur lignaj ŝuoj postulas specialan teknikon, tial iu kiu neniam antaŭe iris sur lignaj ŝuoj havas multan penon per tio kaj rapide perdas siajn lignajn ŝuojn.;calzado;161229;Lignoŝuo;9104536;https://eo.wikipedia.org/w/index.php?oldid=9104536
Komence botoj havis praktikan rolon, kiel la plejmultaj vestaĵoj. Rajdantoj portis botojn por protekti krurojn kaj piedojn .;calzado;164056;Boto;9155804;https://eo.wikipedia.org/w/index.php?oldid=9155804
Tia praktika rolo ekzistas ankoraŭ nun pri pluvbotoj (plej ofte el kaŭĉuko ) kaj neĝbotoj (kun interna natura aŭ artefarita felo ).;calzado;164056;Boto;9155804;https://eo.wikipedia.org/w/index.php?oldid=9155804
"Fruaj botoj konsistis el apartaj kruringo , plandumo, kaj supraĵo, uzataj por pli bone ŝirmi la maleolojn ol ŝuo aŭ sandalo . La plej malnovaj atestoj pri botoj vidiĝas en la murpentraĵoj (15.000–13.000 a. K.) de la hispana Kaverno Altamira , kie estas bildoj de ĉasistoj kun botecaj piedvestoj. El la Bronzepoko kaj Ferepoko restas botformaj vazoj, kiuj pruvas la ekziston de responda piedvesto. Anatolia bota formo, kiu longis ĝis malmulte super la maleolo, datiĝas de la tempo de Kārum Kaniš (proksimume 2000 a. K.) Ĝi estis tre populara ankaŭ poste en la hitita imperio kaj ankoraŭ en la novanatolia tempo sub Tiglatpilezero la 3-a . En Mezopotamio, aliflanke, uziĝis altaj botoj kun oblikve detranĉitaj kruringoj, kiuj kutime atingis la genuo. Ili daŭris senŝanĝe ĝis la tempo de Sanĥerib , kaj Aŝurbanipal . Mankas vortaj priskriboj, sed reliefaj bildoj atestas. Tamen el pli fruaj tempoj troviĝas sumeraj ekonomiaj tekstoj de la 3-a dinastio de Ur kaj la fru-babilona epoko de Isin , kvankam tiam bildaj prezentoj estas maloftaj kaj neklaraj. Tiel ekzemple registriĝas, ke mastrumisto ricevis šuhub 2 –ha x (m)–ba–an (""Hamban–stilajn botojn""). La asiriaj botoj estas menciitaj ankaŭ en Jesaja 9,4 kvankam botoj ne ekzistis en biblia Israelo.";calzado;164056;Boto;9155804;https://eo.wikipedia.org/w/index.php?oldid=9155804
La pantoflo plej ofte estas pli malpeza kaj supla ol ordinara ŝuo . La pantoflo ne estas por uzi en malseka ĉirkaŭaĵo por tio oni uzas zoriojn .;calzado;202222;Pantoflo;9077130;https://eo.wikipedia.org/w/index.php?oldid=9077130
Pantoflo antaŭ ĉio estas por ke la portanto havas pli da komforto : mola kaj varma je la piedoj, kaj facila por sur- kaj demeti. Estas pantofloj kiuj ĉirkaŭfermas la kalkanojn kaj pantofloj kiuj ne faras tion. Babuŝo neniam ĉirkaŭfermas la kalkanojn.;calzado;202222;Pantoflo;9077130;https://eo.wikipedia.org/w/index.php?oldid=9077130
La unuaj pantofloj, kiel la vorto mem, venas el okcidenta Azio. La plej malnovaj pantofloj estis elfositaj el koptaj tomboj (2-8-aj jarcentoj). Ili disvastiĝis tra la Bizanca imperio al Italio, poste en la 15-a jarcento en tuta Eŭropo.;calzado;202222;Pantoflo;9077130;https://eo.wikipedia.org/w/index.php?oldid=9077130
Mokaseno estas vorto kiu originas de indiana lingvo el Norda Ameriko , kaj havas la signifon: ŝuo farita el cervofelo (aŭ alia mola ledo). Ĝia plandumo kaj flankoj estas faritaj el unu peco, kunstebita je la supro, kelkfoje per uzo de aldona peco por la supraĵo. La plandumo estas supla kaj la supraĵo ofte estas ornamita per brodaĵo, aŭ alia dekoracio. Ĝi estas la historia piedvesto de la indianaj triboj de Norda Ameriko kaj ankaŭ portita de ĉasistoj, komercistoj kaj setlantoj. Estas multaj tipoj de mokasenoj de diversaj triboj .;calzado;202342;Mokaseno;9091585;https://eo.wikipedia.org/w/index.php?oldid=9091585
"La vorto eniris Esperanton el la franca lingvo , kie ĝi havas la formon babouche , ĝi devenas de la persa pāpūŝ , kiu havas ĝeneralan signifon similan al "" ŝuo "" ( pā = piedo, pūŝīdan = surmeti, kovri). Ankaŭ la turka lingvo posedas vorton papuş .";calzado;202525;Babuŝo;9045206;https://eo.wikipedia.org/w/index.php?oldid=9045206
Tabioj ( japane : 足袋) estas tradiciaj japanaj ŝtrumpetoj . Ili altas ĝis la maleolo kaj apartigas la piedan dikfingron de la aliaj piedfingroj . Oni portas tabiojn, kiam oni portas getaojn aŭ zoriojn kun tradicia kimono aŭ yukata . Tabioj estas plej ofte blankaj, ĉefe kiam temas pri oficialaj eventoj. Foje tamen viroj portas nigrajn tabiojn kaj virinoj buntajn.;calzado;206178;Tabioj;9061682;https://eo.wikipedia.org/w/index.php?oldid=9061682
Jika-tabioj estas samformaj ŝuoj, faritaj el pli dikaj fibroj kaj kun plandumo el kaŭĉuko . Jika-tabiojn portas konstrulaboristoj, ĝardenistoj kaj rikiŝistoj .;calzado;206178;Tabioj;9061682;https://eo.wikipedia.org/w/index.php?oldid=9061682
Getaoj ( japane : 下駄) estas japanaj ŝuoj , simila al lignoŝuoj kaj sandaloj . Ili estas portataj kun tradiciaj japanaj vestoj, kiel kimonoj , sed ankaŭ kun okcidentstilaj vestoj.;calzado;206179;Getaoj;9070388;https://eo.wikipedia.org/w/index.php?oldid=9070388
Ekzistas pluraj tipoj de getaoj. La eksterlande plej konataj estas farita el kruda lignotabulo, nomita dai (台, subteno), sur kiu estas metita la piedo , kiu estas tenita de rimeno , kiu pasas inter la unua kaj dua piedfingroj. La dai estas fiksita sur du vertikalaj lignopecoj, nomitaj ha (歯, la dentoj ). La klakado de tiuj lignopecoj surplanke estas tre tipa kaj estas unu el tiuj, kies malaperon bedaŭras multaj japanoj.;calzado;206179;Getaoj;9070388;https://eo.wikipedia.org/w/index.php?oldid=9070388
Formdetaloj kaj materialo de la eroj povas esti tre diversa. Tiel kamparanaj getaoj estas el kruda ligno kun kanaba rimeno kaj malaltaj ha-oj por ne enkotiĝi surpade. Kontraŭe gejŝaj getaoj estas lakitaj kun silka rimeno. Dum la Edo-epoko oiran-oj aŭ tayu-oj portis tre altajn getaojn, kiuj ŝajnigis ilin misteraj kaj malhelpis paŝadon senhelpan. Eblas ankoraŭ vidi tiajn getaojn dum kabuko -prezentado.;calzado;206179;Getaoj;9070388;https://eo.wikipedia.org/w/index.php?oldid=9070388
Oni nomas ankaŭ getaoj diversajn aliajn similajn sandalojn kun plasta plandumo, kiu similas al sandaloj el rizo - pajlo ( zorioj ).;calzado;206179;Getaoj;9070388;https://eo.wikipedia.org/w/index.php?oldid=9070388
Waraji-oj ( japane : 草鞋) estas japanaj sandaloj el rizopajlo. Ili havas plandumon kaj rimenojn por fiksi la waraji-ojn ĉirkaŭ la maleolo .;calzado;206182;Waraji;8567590;https://eo.wikipedia.org/w/index.php?oldid=8567590
En malnova Japanio, waraji-oj estis la ŝuoj de la popolo. Solide fiksitaj al piedoj, sekiĝante rapide, ili ebligos paŝadon en la rizejoj . Pro la malalta kosto kaj la solideco ili estis ŝatataj de budhismaj monaĥoj kaj de samurajoj , kiuj devis paŝi longdistance. Ekzistis ankaŭ modeloj por ĉevaloj .;calzado;206182;Waraji;8567590;https://eo.wikipedia.org/w/index.php?oldid=8567590
Bata (ĉehe kaj slovake Baťa ) estas la plej granda produktisto de ŝuoj . La ĉefsidejo de la konzerno troviĝas hodiaŭ en Lausanne .;calzado;212413;Bata (konzerno);9469387;https://eo.wikipedia.org/w/index.php?oldid=9469387
En la konzerno laboras pli ol 40.000 personoj, la konzerno havas 40 produktadejon en 26 landoj kaj prozorgas 4.600 ŝuvendejojn. Ekzistas kvar vendaj regionoj: Bata Europe kun sidejo en Lausanne , Bata Asia Pacific-Africa kun sidejo en Singapuro , Bata Latin America kun sidejo en Meksikurbo kaj Bata North America kun sidejo en Toronto .;calzado;212413;Bata (konzerno);9469387;https://eo.wikipedia.org/w/index.php?oldid=9469387
"La ŝufabriko Baťa estis fondita kiel ""T. & A. Baťa"" la 24-an de aŭgusto 1894 fare de Tomáš Baťa (1876-1932) kaj ties gefratoj Antonín kaj Anna. La fondo okazis en la Moravia urbo Zlín , Aŭstrio-Hungario . Antonín kaj Anna retiriĝis unu jaron poste el la gvidado de la entrepreno kaj transdoni la solan respondecon al Tomáš. Post morto de Antonín en 1908, Tomáš transprenis la entreprenon tute.";calzado;212413;Bata (konzerno);9469387;https://eo.wikipedia.org/w/index.php?oldid=9469387
Per enkonduko de fabrika produktado kaj livero al etkomercistoj, Baťa revoluciis la ŝuproduktadon, kiu okazis pli frue en malgrandaj unupersonaj metiejoj.;calzado;212413;Bata (konzerno);9469387;https://eo.wikipedia.org/w/index.php?oldid=9469387
La entrepreno komencis eksporti en 1909 kaj ene de kelkaj jaroj liveris jam al ĉiu mondoparto. Baťa estis unu el la malgrandaj entreprenoj, kiuj povis plenumi grandiĝantajn dezirojn dum la unua mondmilito je militistaj boto . La fabrikoj estis tiel organizitaj en eksterlando, ke ili estis sendependaj de la centra entrepreno. La entrepreno agadis sociale, apogis bonstaton de la laboristoj.;calzado;212413;Bata (konzerno);9469387;https://eo.wikipedia.org/w/index.php?oldid=9469387
Ĝis 1930, Baťa iĝis gvidanto de la monda ŝumerkato. Tomáš Baťa mortis en 1932 pro aviadila akcidento, lia duonfrato Jan Antonín Baťa iĝis la nova gvidanto de la konzerno.;calzado;212413;Bata (konzerno);9469387;https://eo.wikipedia.org/w/index.php?oldid=9469387
Post la german okupado de Ĉeĥoslovakio en marto de 1939, Jan Antonín Baťa estis arestita, sed baldaŭ liberigita. Post mallonga restado en Usono, li kaj la familio setlis en Brazilo. Li tie rekonstruis la entreprenon helpe de la eksterlandaj produktejoj. Jan Antonín Baťa fondis ankaŭ plurajn urbojn kiel Bataiporã , Bataguassu , Batatuba , Anaurilândia kaj Mariaopolis .;calzado;212413;Bata (konzerno);9469387;https://eo.wikipedia.org/w/index.php?oldid=9469387
La juna filo de Tomáš Baťa ( Tomáš J. Baťa ) agadis en Kanado, la familio post la dua mondmilito luktis por la posedaj rajtoj de la firma heredaĵo.;calzado;212413;Bata (konzerno);9469387;https://eo.wikipedia.org/w/index.php?oldid=9469387
Jan Antonín Baťa mortis en 1965 kaj Tomáš J. Baťa (Thomas J. Bata) plukonstruis la konzernon. Post la kolapso de komunisma reĝimo en 1989, Tomáš J. Baťa vizitis Ĉeĥoslovakion kaj reaĉetis la restaj firmaopartojn.;calzado;212413;Bata (konzerno);9469387;https://eo.wikipedia.org/w/index.php?oldid=9469387
"La Bata-familio kaj konzerno estas sociale tre aktiva, precipe en Kanado. En Toronto funkcias la Bata Shoe Museum . Thomas J. Bata apogis grandanime la Trent University en Peterborough , la tiea universitata biblioteko estis nomata post li ""Thomas J. Bata Library"". La Bata-fondaĵo apogas kulturajn kaj karitativajn celojn.";calzado;212413;Bata (konzerno);9469387;https://eo.wikipedia.org/w/index.php?oldid=9469387
Neĝŝuo (aŭ neĝmarŝilo ) estas ilo por promeni aŭ migri sur neĝo , konata delonge en ofte neĝkovritaj regionoj. Ili estas kutime rakedformaj (tial en kelkaj lingvoj ilia nomo estas neĝrakedoj ) kaj fiksitaj al kutimaj ŝuoj . La pezo de la persono, kiu portas ilin estas disigita sur pli granda surfaco kaj tial la piedoj malpli enprofundiĝas en la neĝon.;calzado;227537;Neĝŝuo;9104534;https://eo.wikipedia.org/w/index.php?oldid=9104534
Deveno kaj aĝo de neĝŝuoj estas nekonataj. La greka historiisto kaj filozofo Strabono (ĉ. -63 ĝis ĉ. 24 ) skribis, ke la loĝantoj de la Kaŭkazo uzis platajn ledojn por ne enprofundiĝi en neĝon, kaj ke armenoj uzis samcele rondajn lignoplatojn . Certas, ke homoj jam antaŭ tre longe uzis neĝŝuojn.;calzado;227537;Neĝŝuo;9104534;https://eo.wikipedia.org/w/index.php?oldid=9104534
Preskaŭ ĉiu indiĝena tribo de Nord-Ameriko inventis propran tipon de neĝŝuo. La plej simplaj troviĝas plej norde. Inuitoj havas du malsamajn tipojn: unu estas triangula kaj ĉirkaŭ 45 cm longa, la alia ronda. Pli sude la neĝŝuoj iĝas malpli larĝaj kaj longaj, la plej longajn uzis la Cree -indianoj, ili estis preskaŭ 2 metrojn longaj. Kutima neĝŝuo similas al tenisrakedo .;calzado;227537;Neĝŝuo;9104534;https://eo.wikipedia.org/w/index.php?oldid=9104534
Artefaritaj naĝiloj estas speco de ŝuoj kiuj similas la naturajn naĝilojn de fiŝo. Faritaj el kaŭĉuko aŭ plasto, ili helpas homojn movi tra la akvo. Oni uzas ilin en akvaj sportoj, ekz. naĝado kun spirtubo , submara plonĝado, trejnado aŭ ekzercado por plibonigi naĝan kondiĉon, ktp.;calzado;237456;Naĝilo (artefarita);9114014;https://eo.wikipedia.org/w/index.php?oldid=9114014
Naĝiloj ebligas naĝi sen movi la brakojn, do plonĝisto povas uzi la manojn por porti iun ilon, ekz. fiŝlancon , aŭ fotilon en hermetika ujo. Kiam oni piediras per tiuj ŝuoj, ekzemple en plaĝo por eniri en la maron , oni devas piediri reen, ĉar alimaniere la naĝila parto de la naĝilo malhelpos la piediradon.;calzado;237456;Naĝilo (artefarita);9114014;https://eo.wikipedia.org/w/index.php?oldid=9114014
Lignoplanduma ŝuo (en la angla clog , sveda träskor , dana træsko ) estas pantofleca ŝuo kun ligna plandumo portita kaj de viroj kaj de virinoj .;calzado;256529;Lignoplanduma ŝuo;8326693;https://eo.wikipedia.org/w/index.php?oldid=8326693
Lignoplandumaj ŝuoj evoluis el apartaj laborlignoŝuoj . Ili devenas el Anglio kaj disvastiĝis tra Svedio en la tuta Eŭropo . En la komenco la antaŭa supera parto estis el ledo , lino aŭ rafio . Nuntempe ili ekzistas el ĉiaj materialoj, kiuj estas kiel ĉiam najlitaj flanke al la ligna plandumo. Tia facila kaj materialmalriĉa konstrumaniero igis lignoplandumajn ŝuojn malmultekostaj.;calzado;256529;Lignoplanduma ŝuo;8326693;https://eo.wikipedia.org/w/index.php?oldid=8326693
En la 1970-aj kaj 1980-aj jaroj lignoplandumaj ŝuoj spertis veran bumon kaj estis portitaj kiel laborŝuo de kuracistaj dungitoj. Ankaŭ kuirejaj dungitoj mase portis tiajn ŝuojn.;calzado;256529;Lignoplanduma ŝuo;8326693;https://eo.wikipedia.org/w/index.php?oldid=8326693
Ekde la 1980-aj jaroj lignoplandumajn ŝuojn anstataŭis samtipaj sed tute plastaj ŝuoj, kiuj plej ofte estas truitaj por malhelpi ŝvitadon de la piedoj.;calzado;256529;Lignoplanduma ŝuo;8326693;https://eo.wikipedia.org/w/index.php?oldid=8326693
Piedvesto ( piedvestaĵo , ŝuaĵo ) estas produkto el ledo , gumo aŭ aliaj materioj, kiu ŝirmas piedojn antaŭ la malfavoro de vetero kaj antaŭ mekanikaj vundoj. Tio estis ĝia originala funkcio. Hodiaŭ plenumas ankaŭ aliajn taskojn, unue beligas.;calzado;266962;Piedvesto;9118307;https://eo.wikipedia.org/w/index.php?oldid=9118307
Boteto estas alta, kutime fortika ŝuo , kiu superas la maleolon , sed ne atingas la genuon (kiel faras botoj ). Ŝtal ĉapaj botetoj (aŭ laĉhava aŭ senlaĉaj) estas kutime portataj de laboristoj, ekzemple konstruistoj, meblotransportistoj, ĝardenistoj ktp, por protekti la piedojn kontraŭ falantaj objektoj aŭ akraĵoj.;calzado;267627;Boteto;7870628;https://eo.wikipedia.org/w/index.php?oldid=7870628
Botetoj povas esti aŭ laĉhavaj aŭ senlaĉaj. Aŭstraliaj laborbotetoj kutime estas senlaĉaj kaj havas elastaĵojn ĉe la flankoj por facile sur- aŭ demeti ilin.;calzado;267627;Boteto;7870628;https://eo.wikipedia.org/w/index.php?oldid=7870628
En Britio populara stilo ĉe laboristoj nomiĝas angle rigger boots kaj havas nek laĉojn nek elastaĵojn.;calzado;267627;Boteto;7870628;https://eo.wikipedia.org/w/index.php?oldid=7870628
Ankaŭ punkuloj , gotikuloj kaj razkapuloj emas porti botetojn, ofte ŝtalĉapaj kaj kun aliaj metalmuntaĵoj por doni al ili agreseman aspekton.;calzado;267627;Boteto;7870628;https://eo.wikipedia.org/w/index.php?oldid=7870628
Baletaj ŝuoj estas malpezaj ŝuoj dizajnitaj speciale por baleta dancado . Ili estas farita el mola ledo , kanvaso aŭ atlaso , kaj havas flekseblajn maldikajn plandumojn . Baletaj ŝuoj estas tipe haveblaj en rozkolora, nigra, griza kaj blanka koloroj. Tradicie, virinoj vestas rozkolorajn ŝuojn kaj viroj vestas blankajn. En la alia tradicio, viroj vestas nigrajn ŝuojn kaj virinoj vestas blankajn aŭ nigrajn ŝuojn. Karne aŭ flavo-brune kolorigita ŝuoj, estas modestaj kaj tial donas aspekton de dancado nudpiede, ili estas uzataj de ambaŭ viroj kaj virinoj.;calzado;287989;Baletaj ŝuoj;6567958;https://eo.wikipedia.org/w/index.php?oldid=6567958
Komencantaj baletistinoj, kaj baletistinoj pli junaj ol proksimume dek du jaroj de aĝo, vestas nur baletajn ŝuojn dum trejnado kaj plenumado, dum kiam pli plibonigitaj inaj dancistoj povas vesti baletajn ŝuojn aŭ surfingrebligajn ŝuojn depende de la situacio. Junaj dancistoj estas tipe limigitaj al baletaj ŝuoj ĝis kiam iliaj ostoj estas ostiĝintaj kaj iliaj muskoloj estas sufiĉe fortaj por dancado surfingre . Virdancistoj preskaŭ ĉiam vestas molajn baletajn ŝuoj, ĉar ili malofte dancas surfingre.;calzado;287989;Baletaj ŝuoj;6567958;https://eo.wikipedia.org/w/index.php?oldid=6567958
Baletaj lernejoj tipe gvidas teknikajn klasojn, en kiuj dancistoj vestas nur baletajn ŝuojn, kaj surfingrajn klasojn en kiuj dancistoj unue vestas baletajn ŝuojn por varmiĝi je la manrelo , kaj poste ŝanĝas ilin al surfingrebligajn ŝuojn por la centraj ekzercoj. Multaj baletaj lernejoj postulas minimume kvin jarojn de trejnado antaŭ permeso al iliaj dancistoj al iri surfingre.;calzado;287989;Baletaj ŝuoj;6567958;https://eo.wikipedia.org/w/index.php?oldid=6567958
Simile al multaj aliaj specoj de dancaj ŝuoj , baletaj ŝuoj povas havi fendo-planduman aŭ pleno-planduman konstruon. La fendita plandumo konsistas el du apartaj partoj, unu sub fingroj kaj la alia sub kalkano , por pli granda fleksebleco. La pleno-planduma konstruo estas pli simila al tiu de la kutimaj, ne-dancaj ŝuoj.;calzado;287989;Baletaj ŝuoj;6567958;https://eo.wikipedia.org/w/index.php?oldid=6567958
"Elastaj rubandoj estas uzataj por helpi fiksi baletajn ŝuojn al la piedoj. Ĉiu ŝuo povas esti fiksita per sola rubando trans la arko de la piedo, aŭ per du rubandoj kiuj kruciĝas je ""X"" formo je la supro de la arko. Fendo-plandumaj ŝuoj tipe havas du rubandoj, dum kiam pleno-plandumaj ŝuoj kutime havas nur unu rubandon. Ĉe solo-rubandaj ŝuoj, plejparto de fabrikantoj de baletaj ŝuoj alfiksas la rubandojn al la ŝuoj dum la fabrikada procezo. Ĉe duopo-rubandaj ŝuoj, fabrikantoj tipe alfiksas unu finon de ĉiu rubando al la ŝuo kaj lasas al la aĉetanto alfiksi la liberajn finojn de la rubandoj por optimala adapto.";calzado;287989;Baletaj ŝuoj;6567958;https://eo.wikipedia.org/w/index.php?oldid=6567958
Piedpintaj ŝuoj estas ellaboritaj de la deziro ke dancistoj aperu kvazaŭ senpezaj kaj la ŝuoj kapabligas la dancistojn danci sur la finoj de iliaj piedfingroj por sufiĉe grandaj periodoj de tempo. Ili estas normale vestataj de inaj dancistoj, kvankam viraj dancistoj povas vesti ilin por nekutimaj roloj, ekzemple la malbelaj duonfratinoj en Cindrulino , aŭ viroj plenumantaj virinajn rolojn en iuj dancaj kompanioj kiel ekzemple Les Ballets Trockadero de Montekarlo kaj Grandiva.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
En 1681 , dudek jaroj post kiam Ludoviko la 14-a fondis la Reĝan Akademion de Danco , virinoj komencis aperi sur la sceno. Tiam, la normaj virinaj baletaj ŝuoj havis kalkanumojn . En mezo de la 18-a jarcenta dancisto Marie Camargo estis la unua kiu vestis senkalkanumajn ŝuojn, kapabligantajn al ŝi plenumi saltojn kiuj devus esti malfacilaj aŭ neeblaj en la pli kutimaj ŝuoj de la epoko. Post la franca revolucio de 1789 kalkanumoj estis plene eliminitaj de normaj baletaj ŝuoj. Ĉi tiuj plato-fundaj antaŭaĵoj de la modernaj surfingrebligaj ŝuoj estis fiksitaj al la piedoj per rubandoj kaj kunigitaj sulkoj sub la piedfingroj por kapabligi dancistojn salti, turniĝi kaj plene etendi la piedojn.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
"La unuaj dancistoj plialtiĝis supren sur siajn piedfingrojn kun la helpo de invento de Charles Didelot de 1795. Lia ""flugaparato"" levis dancistojn supren, permesante al ili stari sur siaj piedfingroj antaŭ laso de la tero. Ĉi tiu heleco kaj aera kvalito estis bone pritaksita de la spektantoj, kaj tiel dancordigistoj komencis serĉi manierojn aldoni plian surfingradon en siajn laborojn.";calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
Kun progreso de dancoj en la 1800-aj, la emfazo je teknika ekzerciteco kaj scipovo pligrandiĝis, kaj estis la deziro danci surfingre sen la helpo de dratoj. Kiam Marie Taglioni unua dancis La Sylphide surfingre, ŝiaj ŝuoj estis nenio pli ol modifitaj atlasaj pantofloj, la plandumoj estis el ledo kaj la flankoj kaj piedfingroj estis flikitaj por helpi al la ŝuoj teni sian formon. Ĉar la ŝuoj de ĉi tiu periodo ne donis subtenon, dancistoj devis enmeti molaĵojn je iliajn piedfingrojn por komforto kaj fidi je la forteco de siaj piedoj kaj maleoloj por subteno.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
La venonta esence malsama formo de piedpintaj ŝuoj aperis en Italio en fino de la 19-a jarcento . Dancistoj, inter ili Pierina Legnani , vestis ŝuojn kun firma, plata platformo je la antaŭa fino de la ŝuo, anstataŭ la pli akre eltirita piedfingro de pli fruaj modeloj. Ĉi tiuj ŝuoj ankaŭ inkluzivis skatolon el multaj tavoloj de teksaĵo por enhavi la piedfingrojn, kaj pli firmajn plandumojn. Ili estis konstruitaj sen najloj kaj la plandumoj estis nur plimalmoligitaj je la piedfingroj, farante ilin preskaŭ silentajn.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
La naskiĝo de la modernaj piedpintaj ŝuoj estas ofte atribuita al Anna Pavlova , rusia baletistino de la komenco de la 20-a jarcento, kiu estis unu el la plej famaj kaj influaj dancistoj de sia tempo. Pavlova havis aparte altajn, arkigitajn instepojn , kaj pro tio ŝi estis pli vundebla dum dancado surfingre. Ŝi ankaŭ havis sveltajn piedojn pro la ekscesa premo aplikita al ŝiaj grandaj piedfingroj. Por kompensi por ĉi tion, ŝi devis enigi malmoligitajn ledajn plandumojn en siajn ŝuojn por aldona subteno kaj devis platigi kaj hardi la piedfingran areon al formo de skatolo. Ĉi tiu praktiko faris dancadon surfingre pli facilan por ŝi, kaj ĝi estis estimita per ŝiaj samtavolanoj kiel friponanta.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
Ĉiu dancisto havas unikajn piedoj, de diversaj piedfingraj longo kaj formo, arka fleksebleco kaj mekanika forteco . Sekve, plejparto de fabrikantoj de la ŝuoj produktas pli ol unu modelon de la ŝuoj, kun ĉiu modelo donanta malsaman adaptiĝon. Sendistinge de la fabrikanto aŭ modelo, tamen, ĉiuj surfingrebligaj ŝuoj komunigas du gravajn strukturajn trajtojn kiuj kapabligas la dancistojn danci sur finoj de iliaj piedfingroj:;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
La eksteraĵo de surfingrebliga ŝuo estas kovrita per teksaĵo kiu kaŝas la skatolon kaj la aliajn enajn strukturajn erojn kaj donas belan aspekton al la ŝuo. Plejparto de surfingrebligaj ŝuoj estas kovrita per atlaso , sed iuj estas haveblaj kun kanvasaj eksteraĵoj. Surfingrebligaj ŝuoj estas plej ofte haveblaj en helaj rozkoloraj koloroj kaj malpli kutime en nigra aŭ blanka koloroj. Surfingrebligaj ŝuoj povas ofte esti menditaj de la aliaj koloroj.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
En plej piedpintaj ŝuoj, la plandumo estas konstruita de sola peco de ledo kiu estas alfiksita al la ŝuo kun adhero kaj per kudrado laŭ ĝiaj randoj. La plandumo interkovras kaj fiksas la nesubfalditajn randojn de la ŝua eksteraĵa teksaĵo. Estetika aspekto estas grava por modernaj surfingrebligaj ŝuoj. Por atingi eleganta aspekto, la ŝua pli ornama ekstera teksaĵo estas elstare trajtata, kovranta la maksimuman eblan areon de videblaj surfacoj de la ŝuoj. Pro ĉi tio, la plandumo estas farata el maldika materialo por doni al ĝi minimuman profilon, kaj marĝeno de atlaso estas lerte sulkigita ĉirkaŭ ĝi tiel ke la plandumo kovras nur parton de la fundo de la ŝuo. Surfingrebligaj ŝuoj povas esti fabrikataj kun skrapitaj plandumoj kiuj provizas pli grandan trakcion , aŭ kun pli glataj plandumoj por malpligrandigita trakcio.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
Trunkoj estas tipe farita de ledo , plasto , dika papero (135 ... 300 g/m ) , aŭ tavoloj de glue-hardita ŝtofo. La fleksebleco de trunko estas difinita per ĝia dikeco kaj la speco de materialo uzata. Trunka dikeco povas esti la sama tra ĝia longo aŭ ĝi povas variiĝi laŭ ĝia longo por havi malsamajn firmecojn je malsamaj lokoj. Tranĉoj povas esti faritaj trans la trunko je loko de duon-surfingrado por pligrandigi eblecon de travolviĝo. Ankaŭ, trunka dikeco povas ŝanĝiĝi je iu punkto laŭ ĝia longo por ke realigi malsamajn firmecojn pli supre kaj pli sube de la loko de ŝanĝiĝo. Normaj surfingrebligaj ŝuoj tipe havas plenan trunkon, en kiuj la trunko estas de la plena longo de la plandumo, aŭ havas parte (ekzemple, je duono aŭ tri kvaronoj) longajn trunkojn. Multaj surfingrebligaj ŝuaj fabrikantoj ofertas elekton de trunkaj materialoj, kaj iuj konstruas ŝuojn kun agorditaj trunkoj de varianta malmoleco kaj longo.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
Dancistoj iam vestas malsaman modelojn de surfingrebligaj ŝuoj por malsama pecoj de spektaklo. En ĉi tiaj okazoj, la dancordigo ofte statigas la specon de trunko postulata. Lirika stilo povas bezoni pli molajn ŝuojn, kaj agresema stilo kun multaj turnoj estas pli facile plenumata en pezaj malmolaj ŝuoj.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
Piedpintaj ŝuoj havas du teksaĵajn rubandojn kaj elastan rubandon por esti fiksitaj al la piedo. La plejparto de la fiksado de ŝuoj al la piedoj estas farata per la teksaĵaj rubandoj. La du teksaĵaj rubandoj estas volvitaj ĉirkaŭ la dancista maleolo en kontraŭaj direktoj, interkovrante unu la alian en formo de kruco je la antaŭo. La randoj estas tiam kunnodigitaj , kaj la nodo estas tiam metita sub la rubando en la eno de la maleolo por kaŝi ĝin de vido. La randoj de la teksaĵaj rubandoj estas neniam nodigataj kiel banto , ĉar banta nodo povas ne kunteniĝi esperinde. La elasta rubando, kiu trairas la antaŭon de la maleolo pli sube de la teksaĵaj rubandoj tenas la kalkanumon de la ŝuo en loko ĉe la piedo kiam la dancisto estas surfingre.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
La situoj kie la rubandoj estas alfiksitaj al ŝuo estas grava, ĉar malĝusta lokigo povas rezulti je malbone almuntita ŝuo. Optimala lokigo dependas de la atributoj de la piedo, kaj do la rubandoj ne povas esti ligitaj al la ŝuo dum fabrikada procezo. Post ekhavo de nova paro de surfingrebligaj ŝuoj, dancisto devas difini la konvenajn situoj por la rubandoj kaj tiam alkudri ilin.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
Dancistoj ofte mildigas novajn surfingrebligajn ŝuojn por plibonigi ilian adaptiĝon kaj tial elimini la malkomforton kaŭzitan per novaj ŝuoj. Diversaj manieroj estas uzataj, ekzemple batado de ili per malakraj objektoj, vestado de ili kun malsekigitaj piedfingraj skatoloj, aŭ simple tolerado de la malkomforto kaj dancado en ili ĝis kiam ili nature mildiĝos.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
Post kiam surfingrebliga ŝuo estas preparita, komune ĝia piedfingra skatolo ankoraŭ ne provizas komforton por la dancista piedo. Pro tio, dancistoj ofte vestas piedfingrajn molajn enmetojn , kiu estas teksaĵo-kovritaj ĝelaj molaĵoj kiuj enkapsuligas la piedfingrojn. Ankaŭ, ĝelaj piedfingraj spacigiloj estas iam enigitaj inter la piedfingroj por ĝustigi piedfingran aranĝon.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
Dum la normala uzado, estas du specoj de eluzado de surfingrebliga ŝuo kiu difinas ĝian utilan vivperiodon. La plej grava estas eluzado de la trunko. Pri tio ke la korpo de la ŝuo estas ripete fleksiĝas, la trunko pomalmulte malfirmigas kaj perdas sian kapablon provizi subtenon. Surfingrebliga ŝuo estas jam ne servokapabla kiam la trunko rompiĝas aŭ iĝas tro molan por provizi subtenon.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
La alia grava speco de eluzado engaĝas la eksteraĵan teksaĵon. Dum surfingrado, la antaŭa edro kaj funda rando de la piedfingra skatolo estas frotata je la sceno. Ĉi tiu frotado povas fari truon en la ŝua ekstera teksaĵo, per tio eksponante la piedfingran skatolon kaj kreante laksajn, fibriĝitajn teksaĵajn randojn. Malsimile al okazo de malfirmiĝinta trunko, difektita ekstera teksaĵo ne influas la eblecon surfingriĝi per la ŝuo. Tamen pro ĝia malbela aspekto, ŝuo kun difektita teksaĵo povas esti uzata nur por trejnado kaj provludado.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
Sub modera uzado, paro de surfingrebligaj ŝuoj estos tipe eluzata dum inter 10 kaj 12 horoj de vestado. Por studantoj de dancado, ĉi tio ofte estas en semajnoj aŭ monatoj de uzo de paro de surfingrebligaj ŝuoj. Profesiaj dancistoj tipe eluzas surfingrebligajn ŝuojn multe pli rapide, nova paro povas eluziĝi dum sola spektaklo.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
Duon-surfingraj ŝuoj, kiu estas ankaŭ diverse nomataj kiel antaŭ-surfingraj aŭ molo-blokaj ŝuoj, havas karakterizojn de ambaŭ baletaj ŝuoj kaj surfingrebligaj ŝuoj. Ĝia ekstera aspekto similas al tiu de surfingrebligaj ŝuoj kaj ili havas piedfingran skatolon, kvankam la skatolo estas pli mola kaj la flankoj de la skatolo estas tipe ne tiel profundaj kiel tiuj de surfingrebligaj ŝuoj. Malsimile al surfingrebligaj ŝuoj, duon-surfingraj ŝuoj ne havas trunkon, tiel ili ne provizas la subtenon necesan por surfingrado, kaj do studantoj de dancoj tipe ricevas fortajn avertojn pri la danĝeroj de starado surfingre en duon-surfingraj ŝuoj.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
Duon-surfingraj ŝuoj estas iam uzataj por trejni dancistojn por kiuj la surfingra tekniko estas nova. En ĉi tiu okazoj, la duon-surfingraj ŝuoj servas por alkutimigi la dancistojn al la sento de portado de surfingrebligaj ŝuoj en preparado por dancado surfingre en surfingrebligaj ŝuoj. Duon-surfingraj ŝuoj povas ankaŭ esti uzataj en spektakloj en kiuj la aspekto de surfingrebligaj ŝuoj estas dezirata, sed surfingrado ne estas plenumata.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
Notu, ke dum kiam la surfingrebligaj ŝuoj estas bezonataj por surfingrado (laŭ la nuna danca tekniko), suon-surfingraj ŝuoj tute ne nepras por duon-surfingrado . Duon-surfingrado povas facile esti farita en molaj baletaj ŝuoj aŭ eĉ nudpiede.;calzado;290622;Piedpintaj ŝuoj;9258642;https://eo.wikipedia.org/w/index.php?oldid=9258642
Bastoŝuo estas malalta ŝuo , plektita el bastrubandoj ( tiliaj , salikaj , ulmaj kaj aliaj) aŭ el betulŝelo . Bastoŝuo fiksiĝas al la piedo per laĉoj , torditaj el la samaj bastrubandoj, el kiuj bastoŝuo mem estas farita.;calzado;291575;Bastoŝuo;7946494;https://eo.wikipedia.org/w/index.php?oldid=7946494
Por elfari unu bastoŝuon necesas sep bastrubandoj, ĉiu longa je du metroj . Larĝeco de unu bastrubando egalas proksimume al larĝeco de polekso de viro , kiu mem elfaradis bastrubandojn kaj poste mem plektis bastoŝuojn.;calzado;291575;Bastoŝuo;7946494;https://eo.wikipedia.org/w/index.php?oldid=7946494
En antaŭindustriaj socioj ŝuistoj laboras kaj laboris en malgrandaj atelieroj ofte hejme aŭ ĉestrate en tre malgrandaj ĉambretoj, ekzemple sub ŝtuparo, bazara budo ktp. En nunaj industriaj socioj eblas ŝuistoj kiuj ĉefe riparas, sed ne fabrikas, ŝuojn en ankaŭ malgrandaj, sed ofte modernaj kaj sufiĉe luksaj, atelieroj. Tiuj ŝuistoj kiuj fabrikas ŝuojn laboras plej ofte en fabrikoj. En tradiciaj regionoj kun grandaj ŝufabrikoj ofte privatuloj povas labori hejme aŭ en malgrandaj atelieroj por fari specifajn taskojn por la apudaj fabrikoj.;calzado;296163;Ŝuisto;9218133;https://eo.wikipedia.org/w/index.php?oldid=9218133
En kamparaj socioj, kiel ekzemple en Hispanio ĝis la fino de la 20a jarcento , kutimis, ke kriplitaj homoj, ĉefe lamuloj , dediĉis sin al profesio de ŝuriparado hejme, ĉar ili ne povis tiom facile labori en kamparo kiel kultivistoj.;calzado;296163;Ŝuisto;9218133;https://eo.wikipedia.org/w/index.php?oldid=9218133
Pluvboto aŭ gumboto estas genu- ĝis sur -alta akvorezista boto . Tradicie tiaj botoj estis faritaj el gumo , sed nuntempe oni uzas ankaŭ termoplastajn plaston .;calzado;296282;Pluvboto;8789439;https://eo.wikipedia.org/w/index.php?oldid=8789439
En Eŭropo oni unue ne multe interesiĝis pri akvorezistaj botoj aŭ aliaj vestaĵoj el gumo. Kiam varmegis la botoj gluis ĉe la korpo kaj kiam malvarmegis ili truiĝis. Nur kiam en 1840 Charles Goodyear hazarde malkovris, ke kaŭĉuko miksita kun sulfuro kaj fulgo kaj varmigita, vulkaniziĝas kaj iĝas daŭre elasta kaj ne gluema, gumproduktoj kaj do ankaŭ gumbotoj trovis amase aĉetantojn.;calzado;296282;Pluvboto;8789439;https://eo.wikipedia.org/w/index.php?oldid=8789439
"La usonano Hiram Hutchinson aĉetis licencon de Charles Goodyear por la fabrikado de gumbotoj. Li fondis en la jaro 1853 en Francujo fabrikon por gumbotoj, ĉar la tiama ĉefe kamparana Eŭropo ŝajnis al li ege bona merkato. Liajn botojn li vendis sub la marko ""A l'Aigle"" (de la aglo ), kiel tributo al la usona blazonbesto. La firmao Hutchinson ekzistas ankoraŭ nuntempe kaj fabrikas altkvalitajn gumbotojn por profesia kaj feritempa uzo. La markonomo estis plilongigita al ""Aigle"" .";calzado;296282;Pluvboto;8789439;https://eo.wikipedia.org/w/index.php?oldid=8789439
"Henry Lee Norris translokiĝis en la jaro 1855 de Ameriko al Skotlando por tie krei gumbotfabrikon. Post unu jaro ekfabrikiĝis la unuaj botoj de la North British Rubber Company en Edinburgo . Ĉirkaŭ 100 jarojn poste, vintre 1955 , la firmao prezentis tiam novmodelan verdan gumboton, kiu ankoraŭ nuntempe estas konata kiel "" Original Hunter Boot"" ( Hunter estas la kortumliveristo de la brita reĝfamilio).";calzado;296282;Pluvboto;8789439;https://eo.wikipedia.org/w/index.php?oldid=8789439
"En la jaro 1927 la franco Claude Chamot volis produkti gumbotojn. Li elektis kiel markonomon ""Le Chameau"" (la kamelo ), kromnomon, kiun li ricevis infanaĝe. Ankaŭ tiu firmao daŭre produktas altkvalitajn gumbotojn.";calzado;296282;Pluvboto;8789439;https://eo.wikipedia.org/w/index.php?oldid=8789439
Dum jardekoj pluvbotoj estis la preferata piedvesto dum malseka vetero, ĉar kutimaj ŝuoj plej ofte havis ledajn plandumojn , kiuj moliĝis dum pluvado. En la 1980-aj jaroj tamen gumbotoj elmodiĝis, almenaŭ por feritempa kaj sporta uzoj, ĉar vendiĝis akvorezistaj ŝuoj el Gore-Tex kaj aliaj novaj materialoj. Nur fine de la 1990-aj jaroj gumbotoj denove modiĝis. Post kiam komence de la 2000-aj jaroj steluloj portis gumbotoj dum diversaj eventoj, ili iĝis ekde la jaroj 2005 moda aldonaĵo, kiu pli kaj pli disvastiĝis. Ĉefe virinaj modeloj estas nuntempe tre koloraj aŭ buntaj.;calzado;296282;Pluvboto;8789439;https://eo.wikipedia.org/w/index.php?oldid=8789439
Galoŝo estas pied-vesto, plej ofte kaŭĉuka, kiun oni metas sur la ŝuojn por ŝirmi ilin kontraŭ malsekeco, malpureco, neĝo ktp.;calzado;298153;Galoŝo;9058554;https://eo.wikipedia.org/w/index.php?oldid=9058554
Gamaŝo estas senplanda piedvesto, uzata por ŝirmi la supron de la ŝuo kaj la malsupran parton de la kruro. Gamaŝoj estas faritaj el tuko, tolo, kaŭĉuko aŭ ledo . Oni laĉis aŭ butonis ilin de la flanko. Ili ŝirmu kontraŭ vundiĝoj, malpuraĵoj, malvarmo kaj malsekeco. Historie gamaŝoj estis uzataj precipe de soldatoj, nuntempe ĉefe ĉe diversaj sportoj kaj por ĉevaloj.;calzado;298158;Gamaŝo;9088966;https://eo.wikipedia.org/w/index.php?oldid=9088966
La vorto devenas de hispana nomo por specifa ledo-speco gaudamaci , kies origino estas la araba gild gadamasiy („Ledo el Ghadames “). En la 17-a jarcento la vorto eniris la francan kiel gamache kaj de tie en aliaj lingvoj.;calzado;298158;Gamaŝo;9088966;https://eo.wikipedia.org/w/index.php?oldid=9088966
Ŝukorno aŭ enŝuigilo estas ilo uzata por facile glitigi la piedon en ŝuon . Speciale se temas pri novaj ŝuoj, kiuj ankoraŭ ne estas 'uzadaptitaj', ĝi povas esti utila instrumento. Tial ŝukorno ofte estas uzata en ŝuvendejoj.;calzado;392232;Ŝukorno;9118130;https://eo.wikipedia.org/w/index.php?oldid=9118130
Ŝukorno ankaŭ preventas difektiĝon de la kalkanuma ŝuparto dum surmetado de la ŝuo. Per ŝukorno la piedo pli facile ŝoviĝas en la ŝuon kaj la kalkanparto de la ŝuo ne faldiĝas internen, pro kio la postaĵo restas firma.;calzado;392232;Ŝukorno;9118130;https://eo.wikipedia.org/w/index.php?oldid=9118130
En antaŭaj tempoj la ilo estis farita el laŭlonge segita korno sed nuntempe plejofte el metalo aŭ plasto .;calzado;392232;Ŝukorno;9118130;https://eo.wikipedia.org/w/index.php?oldid=9118130
Ĝi konsistas el ligna breto kiu estas subtenata je unu flanko tiel ke ĝi staras iom oblikve rilate al la horizontala ebeno. Je la plej alta flanko estas noĉo en kiu oni povas meti la kalkanumon de la piedvesto. Per la alia piedo oni ekstaras sur la breto post kio oni povas detiri la boton aŭ ŝuon.;calzado;436546;Bototirilo;8775607;https://eo.wikipedia.org/w/index.php?oldid=8775607
La kalkanumo estas la parto de ŝuo aŭ alia piedvesto kiu troviĝas je la suba flanko de ĝi, tutposte. La kalkanumo estas muntita sur la plandumo , aŭ tuj lokita kaj muntita sur la tordiĝinta ledo, aŭ formas unu tutaĵon kun la plandumo. Dum starado aŭ irado la tuta korpopezo premas sur la kalkanumo kaj plandumo.;calzado;436690;Kalkanumo;8260766;https://eo.wikipedia.org/w/index.php?oldid=8260766
La plandumo de ŝuo aŭ alia piedvesto kiel boto estas la parto kiu troviĝas sub ĝi kaj direkte kontaktas la grundon sur kiu la portanto iras. Bonaj plandumoj estas firmaj kaj tamen fleksiĝemaj.;calzado;437032;Plandumo;9101905;https://eo.wikipedia.org/w/index.php?oldid=9101905
Vadpantalono estas pantalono el likimuna materialo. La krurumoj de la pantalono finiĝas en botoj , dum la supra parto etendiĝas ĝis tuj sub la akseloj . La tuto estas suprentenata per ŝelkoj . Antaŭe tia pantalono ofte estis farita el kaŭĉuko aŭ kanvaso , nuntempe el polivinilklorido .;calzado;471900;Vadpantalono;9201214;https://eo.wikipedia.org/w/index.php?oldid=9201214
Vadpantalonoj estas uzataj por fari laboron en sufiĉe profunda akvo, sen fariĝi malseka. Ekzemple en la fiŝhokado , riparaj kaj ŝipsavaj laboroj, la kontraŭbatalo de ondatroj kaj helpo dum inundoj .;calzado;471900;Vadpantalono;9201214;https://eo.wikipedia.org/w/index.php?oldid=9201214
Vadbotoj estas botoj de maksimuma longo. La kruringoj finiĝas tuj sub la ingvenoj . Vadbotoj estas uzataj por laboroj en ne tro profunda akvo, koto , sterko k.t.p. Ili ofte estas uzataj de fiŝhokantoj kaj aliaj fiŝistoj, kaj ankaŭ de biologoj kaj aliaj personoj kiuj laboras en nomitaj medioj. Vadpantalono estas uzata por laboroj en pli profunda akvo.;calzado;471964;Vadbotoj;8493085;https://eo.wikipedia.org/w/index.php?oldid=8493085
Platformaj ŝuoj (konataj ankaŭ kiel dancobotoj , aŭ simple platformoj ) estas ŝuoj , botoj , aus sandaloj kun dikaj plandoj. Ili estas ofte faritaj el korko , plasto , gumo, aŭ ligno (platformaj ŝuoj kun lignaj plandoj estas, teknike, ankaŭ lignoŝuoj ). Platformaj ŝuoj estis uzitaj, pro tialoj kiaj modo aŭ por aldoni (ŝajnige) altecon, en variaj kulturoj ekde la Antikveco .;calzado;497306;Platforma ŝuo;8407675;https://eo.wikipedia.org/w/index.php?oldid=8407675
Irilo laŭ esperantlingvaj vortaroj kiel PIV aŭ Reta Vortaro estas longa stango, provizita per altefiksita piedapoga hoko, kiun oni uzas pare por iri starante alte super la tero kaj por trapasi marĉojn. Sinonime foje uzatas la neologismo stilzo (el la ĝermana lingvaro , ekzemple germane Stelze , nederlande stelt kaj angle stilt ). Ofte iriloj uzatas akrobatike aŭ kiel parto de rola alivestiĝo en cirko , teatro aŭ karnavalo .;calzado;547473;Irilo;9049378;https://eo.wikipedia.org/w/index.php?oldid=9049378
Piedĉifono estas peco de lina , kotona , flanela (por vintro) aŭ alia teksaĵo , kutime havanta la formon de kvarangulo aŭ kvadrato . Piedĉifonojn uzis soldatoj anstataŭ ŝtrumpojn aŭ ŝtrumpetojn volvante ilin ĉirkaŭ siajn piedojn por sorbi ŝviton kaj protekti kontraŭ humido kaj malvarmo. Ĉifonitajn piedojn oni metis en botojn aŭ aliajn soldatŝuojn. En Orienta Eŭropo piedĉifonoj estis vaste uzataj en armeoj ankoraŭ en la 21-a jarcento .;calzado;555872;Piedĉifono;8406566;https://eo.wikipedia.org/w/index.php?oldid=8406566
La avantaĝoj de piedĉifono estas pli granda fortikeco ol ĉe ŝtrumpeto kaj universala grandeco. Ĝi sekiĝas pli rapide post lavado kaj eblas ĝin memfari el ajna teksaĵo. Tamen surmeti piedĉifonon estas pli komplike kaj daŭras multe pli longe ol ĉe ŝtrumpeto, kaj malĝuste volvita povas kaŭzi frot-vundojn. Ĝi estas ankaŭ pli granda ol ŝtrumpeto kaj postulas pli grandajn ŝuojn. Ne eblas uzi piedĉifonojn sen ŝuoj, kiuj fikstenas ilin.;calzado;555872;Piedĉifono;8406566;https://eo.wikipedia.org/w/index.php?oldid=8406566
Futbalŝuoj estas piedvestoj uzataj en futbalo . Futbalŝuoj dezajnitaj por herbaj futbalejoj (d) havas butonojn sur la plandumo por pli firme teni sin sur la grundo . Futbalŝuoj, multe progresinte de simpla komenco, nuntempe estas celo de multe da esplorado, evoluigo, financo, kaj reklamado ene de industrio tutmonda. Komence ili estis botoj kaj oni ankoraŭ foje nomas ilin tiel, sed modernaj futbalŝuoj jam ne estas veraj botoj ĉar ili ili ne kovras la maleolon. La baza dezajno kaj aspekto konverĝis kun aliaj sportoŝuoj ekde la jardeko de 1960.;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Ekster futbalo, futbalŝuojn ofte uzas en rugbeo 15 kaj rugbeo 13 ludantoj en specifaj pozicioj prefere ol specialajn rugbeŝuojn. Ili uziĝas ankaŭ en aliaj surherbaj sportoj kiel kanada bastonludo , iomano , kvidiĉo (d) kaj eĉ ŝnurtira lukto .;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Dum la 19-a jarcento futbalo fariĝis popularega en Britujo . La ludantoj uzadis siajn pezajn, malmolajn laborbotojn. Tiuj estis la plej fruaj botoj kun ŝtala fingroŝirmilo ĉe la pinto, longaj laĉoj, kaj alta kruringo. Ili havis ankaŭ metalajn butonojn malsupre por pli stabila kaj firma kontakto kun la grundo. En la malfrua parto de la jarcento oni dezajnis la unuan specife futbalan boton, faritan el dika kaj peza ledo kaj altan ĝis la maleolo por pli bona protekto. Ĝia pezo estis 500 g kaj duobliĝis kiam la boto malsekiĝis.;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Ĉi tiam la stilo de futbalŝuoj restis tre simpla, kvankam kelkaj famaj produktantoj, interalie Gola , Hummel kaj Valsport , fariĝis ĉiam pli popularaj dum la intermilita periodo.;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Post la Dua Mondmilito, la dezajno de futbalŝuoj draste ŝanĝiĝis. Unue sudamerikaj futbalistoj komencis uzi pli malpezajn kaj flekseblajn ŝuojn, kiuj poste eniris la atenton de aliaj kontinentoj. Tiu dezajno celis plibonigi regadon de la pilko kaj forton de piedbatoj anstataŭ protekton de la piedo. En 1954 Adi Dassler enkondukis ŝraŭbeblajn butonojn, kiuj donis al la okcident-germana teamo percepteblan avantaĝon dum la multepluva Mondpokalo de tiu jaro. (Disputas la aserton, ke unue Dassler eltrovis ŝraŭbeblajn butonojn, lia frato Rudolf Dassler , fondinto de Puma .);calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Post 1960 multaj futbalŝuoj estis dezajnataj kun pli malalta rando, malplia pezo, kaj plia flekseblo. Tio permesis, ke la plej bonaj eŭropaj kaj sudamerikaj futbalistoj moviĝu kaj ŝanĝu sian direkton pli rapide. Al la batalo aliĝis Mitre , Joma kaj Asics . Adidas iĝis la plej sukcesa produktanto, ties ŝuoj estis surhavataj de 75% de la ludistoj en la Futbala Mondpokalo 1966 .;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Post 1970 aperis multaj grandaj progresoj kaj aliiĝoj en futbalŝua dezajno, interalie malplia pezo kaj diverso de koloroj. Reklamprezentado de ŝuoj fariĝis pli ĝenerala. Adidas regis la merkaton kaj enkondukis novajn teknologiojn, ekzemple remburon por protekti la kalkanon. En 1979 ĝi firmigis sian statuson per futbalŝuo, kiu poste fariĝis la plej vendita de la tuta historio, la Copa Mundial . Dum tiu tempo ekuziĝis kelkaj el la plej oftaj specoj de besta ledo: kangurua , bovida , kaj bova . Diadora eniris la merkaton. En la jardeko de 1980 la teknologiaj progresoj daŭris kaj eniris la merkaton Umbro , Lotto , kaj Kelme .;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Novaj specoj de plandumoj estis enkondukitaj por plibonigi la ekvilibron de la futbalisto. La Adidas Predator , dezajnita de la aŭstraliano Craig Johnston fine de la antaŭa jardeko, estis prezentita en 1994 kaj ĝuis tujan sukceson. Mizuno , Reebok , Uhlsport , kaj Nike ekproduktis futbalŝuojn. La unua ŝuo de Nike, Nike Mercurial Vapor , impaktis tuj kiam oni prezentis ĝin en 1998 kaj Ronaldo uzis ĝin en la Futbala Mondpokalo 1998 .;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Oni enkondukis laseran teknologion kaj tiel produktis la unuan tute individuan futbalŝuon en 2006. Ankaŭ la unua senlaĉa ŝuo, la Lotto Zhero Gravity, estis prezentita en 2006.;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
En la nuna erao, kiam la ludritmo fariĝas pli rapida kaj la ludantoj pli teknikemaj, produktantoj daŭre enkondukis novajn progresojn en teknologio, interalie pli malpezajn ŝuojn el alternativaj materialoj. Individuigo de ŝuoj fariĝis pli konata pro la disvastiĝo de la interreto. Senlaĉaj ŝuoj forte populariĝis post kiam Adidas prezentis la Ace PureControl en 2016.;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Futbalŝuoj estas farataj el diversaj materialoj, interalie kaŭĉuko, sintezaĵoj kiel nilono kaj poliuretano, kaj ledo. La plej populara materialo, pro teksturo kaj eltendaŭro, estas ledo farita el la haŭto de kanguruo . Iuj futbalistoj preferas sintezajn materialojn eĉ pli eltenajn kaj malpli multekostajn. Multaj produktantoj ĉesigas la uzadon de kangurua ledo pro la kritikoj de aktivuloj kontraŭ la buĉado de kanguruoj.;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
"Laŭ la ludejtipo, sporto, kaj eĉ la strategia pozicio de la ludanto, diversaj ŝuaj stiloj kaj precipe diversaj butonaj aranĝoj estas uzeblaj. Sur malmolaj ludejoj, amatoroj ofte uzas ordinaran sportoŝuon aŭ plastobutonan (t.e. ""firmagrundan"" aŭ kun ""muldita plandumo ""), en la plejparto de sportoj kaj pozicioj tio sufiĉas, kvankam en dikherbaj aŭ malsekaj ludejoj ŝraŭbeblaj butonoj estas rekomendindaj por firma kontakto, ili povas esti metalaj, kaŭĉukaj, aŭ plastaj, kaj la ŝuoj povas nomiĝi ""malfirmagrundaj"". Ludante sur tia ludejo, kelkaj ludantoj preferas havi sur la nedomina (apoga) piedo ŝuon kun ŝraŭbeblaj butonoj por firmeco, kaj sur la domina (pilkobata) piedo ŝuon kun mallongaj kaŭĉukaj aŭ plastaj butonoj por akurateco. La plejparto tamen preferas havi samspecajn ŝuojn sur ambaŭ piedoj.";calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Por arenfutbalo oni uzas plankajn ŝuojn. Tiuj havas kaŭĉukajn plandumojn celitajn al maksimume firma kontakto kun la planko. Iuj estas dezajnitaj surbaze de firmagrundaj futbalŝuoj kaj iuj estas dezajnitaj speciale por arenaj ludejoj. Sur artefarita herbo iuj ludistoj uzas ordinarajn firmagrundajn futbalŝuojn. Sed, ĉar la uzelteno sur artefarita herbo estas multe malpli longa, Nike kaj aliaj produktantoj evoluigis specialajn futbalŝuojn por tiaj ludejoj, kun malpli grandaj, rondaj butonoj.;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Por rugbeo 15 , oni preferas ŝraŭbeblajn butonojn, precipe en la pozicioj de prop , hooker , kaj lock , kie plia firmo estas bezonata por disputataj skrumoj. La butonoj devas havi longon de maksimume 21 mm. Rugbeaj ŝuoj ofte pezas pli ol en aliaj futbalaj variantoj. Unu el la plej rimarkeblaj malsamoj inter futbalaj kaj rugbeaj ŝuoj estas la aranĝo de la butonoj - rugbeŝuoj ordinare havas ne malpli ol naŭ butonojn dum futbalŝuoj povas havi ĝis minimumo de ses. Iuj rugbeŝuoj havas altan randon ĉirkaŭ la maleolo, sed tiu tipo malplipopulariĝas, precipe ĉe la elita nivelo. Estas pluraj tipoj de rugbeŝuoj, celitaj por ludantoj en malsamaj pozicioj.;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Ŝraŭbeblaj butonoj estas malpermesitaj en iuj ligoj de aŭstralia futbalo ekde la jardeko de 1990 pro la ofteco de gravaj vundoj al ludantoj pro kolizioj kun la metalo. Antaŭ tio, preferoj pri ŝraŭbeblaj butonoj estis plejparte laŭ la vetero. Arbitraciantoj devas kontroli ĉiujn ŝuojn antaŭ la ekludo por scii, ĉu la butonoj estas difektitaj, por eviti vundojn.;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Lastatempe oni fabrikas mulditajn plandumojn kun butonoj en pluraj direktoj, teorie por maksimumigi firmecon kaj minimumigi maleolajn vundojn. Ili tamen estas kritikataj kiel kaŭzo de foje gravaj vundoj al ludantoj. La angla klubo Manchester United eĉ malpermesis ke ĝiaj sportistoj uzu tiajn ŝuojn post kiam Wayne Rooney kaj David Beckham , interalie, suferis plurfojajn metatarsajn vundojn.;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Komence, nur nigraj futbalŝuoj estis haveblaj, sed lastatempe ili vendiĝas en diversaj koloroj , ekzemple bluo, verdo, ruĝo, blanko, flavo, orkoloro, kaj eĉ rozkoloro. Famaj firmaoj, interalie Nike , Adidas , Puma ktp., havas la plej altan vendadon. La ĉeflokaj ŝuoj estas Phantom VNM, Phantom VSN, Tiempos kaj The Nike Mercurial Vapor uzitaj de Cristiano Ronaldo kaj aliaj. La germana firmao Adidas produktas la serion Predator , uzitan de David Beckham , Gary Neville (d) , kaj Steven Gerrard , kaj la longvivan Copa Mundial . La tuta Germana nacia teamo uzis ŝuojn de Adidas dum la Futbala Mondpokalo 2006 . Alia germana entrepreno, Puma , havas kiel ĉefstatusajn produktojn Puma King Platinum, Puma Future kaj Puma One. Ŝuoj de Puma estis uzataj de lastatempaj futbalistoj kiel Sergio Agüero , kaj legenduloj kiel Pelé , Johan Cruyff kaj Diego Maradona .;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Multaj futbalistoj individuigas siajn ŝuojn por celi certan aspekton kaj fari ilin pli distingeblaj en la kluba vestejo. Nuntempe ili ofte havas kiel personan signon nomon aŭ la komencliterojn de la nomo de la posedanto, la nomojn de ties infanoj, numeron, kluban blazonon, aŭ flagon. Multaj vendantoj, interalie Nike, Adidas, kaj Puma, ofertas plurajn elektojn kaj kolorojn por individuigi futbalŝuojn per brodmaŝinoj.;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
McArthur, Ian. (1995) Elegance Borne of Brutality: An eclectic history of the football boot . London: Two Heads Publishing. ISBN 1-897850-76-X .;calzado;817318;Futbalŝuo;9333476;https://eo.wikipedia.org/w/index.php?oldid=9333476
Havaianas estas brazila varmarko de zorio . La varmarko estas produktita de la kompanio Alpargatas S.A., kiu estas parto de la konzerno Camargo Corrêa . La havaianoj estas produktitaj en unu sola fabriko, en Paranaiba en nordorienta Brazilo .;calzado;832286;Havaianas;9501036;https://eo.wikipedia.org/w/index.php?oldid=9501036
La havaianoj estas faritaj el mola kaŭĉuka aŭ plasta piedmarŝilo, tre malofte oni uzas alian materialon, kiel ledon aŭ ŝtofon . Antaŭ la internacia markigo, Ili estis tre ĉipaj kaj tre demokratiaj. Fakte, ili nur protektas la plandojn . Kiel simpla marŝprotektilo por ne tuŝi la grundon , evitante la kontakton kun tero, sablo, ŝtonoj, akvo kaj (mal)varmeco el grundo, ĝi estas tre simpla kaj utila ŝirmaĵo por la kondiĉoj de lando kiel Brazilo.;calzado;832286;Havaianas;9501036;https://eo.wikipedia.org/w/index.php?oldid=9501036
"La unuaj havaianaj zorioj estis produktitaj en 1962 , inspiritaj per la japanaj zori -sandaloj . Ĉi tiuj sandaloj estas faritaj el riztigoj , dum la havaianoj estas faritaj el kaŭĉuko . Malgraŭ la japana inspiro, la nomo elektita estis Havaianas , kiu signifas ""de Havajo "" aŭ ""havajanoj"" en la portugala .";calzado;832286;Havaianas;9501036;https://eo.wikipedia.org/w/index.php?oldid=9501036
"La havaianoj estis venditaj je malalta prezo, kaj estis konsideritaj dum multaj jaroj kiel ""piedvestojn de malriĉuloj"" .";calzado;832286;Havaianas;9501036;https://eo.wikipedia.org/w/index.php?oldid=9501036
La unuaj modeloj inkluzivis blankajn plandumojn kaj bluajn, nigrajn aŭ flavajn rimenojn. En 1991 , la administro decidis plaĉi al pliaj celgrupoj de konsumontoj, kaj komencis surmerkatigi la modelojn Havaianas Sky , en pli fortaj kolornuancoj, kun pli altaj kalkanumoj kaj je pli alta prezo .;calzado;832286;Havaianas;9501036;https://eo.wikipedia.org/w/index.php?oldid=9501036
La vendoreklamado de la nova modelo inkluzivis reklamojn havantajn lokajn artistojn. La nova reklamkampanjo estis sukcesa, kaj la kompanio komencis vastigi siajn agadojn kaj disvolvi pliajn modelojn. Tio, ekzemple, estis komencita dum la Atlanta Olimpikoj (1996) en la merkatado de alia modelo, Havaianas Olimpic . Eĉ en la sekvaj jaroj, la firmao uzis brazilajn famulojn por antaŭenigi vendadon , inter aliaj, la futbalisto Raí kaj la aktoroj Claudia Abreu, Fabio Assonsão, Reinaldo Janchini, Fernanda Lima , Rodrigo Santoro, Deborah Sekou kaj Marcos Palmeira ĉefrolis en ĝiaj reklamvideoj .;calzado;832286;Havaianas;9501036;https://eo.wikipedia.org/w/index.php?oldid=9501036
En 1997 , oni komencis eksporti havaianoj al aliaj landoj, la unua el kiuj estis Francio . Antaŭ 2006 , havaianoj jam estis eksportitaj al pli ol 80 landoj .;calzado;832286;Havaianas;9501036;https://eo.wikipedia.org/w/index.php?oldid=9501036
En 2006, proksimume 162 milionoj havaianoj estis venditaj . Ĉirkaŭ 90% el ili estis venditaj en Brazilo sole, kaj la cetero ĉirkaŭ la mondo. En 2011 , vendado saltis al proksimume 210 milionoj da havaianoj, kaj la procento de havaianoj surmerkatigitaj ekster Brazilo altiĝis al 15% . En 2013 , la valoro de la varmarko Havaianas estis taksita je 422 milionoj da dolaroj .;calzado;832286;Havaianas;9501036;https://eo.wikipedia.org/w/index.php?oldid=9501036
Sportaj ŝuoj , sportŝuoj aŭ kurŝuoj (en la usona angla athletic shoes , sports shoes aŭ sneakers , IFA : /ˈsniːkərz/ , en la brita angla trainers , IFA : /ˈtreɪnə(r)z/ ) estas ŝuoj, kiuj estas principe taŭgaj por sporto sed ankaŭ por ĉiutaga uzado nerilata al sporto.;calzado;835458;Sportŝuo;9104283;https://eo.wikipedia.org/w/index.php?oldid=9104283
Tiaj ŝuoj estas faritaj el fleksebla materialo, kutime kun kaŭĉuka plando kaj supra parto el ledo aŭ kanvaso . Ili kutime havas laĉojn , teksajn suprojn kaj molajn plandojn faritajn el kaŭĉuko aŭ artefaritaj materialoj, kaj Ili ofte estas en laŭmodaj koloraj kombinaĵoj. Sportŝuoj similas al tradiciaj sportaj kaj ŝtofaj ŝuoj kiel tenisaj kaj gimnastikaj ŝuoj. Modernaj sportŝuoj ankaŭ povas havi lapfermilon kaj pli fortajn plandojn kaj esti faritaj el mola ledo. Nuntempe, sportŝuoj estas tre populara formo de piedvesto .;calzado;835458;Sportŝuo;9104283;https://eo.wikipedia.org/w/index.php?oldid=9104283
La vorto sneakers estas origine usona angla kaj signifas ŝuojn, kiujn oni povas facile kaj trankvile enŝteliri. Kuradaj kaj trejnaj ŝuoj estas ofte ankaŭ nomataj trejnistoj en la angla.;calzado;835458;Sportŝuo;9104283;https://eo.wikipedia.org/w/index.php?oldid=9104283
"La termino ""ŝuoj"" ankaŭ povas priskribi iujn specojn de sportaj ŝuoj destinitaj por certaj sportaj agadoj, kiel kur ŝuoj, basketbalaj ŝuoj, tenisaj ŝuoj aŭ kructrejnaj (angle Crosstraining ) ŝuoj.";calzado;835458;Sportŝuo;9104283;https://eo.wikipedia.org/w/index.php?oldid=9104283
Kelkaj ekzemploj de bonkonataj kaj popularaj markoj de tiaj sportŝuoj estas: Air Jordan , Adidas , Asics , Converse (faristo de All-Stars ), Fila , Mizuno , Keds , New Balance , Nike , Puma AG , Reebok , Skechers , Vans kaj DC Shoes .;calzado;835458;Sportŝuo;9104283;https://eo.wikipedia.org/w/index.php?oldid=9104283
Sportŝuoj fariĝis grava parto de hiphopo - (ĉefe Puma, Nike kaj Adidas) kaj rokenrolo - ( Converse kaj Macbeth) kulturo ekde la 1970-aj jaroj .;calzado;835458;Sportŝuo;9104283;https://eo.wikipedia.org/w/index.php?oldid=9104283
JD Sports Fashion plc , (pli bone konata kiel JD Sports aŭ simple JD ) estas brita sporta kaj moda podetala firmao origine establita en Bury , Greater Manchester , Anglio . Ĝi estas listigita en la London Stock Exchange kaj estas parto de la indekso FTSE 100 .;calzado;835574;JD Sports;8984331;https://eo.wikipedia.org/w/index.php?oldid=8984331
La firmao estis fondita fare de John Wardle kaj David Makin. La nomo de la firmao estas nomita laŭ la unua litero de la nomo de la fondintoj (Johano kaj Davido - JD). La unua butiko estis establita en Bury, Distrikto Manĉestro en 1981 kaj dua branĉo estis establita en 1983 en Arndale, Manĉestro. Post la aĉeto de 55% de la akcioj de la firmao fare de la Grupo Pentland , en interkonsento kun valoro de 44,6 milionoj da pundoj, la branĉoj de la firmao disvastiĝis rapide tra Anglio kaj poste en la tuta mondo .;calzado;835574;JD Sports;8984331;https://eo.wikipedia.org/w/index.php?oldid=8984331
En decembro 2001, JD Sports akiris la First Sport-ĉenon de la 209-butika vesta giganto Blacks Leisure Group. En oktobro 2005, 70 Allsports-filioj estis akiritaj. Post tio, longa serio de kromaj akiroj estis faritaj:;calzado;835574;JD Sports;8984331;https://eo.wikipedia.org/w/index.php?oldid=8984331
Preter la aĉetoj kiujn ĝi faris en Usono , JD Sports forĝis kelkajn partnerecojn kaj aĉetojn eksterlande. Antaŭ majo 2009, ĝi establis 75 branĉojn en Francio. En 2016, la unua branĉo estis malfermita en Kuala-Lumpuro , Malajzio. Ĉe la fino de tiu jaro, la firmao faris unu el ĝiaj plej signifaj aĉetoj ekster Eŭropo kiam ĝi aĉetis 80% de la akcioj de la aŭstralia ĉeno Next Athleisure por 6.6 milionoj USD. Ĉi tiu paŝo aĉetis al ŝi tenon sur signifa parto de la vestaĵo kaj piedvesto merkato en Aŭstralio.;calzado;835574;JD Sports;8984331;https://eo.wikipedia.org/w/index.php?oldid=8984331
JD Sports estas la oficiala provizanto kaj sponsoro de kelkaj futbalteamoj, ludantoj kaj kluboj en Anglio rekonitaj fare de la internacia unuiĝo .;calzado;835574;JD Sports;8984331;https://eo.wikipedia.org/w/index.php?oldid=8984331
La Grupo Pentland ( angle Pentland Group ) estas brita sporta kaj moda podetala firmao , establita en 1932 en Liverpool . La firmao posedas firmaojn kiel ekzemple JD Sports kaj Speedo . Nuntempe, la ĉefsidejo de la firmao situas en Londono kaj ĉirkaŭ 50 mil homoj estas dungitaj de ĝi tutmonde .;calzado;835827;Grupo Pentland;8984353;https://eo.wikipedia.org/w/index.php?oldid=8984353
La kompanio evoluis iom post iom. En 1981 ŝi aĉetis 55% de la akcioj de la kompanio Reebok . En 1983/84 Reebok respondecis pri 70% de la spezo de la firmao. En 1985 , la akcioj de Reebok estis proponitaj por vendo en la Novjorka Borso .;calzado;835827;Grupo Pentland;8984353;https://eo.wikipedia.org/w/index.php?oldid=8984353
La firmaon estis listigita por komerco sur la Londono-Borso en 1964 , sed retenis proprieton de la plej multaj el la akcioj. En 1989 la unua publika oferto estis farita, sed en 1999 la familiaj reprezentantoj reaĉetis la akciojn kiuj estis eldonitaj al publiko. En 2019 la posedanto Stephen Robin estis la plej granda impostpaganto en Britio kaj li pagis 181,6 milionojn da pundoj .;calzado;835827;Grupo Pentland;8984353;https://eo.wikipedia.org/w/index.php?oldid=8984353
En 2005 la firmao akiris 45% de JD Sports kaj poste ĝi pliigis sian parton al 55%. Ĉi tiu firmao havas 2,500 butikojn tutmonde.;calzado;835827;Grupo Pentland;8984353;https://eo.wikipedia.org/w/index.php?oldid=8984353
En 2016 , la vendo de la firmao atingis 2,4 miliardojn da pundoj kaj profitoj pliiĝis al 220 milionoj.;calzado;835827;Grupo Pentland;8984353;https://eo.wikipedia.org/w/index.php?oldid=8984353
En 2018 , estis anoncita la kreadon de ŝuaĵa komunentrepreno kunlabore kun Lacoste . En la sama jaro, JD Sports aĉetis ĉenon de butikoj en Kalifornio kiu inkludas 83 butikojn.;calzado;835827;Grupo Pentland;8984353;https://eo.wikipedia.org/w/index.php?oldid=8984353
New Balance Athletic Shoe, Inc. (Nova Ekvilibro, akronime konata laŭ ĝia emblemo NB ) estas transnacia firmao , kiu produktas sportŝuojn , vestaĵojn kaj akcesoraĵoj. Ĝia ĉefsidejo estas en Bostono , Usono .;calzado;836027;New Balance;9134133;https://eo.wikipedia.org/w/index.php?oldid=9134133
La firmao estis establita en 1906 , kaj unue estis nomita New Balance Arch Support Company. Ĝia fondinto estis William G. Riley, 33-jaraĝa angla enmigrinto, kiu specialiĝis pri farado de arksubtenoj por preskriba piedvesto por malpezigi pieddoloron al homoj suferantaj de piedproblemoj . En 1933 , post la Granda Depresio en Usono, Arthur Hale aliĝis al la firmao kiel partnero .;calzado;836027;New Balance;9134133;https://eo.wikipedia.org/w/index.php?oldid=9134133
"En 1938 , la firmao unue produktis kurŝuon por loka kurklubo. Kaj al la kvardekaj jaroj de la 20-a jarcento, ĝi komencis produkti diversajn specojn de sportaj ŝuoj . En 1950 Riley retiriĝis de la firmao kaj en 1953 Hale vendis la komercon al sia filino Eleanor kaj ŝia edzo, Paul Kidd . En 1956 la firmao ŝanĝis sian nomon al ""New Balance Orthopaedic Laboratory"" kaj en 1963 ĝi finfine ĉesis produkti arksubtenojn kaj ortopediajn ŝuojn kaj moviĝis por temigi ekskluzive sportŝuojn.";calzado;836027;New Balance;9134133;https://eo.wikipedia.org/w/index.php?oldid=9134133
"En 1960 , la firmao produktis la ""Trackster"" modelon, unu el la unuaj popularaj kurŝuoj kun ŝokosorba plandumo ondforma . La ŝuoj estis sukcesaj ĉie en Usono kaj post la sukceso, New Balance fariĝis de malgranda loka firmao en gravan firmaon en la kampo de sportŝuoj.";calzado;836027;New Balance;9134133;https://eo.wikipedia.org/w/index.php?oldid=9134133
Vendado komencis malkreski ĝis 1972, kiam 28-jaraĝa Jim Davis aĉetis la firmaon. En tiu tempo, la firmao dungis ses homojn kiuj faris 30 parojn da ŝuoj ĉiutage. La produktoj estis venditaj ĉefe per poŝta mendo aldone al kelkaj podetalistoj tra Usono . La Boston-areo iĝis la centro de la 1970a kuranta ekprospero kaj de maratonoj , kaj la produktserio disetendiĝis kaj vendo kreskis rapide.;calzado;836027;New Balance;9134133;https://eo.wikipedia.org/w/index.php?oldid=9134133
En 1978 New Balance malfermis sian unuan oficejon ekster Usono, en Londono , kaj en 1980 la tutmonda vendado de la firmao atingis cent milionojn da dolaroj .;calzado;836027;New Balance;9134133;https://eo.wikipedia.org/w/index.php?oldid=9134133
"Chuck Taylor All-Stars , aŭ Converse All Stars , estas kanvasa sportŝuo produktita de la usona kompanio Converse . La ŝuoj unue estis produktitaj en 1917 kiel basketbalŝuoj , kaj la antaŭa nomo estis ""All Star"".";calzado;836034;Chuck Taylor All-Stars;9514706;https://eo.wikipedia.org/w/index.php?oldid=9514706
Kun vendoj de 750 milionoj da ŝuoj komence de la 21-a jarcento, Allstar estas la plej vendita ŝuo en la historio .;calzado;836034;Chuck Taylor All-Stars;9514706;https://eo.wikipedia.org/w/index.php?oldid=9514706
"Unue la All-Stars ŝuoj ne estis tiom popularaj ĝis la fruaj 1920-aj jaroj. En 1921 , basketbaloludanto Charles (""Chuck"") Taylor ""adoptis"" ilin kiel siajn ŝatatajn ŝuojn . Li estis tre imponita kun la dezajno de la ŝuoj, kaj fariĝis vendoreklamanto por la produkto . Post pluraj malgrandaj ŝanĝoj en la dezajno de la ŝuoj, la subskribo de Chuck Taylor estis stampita sur ili, kaj ilia nomo estis ŝanĝita al la nuna nomo - Chuck Taylor All Star .";calzado;836034;Chuck Taylor All-Stars;9514706;https://eo.wikipedia.org/w/index.php?oldid=9514706
Ĝis 1966 , la All-Star ŝuoj estis produktitaj kaj venditaj nur en nigraj kaj blankaj koloroj. Poste, meze de tiu jaro ĝis nuntempe, la kompanio Converse komencis surmerkatigi la ŝuojn en diversaj koloroj, en altaj kaj malaltaj tranĉoj. Krome, ekzistas ankaŭ diverseco en la materialoj el kiuj la ŝuo estas farita .;calzado;836034;Chuck Taylor All-Stars;9514706;https://eo.wikipedia.org/w/index.php?oldid=9514706
En la malfruaj 1970-aj jaroj , la ŝuoj estis precipe ŝatataj de la punka komunumo , kiu adoptis ilin kiel modon .;calzado;836034;Chuck Taylor All-Stars;9514706;https://eo.wikipedia.org/w/index.php?oldid=9514706
Converse ( IFA kɒnvərs ) estas usona modfirmao, konata ĉefe pro la sportŝuoj All-Star kiujn ĝi produktas. La firmao estis fondita en 1908 kaj ekde 2003 estas filio de Nike .;calzado;836039;Converse (marko);9514707;https://eo.wikipedia.org/w/index.php?oldid=9514707
En ĝiaj fruaj tagoj, la firmao produktis vintrajn ŝuojn kun kaŭĉukaj plandumoj , sed dum Dua Mondmilito , ĝi ŝanĝis al produktado de ŝuoj por la Usona Armeo . Komence, Converse estis unu el la nuraj produktantoj de sportŝuoj kaj dum pli ol jarcento dominis la usonan sportŝuomerkaton . De la 1970-aj jaroj, la firmao perdis sian ekskluzivan pozicion, kiam konkurantaj firmaoj ofertis siajn proprajn ŝuodezajnojn al la konsummerkato.;calzado;836039;Converse (marko);9514707;https://eo.wikipedia.org/w/index.php?oldid=9514707
Converse ofte kunlaboras kun aliaj markoj kiel ekzemple John Varvatos por specialaj eldonoj. Converse-ŝuoj havas elstarajn markostampojn: la stelsimbolo de la firmao, la All-Star kaŭĉuka plando, la rondeta kaj glata piedfingrujo kaj la envolva bando. En 2019, Converse vendis produktojn pere de 109 podetalbutikoj en Usono, kiujn ĝi posedas. Krome, ĝi posedas 63 butikojn en internaciaj merkatoj .;calzado;836039;Converse (marko);9514707;https://eo.wikipedia.org/w/index.php?oldid=9514707
"En 1986 , Converse publikigis la ""Armilon"" basketbalŝuon . Ĝi estas produktita en du koloraj aroj, kiuj kongruas kun la ĉemizkoloroj de basketbalteamoj. La unika aspekto de ĉi tiu ŝuo estas la leda konstruo ĉie, inkluzive de la interna kalkanumo, kiu estas tre remburita por komforto. Converse reeldonis la klasikan ""armilon"" (kiun Kobe Bryant kaj Andre Miller portis iom da fojoj en 2002). La sukcesa modelo estis sekvite de ""The Loaded Weapon"" (La Ŝarĝita Armilo) en 2003, ""The 86 Weapon"" en 2008 (kaj la Foreman-versio en 2009, kaj la John Varvatos-versio en 2012), ""The EVO Weapon"" en 2009, kaj ĝia posteulo ""The Star Player EVO"" en 2010.";calzado;836039;Converse (marko);9514707;https://eo.wikipedia.org/w/index.php?oldid=9514707
"La unuaj ludantoj de ""The Weapon (La Armilo)"" estis Larry Bird kaj Magic Johnson , kiuj estis prezentitaj elstare en Converse reklamo. Ili ankaŭ estis portitaj fare de Axl Rose en la muzikfilmeto Estranged de Guns N' Roses .";calzado;836039;Converse (marko);9514707;https://eo.wikipedia.org/w/index.php?oldid=9514707
"Pluraj specialeldonaj Converse-ŝuoj estis kreitaj por famuloj, inkluzive de DC Comics , Pink Floyd , The Ramones , AC/DC , Sid Wilson (""#0""), Sailor Jerry, Metallica , The Clash , Doctor Seuss , Grateful Dead , Ozzy Osbourne , Jimi Hendrix , Miley Cyrus , Drew Brophy, Nirvana , Bad Meets Evil, Green Day , Matt & Kim, Black Sabbath kaj Super Mario . Tri novaj dezajnoj estis kreitaj inspiritaj de The Who . Krome, Converse aperis en pli ol 650 filmoj ĝis nun, inkluzive de Back to the Future , Me, Robot , Grease kaj Me and the Guys . David Tennant (kiu rolis kiel la deka Doktoro) portis gamon da Converse en la sciencfikcia spektaklo Doctor Who , inkluzive de ruĝaj kaj kremaj versioj. En julio 2018 , Converse formis teamon kun BT21, miljara popfiguro kreita de la sudkorea repgrupo BTS .";calzado;836039;Converse (marko);9514707;https://eo.wikipedia.org/w/index.php?oldid=9514707
Skechers USA, Inc. estas usona ŝu - kompanio fondita de Robert Greenberg kaj lia filo Michael en 1992 , tuj post kiam Robert forlasis la kompanion LA Gear - kiun li ankaŭ fondis en 1983 .;calzado;836063;Skechers;8984370;https://eo.wikipedia.org/w/index.php?oldid=8984370
"La unuaj produktoj de la firmao estis botoj en la stilo de "" Dr. Martens "" botoj kaj rultabulaj ŝuoj . Tra la jaroj, la firmao konsiderinde disvolviĝis, kaj hodiaŭ ĝi vendas centojn da malsamaj ŝuomodeloj de malsamaj tipoj, por viroj, virinoj kaj infanoj egale .";calzado;836063;Skechers;8984370;https://eo.wikipedia.org/w/index.php?oldid=8984370
La ŝuoj de la firmao estas produktitaj en fabrikoj situantaj en Ĉinio , Meksiko , Brazilo kaj Rumanio .;calzado;836063;Skechers;8984370;https://eo.wikipedia.org/w/index.php?oldid=8984370
Aktuale en 2012 , Skechers-ŝuoj estis venditaj en pli ol 100 landoj ĉirkaŭ la mondo tra reto de proksimume 30 distribuistoj.;calzado;836063;Skechers;8984370;https://eo.wikipedia.org/w/index.php?oldid=8984370
Ekde 1999 , la akcioj de la kompanio estas interŝanĝitaj ĉe la Novjorka Borso sub la simbolo - SKX.;calzado;836063;Skechers;8984370;https://eo.wikipedia.org/w/index.php?oldid=8984370
Por esti perceptita kiel moda marko , Skechers tra la jaroj reklamis ĝiajn produktojn per diversaj famuloj.;calzado;836063;Skechers;8984370;https://eo.wikipedia.org/w/index.php?oldid=8984370
En 2012, Skechers devis aranĝi grupan juragon kontraŭ ili, kiu baziĝis sur la aserto, ke ili trompis siajn klientojn en la reklamkampanjo de la ŝuomodelo Shape-Ups, por sumo de 40 milionoj da dolaroj .;calzado;836063;Skechers;8984370;https://eo.wikipedia.org/w/index.php?oldid=8984370
"Vans estas usona firmao kiu ĉefe produktas ŝuojn por rultabulantoj kaj sportŝuojn de tipo ""sneakers"", kaj en la lastaj jaroj komencis produkti pli diversajn sportŝuojn. Vans-ŝuoj longe estas popularaj de surfantoj kaj rultabulantoj.";calzado;836065;Vans;8984371;https://eo.wikipedia.org/w/index.php?oldid=8984371
"La kompanio Vans estis fondita en 1966 en Suda Kalifornio fare de Paul Van Doorn kaj ĝia plena nomo estis ""Van Doorn's Rubber Company"" . La firmao merkatas en la kampoj de surfado , rultabulado kaj punkroko . En la lastaj jaroj Vans komencis surmerkatigi pliajn erojn kiel ekzemple vestaĵoj kaj rultabulaj akcesoraĵoj.";calzado;836065;Vans;8984371;https://eo.wikipedia.org/w/index.php?oldid=8984371
En la 1980-aj jaroj , Vans perdis multajn el siaj ĉefaj klientoj, kiam ĝi komencis produkti basketbalajn kaj futbalajn ŝuojn. Kiel rezulto, la korporacio bankrotis en 1984 . Kvar jarojn post la resaniĝo de bankroto, Vans estis aĉetita fare de la bankfirmao McCown De Leeuw, kaj faris publikan deklaron pri ĝi en 1991 .;calzado;836065;Vans;8984371;https://eo.wikipedia.org/w/index.php?oldid=8984371
"Vans malfermis podetalbutikojn enhavantajn sufiĉe grandajn ""endomajn"" rulglitejojn , la unua malfermaĵo estanta en 1998 ĉe la rulglitejo nomata ""The Block at Orange. Ĉi tiu vendejo, same kiel alia lokoj en Orlando , estas la solaj kiuj restas malfermitaj ĝis hodiaŭ.";calzado;836065;Vans;8984371;https://eo.wikipedia.org/w/index.php?oldid=8984371
"En 2004 , la firmao estis aĉetita de ""VF Corporations"", produktantoj de Levi's ĝinzo kaj Wrangler ĝinzo , je valoro de 396 milionoj da dolaroj .";calzado;836065;Vans;8984371;https://eo.wikipedia.org/w/index.php?oldid=8984371
Vans estas la sponsoroj de la Vans and Arpad Tour , festivalo de skao / punko kaj ekstremaj sportoj okazigitaj en Usono ĉiujare ekde 1995 .;calzado;836065;Vans;8984371;https://eo.wikipedia.org/w/index.php?oldid=8984371
La plej konata kaj populara Vans-aĵo estas iliaj ŝuoj , kiuj estas plej popularaj en la punka / roka / rultabula sceno kaj en la lastaj jaroj ankaŭ en la emoa kaj hiphopa sceno.;calzado;836065;Vans;8984371;https://eo.wikipedia.org/w/index.php?oldid=8984371
"D-ro Martens (angle Dr. Martens ) estas brita marko de ŝuoj kaj vestaĵoj. La unikaj ŝuoj de Dr. Martens estis origine evoluigitaj de la germana kuracisto D-ro Klaus Maertens kaj estis tre sukcesaj en la germana merkato dum la 1950-aj jaroj . En 1959, la brita piedvestaĵa produktanto ""R. Griggs Group Ltd. aĉetis de Martens la patenton por la produktado de ŝuoj en Britio kaj la 1-an de aprilo 1960 komencis vendi la unuajn ŝuojn sub la marko Dr. Martens.";calzado;836076;Dr. Martens;8984372;https://eo.wikipedia.org/w/index.php?oldid=8984372
La unikaj botoj de Dr. Martens iĝis tra la jaroj speciale popularaj inter razkapuloj en Britio, kaj fine de la 1980-aj jaroj ili ankaŭ populariĝis en la punka sceno, inter diversaj elstaraj muzikistoj, krom diversaj junularaj subkulturoj en la okcidenta mondo .;calzado;836076;Dr. Martens;8984372;https://eo.wikipedia.org/w/index.php?oldid=8984372
Inter la jaroj 1960 kaj 2010, pli ol 100 milionoj da paroj da ŝuoj de Dr. Martens estis venditaj tutmonde, kaj aktuale en 2010, la firmao vendis pli ol 250 malsamajn modelojn de ŝuoj. Inter la jaroj 2009 kaj 2011, la firmao R. Griggs. Group Ltd. malfermis 14 novajn butikojn de Dr. Martens en Britio, Usono kaj Honkongo, kaj la firmao ankaŭ lanĉis vestaĵlinion sub tiu marko dum 2011 .;calzado;836076;Dr. Martens;8984372;https://eo.wikipedia.org/w/index.php?oldid=8984372
Klaus Märtens estis kuracisto en la germana armeo dum Dua Mondmilito . Dum en forpermeso en 1945, li vundis sian maleolskiadon en la bavaraj Alpoj . Li trovis ke liaj normaj armeaj botoj estis tro malkomfortaj sur lia vundita piedo .;calzado;836076;Dr. Martens;8984372;https://eo.wikipedia.org/w/index.php?oldid=8984372
Kiam la milito finiĝis, Klaus Märtens revenis al sia hejmurbo en Germanio, kie li trovis ledon de botisto. Kun ĉi tiu lernado, li rekonstruis la dezajnon kaj plibonigis la botojn kun aerspirantaj plandumoj .;calzado;836076;Dr. Martens;8984372;https://eo.wikipedia.org/w/index.php?oldid=8984372
Komence, Märtens ne havis multe da sukceso en vendado de siaj novaj botoj, ĝis li renkontis maljunan universitatamikon en Munkeno en 1947. Tiu estis Herbert Fünch el Luksemburgio . Fünck estis fascinita per la nova dezajno, kaj la du iris en komercon kune. Uzante forĵetitan kaŭĉukon de la germana aerarmeo kaj komfortajn plandojn, tio estis grandega sukceso kun virinoj pli ol 40, respondecante pri 80% de vendo en la unua jardeko .;calzado;836076;Dr. Martens;8984372;https://eo.wikipedia.org/w/index.php?oldid=8984372
Fondita en 1916 de U.S. Rubber, ties originala ŝua desegno estis la unua amaso-merkatumita kanvason-pinta sportŝuo . La marko estis vendita al Stride Rite en 1979, kiu estis akirita de Wolverine World Wide en 2012. Ekde februaro 2023, Keds estis posedita kaj funkciita de Designer Brands.;calzado;836079;Keds;8984373;https://eo.wikipedia.org/w/index.php?oldid=8984373
La ŝuoj iĝis tre popularaj de la 1960-aj jaroj ĝis la 1980-aj jaroj. Ili malkreskis en populareco en la 1990-aj jaroj. Ili iĝis tre popularaj denove en la 2000-aj jaroj kaj 2010-aj jaroj. Keds kutime produktas sportŝuoj .;calzado;836079;Keds;8984373;https://eo.wikipedia.org/w/index.php?oldid=8984373
La ŝuoj estis portita de famuloj inkluzive de Marilyn Monroe , Jackie Kennedy Onassis , Katharine Hepburn , Paul Newman , Humphrey Bogart , Kristen Stewart kaj Natalie Portman .;calzado;836079;Keds;8984373;https://eo.wikipedia.org/w/index.php?oldid=8984373
DC Shoes estas marko kaj usona piedvesta firmao bazita en Vista , Kalifornio . La firmao estis fondita en 1994 fare de Ken Block kaj Danny Way. Ĝi specialiĝas pri produktado de rultabulaj ŝuoj, neĝtabulaj botoj kaj vestaĵo . La firmao produktas ŝuojn por ĉiuj specoj de ekstremaj sportoj , inkluzive de rultabulado , neĝtabulado, BMX-bicikloj , motokroso , ralia vetkuro kaj surfado , por viroj kaj virinoj de diversaj aĝgrupoj .;calzado;836080;DC Shoes;9201824;https://eo.wikipedia.org/w/index.php?oldid=9201824
Blundstone Footwear (/ˈblʌndstən/ BLUND-stən) estas aŭstralia piedvesta marko kaj firmao, kiu estas bazita en Hobarto , Tasmanio , kun plejparto de fabrikado estanta farita eksterlande ekde 2007 . Fama produkto de la firmao estas ties senlaĉa , maleolo-longaj botoj kun elasta flanko. La malaltaj botoj estas aprezataj pro iliaj fleksebleco kaj daŭrigeco.;calzado;836083;Blundstone;8984375;https://eo.wikipedia.org/w/index.php?oldid=8984375
"La firmao estis fondita en la 19-a jarcento fare de setlantoj kiuj venis de Anglio al la insulo Tasmanio . En 1853 la fratoj Jacob kaj Thomas Cuthbarston venis de Anglio al Hobarto en Tasmanio kie ili establis ŝuproduktadon kaj importan komercon. En tiu tempo, Johano kaj Eliza Blandstone fondis la Blandstone Firmaon en Hubert. En 1932, la piedvestfirmao posedata fare de la Catberstone-familio aĉetis Blendstone kaj unuigis la firmaojn sub la nomo ""Blandstone"".";calzado;836083;Blundstone;8984375;https://eo.wikipedia.org/w/index.php?oldid=8984375
En 2007 , la firmao anoncis la translokigon de parto de siaj operacioj de siaj fabrikoj en Nov-Zelando kaj Aŭstralio al Tajlando kaj Barato . Kiel rezulto, 360 laborlokoj estis eliminotaj. Sekvante premon de la sindikatoj, kiuj sciigis ke ili bojkotos la produktojn de la firmao, la planoj estis nuligitaj.;calzado;836083;Blundstone;8984375;https://eo.wikipedia.org/w/index.php?oldid=8984375
"La firmao estis fondita en 1896 en la vilaĝo de Kutztown en Pensilvanio en Usono. La nomo de la firmao ricevis post la rivero kiu fluas tra la vilaĝo kaj estas nomita fare de la indianoj ""Sak-o-ni"" kaj la signifo de la nomo estas ""la superflua fluo de akvo"". En 1910 , Avraham Hyde, rusa enmigrinto, malfermis ŝuproduktadfirmaon en Kembriĝo , Masaĉuseco , kaj en 1972 li aĉetis Sacconi, kiu komencis produkti de Kembriĝo. En la 1990-aj jaroj , Sacconi iĝis la ĉefa marko de Hyde kaj la firmao oficiale ŝanĝis sian nomon al Sacconi .";calzado;836085;Saucony;8984376;https://eo.wikipedia.org/w/index.php?oldid=8984376
En 2005 ĝi estis akirita fare de la Stride Rite Corporation , kiu poste estis akirita fare de la ŝuĉeno Payless ShoeSource.;calzado;836085;Saucony;8984376;https://eo.wikipedia.org/w/index.php?oldid=8984376
Foot Locker Retail, Ltd. estas usona podetala firmao kiu vendas sportvestojn kaj piedvestojn, funkciigante en 28 landoj. Ĝia ĉefsidejo situas en Meza Manhatano en Novjorko .;calzado;836087;Foot Locker;9202186;https://eo.wikipedia.org/w/index.php?oldid=9202186
"Kvankam fondita en 1974 , kaj reestablita kiel aparta firmao en 1988 , la radikoj de Foot Locker devenas de 1879 , ĉar ĝi estas posteulkorporacio al la Woolworths Firmao, kiam multaj el ĝiaj elstaraj butikoj estis origine Kinney kaj Woolworths butikoj. La firmao funkciigas la ""Foot Locker"" markon, ĉenon de sportŝuaĵbutikoj (kune kun la ""Kids Foot Locker"" kaj ""Lady Foot Locker"" butikoj), same kiel aliajn sportfiliojn , kiel ekzemple ""Champs Sports"" ‏ ""Footaction USA"", ""House of Hoops""‏ kaj ""Eastbay"", kiu posedas la rajtojn al la retejo Final-Score . La firmao ankaŭ estas fama pro la uniformoj de siaj dungitoj en la ""Foot Locker"" ĉeno, ĝia frontmontra ĉeno, kiuj estas similaj al la uniformoj de la sportaj arbitraciistoj .";calzado;836087;Foot Locker;9202186;https://eo.wikipedia.org/w/index.php?oldid=9202186
Laŭ la dokumentoj prezentitaj de la kompanio al la Usona Komisiono pri Valorpaperoj kaj Interŝanĝoj ), la 28-an de januaro 2017 , Foot Locker havis 3 363 vendejojn, ĉefe en komercaj centroj kaj komercaj centroj, en Usono , Kanado , Eŭropo kaj Azio . Preskaŭ 70% de ĝiaj produktoj venditaj en vendejoj estas de Nike .;calzado;836087;Foot Locker;9202186;https://eo.wikipedia.org/w/index.php?oldid=9202186
Tecnica Group Sp A. estas grupo de fabrikantoj de ŝuoj kaj sportaj produktoj el Italio . La markoj de la firmao inkluzivas Tecnica ,‏ Nordica Moon Boot ,‏ Rollerblade ,‏ Lowa kaj Blizzard Sport .;calzado;836663;Tecnica (firmao);9027055;https://eo.wikipedia.org/w/index.php?oldid=9027055
"La fondinto de la grupo Giancarlo Zanatta fondis en 1960 ""Calzaturificio Tecnica Spa"" por la produktado de laborŝuoj . En 1970 , produktado de la Moon Boot-marko komenciĝis. En 1993 la kompanio akiris kontrolon de la germana fabrikanto Lowa . Post la aĉeto de Dolomite, la firmao eniris la kampon de fabrikado de vintraj sportaj produktoj kaj estas inkluzivita inter la gvidantoj en la mondo en ĉi tiu kampo.";calzado;836663;Tecnica (firmao);9027055;https://eo.wikipedia.org/w/index.php?oldid=9027055
Air Jordan estas usona marko kiu produktas basketbalŝuojn kaj sportajn vestaĵojn de Nike . Air Jordan, fondita en Ĉikago , estis establita por basketbaloludisto Michael Jordan dum lia tempo kiel ludisto en la Chicago Bulls . La originaj sportŝuoj (sneakers) estis produktitaj por Michael Jordan ĉe la fino de 1984, kaj ekde la 1-a de aprilo 1985 ili estis venditaj en la merkato .;calzado;836668;Air Jordan;9440786;https://eo.wikipedia.org/w/index.php?oldid=9440786
"La Jordan emblemo, konata kiel la ""Jumpman"" (angle por Saltulo), prenita de foto de Jacobus Rentmeester, kiam Jordan ludis por Teamo USA en la Somera Olimpiko 1984 .";calzado;836668;Air Jordan;9440786;https://eo.wikipedia.org/w/index.php?oldid=9440786
"En 1984 , la firmao ""Nike"" trovis sin en malfacilaĵoj en la sporta merkato. Unu el la ĉefproblemoj kiujn la firmao renkontis estis ke ĝi estis nekapabla vendi siajn produktojn en la mondo de korbopilko . La sukceso kiun la firmao ĝuis en la antaŭaj jaroj malkreskis, kaj ĝi komencis serĉi manierojn penetri novajn merkatojn. En tiu tempo, Michael Jordan estis selektita tria en la NBA -rangotabelo fare de la Chicago Bulls . La firmao decidis subskribi la ludiston al reklama kontrakto de 2,5 milionoj da dolaroj por periodo de kvin jaroj .";calzado;836668;Air Jordan;9440786;https://eo.wikipedia.org/w/index.php?oldid=9440786
"Jordan iĝis unu el la plej elstaraj ludantoj en la ligo jam en siaj fruaj tagoj kaj ""Nike"" profitis la sukceson por liberigi novan modelon de ŝuoj: ""Air Jordan"", nomita laŭ li. La ŝuoj revoluciis la merkaton de sportaj ŝuoj. Antaŭ la liberigo de la ""Air Jordan"" ŝuoj, la plej multaj el la ŝuoj sur la merkato estis en solida blanka koloro. La ""Air Jordan"" ŝuoj, aliflanke, estis en brila nigra-ruĝa koloro kaj iris kontraŭ tiu ĉi konvencio . La National Basketball Association (NBA) malpermesis al ĝiaj ludantoj porti la ŝuojn en oficialaj ludoj pro ilia nekutima koloro. Malgraŭ tio, Jordan forte ludis kaj estis monpunita ĝis kvin mil dolaroj ĉiufoje kiam li faris tion. ""Nike"" pagis la monpunojn por ke Jordan daŭre portu ilin, kaj la ŝuoj restus en la publika okulo.";calzado;836668;Air Jordan;9440786;https://eo.wikipedia.org/w/index.php?oldid=9440786
"La konflikto, kombinita kun la granda sukceso de Jordan kiel ludanto en la ligo tiun jaron, rezultigis la ŝuojn iĝantajn bonkonata nomo. La merkatigo de la ŝuo estis tiel sukcesa ke la firmao estis nekapabla renkonti la postulon, kaj la alta postulo eĉ kaŭzis la disvastiĝon de la ""ŝuo-rabo-"" fenomeno, en kiu homoj estis prirabitaj de siaj Air Jordan-ŝuoj per armila forto.";calzado;836668;Air Jordan;9440786;https://eo.wikipedia.org/w/index.php?oldid=9440786
"Sekvante la sukceson, Nike kreis apartan sekcion nomitan ""Jordan Brand"" (varmarko Jordan) kiu traktas la merkatigon de sportvestaĵoj . La firmao daŭre produktis novajn modelojn de la ŝuoj ĉiujare eĉ post la emeritiĝo de Jordan de aktiva ludado. En 2009, la firmao publikigis 23 malsamajn modelojn de la ŝuoj. Kun la tempo, la ŝuoj iĝis kolektaĵo, kun la unua modelo en la serio, kiu vendiĝis por 70 USD dum sia liberigo, vendante por centoj da dolaroj.";calzado;836668;Air Jordan;9440786;https://eo.wikipedia.org/w/index.php?oldid=9440786
En 2013 , la marko Air Jordan alportis al Nike $ 2.25 miliardojn. Kompare kun la malalta punkto de 1984, en 2013 Nike kontrolis 92% de la sportŝua merkato .;calzado;836668;Air Jordan;9440786;https://eo.wikipedia.org/w/index.php?oldid=9440786
Ekde ilia enkonduko al la ŝuaĵmerkato, Air Jordans evoluis de originaj basketbalŝuoj al multuzaj modeloj, inkluzive de trejnado kaj hazardaj ŝuoj, sed ankaŭ retroa linio (reeldonoj de la origina Air Jordans I-XXIII). Air Jordan nuntempe sponsoras ĉirkaŭ 19 aktivajn NBA- ludantojn, inkluzive de Chris Paul, Carmelo Anthony kaj iama NBA-ludanto Ray Allen.;calzado;836668;Air Jordan;9440786;https://eo.wikipedia.org/w/index.php?oldid=9440786
"En 2020 , paro de uzitaj ""Air Jordan"" sportŝuoj, portitaj fare de Michael Jordan en 1985 , estis venditaj ĉe aŭkcio por 615,000 USD .";calzado;836668;Air Jordan;9440786;https://eo.wikipedia.org/w/index.php?oldid=9440786
Ekde 2016, Air Jordan iĝis la sola provizanto de ekipaĵo por la usona futbala teamo de la Universitato de Miĉigano .;calzado;836668;Air Jordan;9440786;https://eo.wikipedia.org/w/index.php?oldid=9440786
Tio markis la unuan projekton de la marko en sporton krom basketbalo. Aktuale en 2018, Air Jordan estis la ekipaĵprovizanto por la futbalteamoj de la Universitato de Norda Karolino ĉe Chapel Hill , la Universitato de Oklahomo kaj la Universitato de Florido .;calzado;836668;Air Jordan;9440786;https://eo.wikipedia.org/w/index.php?oldid=9440786
En 2018, la marko sponsoris futbalteamon por la unua fojo, kiam franca klubo Paris Saint-Germain prezentis la firmaan emblemon sur siaj tripartaj vestaĵoj, portitaj en la 2018-19-eldono de la Ligo de Ĉampionoj de UEFA . De la sezono 2021-22, la hejma ĉemizo de Paris Saint-Germain uzas la emblemon de Air Jordan anstataŭ la emblemon de Nike .;calzado;836668;Air Jordan;9440786;https://eo.wikipedia.org/w/index.php?oldid=9440786
Ekde la 2020-21 NBA-sezono, la ĵerzoj havis la emblemon de Air Jordan sur la ĵerzoj, la ĵerzoj de Charlotte Hornets havis la emblemon en antaŭaj sezonoj.;calzado;836668;Air Jordan;9440786;https://eo.wikipedia.org/w/index.php?oldid=9440786
Air Jordan kunlaboras kun multaj markoj kaj artistoj, inkluzivanta famulojn Drake , Billie Eilish , J Balvin , DJ Khaled , Eminem , Nicki Minaj kaj Mark Wahlberg .;calzado;836668;Air Jordan;9440786;https://eo.wikipedia.org/w/index.php?oldid=9440786
"Birkenstock Grupo B.V. & Co. KG germana ŝufabrikisto bazita en Linz am Rhein . La firmao estas reprezentita internacie kun la samnoma ŝumarko kaj estas konata tutmonde kun la ""Birkenstock sandalo "" kaj ĝiaj variaĵoj.";calzado;836732;Birkenstock;9297268;https://eo.wikipedia.org/w/index.php?oldid=9297268
La Birkenstokaj ŝuoj estas konataj pro sia tipa korka plandumo farita kun tavoloj de sŭedo kaj juto , kiu konformiĝas al la formo de la plando .;calzado;836732;Birkenstock;9297268;https://eo.wikipedia.org/w/index.php?oldid=9297268
Fondita en 1774 de Johann Adam Birkenstock kun ĉefsidejo en Neustadt (Wied) , Rhineland-Palatinate , Germanio , la originala celo de la firmao estis fabriki ŝuojn kiuj subtenas kaj rondiras la piedon, komparita al la plataj plandumoj de multaj ŝuoj de tiu tempo. En 1896 la piedbedo (germane Fussbett) estis desegnita, kiu malpezigas la piedojn, genuojn kaj koksojn. Ekde 1925, Birkenstok-oj estis venditaj ĉie tra Eŭropo .;calzado;836732;Birkenstock;9297268;https://eo.wikipedia.org/w/index.php?oldid=9297268
En 1966, Margot Fraser unue alportis Birkenstokoj al Usono . En Usono, ili estis vendita en porsanaj vendejoj, kaj tiel fariĝanta asociita kun hipioj en la 1970aj jaroj .;calzado;836732;Birkenstock;9297268;https://eo.wikipedia.org/w/index.php?oldid=9297268
Kiam fremdaĵo (ekzemple sablero) envenas aperturon de konko , tio kaŭzas la konkulon ekkovradi ĝin per perlaĵo, por malpliigi la iritiĝon. Post multaj jaroj, la grajno estas tiel dike kovrita, ke ĝi bele briletas. Perlo ekestis.;joyas;3340;Perlo (globeto);9421987;https://eo.wikipedia.org/w/index.php?oldid=9421987
Krom naturaj perloj disponeblas ankaŭ kultivitaj perloj . Tiucele oni intence enmetas fremdaĵon en la konkulon kaj atendas la kovradon per perlaĵo.;joyas;3340;Perlo (globeto);9421987;https://eo.wikipedia.org/w/index.php?oldid=9421987
Pro la perceptado de perlo kiel tre bela, malofta objekteto, oni ankaŭ povas nomi elstare belan artoverk(et)on perlo .;joyas;3340;Perlo (globeto);9421987;https://eo.wikipedia.org/w/index.php?oldid=9421987
Kronadjuveloj estas juveloj , kiuj apartenas al la suvereno kaj pasas al la venonta suvereno kaj simbolas la rajton regi. Ili kutime inkludas unu aŭ pli kronojn , sceptrojn , globojn , glavojn kaj ringojn .;joyas;12087;Kronadjuveloj;8289101;https://eo.wikipedia.org/w/index.php?oldid=8289101
Guldhornene , la oraj kornoj aŭ la kornoj de Gallehus estis Dania nacia trezoro konsistanta el du artefaritaj oraj kornoj , unu mallonga kaj unu longa. Origine ili ambaŭ longis, sed unu el la pintoj jam mankis, kiam oni retrovis la kornon. La kornojn oni malkovris unuope je du malsamaj fojoj.;joyas;29784;Oraj kornoj;9514306;https://eo.wikipedia.org/w/index.php?oldid=9514306
"La oraj kornoj famiĝis ĉefe pro ilia 1802-a perdiĝo. Tiu ĉi okazaĵo donis inspiron kaj nomon al la eble plej fama poemo en la dana lingvo - ""Guldhornene"" de Adam Oehlenschläger . La poemon li skribis tuj post la ŝteliĝo de la kornoj. Kuriozaĵo estas ke Oehlenschläger tiam fakte loĝis en la sama kvartalo kiel Heidenreich, kiu ŝtelis kaj forfandis la kornojn. Perdiĝinte la oraj kornoj iĝis simbolo de la ""glora pasinteco de Danio"", helpe de la poemo de Oehleschläger. Ilin reklamcele uzis pluraj grupoj, inter tiuj feministoj komence de la 20-a jarcento , kaj la nazioj .";joyas;29784;Oraj kornoj;9514306;https://eo.wikipedia.org/w/index.php?oldid=9514306
La longan kornon trovis knabino Kirsten Svendsdatter la 20-an de julio 1639 , kaj la mallongan bienetisto Erich Lassen la 21-an de aprilo 1734 . Ambaŭ kornoj trovitis en kampo ĉe Gallehus , proksime de Møgeltønder en Sud-Jutlando .;joyas;29784;Oraj kornoj;9514306;https://eo.wikipedia.org/w/index.php?oldid=9514306
Kirsten leterskribis al la reĝo kaj rericevis jupon kiel salajron, dum Erich liveris la mallongan kornon al la grafo de Schackenborg , kiu siavice pluliveris la ĝin al la reĝo kaj ricevis 200 rigsdaler (tiamaj moneroj) rekompence. La kornojn tiele ekposedis la reĝa familio, kaj ili estis konservataj en la Dana Reĝa Artoĉambro en Christiansborg (kie nun troviĝas la Dana Ŝtata Arkivejo ).;joyas;29784;Oraj kornoj;9514306;https://eo.wikipedia.org/w/index.php?oldid=9514306
La longan kornon donis Kristiano la 4-a al sia filo princo Christian, kiu uzis ĝin kiel trinkkornon . Ĝi longis 75,8 centimetrojn je la plejlonga kurbo, kaj la diametro de la malpinta truo estis 10,4 cm, la pezo de la korno 3,2 kilogramoj.;joyas;29784;Oraj kornoj;9514306;https://eo.wikipedia.org/w/index.php?oldid=9514306
La oraj kornoj famiĝis, kaj sciencisto pri antikveco Ole Worm jam en 1641 eldonis raporton pri la longa: De aureo cornu . En ĝi troveblas unu el du postkonservitaj desegnoj de la kornoj. La kornojn ankaŭ menciis la ĵurnalo de la Franca Scienca Akademio, Journal des Savants , en 1678 . La mallongan kornon priskribis arkivisto Richard Joachim Paulli en 1734 .;joyas;29784;Oraj kornoj;9514306;https://eo.wikipedia.org/w/index.php?oldid=9514306
"La kornoj devenis de la ĝermana ferepoko en la 5-a jarcento . Unu el ili portis sub la malpinta truo runan skribaĵon en la praskandinava : ""Ek hlewagastiR holtijaR horna tawido"" (tradukebla kiel ""Mi Lægæst, arbarulo, kornon faris"", aŭ: ""Mi, Lægæst, filo de Holt, kornon faris.""). Iuj postulas, ke ĝi apartenis al la reĝo de la Engloj , dum aliaj diras ke Lægæst identas kun Liugast , la migrepoka reĝo en Danio de ĉ. 450 , ""riĉa je oro"". La nobla materialo kaj la ornamo indikas ke la kornoj uzatis ĉe ritualoj dum religiaj ceremonioj . Sed necertas ĉu tiu uzo estis trinkokorna aŭ blovinstrumenta . Dum tiu tempo oni importis trinkajn kornojn el vitro havantajn proksimume la saman formon, kaj oni havis ankaŭ hejmfaritajn trinkokornojn el bovaj kornoj bronzornamitaj. Sed la homoj ankaŭ uzis lignajn blov-instrumentojn similajn al la oraj kornoj. 78 centimetrojn longa ligna blovokorno de la 5-a aŭ 6-a jarcento estis trovita ĉe Holing ĉe Herning en 1993 . La Oraj Kornoj estis enterigitaj, aŭ oferrituale, aŭ trezor-stoke.";joyas;29784;Oraj kornoj;9514306;https://eo.wikipedia.org/w/index.php?oldid=9514306
La orkornaj bildoj rakontas pri mitoj , kun ilustroj de homoj , aliaj bestoj kaj fantaziaj estuloj en pluraj dividitaj partetoj. Granda parto de la mitoj nekonatas por ni hodiaŭ, sed ili ŝajne parte devenas de la Mediteranea areo , dum aliaj estas klare Nordiaj aŭ eble Keltaj. La Asismo probable malfrue (post 300) ekestis en Nordio kaŭ eble la Nordiaj dioj origine samis kiel la Keltaj dioj. Ole WORM kredis ke la figuroj montras moralaĵajn alegoriojn , kaj ke la korno devenis de la epoko de Frode Fredegod . Germana kultura kaj priscienca historiisto Willy Hartner per numerologio , astrologio kaj astronomio en 1969 interpretis la diversajn figurojn kiel kaŝitajn runskribaĵojn, kiuj estus astronomiaj notoj pri suneklipso en 413 . Tria interpreto videblas je Alternativinfo.dk Arkivigite je 2004-12-12 per la retarkivo Wayback Machine - ke la figuroj montras la morton de Balder en la Nord-ĝermana mitologio , kaj tiele bildigas la historion pri morto kaj reviviĝo, vidita kiel la ŝanĝiĝantaj sezonoj. Malgraŭ multaj provoj ĝis nun neniu sukcesis liveri konvinkivan deĉifron de la rakontoj sur la Oraj Kornoj.;joyas;29784;Oraj kornoj;9514306;https://eo.wikipedia.org/w/index.php?oldid=9514306
Vespere la 4-an de majo 1802 malriĉa oraĵisto kaj horloĝisto Niels Heidenreich ŝtelis la du kornojn. Heidenreich penetris en la ĉambron pere de du ŝlosiloj : sia loĝejporda ŝlosilo kaj alia, kies tenilon li estis plilongiginta. Jam la sekvan matenon oni registris la ŝtelon, kaj la ŝtelinto estis serĉata per tiutempaj iloj - gazetoj kaj stratkantoj . La rekompenco por riveli la ŝtelinton estis 1000 rigsdaler kaj plena anonimeco.;joyas;29784;Oraj kornoj;9514306;https://eo.wikipedia.org/w/index.php?oldid=9514306
La ŝtelinton malkovris Andreas Holm, ĉefo de la asocio de oraĵistoj. Heidenreich falsnome vendis kelkajn pagodojn al Holm. La pagoda oro entenis latunon , kion Holm rimarkis kaj alarmis la policon. Heidenreich unuafoje liberjuĝatis, sed Holm restis skeptika pri li kaj spionetis lin helpe de kelkaj kolegoj, interalie kiam Heidenreich forĵetis monerajn stampilojn en la urban elfosaĵon. La 27-an de aprilo oni finfine rekaptis Heidenreich, kiu la 30-an de aprilo agnoskis sian krimon.;joyas;29784;Oraj kornoj;9514306;https://eo.wikipedia.org/w/index.php?oldid=9514306
Bedaŭrinde li tiam antaŭlonge fandis la orajn kornojn en sia propra kuirejo. La oron li poste uzis por produkti juvelojn , ŝuajn fermilojn kaj falsajn pagodojn. Heidenreich estis kondamnita pro ŝtelo la 10-an de julio , kaj li poste restis en pundomo ĝis 1840 . 4 jarojn poste li mortis. La aĉetintoj de la korna oro redonis la oron, post kiam oni refandis ĝin en la Reĝa Dana Monerejo . Tiel la orkorna oro fine malaperis - entute 7,5 kiloj. Eble resteto da tiu oro tamen konserviĝas en paro da orelringoj , kiujn Heidenreich donacis al sia najbaro. Tiuj orelringoj hodiaŭ troveblas en muzeo en Ringe en insulo Fyn .;joyas;29784;Oraj kornoj;9514306;https://eo.wikipedia.org/w/index.php?oldid=9514306
Post la ŝtelo oni faris kopiojn de la du kornoj laŭ desegnoj kaj memoroj de atestantoj. En la 1980-oj oni faris novajn kopiojn. En la nova eldono la kornoj iomete vinditas, eble pli simile al la originaloj. Ambaŭ paroj kopiaj hodiaŭ montratas en la Dana Nacia Muzeo .;joyas;29784;Oraj kornoj;9514306;https://eo.wikipedia.org/w/index.php?oldid=9514306
La 17-an de septembro 2007 ŝtelistoj eniris la unuan etaĝon de la muzeo uzante ŝtupetaron kaj forrabis ne nur la kopiojn de la oraj kornoj, sed ankaŭ aliajn valorajn objektojn, inter ili juvelojn faritajn de Heidenreich el la oro de la originalaj kornoj. Oni rapide retrovis ilin la 19-an de septembro .;joyas;29784;Oraj kornoj;9514306;https://eo.wikipedia.org/w/index.php?oldid=9514306
Sukceno (C 12 H 20 O ) estas malmola flava diafana fosilia rezino , el kiu oni faras diversajn objektojn, precipe enbuŝaĵojn de pipoj kaj ornamaĵojn. Ĉar ĝi havas biologian devenon, ĝi ne apartenas al la veraj, multekostaj juvelŝtonoj, sed al ornamjuveloj.;joyas;31847;Sukceno;9434993;https://eo.wikipedia.org/w/index.php?oldid=9434993
El ĝi oni konstruis en 1716 la faman Sukcenan Ĉambron en Sankt-Peterburgo , kies sukcenoj malaperis dum la Dua Mondmilito . Helpe de la germana ŝtato , la sukcena ĉambro estis rekonstruita post retrovo de malnovaj fotoj pri la ĉambro kaj estis solene inaŭgurita la 31-an de majo de 2003 .;joyas;31847;Sukceno;9434993;https://eo.wikipedia.org/w/index.php?oldid=9434993
Pro la biologia deveno, ĝi kelkfoje entenas malpuraĵojn kaj tre rare eĉ insektojn el tre fora tempoepoko. Ĝi estiĝis ĝis 260 milionojn da jaroj antaŭ nia tempo.;joyas;31847;Sukceno;9434993;https://eo.wikipedia.org/w/index.php?oldid=9434993
Kutime sukceno kolore varias de flava ĝis bruna, kies ĉefa trovejo estas la baltmaraj landoj . Krome ekzistas blueca varianto, ĝia ĉeftrovejo estis Dominika Respubliko ĝis en 2008 oni trovis grandan sukcenokampon apud El Soplao en norda Hispanio.;joyas;31847;Sukceno;9434993;https://eo.wikipedia.org/w/index.php?oldid=9434993
El la malnovgreka vorto por sukceno (ἤλεκτρον), devenas (tra naciaj lingvoj) la Esperanta vorto elektro , ĉar la unua konata elektro estas la statika , kiu aperas, se oni frotas ekzemple sukcenon per ŝtofo aŭ besthaŭto, tiu fenomeno nomiĝas triboelektro .;joyas;31847;Sukceno;9434993;https://eo.wikipedia.org/w/index.php?oldid=9434993
La romianoj havis gravan vojon nomatan laŭ sukceno en Panonio , ĉar la komercistoj tie liveris la sukcenojn el Balta Maro ĝis Italio .;joyas;31847;Sukceno;9434993;https://eo.wikipedia.org/w/index.php?oldid=9434993
Dominga sukceno estas sukceno el Domingo derivita el rezino de formortinta arbo Hymenaea protera . Dominga sukceno diferenciĝas el la Balta sukceno je tio ke ĝi estas preskaŭ ĉiam travidebla, kaj ĝi havas pli altan nombron de fosiliaj inkludaĵoj.;joyas;31847;Sukceno;9434993;https://eo.wikipedia.org/w/index.php?oldid=9434993
La karato estas mezurunuo de aŭ la pureco de karaj metaloj (mallongigo: kt ), aŭ la maso de gemoj (mallongigo: CD ).;joyas;36564;Karato;9092563;https://eo.wikipedia.org/w/index.php?oldid=9092563
Straso estas senvalora au vitra imitaĵo de oro , perloj , diamantoj kaj simile. Sinonimo estas falsbrilaĵo .;joyas;49871;Straso;9016439;https://eo.wikipedia.org/w/index.php?oldid=9016439
Fibula Praenestina (Broĉo de Praeneste) estas ora broĉo sur kiu estas gravurita skribaĵo Manios med fhefhaked Numasioi (Manio min faris por Numerio). La skribaĵo aperas dekstre-maldekstre. Dum multaj jaroj, oni kredas ke ĝi estas la plej malnova teksto konservita de latina lingvo , datita de ĉirkaŭ -600 . La fibula estis trovita apud Palestrina (tiame Praenestre) kaj estas en la Museo Preistorico Etnografico Luigi Pigorini en Romo .;joyas;56947;Fibula praenestina;8912486;https://eo.wikipedia.org/w/index.php?oldid=8912486
Oni notu ke pri la aŭtenteco de la surskribaĵo de la fibula ĉiam estis suspektoj, kaŭzataj ĉefe pri la cirkonstancoj de ĝia trovo. Margherita Guarducci , diras ke la fibula estas falsa, produkto de la kunlaboro de klerulo Wolfgang Helbig kaj la antikvaĵisto Francesco Martinetti en la 19-a jarcento .;joyas;56947;Fibula praenestina;8912486;https://eo.wikipedia.org/w/index.php?oldid=8912486
"Nun (2011) fakula teamo kreita ĉe la Universitato de Romo ""La Sapienza"", ene de Instituto por nanostrukturoj de la Nacia Koncilio por Esploro, unuanime konfirmas la historian aŭtentikecon kaj, do, la kulturan valoron de la Fibula Praenestina.";joyas;56947;Fibula praenestina;8912486;https://eo.wikipedia.org/w/index.php?oldid=8912486
La fibula longas 10,7 cm, sin prezentas profile kiel ronda bosnajlo. Ĝia arkaika latino estas kuranta de dekstra flanko al la maldekstra, jen la teksto:;joyas;56947;Fibula praenestina;8912486;https://eo.wikipedia.org/w/index.php?oldid=8912486
kies transskribo estas: MANIOS MED FHE FHAKED NUMASIOI. En klasika latino estus: MANIUS ME FECIT NUMERIO, aŭ « Manio min faris por Numerio».;joyas;56947;Fibula praenestina;8912486;https://eo.wikipedia.org/w/index.php?oldid=8912486
• arkaika morfologio, kun la nominativo en –os, dativo en oi, la persona unuapersona pronomo akuzativigita per med, la perfekto de la verbo facere liverita duoblita. • La arkaika formo de la literoj, similaj al tiuj de la grekaj skribaĵo de Kuma;joyas;56947;Fibula praenestina;8912486;https://eo.wikipedia.org/w/index.php?oldid=8912486
"• la teksto estas certe antaŭa al la "" rotacismo "", tio estas la transformiĝo al –R de la –S kiel en (numaSioi).";joyas;56947;Fibula praenestina;8912486;https://eo.wikipedia.org/w/index.php?oldid=8912486
La teksto antaŭas ankaŭ la “febliĝojn” kiuj trafos la vokalojn de la silaboj sinsekvaj al la unua (per kiu A fariĝos E en silabo fermita;joyas;56947;Fibula praenestina;8912486;https://eo.wikipedia.org/w/index.php?oldid=8912486
La Koh-i-noor („Lumomonto“) (La telugua : కోహినూరు, la persa : کوہ نور, la urdua : کوہ نور, la hinda : कोहिनूर, la bengala : কহিনূর) estas ĉirkaŭ 110- karata diamanto , kiu nun estas ero de la britaj kronjuveloj .;joyas;93830;Koh-i-noor;8270560;https://eo.wikipedia.org/w/index.php?oldid=8270560
Probable Koh-i-noor estas la plej malnova ankoraŭ ekzistanta diamanto, kiun iam ajn tuŝis homaj manoj. Ĝia spuro komencis antaŭ pli ol 5000 jaroj, kiam oni jam menciis ĝin en malnova Sanskrit -eposo.;joyas;93830;Koh-i-noor;8270560;https://eo.wikipedia.org/w/index.php?oldid=8270560
"Poste oni parolis pri ĝi en la jaro 1304 , kiam Sultano Ala-ad-Din ŝtelis ĝin de la hindreĝo de Malwa ( Hindio ). Poste la gemo ornamis la okulon de la pavo de la tiama mondfama belega pavotrono en Delhi . Dum la konkero de Delhi en 1739 akiris ĝin Nadir , ŝaho de Persio . La legendo rakontas ke haremvirino diris al la konkerinto, ke la diamanto estis kaŝita en la turbano de la mogulo . Post tio la ŝaho proponis dum festo interŝanĝi la turbanojn. Tiu gesto estis simbolo de ĉiama amikeco kaj frateco, malakcepto estintus afronto. Kiam la ŝaho malvolvigis la diamanton el la tuko, li ŝajne ĝoje kriis ""koh-i-noor"".";joyas;93830;Koh-i-noor;8270560;https://eo.wikipedia.org/w/index.php?oldid=8270560
"En la jaro 1747 la persa ŝaho estis murdita kaj ""Koh-i-noor"" iris al la trezorejo de Panĝabo . Kiam la lando estis aneksita en 1849 al la brita Hindio, la juvelo estis donita kiel kompenso por la Sikh-militoj al la Brita Kompanio de Orientaj Indioj .";joyas;93830;Koh-i-noor;8270560;https://eo.wikipedia.org/w/index.php?oldid=8270560
Dum la jaro 1850 oni transdonis la juvelŝtonon al la brita reĝino Viktoria okaze de la 250-jara fondojubileo de la Brita Kompanio de Orientaj Indioj, kiu tiel ricevis ĉiujn rajtojn pri komerco kun Hindio kaj la spicinsuloj. Reĝino Viktoria tranĉadigis la Koh-i-noor-diamanton denove, post kiam iuj parolis pri malsufiĉa fajro. Tiel la iama 186-karata diamanto estis malgrandigita je la nuna grandeco de 108,93 karatoj.;joyas;93830;Koh-i-noor;8270560;https://eo.wikipedia.org/w/index.php?oldid=8270560
Por la kronado de la Reĝino Mary en 1911 , la Koh-i-noor estis enmetita kiel centra ŝtono en la kronon kaj fine en 1937 ĝi estis translokigita en la kronon de la Reĝino Elizabeta , la posta reĝpatrino.;joyas;93830;Koh-i-noor;8270560;https://eo.wikipedia.org/w/index.php?oldid=8270560
La ĉeĥaj kronadjuveloj estas objektoj servintaj por kronado de ĉeĥaj reĝoj . Ili estas metitaj daŭre en Praga burgo , kaj ankaŭ ili povas esti ekspoziciataj sole sur ties teritorio. Sed tio okazas dum tute eksterordinaraj eventoj. En la 20-a jarcento tiel okazis naŭfoje, lastfoje en oktobro de 1998 okaze de la 80-a datreveno de la estiĝo de la memstara Ĉeĥoslovaka respubliko . Por ekspozicii la kronadjuvelojn rajtas decidi eksplicite la prezidento de la respubliko.;joyas;103041;Ĉeĥaj kronadjuveloj;9336579;https://eo.wikipedia.org/w/index.php?oldid=9336579
Kronadkamerao en la templo de sankta Vito , en kiu la kronadjuveloj de la ĉeĥaj reĝoj estas metitaj, estas sen troigado la plej malmulte alirebla loko en Praga burgo. La pordo, same kiel blendŝranko por la juveloj , havas sep serurojn kaj al ties malfermo kaj ekpreno de la juveloj devas kunveni sep tenantoj de ŝlosiloj : la prezidento de respubliko, prezidanto de la registaro , ĉefepiskopo praga, prezidanto de Deputita parlamentejo, prezidanto de Senato , dekano de Metropolita kapitulo ĉe la sankta Vito kaj primatoro de la ĉefurbo Prago .;joyas;103041;Ĉeĥaj kronadjuveloj;9336579;https://eo.wikipedia.org/w/index.php?oldid=9336579
Tradicio de la sep ŝlosiloj estiĝis post tio, kiam en la jaro 1791 reĝo Leopoldo la 2-a plenumis peton de la ĉeĥa nobelaro pri redono de la juveloj el Vieno ĝis Prago. Ili estis metitaj en la templo de la sankta Vito en Praga burgo, ĉifoje en krona arkivo malantaŭ Sanktvenceslaa kapelo , malantaŭ fera pordo kun kvin seruroj. Estis rezervita por ili aparta ŝranko, havigita per pluaj du seruroj, al kiuj ricevis ŝlosilojn du gardistoj de la juveloj. Eĉ kiam pli malfrue loko de ilia metigo ŝanĝiĝis, la tradicio kun la sep ŝlosiloj konserviĝis.;joyas;103041;Ĉeĥaj kronadjuveloj;9336579;https://eo.wikipedia.org/w/index.php?oldid=9336579
La valoro de la Ĉeĥaj kronadjuveloj estas en la mono entute neesprimebla. Krom la neĝistaksita historia valoro ili fariĝis ankaŭ rara simbolo de la ĉeĥa ŝtataneco .;joyas;103041;Ĉeĥaj kronadjuveloj;9336579;https://eo.wikipedia.org/w/index.php?oldid=9336579
La kolekton kreas Sanktvenceslaa krono de Karolo la 4-a , al ĝi apartenanta ujo kaj kuseneto, la reĝa sceptro kaj ties ujo, la reĝa pomo kaj ties ujo, la kronadmantelo kaj pluaj vestaĵaj partoj: zono, maniplo (ornama ŝtofa zono portata sur la maldekstra antaŭbrako, parto de liturgia vesto), stolo (longa zono de ŝtofo aŭ pelto pendanta de ĉirkaŭ la kolo ĝis la genuoj) kaj mustela kolumo. En mezepoko apartenis al la kolekto ankoraŭ ringo , oraj braceletoj kaj kristala ujo por krismo .;joyas;103041;Ĉeĥaj kronadjuveloj;9336579;https://eo.wikipedia.org/w/index.php?oldid=9336579
La sanktvenceslaa krono – la juvelo farita el oro de alta pureco (21 – 22 karatoj ), ornamita per multekostaj ŝtonoj kaj perloj – estas en la kronadkolekto la plej malnova. Ĝi pezas preskaŭ du kilogramojn, kun la kruceto ĝi estas alta 19 cm, ties diametro estas same 19 cm kaj ĉiu el la kvar pecoj de diademo mezuras 14,5 cm. Karolo la 4-a igis fari ĝin en la jaro 1347 por sia kronado, poste li donacis ĝin al la unua patrono de la lando, al sankta Venceslao lasinte ĝin kiel kronon ŝtatan al kronado de pluaj ĉeĥaj reĝoj, al siaj sekvantoj sur la ĉeĥa trono. Sed eble ĝis fino de sia vivo ( 1378 ) li lasis ĝin reguligi kaj plenigi per la plej raraj multekostaj ŝtonoj, kiujn li sukcesis akiri. Tiel do la krono evoluis ĝis sia fina nuntempa aspekto.;joyas;103041;Ĉeĥaj kronadjuveloj;9336579;https://eo.wikipedia.org/w/index.php?oldid=9336579
Per sia formo ĝi ligas al antaŭaj kronoj de Premislidoj kaj la francaj reĝoj. Ĝi havas aspekton de diademo el kvar partoj, sur kies ĉiu supraĵo estas granda lilio . La pecoj estas supre kunigitaj per du arkoj, sur kiuj estas alfiksitaj ornamaĵoj de pli aĝa gemo (de diademo aŭ zono). Sur la supraĵo de la krono, kie la arkoj kruciĝas, estas lokigita ora kruceto kun safira kameo, juvelŝtono ornamita per gravuraĵo. Sur la krono estas sume 19 safiroj, 44 spineloj , 1 rubeno , 30 smeraldoj kaj 20 perloj .;joyas;103041;Ĉeĥaj kronadjuveloj;9336579;https://eo.wikipedia.org/w/index.php?oldid=9336579
Laŭ ordono de Karolo la 4-a la nova reĝa krono estis lokigonta por daŭraj tempoj en katedralo de sankta Vito . Sed jam la unua sekvanto de Karolo, lia filo Venceslao la 4-a , evidente komence de la 15-a jarcento lasis transporti la krondjuvelojn en burgon Karlštejn , kie ili devis esti en maltrankvilaj tempoj de disputoj je regado pli bone ŝirmataj. Ekde tiuj tempoj la loko de ilia lokigo multfoje ŝanĝiĝis, kutime dum politikaj maltrankviloj, bataloj je la ĉeĥa trono kaj milita danĝero. La ŝtorma 17-a jarcento alportis draman sorton ankaŭ al la krondjuveloj. Ili multfoje ŝanĝis lokon de la metigo. Por ia tempo ili aperis denove en la templo de sankta Vito, poste en oficejo de Tabuloj landaj, poste en Malnovurba urbodomo. En tempoj de endanĝerigo de Prago ili trovis kaŝejon eĉ en České Budějovice . Pli daŭran lokon destinis por ili la reganta dinastio de habsburgoj en Vieno, kie ili restis preskaŭ ĝis fino de la 18-a jarcento . Sed se ili estis metitaj kie ajn, en Karlštejn aŭ en Vieno, al la reĝa kronado ili estis ĉiam transportataj ĝis Bohemio en Pragan burgon.;joyas;103041;Ĉeĥaj kronadjuveloj;9336579;https://eo.wikipedia.org/w/index.php?oldid=9336579
La ora reĝa pomo kaj la sceptro , kiuj estas hodiaŭ nedisigeblaj partoj de la Ĉeĥaj kronadjuveloj, estiĝis en tempo multe pli malfrua ol Sanktvenceslaa krono, kiu okupas en la ĉeĥa kronad-kolekto dominan postenon. Diference de ĝi nek la pomo nek la sceptro ricevis memstaran atentemon. Eĉ malgraŭ tio, ke temas pri tre signifaj kaj unikaj fragmentoj de renesanca oraĵistejo. Ilia estiĝo datiĝas ĝis la unua duono de la 16-a jarcento , plej verŝajne en la tempon de Ferdinando la 1-a , kronita en 1527 kiel ĉeĥa reĝo, en 1531 kiel romia reĝo kaj en 1556 kiel imperiestro.;joyas;103041;Ĉeĥaj kronadjuveloj;9336579;https://eo.wikipedia.org/w/index.php?oldid=9336579
La pomo, farita el oro de alta pureco (18 karatoj), pezas 780 gramojn kaj ĝi estas alta 22 cm. Ĝi konsistas el du platigitaj duongloboj kunigitaj per ornamita ringo kaj finitaj per proporcie granda kruco . Sur rondo sub la kruco estas skribaĵo DOMINE IN VIRTUTE TUA LETABITUR REX ET SUPER SALUTARE TUAM EXULTABIT ( Sinjoro, el via potenco ĝojas la reĝo kaj el via helpo li ĝojas. ) En la riĉa ornamaĵaro de la pomo dominas multekostaj juvelŝtonoj kaj perloj, ĉarma detalo estas etaj staturetoj de ses sfingoj , kiuj kreas soklon de kruco. Okulfrape validiĝas kolora harmonio de oro, ruĝaj spineloj kaj bluaj safiroj, diversloke emfazita per buntaj emajloj. La surfacon de ambaŭ hemisferoj kovras treege precize orbatita reliefo kun figuraj aperaĵoj, kiuj per siaj temoj rilatas al la kronado. Sur la supra duonglobo estas scenoj el historio de Davido – Krismado de Davido je reĝo kaj Batalo de Davido kontraŭ Goljato , sur la suba duonglobo estas aperaĵoj el Genezo , montrantaj Adamon genuanta antaŭ Kreinto, Enkondukon de Adamo en edenon kaj Kreinton avertanta Adamon kaj Evon antaŭ la arbo de ekkono. La eluzo de tiu ĉi areo por la figura ornamaĵo estas unika kaj ĝi signas grandan signifon, kiu ties mendinto aldonis al tiu ĉi insigno.;joyas;103041;Ĉeĥaj kronadjuveloj;9336579;https://eo.wikipedia.org/w/index.php?oldid=9336579
La sceptro , farita el oro de la sama pureco kiel la pomo, estas 67 cm longa kaj ĝi pezas 1013 gramojn. En la kolekto de Ĉeĥaj krondjuveloj ĝi anstataŭis sceptron el la tempo de Premislidoj . Ĝi estas ornamita per kvar safiroj , kvin spineloj kaj sesdek du perloj . Ĝi konsistas el kelke da diference formitaj partoj kunigitaj per ringoj , en loko de mantenilo karakterizita per vico da perloj. La tutan areon de ĉiuj partoj kovras fajne cizelita dekoraĵo kun superanta motivo de vinberaj alfiksaĵoj, folietoj, floroj aŭ akantoj . Kelkaj partoj estas plie tegitaj per bunte kolorigitaj emajloj. La epika supraĵo de la sceptro estas kreita je maniero de ia floro per ogiva aera tigo kun fajnaj akantoj, el kiuj elkreskas multekostaj ŝtonoj kaj perloj.;joyas;103041;Ĉeĥaj kronadjuveloj;9336579;https://eo.wikipedia.org/w/index.php?oldid=9336579
"La kronadmantelo, konservata en Praga burgo , estas garita el luksa silka ŝtofo. Dezajnon de la ŝtofo kreas silka fadeno ĉirkaŭvolvita per plata ora drateto . La mantelo estas duonronda, sen manikoj, malantaŭe plilongigita en rubandon. Ĝi estas 312 cm larĝa, ekde kola dekoltaĵo ĝis fino de la rubando 236 cm longa. Sur la tuta perimetro ĝi estas borderita per mustelo – vintra blanka pelto de ermino – por sia rareco multfoje konsiderata kiel pelto de reĝoj. Esploristoj sukcesis pli precize envicigi ĝin en komencon de la 17-a jarcento , plej verŝajne en la tempon de Ferdinando la 2-a . Li estis kronita kiel ĉeĥa reĝo en la jaro 1617 , kaj kiel estas skribite, li estis onidire vestita en ""orkapan mantelon"". Lasta reĝo, kiu estis en la ĉeĥa kronadmantelo vestita, estis Ferdinando la 5-a dum kronado kiel ĉeĥa reĝo en la jaro 1836 . Ekde tiu tempo la kronadmantelo havas signifon sole historian kaj artehistorian. Sed ĝi plenigas la unikan vicon de la vestoj de regintoj kaj iliaj fragmentoj, kiuj en Praga burgo konserviĝis ekde la 10-a jarcento .";joyas;103041;Ĉeĥaj kronadjuveloj;9336579;https://eo.wikipedia.org/w/index.php?oldid=9336579
Fingroringo estas fingra aŭ piedfingra ornamaĵo . Kutime la fingroringoj estas surmetataj sur la kvara manfingro (tial nomata ringofingro ) kaj sur la dua piedfingro .;joyas;204011;Fingroringo;9272350;https://eo.wikipedia.org/w/index.php?oldid=9272350
Piedfingra ringo' estas ornama ringo farita el metaloj aŭ ne-metaloj teorie uzitaj sur iu ajn el la piedfingroj . La dua piedfingro de ĉiu piedo estas plej ofte uzata, ĉar proporcie ĝi estas la plej longa kaj do la plej facila piedfingro por enigi ringon kaj resti sen esti konektita al io alia. En plej multaj okcidentaj landoj tia ringo estas relative nova moda akcesoro, kaj tipe havas neniun simbolan signifon. Ĝi estas kutime portata nudpiede, aŭ kun sandaloj , aŭ kun zorioj .;joyas;204011;Fingroringo;9272350;https://eo.wikipedia.org/w/index.php?oldid=9272350
La plej malnovaj ringoj havas pli ol 21.000 jarojn, kiel ekzemple la fingroringoj el mamuta eburo trovitaj en Pavlov , Ĉeĥio .;joyas;204011;Fingroringo;9272350;https://eo.wikipedia.org/w/index.php?oldid=9272350
Ringojn oni uzis ne nur kiel ornamaĵo, sed ankaŭ kiel pagilo . Ringa mono certe estis unu el la pramonoj.;joyas;204011;Fingroringo;9272350;https://eo.wikipedia.org/w/index.php?oldid=9272350
Unuaj ringoj estis faritaj el ostoj , eble ankaŭ el ligno kaj fungoj , ŝtono , sukceno kaj poste el bronzo , kupro , fero kaj nobelaj metaloj . Tra la kulturhistorio ringoj ricevis la plej diversajn signifojn.;joyas;204011;Fingroringo;9272350;https://eo.wikipedia.org/w/index.php?oldid=9272350
Ringoj estis simboloj de regantoj, signalis anon de kredokomunoj, estis iloj por ritoj aŭ montris donitan geedziĝpromeson. Sed aldone ringoj ĉiam ankaŭ estis ilo por memprezentado.;joyas;204011;Fingroringo;9272350;https://eo.wikipedia.org/w/index.php?oldid=9272350
Broĉoj estas tre diversaj pri grandeco, materialo kaj formo, sed la fiksilo, plej ofte metala, similas ĉe plej multaj broĉoj. Broĉoj estas ofte manfaritaj ŝatokupe , sed ekzistas ankaŭ broĉoj el altvalora metalo ( oro , arĝento , ...) kaj ornamitaj de gemoj .;joyas;213396;Broĉo;9093754;https://eo.wikipedia.org/w/index.php?oldid=9093754
Laŭ Francisko Azorín Broĉo estas Metalaĵo, foje tre arta, formita per du pecoj kunfikseblaj, uzata por kunigi vestorandojn aŭ por alia simila celo. Li indikas etimologion el latina brochus (dento elstara).;joyas;213396;Broĉo;9093754;https://eo.wikipedia.org/w/index.php?oldid=9093754
Orelringo estas juvelo portita ĉe la orelo . Ĝi estas fiksita aŭ danke al truo en la orelo aŭ per fiksilo , kiu premas la orelon deambaŭflanke.;joyas;214821;Orelringo;9472134;https://eo.wikipedia.org/w/index.php?oldid=9472134
Orelringoj estas ofte el metalo , sed ankaŭ el osto , ligno , plasto aŭ aliaj malmolaj materialoj. La ornamaĵoj povas esti el preskaŭ ĉiuj materialoj inkluzive de vitro , gemoj kaj perloj . Formoj varias de malgrandaj ringoj aŭ pingloj ĝis grandaj pendaĵoj. Grandecon kaj pezon limas la ŝarĝeblo de la orelo kaj de la oreltruo.;joyas;214821;Orelringo;9472134;https://eo.wikipedia.org/w/index.php?oldid=9472134
La plej malnova orelringo trovita estas ĉirkaŭ 7500 ĝis 8200 jarojn aĝa kaj estis farita en la urbo Chifeng , interna Mongolio , Ĉinio . Oni trovis plurajn parojn de orelringoj, faritaj el jado kaj 2,5 ĝis 6 cm grandaj.;joyas;214821;Orelringo;9472134;https://eo.wikipedia.org/w/index.php?oldid=9472134
Dum la mezepoko ĉe kelkaj gildoj aparteno estis montrita per orelringo. Ĝi estis el oro kaj utilis kaze de morto kiel pago al la entombiganto. Al pekinto kontraŭ la gildaj reguloj oni forŝiris la ringon, disŝirante la orelon, tiel markante la ulon tutvive (vidu de:Schlitzohr ).;joyas;214821;Orelringo;9472134;https://eo.wikipedia.org/w/index.php?oldid=9472134
Multaj maristoj portis orajn orelringojn por iam pagigi sian kristanan entombigon . Se li mortus ekzemple pro ŝipdrono kaj lia korpo estus trovita de kristanoj, tiam ili povus pagi la entombigon per la oro de la ringo.;joyas;214821;Orelringo;9472134;https://eo.wikipedia.org/w/index.php?oldid=9472134
Orelringoj estis kaj estas portataj en multaj kulturoj de kaj viroj kaj virinoj, tamen plej ofte de virinoj. En kelkaj kulturoj la oreltruoj estas pligrandigitaj per pli kaj pli grandaj kaj pezaj orelringoj. Foje ankaŭ oni eltranĉas pecon de la orelo por enmeti grandan ringon au aliforman juvelon en la orelon.;joyas;214821;Orelringo;9472134;https://eo.wikipedia.org/w/index.php?oldid=9472134
En Eŭropo orelringoj estis malkutimaj ĉe viroj dum la 19-a kaj 20-a jarcentoj . Pri la 17-a kaj 18-a jarcentoj ekzistas multaj rakontoj kaj portretoj, kiuj montras, ke ankaŭ viroj portis ilin. Ekde la 1970-aj jaroj pli kaj pli da knaboj kaj viroj portis orelringon ĉe la maldekstra orelo. Poste, ĉirkaŭ la fino de la 1980-aj jaroj viroj ekportis orelringojn ĉe ambaŭ oreloj. Tiun popularigon kaŭzis ĉefe junuloj.;joyas;214821;Orelringo;9472134;https://eo.wikipedia.org/w/index.php?oldid=9472134
Brakringo aŭ braceleto estas juvelo portata je la brako , plej ofte je la manradiko kaj ĉirkaŭas ĝin.;joyas;217229;Brakringo;9411698;https://eo.wikipedia.org/w/index.php?oldid=9411698
Brakringo estas farita aŭ el flekseblaj materialoj, ekzemple ĉeneroj el metalo , perloj , konkoj , gemoj , kornoj , bestodentoj , ledo , gumo aŭ plasto aŭ ringforma el malmolaj materialoj, kiel metalo , ligno , ktp.;joyas;217229;Brakringo;9411698;https://eo.wikipedia.org/w/index.php?oldid=9411698
Kol- kaj brakjuveloj kiel simplaj aŭ plureraj ĉenoj jam ekzistis dum la Ŝtonepoko . Post malkovro de la prilabormaniero de kupro kaj bronzo oni kreis el tiuj materialoj rulaĵetojn, platetojn, perlojn, ringojn kaj diskojn. Ju pli oni uzis metalon, des malpli oni uzis organikaĵojn kiel bestajn dentojn aŭ sukcenon .;joyas;217229;Brakringo;9411698;https://eo.wikipedia.org/w/index.php?oldid=9411698
Aldoniĝis vitraĵoj kaj poste plasto kiel materialoj por krei brakringojn. Pro la uzo de altvaloraj materialoj juveloj iĝis samtempe valoraĵoj. Arĝentaj brakringoj estis iam uzataj kiel pagiloj .;joyas;217229;Brakringo;9411698;https://eo.wikipedia.org/w/index.php?oldid=9411698
Brakringo utilis ankaŭ kiel ŝirmiloj al ĉasantoj kaj batalantoj . Ĉe la keltoj , viroj portis brakringojn por indiki aristokratecon aŭ potencan pozicion.;joyas;217229;Brakringo;9411698;https://eo.wikipedia.org/w/index.php?oldid=9411698
Brakringoj utilas ne nur kiel juvelo, sed foje havas alian funkcion, kiel ekzemple brakringoj, kiuj fiksas brakhorloĝojn je la pojno .;joyas;217229;Brakringo;9411698;https://eo.wikipedia.org/w/index.php?oldid=9411698
Sufiĉe nova afero estas la tiel nomataj sportobrakringoj , kiuj plej ofte estas faritaj el malpeza akvorezista neopreno al kiuj oni povas fiksi iPod -on aŭ alian muzikludilon , tiel ke eĉ sportumante oni povas uzi ĝin.;joyas;217229;Brakringo;9411698;https://eo.wikipedia.org/w/index.php?oldid=9411698
En malsanulejoj pacientoj ricevas pacientobrakringon kun informoj por eviti misoperaciojn aŭ eraran donon de kuraciloj . Foje tiuj brakringoj surhavas strikodojn . Ĉe novnaskitoj la koloro (roza aŭ helblua) ofte indikas la sekson de la bebo .;joyas;217229;Brakringo;9411698;https://eo.wikipedia.org/w/index.php?oldid=9411698
En naĝejoj plastaj brakringoj kun ŝlosilo ebligas aliron de specifaj ĉambroj aŭ malfermon de ŝrankoj.;joyas;217229;Brakringo;9411698;https://eo.wikipedia.org/w/index.php?oldid=9411698
"Torko (el la latina : torquere = „turni“) estas kolringo malfermita antaŭe portita de keltoj dum "" La Tène ""-epoko kaj poste kiel honora insigno de romiaj soldatoj . Ĝi konsistas el dika ronda metalstango, simila al turnita kordo , plej ofte kun globoj aŭ bestkapoj ĉe la ekstremaĵoj kaj pli-malpli ornamita.";joyas;217430;Torko;8963703;https://eo.wikipedia.org/w/index.php?oldid=8963703
Juvelo estas dekora objekto belig-cele metata sur homon , alian beston , aŭ eĉ alian objekton. Ofte ĝi estas farita el valora metalo ( oro , arĝento , bronzo ) sen aŭ kun kromaj gemoj ( diamanto , rubeno ktp.). Juveloj ankaŭ povas konsisti el aliaj materialoj ( sukceno , perloj , plumoj , k.a.).;joyas;234402;Juvelarto;9447196;https://eo.wikipedia.org/w/index.php?oldid=9447196
Laŭ Francisko Azorín Juvelo estas Altvalora ornamilo (orelringo, fingringo, kolringo, k.c.). Li indikas etimologion el la latina jocus, jovalis (ludo, ludeto).;joyas;234402;Juvelarto;9447196;https://eo.wikipedia.org/w/index.php?oldid=9447196
Juvelarto estas arto, tekniko kaj agado rilataj al produktado kaj vendado de juveloj. La rilataj ejoj estus juvelateliero pri la produktado kaj juvelvendejo aŭ eĉ juvelejo pri la vendejo . La respektivaj profesiuloj estus juvelfaristo kaj juvelvendisto , foje sama isto dediĉas sin al ambaŭ agadoj, tiam ĝi estus juvelisto .;joyas;234402;Juvelarto;9447196;https://eo.wikipedia.org/w/index.php?oldid=9447196
Ekde sia ekesto homo uzis juvelojn. Dum la frua ŝtonepoko oni uzis interalie lupajn dentojn kiel juvelojn. Antaŭ la disvastiĝo de la mondaj religioj , kaj ankoraŭ en kelkaj paganaj socioj, la mortintoj entombigitis akompane de siaj juveloj. Per la bronzepoka evoluo de metala prilaborado kaj la posta utiligo de daŭre novaj materialoj (precipe dum kaj post la 20-a jarcento ) la aĵaro por juvelfarado multe plivastiĝis. Kun pli valoraj entenaĵoj juveloj mem iĝis valoraj, komerce interŝanĝeblaj. Grandan juvelaron sen konata posedanto oni kelkfoje nomas trezoro .;joyas;234402;Juvelarto;9447196;https://eo.wikipedia.org/w/index.php?oldid=9447196
Dum la unua mezepoko eŭropaj landoj kontaktis kun orientaziaj landoj, kiuj estis tre disvolvigitaj en tiu arto. Tiam komencis tutmonda komerco pri tiuj produktoj. Post la malkovro de Ameriko komencis eniri en Eŭropon grandajn kvantojn kaj de valoraj metaloj , kiel oro kaj arĝento, kaj de gemoj . Tio akcelis la produktadon kaj ĉefe la antaŭenigon de la juvelarto en Eŭropo.;joyas;234402;Juvelarto;9447196;https://eo.wikipedia.org/w/index.php?oldid=9447196
Nuntempe plej altrangaj juveloj estas realigitaj el titanio . Tiuj juveloj uzas la karakterojn de la titanio (rigideco kaj malpezeco) por atingi novajn dimensiojn (florojn de 18 cm de diametro , grandaj papiliojn, ktp). Tiuj juveloj el titanio estas produktitaj per gluado (ciro) kaj poste poluritaj.;joyas;234402;Juvelarto;9447196;https://eo.wikipedia.org/w/index.php?oldid=9447196
"Ekzistas krome la juvelarto de ""fantazio"", kiu uzas metalojn nevalorajn kiel kupro , latuno , stano , zamako , kaj lastatempe kirurgia ŝtalo aŭ la jam menciita titanio . Tiu duaranga parto de la juvelarto uzas ankaŭ nemetalajn materialojn kiel kristalo ( strass ), émajlo, kaj naturajn materialojn kiel ligno , korno , plumoj , kaj sintezitaj rezinoj . Tiuj juveloj estas foje tre kreivaj kaj inventivaj kaj povas esti multekostegaj.";joyas;234402;Juvelarto;9447196;https://eo.wikipedia.org/w/index.php?oldid=9447196
La fabrikado de juveloj estis tradicie farita permane de manartistoj, tre bonkvalite kaj solide. Tio implicas formadon, faldadon, fajladon , segadon kaj lutadon ( veldado aŭ brazado) de pecoj de valoraj metaloj aŭ gemoj. Polurado kutime estas la lasta tasko.;joyas;234402;Juvelarto;9447196;https://eo.wikipedia.org/w/index.php?oldid=9447196
Pro antaŭenigo de inventoj eblis uzado de pli precizaj fornoj kaj teknikoj de lutado, kiel uzado de elastomeroj aŭ la fonto de perdita ciro , oni uzas acidojn por forigi borakson , kiu estas likvido uzata por lutado kaj kiu poste solidiĝas. Polurado povas esti ĉu mekanika ĉu kemia, kio estas pli danĝera.;joyas;234402;Juvelarto;9447196;https://eo.wikipedia.org/w/index.php?oldid=9447196
Inter homoj juveloj estas dekoraĵoj, kiujn oni rekte-nerekte portas sur la korpo . Ĉefcelo de la juveloj estas fari personon pli atraktiva interne de socio, aŭ elmontri statuson de la homo.;joyas;234402;Juvelarto;9447196;https://eo.wikipedia.org/w/index.php?oldid=9447196
La krono de Sankta Eduardo ( anglalingve St Edward's Crown ) estas unu el la britaj monarkaj juveloj . Ĝi estas la oficiala kroniga krono , kaj estas ekskluzive uzata por la kronigo de nova reĝo . La nuna krono estis ellaborita en 1661 , okaze de la kronigo de Karlo la 2-a . Antaŭe ekzistis alia krono, kiu estis detruita dum la angla enlanda milito pro ordono de Oliver Cromwell . La krono estis laŭdire farita el oro de la krono de Sankta Eduardo la Konfesanto , angla reĝo de la 11-a jarcento .;joyas;234752;Krono de Sankta Eduardo;8547562;https://eo.wikipedia.org/w/index.php?oldid=8547562
Magatama aŭ magatamo (japane) kogok (koreie) estas gutoforma ornamaĵo el jado , kun truoj sur la pli dika fino. Ĝi disvastiĝis ĉefe en Japanio , ĝia formo venas verŝajne el praa formo de la bestodentaj pendaĵoj.;joyas;245883;Magatama;8340491;https://eo.wikipedia.org/w/index.php?oldid=8340491
En Japanio, oni pretigis magatamojn ekde la neolitiko, ili popularis ĉefe en la epoko de la holmotombejoj (kofun, 3-4-a jc.), tiutempe ankaŭ en Koreio.;joyas;245883;Magatama;8340491;https://eo.wikipedia.org/w/index.php?oldid=8340491
La magatamo iĝis - krom la glavo kaj spegulo - unu el la tri japanaj imperiaj regsignoj.;joyas;245883;Magatama;8340491;https://eo.wikipedia.org/w/index.php?oldid=8340491
"Giloŝo - france guillochis - laŭ PIV estas ""ornamo, precipe pri juveloj, artaĵetoj el oro aŭ arĝento, per gravurado aŭ enĉizado de retaĵe interplektitaj streketoj"". Do temas pri speciala ornamaĵo el aro da linioj, la unuopaj linioj ofte konsistigas kvazaŭ ŝnuroj fermitajn elipsojn aŭ cirklojn kaj kreas kutime simetriajn interplektaĵojn.";joyas;271729;Giloŝo;9126125;https://eo.wikipedia.org/w/index.php?oldid=9126125
Krom dekori juvelojn kaj aliajn artaĵetojn, giloŝoj estis gravaj elementoj dum la presado de monbiletoj , akcioj , pasportoj kaj aliaj gravaj oficialaj dokumentoj, kiuj celis malfaciligi ties falsadon : Tre malfacilis ekzakte reprodukti la giloŝajn ornamojn gravuritajn sur la metalaj platoj uzitaj por presado.;joyas;271729;Giloŝo;9126125;https://eo.wikipedia.org/w/index.php?oldid=9126125
Krome giloŝoj ankaŭ en oficialaj dokumentoj havis dekoran funkcion kaj servis al kreo de areoj nuancitaj je heleco: ju pli densa estis la reto de linioj, des pli malhela ŝajnis la presita areo.;joyas;271729;Giloŝo;9126125;https://eo.wikipedia.org/w/index.php?oldid=9126125
Por la gravurado de giloŝoj sur metalajn artaĵetojn la juvelistoj ekde la 17-a jarcento uzis apartajn aparatojn, kiuj turnis la dekorendan artaĵon dum la gravurado. Giloŝaj dekoroj aparte ŝatatis sur skatoloj, trinkiloj, fermiloj de poŝhorloĝoj kaj metalaj skribiloj. Aparte bone videblaj ili estis sur arĝentaj fingringoj .;joyas;271729;Giloŝo;9126125;https://eo.wikipedia.org/w/index.php?oldid=9126125
Komence de la 21-a jarcento , giloŝado estas formortanta metio. Dum la dua duono de la 20-a jarcento , kiam la homa labortempo ĉie en la mondo iom post iom iĝis pli multekosta, la tre temporaba dekoro iĝis pli kaj pli neekonomia. Nun pli kaj pli la metio prezentiĝas en muzeoj pri metilaborado, anstataŭ en produktantaj metiejoj.;joyas;271729;Giloŝo;9126125;https://eo.wikipedia.org/w/index.php?oldid=9126125
"Planta eburo , ankaŭ nomata Tagŭa , estas la semo de palmo, jam konata dum la antikva epoko . La latina botanika nomo de la palma specio estas Phytelephas macrocarpa , en Esperanto ""planta eburo kun granda frukto"". Palmoj de tiu specio aparte kreskas en Ekvadoro , sed ankaŭ en Panamo , Kolombio , Brazilo kaj Peruo . Ili preferas humidajn lokojn kaj kreskas ĝis alteco de 1800 metroj super la marnivelo . La palmaj folioj, kiuj longas proksimume 6 metrojn kaj larĝas ĝis 1 metron, bone taŭgas por tegi tegmentojn, kaj el la foliaj fibroj produkteblas ŝnuroj kaj balailoj. La floraro de la pli malofta vira planto estas simpla spadiko kun floroj proksimaj unu al la alia, kiuj post sia plena malfermo povas disvastigi intensan odoron tra la arbaro. La inaj plantoj produktas ĉiujare proksimume 20 fruktojn en la grandeco de homa kapo, kiuj pendas rekte ĉe la palma trunko kaj nur rikolteblas per hako de forta tranĉilo. La fruktoj havas plurajn ĉambrojn kaj, en pulpo , entenas po frukto plurajn dekojn da nuksoj , kiuj povas atingi la grandon de kokina ovo.";joyas;279939;Planta eburo;8497560;https://eo.wikipedia.org/w/index.php?oldid=8497560
Post la rikolto, la nuksoj ankoraŭ molas kaj enhavas trinkeblan likvaĵon, kiu acidetas iom kompareble al vino. La nuksoj elektitaj por plua prilaboro plurajn monatojn estas sekigataj per sunradioj. Pli kaj pli ili solidiĝas, ĝis fine ili atingas firmecon kompareblan al osto. Sub bruna-nigra ekstera haŭto nun ekzistas blankeca materialo, simila al eburo , kiun eblas segi, ĉizi, poluri kaj ankaŭ farbigi. Ĝi estas mirige rezista kontraŭ batado aŭ frotado. Se ĝi sen la haŭto estas pli longan tempon eksponata al sunradioj, ĝi iom plimalheliĝas. Per hejtado en fajro eblas al la surfaco doni helbrunan koloron kun iom marmoreca strukturo.;joyas;279939;Planta eburo;8497560;https://eo.wikipedia.org/w/index.php?oldid=8497560
La indiĝenoj en la regionoj, kie kreskas la Tagua-palmo, jam ekde multaj generacioj produktas diversajn dekorajn kaj praktikajn aĵojn el ĝi - ekzemple juvelojn, kombilojn aŭ segilojn. Pro la interkontinenta komerco la planta eburo ankaŭ konatiĝis en Eŭropo kaj Azio. Dum proksimume du jarcentoj la plantaj nuksoj estis bonvena anstataŭigo de eburo , kiu protekte al la afrikaj elefantoj ne plu rajtas esti komercata. El planta eburo produktiĝis butonoj, ŝakaj figuroj, ludkuboj, pipaj kapoj, infanaj ludiloj kaj multo plia. Famiĝis ankaŭ japanaj miniaturaj ĉizaĵoj nomataj „ Netsuke “. Dum la 1920-aj jaroj la produktaĵoj de varoj el planta eburo atingis sian kulminon, tunoj de la materialo transportiĝis al Eŭropo kaj en fabrikoj produktiĝis aparte butonoj el ĝi. Post la Dua Mondmilito la uzo de la materialo rapide malkreskis, ĉar pli malmulte kostaj plastaj materialoj anstataŭigis ĝin.;joyas;279939;Planta eburo;8497560;https://eo.wikipedia.org/w/index.php?oldid=8497560
Dum la 1960-aj kaj 1980-aj jaroj la natura materialo kadre de ekologiaj movadoj replipopulariĝis. Povraj kolektistoj de planta eburo estis kuraĝigitaj fondi kooperativojn, artaj konkursoj pri la ĉizado de figuroj kaj juveloj subtenis talentajn ĉizistojn kaj levis la artan metion de prilaboro de planta eburo al internacie agnoskita nivelo. Kontraŭe al la uzo de eburo, kies populareco gvidis al drasta malmultiĝo de la elefantoj , la populareco de la Tagua-nukso rekte subtenas la konservon de tropika praarbaro kaj ankaŭ de la bestaroj tie vivanta. Dum la kolekto de la nuksoj la palmoj restas nedifektitaj, kaj male eĉ estas aparte flegitaj, por ke la postkreskanta rikoltaĵo estu abunda. Tie, kie la planta eburo iĝis konvinka ekonomia faktoro por la indiĝenoj, ili forte kontraŭas industriajn penojn malpliigi la tropikan praarbaron.;joyas;279939;Planta eburo;8497560;https://eo.wikipedia.org/w/index.php?oldid=8497560
Bido (aŭ artefarita perlo ) estas eta objekto kun truo en ĝi por fari kolierojn kaj braceletojn . Bidoj povas esti faritaj el multaj diversaj materialoj inkluzive ligno , vitro , kaj gemoj . La sciate plej malnovaj bidoj aĝas pli ol 100 000 jarojn.;joyas;279945;Bido;9073359;https://eo.wikipedia.org/w/index.php?oldid=9073359
Bidoj estas produktataj el diversaj materialoj kaj per plej diversaj metodoj. Rezulto estu etaj, plej ofte rondaj, objektoj, kiuj kutime estas centre traboritaj por ke oni povu vicigi ilin je ŝnuro kaj porti dekore simile al sama uzo de veraj perloj . Krom esti portataj en formo de ĉeno , bidoj ankaŭ ofte estas dekore broditaj sur ŝtofojn.;joyas;279945;Bido;9073359;https://eo.wikipedia.org/w/index.php?oldid=9073359
Bidoj interalie tornatas el la dentoj de dugongo , el alabastro , planta eburo , ligno kaj koraloj .;joyas;279945;Bido;9073359;https://eo.wikipedia.org/w/index.php?oldid=9073359
Vitroperloj apartenas al la plej malnovaj juveloj de la homaro. Ili konsistigas la plej gravan grupon de bidoj . Krom simplaj, aparte malnovaj ekzempleroj ekzistas tre vasta kaj multfaceta gamo de vitroperloj el ĉiuj epokoj de la homa prahistorio .;joyas;279968;Vitroperlo;8943793;https://eo.wikipedia.org/w/index.php?oldid=8943793
La plej ofta speco de la prahistoria vitroperlo estas unukolora, netravidebla perlo, kutime en la koloroj bluo, flavo, bruno aŭ oranĝo. La vitron kolorigis pigmentoj , kutime produktitaj el metalaj oksidoj , kiuj estis kromrezulto dum la prilaboro de metaloj. Krome la prahistoriaj juvelistoj jam konis reduktigajn kaj oksidigajn efikon dum la fanda produktado de vitro.;joyas;279968;Vitroperlo;8943793;https://eo.wikipedia.org/w/index.php?oldid=8943793
Ĉiuj altaj kulturoj de la antikva epoko konis vitroperlojn kiel dekoraĵoj, kaj ankaŭ dum la frua eŭropa mezepoko , ekzemple ĉe la alemanoj aŭ merovidoj , la arta metio prosperis.;joyas;279968;Vitroperlo;8943793;https://eo.wikipedia.org/w/index.php?oldid=8943793
Post la mezepoko aparte la insularo Murano proksime de Venecio famiĝis pro la produktado de altnivelaj vitroperloj.;joyas;279968;Vitroperlo;8943793;https://eo.wikipedia.org/w/index.php?oldid=8943793
Geedziĝa ringo estas ringo el altvalora metalo , kiun oni portas sur la ringofingro de la dekstra aŭ maldekstra mano (diverse en diversaj landoj).;joyas;305981;Geedziĝa ringo;8199019;https://eo.wikipedia.org/w/index.php?oldid=8199019
La geedziĝa ringo simbolas geedzan ligon: la geedzoj portas ĝin kiel signo de la fideleco unu al la alia.;joyas;305981;Geedziĝa ringo;8199019;https://eo.wikipedia.org/w/index.php?oldid=8199019
Ĉe la ortodoksaj kristanoj estas difinite porti la ringon sur la ringofingro de la dekstra mano, sur tiu mano oni portas la geedziĝajn ringojn en la landoj de la Centra-Orienta Eŭropo , katolikoj en Aŭstrio , kaj ankaŭ en Rusio , Serbio , Pollando , Germanio , Ĉilio , Grekio , Kastilio , Hindio , Venezuelo k.a.;joyas;305981;Geedziĝa ringo;8199019;https://eo.wikipedia.org/w/index.php?oldid=8199019
Sur la maldekstra mano la geedziĝajn ringojn oni portas en tiuj landoj kiel Kubo , Brazilo , Francio , Irlando , Kanado , Meksiko , Slovenio , Svedio , Usono , Britio , Italio , Turkio , Japanio , Katalunio , k.a.;joyas;305981;Geedziĝa ringo;8199019;https://eo.wikipedia.org/w/index.php?oldid=8199019
En diversaj fontoj oni diverse priskribas originon de la tradicio de la geedziĝaj ringoj. Kelkaj aŭtoroj opinias ke la tradicio komencas sian historion en la Antikva Egiptio , aliaj – en la Antikva Grekio .;joyas;305981;Geedziĝa ringo;8199019;https://eo.wikipedia.org/w/index.php?oldid=8199019
La tradicio interŝanĝi la geedziĝajn ringojn iras per siaj radikoj en tiun malproksiman tempon, kiam feliĉaj novgeedzoj por la signo de amo kaj fideleco donacis unu al la alia simbolajn geedziĝajn ringojn, plektitaj el kanabo aŭ kano . Ĝuste ili iĝis prototipo de nuntempaj geedziĝaj ringoj. Pli poste en la Antikva Romo aperis moro , laŭ kiu fianĉo , por la signo de akcepto al si de devontiĝo kontraŭ la estonta edzino , dum la interfianĉiĝa rito donacis metalan geedziĝan ringon al la gepatroj de la fianĉino.;joyas;305981;Geedziĝa ringo;8199019;https://eo.wikipedia.org/w/index.php?oldid=8199019
Nuntage la geedziĝa ringo ne perdis sian originan signifon. Ĝia ronda formo, ne havanta komencon nek finon, simbolas senfinecon kaj eternan amon, kaj nobla metalo , el kiu oni tradicie faras la geedziĝajn ringojn, estas simbolo de pureco kaj virgeco. Tamen ja fasono de la geedziĝaj ringoj dum sia multjarcenta historio trasuferis vicon da ŝanĝoj. Kaj se origine la geedziĝaj ringoj estis ekstreme simplaj kaj ne havis iujn ajn ornamojn, do nuntempe ĉiam pli kaj pli junaj paroj elektas por geedziĝa ceremonio ekskluzivaj geedziĝaj ringoj, en kies dezajno ĉeestas kaj aro da multekostaj ŝtonoj kaj elegantaj kombinaĵoj de noblaj metaloj. Kiel regulo, ili estas ringoj el oro , kvankam lastatempe tio ne estas nepra postulo.;joyas;305981;Geedziĝa ringo;8199019;https://eo.wikipedia.org/w/index.php?oldid=8199019
Se tute sekvi tradiciojn, do la unua ringo, kiun fianĉo donacas al la estonta fianĉino, estas fianĉiniga . Ĉi tiun donacon oni laŭ la moro prezentas dum la gefianĉiĝo. Tiun ringon oni portas sur la ringofingro de la dekstra mano. Dum geedziĝa ceremonio fianĉo surmetas sur la saman fingron la geedziĝan ringon, kaj tiam la du ringojn oni portas kune. Aŭ la fianĉinigan ringon oni demetas, kaj anstataŭ ĝi fianĉo surmetas la geedziĝan ringon.;joyas;305981;Geedziĝa ringo;8199019;https://eo.wikipedia.org/w/index.php?oldid=8199019
Rito de interfianĉiĝo unuafoje aperis ĉe la romanoj: fianĉo donacis al la gepatroj de la fianĉino simplan metalan ringon kiel simbolo de devoligoj kaj kapabloj vivteni la fianĉinon.;joyas;305981;Geedziĝa ringo;8199019;https://eo.wikipedia.org/w/index.php?oldid=8199019
Ĉe la judoj la ringo aperis anstataŭ moro enmanigi al fianĉino monereton kiel konfirmo de seriozeco de siaj intencoj.;joyas;305981;Geedziĝa ringo;8199019;https://eo.wikipedia.org/w/index.php?oldid=8199019
Dum multaj jarcentoj al la geedziĝa ringo homoj atribuis preskaŭ mistikan signifon – impresi sorton de geedza paro. Se enpensiĝi, do en la cerbon venos multo da ligitaj kun tio indicoj. Ekzemple perdo aŭ rompo de la geedziĝa ringo perceptiĝas kiel antaŭsciiganto de disiĝo kaj detruo de geedza ligo. Ne necesas doni al iu ajn mezuri sian ringon. Ankaŭ faligi la ringon ĝis aŭ dum geedziĝosoleno oni kalkulas kiel malbona indico. La gepatroj povas transdoni siajn geedziĝajn ringojn nur tiuokaze, se sperto de ilia familia vivo estis sukcesa.;joyas;305981;Geedziĝa ringo;8199019;https://eo.wikipedia.org/w/index.php?oldid=8199019
Iuj opinias ke post la divorco transporto de la estinta geedziĝa ringo sur la maldekstran manon signifas simbolan signon por la mala sekso (multaj post la divorco simple demetas la ringojn) pri sia preteco al novaj rilatoj. Praktike do plej ofte neniun signifon al la ringo sur la ringofingro de la maldekstra mano oni ne atribuas.;joyas;305981;Geedziĝa ringo;8199019;https://eo.wikipedia.org/w/index.php?oldid=8199019
En holivudaj filmoj tre ofte rimarko de ringo ĉe eventuala amindumonto interrompas la eblajn rilatojn, kaj same, kiam amindumonto volas sukcesi, rapide demetas sian ringon kaj kaŝas ĝin, tamen foje la marko de multjara surmetado denuncas ties situacion de nefraŭlo.;joyas;305981;Geedziĝa ringo;8199019;https://eo.wikipedia.org/w/index.php?oldid=8199019
Kelkaj antropologoj rilatas la fakton de solena enmetado de fingro en la ringo kiel simbolo de la aminduma momento kiam peniso same eniras la vaginon . Fakte en kelkaj kulturoj oni faras ripetan manan geston por tiu simbolo.;joyas;305981;Geedziĝa ringo;8199019;https://eo.wikipedia.org/w/index.php?oldid=8199019
Pureca ringo (konata ankaŭ kiel virgeca ringo , promesa ringo aŭ abstina ringo ) estis origine inventita en Usono en la 1990aj jaroj fare de kristanismaj grupoj por propagi seksan abstinon. La portado de la ringo estas ofte kunligita kun virgeca garantiaĵo , subskribata komuna proceso inter gepatroj kaj gefiloj.;joyas;306550;Pureca ringo;8419789;https://eo.wikipedia.org/w/index.php?oldid=8419789
La portanto de la ringo, la subskribanto de la garantiaĵo promesas deteni, rezervi virgecon ĝis geedziĝo.;joyas;306550;Pureca ringo;8419789;https://eo.wikipedia.org/w/index.php?oldid=8419789
"Ringo CTR estas ringo kun la literoj ""CTR"" ofte portata de membroj de la Mormona Eklezio kiel rememorigilo de la devizo Choose The Right (""Elektu La Justecon""), kaj kiel simbolo de ilia religia aliĝo.";joyas;357497;Ringo CTR;8428541;https://eo.wikipedia.org/w/index.php?oldid=8428541
Dum la pasintaj jaroj CTR ringoj fariĝis popularaj ĉe infanoj kaj plenkreskuloj en la Mormona Eklezio. Ili estas hodiaŭ haveblaj en diversaj dizajnoj en pli ol 30 lingvoj.;joyas;357497;Ringo CTR;8428541;https://eo.wikipedia.org/w/index.php?oldid=8428541
La Gizela-kruco estas ligna kruco kovrita per oraj lamenoj, kiun la reĝino Gizela de Hungario pretigis je morto de ŝia patrino Gizela de Burgundio , mortinta en 1006, kaj kiun donacis al la Monaĥejo Niedermünster (Regensburg).;joyas;441546;Gizela-kruco;6740000;https://eo.wikipedia.org/w/index.php?oldid=6740000
La kruco altas 44,5 cm kaj larĝas 32 cm. Ĝi konsistas en kerno el kverkoligno, kiu estas kovrita per orlamenoj. La Kristo-figuro sur la antaŭa flanko konsistas el or-verŝo. LA piedoj de la krucumito, troviĝas figurigo de la kronita Gizela de Hungario kun ŝia patrino. La randoj de la kruco estas kovrita per perloj, juvelŝtonojn kaj emajloj. Inter ili kaj la figor de Jesuo troviĝas la skribo pri la donacinto kaj okazo de la donacado. La malantaŭa flanko de la kruco montras similaĵojn al la antaŭa kun perloj kaj juvelŝtonoj, mallongan skribon kaj kvar evangelistajn simbolojn. Okaze de la 200 emajlaj lamenetoj uzataj sur la du flankoj de la kruco, temas pri reuzado de tiuj, kiuj plej verŝajne estis alkudritaj ornamaĵoj sur ŝtofo.;joyas;441546;Gizela-kruco;6740000;https://eo.wikipedia.org/w/index.php?oldid=6740000
La kruco estis pretigita verŝajne en 1006/1007 eble en Regensburg kaj rezervita ĉe la Monaĥejo Niedermünster (Regensburg), ĉe tombo de Gizela de Burgundio. Ĝi hodiaŭ troviĝas en la trezoro de la Munkena Rezidejo .;joyas;441546;Gizela-kruco;6740000;https://eo.wikipedia.org/w/index.php?oldid=6740000
En juvelarto , stampilo estas ilo por fari kavan aŭ reliefan markon sur nobla metalo , frapante per gravurita matrico . Plej ofte, ĝi konsistas el malmolega rustorezista ŝtalo kun alta dureco . La marko kreita sur la metala objekto por identifi ties originon kaj kvaliton (maso aŭ pureco) nomiĝas stampo .;joyas;449307;Stampilo (noblaj metaloj);9404249;https://eo.wikipedia.org/w/index.php?oldid=9404249
En Francujo , la stampilo responda al oro estas aglokapo, al plateno lupo-(aŭ hundo-)kapo, kaj al arĝento Minerva kapo (kun kasko). Estas specialaj stampiloj pri puraj metaloj (proporcio 999 / 1000), reprezentante hipokampon por oro, pingvenon por plateno kaj amforon por arĝento.;joyas;449307;Stampilo (noblaj metaloj);9404249;https://eo.wikipedia.org/w/index.php?oldid=9404249
Stampo estas marko aŭ serio da markoj frapita sur artikoloj el valoraj metaloj - plateno , oro , arĝento kaj en iuj nacioj, paladio . Historie, stampoj estis aplikitaj de fidinda fakularo: la iamaj gardistoj de metio aŭ je hodiaŭaj tagoj per analizoficejo . Tiuj markoj garantias certan purecon aŭ fajnecon de la metalo.;joyas;449307;Stampilo (noblaj metaloj);9404249;https://eo.wikipedia.org/w/index.php?oldid=9404249
Tiuj stampoj devus ne esti konfuzitaj kun aliaj surskriboj, kiam ili estas nur nombro kiel 750 aŭ 925 (kvankam antaŭvidita por signifi la kvaliton de la metalo), kiu estas farita de iuj fabrikantoj, kaj bedaŭrinde ne ĉiam reflektas la veran purecon de metalo.;joyas;449307;Stampilo (noblaj metaloj);9404249;https://eo.wikipedia.org/w/index.php?oldid=9404249
Malgraŭ la estanta stampado de valoraj objektoj, multaj nacioj postulas, kiel kondiĉo por oficiala stampo, ke la marko de fabrikanto aŭ de komanditanto farita sur artikolo estu marko asertante responsecon kaj fajnecokvaliton. Markoj de responso estas ankaŭ postulataj en Usono malgraŭ la fakto, ke ne estas interkonsento pri oficiala stampo en tiu lando. Tamen, en nacioj havantaj oficialan aranĝon pri stampado, la stampo estas nur aplikata post la analizo de la artikolo, determinante, ke ties pureco laŭiĝas ne nur al la normoj establitaj de leĝo, sed ankaŭ al la postuloj de la fabrikanto pri la metalenhavo.;joyas;449307;Stampilo (noblaj metaloj);9404249;https://eo.wikipedia.org/w/index.php?oldid=9404249
En iuj nacioj, kiel ekzemple Britio , stampado konsistas el pluraj elementoj inkluzive: marko indikante la tipon de metalo, la marko de faristo kaj la jaro de la surskribo. En aliaj nacioj, kiel Svislando , la stampo estas simpla marko sendependa de la metalo aŭ de la pureco, kvankam la marko estas akompanata de marko de responso (kiel la komerca nomo de fabrikanto en UK ) kaj la aserto de la fabrikanto pri pureco. Kelkaj nacioj subskribis multflankan interkonsenton pri stampo de aĵoj el noblaj metaloj konatan kiel la Konvencio de Vieno de 1972 , plia, nedeviga, eĉ oficialaj markoj povas esti frapitaj fare de taksa oficejo: la konvencio havas la rezulton faciligi la importadon inter ŝtatoj, subskribintaj aŭ ne de tiu konvencio. Subskribitaj landoj havas apartan stampon pri si mem apud la konvencia marko, kiu indikas la metalon kaj ĝian purecon.;joyas;449307;Stampilo (noblaj metaloj);9404249;https://eo.wikipedia.org/w/index.php?oldid=9404249
"Oraĵo aŭ juvelartaĵo do ricevos du stampojn: la marko de la fabrikanto ( stampo de majstro ) aŭ importisto ( stampo de respondeco ), kaj la elcentaĵo de metalo (ekz. aglokapo pri novaj oraj aĵoj kun proporcio de 750 milonoj). Atentu, la difino de karato ne estas la sama pri noblaj metaloj kaj pri gemoj . La oficialaj stampoj estas frapataj de provjuristoj sur aĵoj, kiujn ili kontrolis kaj kies metalelcentaĵo estas konforma. En aliaj vortoj, la oficiala stampo estas la oficiala garantio de la ŝtato. La svisa oficiala stampo , ekzemple, ""kapo de San-Bernarda hundo "" estas farita (per matrico ) en Centra oficejo pri kontrolo de noblaj metaloj (en Berno ). En Svislando, ĉiu skatolo el nobla metalo pasas oficialan kontrolon. Ĝi estas analizata, frapata per stampiloj rilatante al taŭga elcentaĵo, al nacia kontrolo (hundokapo) kaj eŭropa kontrolo (pesilo), kaj estas akompanata de atestilo pri la analizo de la centra kontroloficejo de noblaj metaloj. Tiu procezo tiel krei stampon sur kontrolita aĵo nomiĝas stampado .";joyas;449307;Stampilo (noblaj metaloj);9404249;https://eo.wikipedia.org/w/index.php?oldid=9404249
"En Svislando , ekzistas ankaŭ la ""Ĝeneva stampo"" (france, Poinçon de Genève ) , la stampo de kvalito uzata por produktitaj horloĝoj en Kantono Ĝenevo .";joyas;449307;Stampilo (noblaj metaloj);9404249;https://eo.wikipedia.org/w/index.php?oldid=9404249
Fabrikanto de horloĝskatoloj havas permeson frapi noblajn metalojn per sia propra stampo de majstro . Ĉi tiu stampo garantias la originon de la elemento. Plie, estas komprenite, ke kiu krapas markon per sia stampo de majstro responsas ankaŭ pri la metalurgia enhavo de la valoraĵo.;joyas;449307;Stampilo (noblaj metaloj);9404249;https://eo.wikipedia.org/w/index.php?oldid=9404249
Ankaŭ en juvelarto kaj oraĵarto, nun ekzistas la komunaj stampoj por normigi leĝaron pri la kontrolo de noblaj metaloj kaj faciligi la internacian komercon , Aŭstrio , Britio , Finnlando , Norvegio , Portugalio , Svedio , Svislando kaj aliaj subskribis interkonsenton pri la kontrolo kaj la markado de artikoloj el noblaj metaloj. La objektoj, kiuj estis kontrolitaj kaj taksitaj konformaj fare de la kompetenta oficejo de kontraktinta lando ricevas stampon, nomatan komuna stampo , atestante iliajn metalurgian enhavon. Ĉi tiu stampo estas rekonata valide en ĉiuj aliaj kontraktantaj ŝtatoj.;joyas;449307;Stampilo (noblaj metaloj);9404249;https://eo.wikipedia.org/w/index.php?oldid=9404249
"The multi-nivela figuro de la komuna kontrolmarko estas pesilo, supermetita, pri oro, de du sekcantaj cirkloj, pri plateno, de diamantoformo, kaj pri arĝento de marko en la formo de la latina litero ""M"".";joyas;449307;Stampilo (noblaj metaloj);9404249;https://eo.wikipedia.org/w/index.php?oldid=9404249
"La principo de ĉi tiu konvencio estas kiel sekvas: artikolo, kiu portas la oficialan stampon de kontraktinta ŝtato kaj la ""komunan kontrolmarko"" de la konvencio (""pesilo""), ne plu bezonos aŭ novan kontrolon aŭ markadon de aliaj subskribintaj landoj. Plia avantaĝo estas, ke ne plu necesas registri la stampon de majstro en la lando de la celloko.";joyas;449307;Stampilo (noblaj metaloj);9404249;https://eo.wikipedia.org/w/index.php?oldid=9404249
La Ora Madono estas dipatrina skulptaĵo de la katedrala trezoro de Essen . Ĝi estas la plej malnova, plenplastika figuro de Maria en la kristana arto. Krom la Gero-Kruco en la katedralo de Kolonjo ĝi estas unu el la malmultaj konserviĝintaj otidaj grandaj artverkoj. Ankoraŭ nuntempe kelkaj katolikoj adoras la madonon, por ili ĝi estas identiga figuro de la Ruhr-Regiono kaj ties historio.;joyas;479987;Ora Madono;7855148;https://eo.wikipedia.org/w/index.php?oldid=7855148
La Oto-Matilda-Kruco estas procesiokruco el la katedrala trezoro de Essen , kiu estis produktita en la 10-a jarcento kaj ĝis en la plej lasta tempo estis uzata je altaj festotagoj. La nomo de la kruco deduktiĝas de la personoj, kiuj estas bildigitaj sur la enigita fondintotabuleto. Je tiuj ĉi temas pri duko Oto de Ŝvabujo (Ŝvabio) kaj lia fratino, abatino de abatinejo Essen , Matilda la 2-a . La du estis genepoj de imperiestro Oto la Granda kaj konfidatoj de sia onklo, imperiestro Oto la 2-a . La kruco estas per tio unu el la pecoj, kiuj montras la aparte intiman ligon de la Otidoj kun la abatinejo Essen.;joyas;480261;Oto-Matilda-Kruco;9271712;https://eo.wikipedia.org/w/index.php?oldid=9271712
La kruco troviĝas ekde sia ekesto en Essen, nekonsidere forportadojn kaŭzitajn per militoj kaj krizoj. La Inventarium reliquiarum Essendiensium de la 12-a de julio 1627, la plej frua listo de la abatineja trezoro, ne ebligas neriproĉeblan identigon, ĉar ĝi nur registras du krucojn tegitajn antaŭe per multaj juveloj kaj oro, malantaŭe per orumita kupro. Tiu ĉi priskribo ĝustas por ĉiuj kvar procesiokrucoj, kiujn enhavas la katedrala trezoro de Essen. Ankaŭ la Liber ordinarius , kiu reguligis la liturgian uzon de la abatineja trezoro, mencias nur ĝenerale procesiokrucojn. Ĉar donacitaj sanktaĵoj kutime ne estas fordonataj, oni tamen devas supozi, ke la kruco apartenis de ĝia donacado ĝis la civiligo de la abatinejo Essen en 1802 seninterrompe al ĉi tiu. Dum la Tridekjara Milito la abatino fuĝis kun la trezoro al Kolonjo . Dum aliaj krizoj oni verŝajne kaŝis la krucon sur la teritorio de la abatinejo. Dokumentita estas tio ĉi por 1794, kiam la franca armeo antaŭeniĝis al Essen, kaj la abatineja trezoro estis forportata al Steele (nun Essen-Steele ) en la orfejon fondintan de abatino Franciska Kristina de Palatinato-Sulzbach .;joyas;480261;Oto-Matilda-Kruco;9271712;https://eo.wikipedia.org/w/index.php?oldid=9271712
Je la sekularigo la katolika paroĥo Sankta Johano transprenis la abatopreĝejon kune kun ties inventaron kiel paroĥopreĝejon. Dum la Ruhr-Ribelo en 1920 oni portis la tutan abatinujan trezoron tute sekrete al Hildesheim , de kie ĝi estis reportata en 1925 je same kaŝaj cirkonstancoj.;joyas;480261;Oto-Matilda-Kruco;9271712;https://eo.wikipedia.org/w/index.php?oldid=9271712
En la Dua Mondmilito oni portis la katedralan trezoron komence al Warstein , poste al kastelo Albrechtsburg en Meißen kaj ekde tie en bunkron en Siegen . Post militofino ĝin trovis usonaj trupoj, kaj la kruco atingis kune kun la trezoro la landan muzeon en Marburgo kaj pli malfrue kolektejon por forkaŝitaj artverkoj en Kastelo Dyck ĉe Rheydt . De aprilo ĝis oktobro 1949 la katedrala trezoro estis ekspoziciata en Bruselo kaj Amsterdamo , por postsekve esti reportata al Essen.;joyas;480261;Oto-Matilda-Kruco;9271712;https://eo.wikipedia.org/w/index.php?oldid=9271712
Per la fondado de la episkopujo Essen en 1958 kaj la rangaltigo de la abatopreĝejo al katedralo la kruco venis al la episkopujo.;joyas;480261;Oto-Matilda-Kruco;9271712;https://eo.wikipedia.org/w/index.php?oldid=9271712
La kerno de la kruco 44,5 centimetrojn alta kaj 29,5 centimetrojn larĝa konsistas el kverkoligno . Temas pri latina kruco , kies trabofinaĵoj estas maltipe plilarĝigitaj. La trapezoformaj plilarĝigoj memorigas per du kornicoj kaj la surmetita triangulo al kapitelformo simile al tiu de la akena Lotarokruco donacita de la okcidentfranka reĝo Lotaro , kiu datiĝas en la jaro 984 . La fronto de la kruco estas tegita per martelformita orolado . Ĉirkaŭ la fronto de la kruco pasas randa borderaĵo, je kiu la gemoj metitaj sur or- filigrano ĉiufoje estas separataj per du apudaj perloj . Kiel randlinio de la kadro servas maldika perlobendo. Ĉar la pokaze kontraŭe troviĝantaj ŝtonoj estas agorditaj laŭ koloro kaj grando, la ornamo aperas akcentite ordigita kaj klara. Ĉe la malsupra finaĵo de la vertikala krucotrabo troviĝas la fondintoplato el ĉelemajlo . Sur ĉi tiu estas bildigitaj Matilda – nomata Mathild Abba – kaj Oto – nomata Otto Dux , kiuj komune tenas krucon.;joyas;480261;Oto-Matilda-Kruco;9271712;https://eo.wikipedia.org/w/index.php?oldid=9271712
La korpo martelformita el la orolado de la bazoplato montras la suferantan Kristo-n . Pro la antaŭenvolbita ventro kaj la malsimetria desegnaĵo de la supra korpo ekzistas similecoj kun la korpo de la Gerokruco de Kolonjo. Tial oni supozas, ke Kolonjo estas la ekestejo de la Oto-Matilda-Kruco. Tamen ankaŭ oni konsideras Treviron , ĉar la ĉelemajlaj platoj de la kruco devenas el la trevira metiejo de Egberto . Eblas ankaŭ, ke nur la emajloj estis farataj en Treviro, kaj ke la kruco aliloke estis kunmetata. La aŭreolo ĉirkaŭ la iomete klinita kapo de la Krucumito tranĉas la randon, la tri gemoj de la aŭreolo ekstraordinare akcentas la kapteniĝon. Inter la piedoj de la Krucumito starantaj sur piedapogilo kaj la fondintoplato troviĝas ankaŭ la martelformita bildo de serpento, kiu rilatas kun la fera serpento de la vivo Mosea en Nombroj 21,4–9. Laŭ alia interpreto estas prezentata bazilisko , kiu montras sur la Savanton kiel venkinton super la malbono el Ps 91,13: „Sur leonojn kaj aspidojn vi paŝos, vi piedpremos leonidon kaj drakon“. Super la kapo de la korpo troviĝas dua ĉelemajla plato kun la enskribaĵo dividita en tri liniojn IHC NA / ZARENVS / REX IVDEOR („Jesuo de Nazareto, reĝo de la judoj“), la literoj IHC reprezentas je tio la unuajn tri literojn de la Jesua nomo en greka lingvo.;joyas;480261;Oto-Matilda-Kruco;9271712;https://eo.wikipedia.org/w/index.php?oldid=9271712
La dorsflanko de la kruco estas tegita per orumita kuprolado kaj klare pli simple farita. Ĝi montras sur la finaĵoj de la krucotraboj la kvar Evangeliistoj , ĉe la interkruciĝejo estas bildigita la Dia Ŝafido . La bildoj estas interligitoj per vivarbo .;joyas;480261;Oto-Matilda-Kruco;9271712;https://eo.wikipedia.org/w/index.php?oldid=9271712
La ŝlosilo al interpreto kaj datigo de la kruco estas la fondintoplato de la kruco konserviĝinta senŝanĝe ekde sia fariĝo. Ĝi bildigas la dukon de Ŝvabujo (Ŝvabio), Oto, kaj la abatinon Matilda en kortega vestmaniero. Je la montrataj ŝtofoj temas supozeble pri sogdaj silkoŝtofoj, kiuj atingis la Frankan Regnon nur kiel donacoj de la bizanca imperiestro-kortego , similaj ŝtofoj konserviĝis en la trezoro de Essen kiel relikvaj ĉirkaŭvolvaĵo. Ĉar Matilda estis abatino de Essen ekde 973, tamen ne estas montrata en abatina vestaĵo, la pli aktuala literaturo supozas, ke ŝia prezentado en kortega vestaĵo de altranga nobelulo, ke ŝi sin montras kiel fratino de duko Oto kaj ne en sia funkcio kiel abatino. Ankaŭ la manko de regosignoj de duko, ekzemple glavo aŭ lanco ĉe Oto montras, ke la gefratoj estas prezentataj kiel familianoj kaj ne kiel eminentuloj. Matilda alprenas el la manoj de sia frato krucon. Oto tenas la krucon ja ambaŭmane, sed per etenditaj fingroj, dum kiam Matilda ĝin ekprenas per unu mano firme fermita. Ŝia alia mano estas tenata alten en la manartiko, tio ĉi estas dusenca. Unuflanke ĉi tiu gesto estus interpretebla el la bilda kunteksto kiel saluta aŭ akcepta gesto al Oto, tamen ŝiaj fingroj montras trans la bildo al la Krucumito, tiel ke ĝi ankaŭ estas peranta gesto.;joyas;480261;Oto-Matilda-Kruco;9271712;https://eo.wikipedia.org/w/index.php?oldid=9271712
La fondintobildo , aparte pro la manteniĝoj de la gefratoj, pli frue estis interpretata tia, ke Oto donacas la krucon al la abatinejo, kiun estras lia fratino. Tio ĉi tamen ne povas klarigi, kial ne estas prezentata per reĝaj signoj Matilda kiel abatino kaj Oto kiel duko. La ofta tezo, interalie reprezentata ankoraŭ de Pothmann, estas ke komuna donaco de la gefratoj ne konvenas kun la fakto, ke la kruco bildigita sur la fondintoplato ne konformas laŭ formo kun la Oto-Matilda-Kruco. Tio estus tipa afero je mezepokaj prezentadoj de donacado.;joyas;480261;Oto-Matilda-Kruco;9271712;https://eo.wikipedia.org/w/index.php?oldid=9271712
Laŭ pli nova literaturo oni devas rigardi la prezentadon de kruca transdono de Oto al Matilda simbola kaj kun enrigardo en la familian historion de la Liudolfidoj : Per la morto de Oto la 31-an de oktobro 982 estis mortinta la lasta vira ido de reĝino Edgita , unua edzino de Oto la granda. Matilda, kiu estis klare fidela al sia familio, nun per lia morto estis fariĝinta la mastrumanto de la familiaj bienoj. Kiel lasta membro de ĉi tiu familia branĉo ŝi precipe estis devigata prizorgi la mortmemoron de la familio. Tiun ĉi taskon ŝi transprenis de sia frato. Tiusence la fondintobildo estas interpretenda: Ĝi simbolas la volon de Matilda, daŭrigi la heredaĵon de Oto restinta sen gefiloj. Pro tio estas supozende, ke Matilda donacis la krucon memore al sia frato, kiu mortis en la jaro 982 dum la itala milito de imperiestro Oto la 2-a . La donacado okazus mallonge post lia morto, do en 983 aŭ 984, eble ankaŭ samtempe kun la donacado de la triumfokruco supernature granda en la abata preĝejo Sanktaj Petro kaj Aleksandro je Aschaffenburg , en kiu Oto estas entombigata. La pentrita kadrigo de ĉi tiu kruco konformas kun la eĝokadrigo de la Oto-Matilda-Kruco.;joyas;480261;Oto-Matilda-Kruco;9271712;https://eo.wikipedia.org/w/index.php?oldid=9271712
La liturgia enigo de la Oto-Matilda-Kruco en la mezepoko estas preskaŭ ne rekonstruebla. Principe procesiokrucoj estis starigataj ĉe aŭ parte sur la altaron kaj kunportataj dum procesioj. Je tio la flanko kun la korpo kutime montris antaŭen, por ke la procesiantoj ekzemple laŭ Mt 10,38 postsekvu Jesuon. La Liber Ordinarius de Essen el la 14-a jarcento, kiu ĉerpas el antaŭaj tekstoj, tradicias plurajn procesiojn. Eble procesiokruco estis, ankaŭ kompareble kun la imperia kruco el la imperia trezoro , emblemo de la princabatinoj de Essen.;joyas;480261;Oto-Matilda-Kruco;9271712;https://eo.wikipedia.org/w/index.php?oldid=9271712
Apartaĵon, kiun Beuckers pro tio, ke la Oto-Matilda-Kruco estas la plej aĝa kaj plej elspeza el la kvar procesiokrucoj de Essen, rilatigas sur ĉi tiun, tradicias la Liber Ordinarius pri la procesio dum la paska nokto, kiu pasis ekde la Petro-Altaro en la okcidenta alkonstruaĵo de la katedralo tra la klostro al la tombejo de la monaĥinoj, kie la tomboj estis ŝprucigataj per benita akvo, dum kiam la konvento en antifonoj rilatis kun la saviĝo per la kruco. Tia procesio, ankaŭ konata de aliaj monaĥejoj, simbolis la reviviĝon de Jesuo ankaŭ por la mortintoj, kiuj estis enigataj en la liturgion de la paskofesto, ĝi havas porpetan karakteron kaj rilatas kun la mortmemoro . Por la procesio de Essen Liber Ordinarius priskribis devojiĝe de la kutima praktiko, ke la fronta flanko de la kruco estu turnata al la procesiantoj. Je la Oto-Matilda-Kruco tio konsekvencigus, ke la gefratoj bildigitaj sur la fondintoemajlaĵo kvazaŭ estus fariĝintaj partoprenantoj de la procesio, preĝa interfratiĝo trans la morto, kiu estus ekzempla por la funkcio de mortmemora donacaĵo.;joyas;480261;Oto-Matilda-Kruco;9271712;https://eo.wikipedia.org/w/index.php?oldid=9271712
La Oto-Matilda-Kruco estas malgraŭ siaj aĝo kaj arthistoria graveco ne muzeaĵo. La sanktan funkcion de procesiokruco ĝi neniam perdis. Dum la surtronigo de la unua episkopo de Essen la 1-an de januaro 1958 ĝi estis portata antaŭ ĉi tiu, kaj li uzis ĝin ankoraŭ kiel procesiokruco je altaj festotagoj kaj procesioj. Tiun ĉi uzon ĉesigis liaj posteuloj pro konservema prizorgo, por tiaj celoj ekzistas moderna procesiokruco aludantan laŭ aspekto la Oto-Matilda-Krucon. Je la festomeso okaze de la konsekrado de la trezorejo de Essen la 15-an de majo 2009 la pli ol mil jarojn aĝa Oto-Matilda-Kruco servis, reprezentante la katedralan trezoron, kiel altarkruco.;joyas;480261;Oto-Matilda-Kruco;9271712;https://eo.wikipedia.org/w/index.php?oldid=9271712
La Lotarokruco estas otida procesiokruco en la katedrala trezoro de Akeno . Ĝi estis farita proksimume en la jaro 1000 supozeble en Kolonjo , ĝin verŝajne donacis imperiestro Oto la 3-a . Ĝis nun ekleziuloj uzas ĝin dum altaj festotagoj en la liturgio . Sian nomon la kruco havas pro reĝo Lotaro la 1-a de la Franka Regno, kies nomo troviĝas en la ĉirkaŭskribaĵo de enlaborita sigelo ŝtono.;joyas;480618;Lotarokruco;8366192;https://eo.wikipedia.org/w/index.php?oldid=8366192
La imperia kruco apartenas al la imperia trezoro de la Sankta Romia Imperio . Ĝi estas kava kaj servis por la gardado de la du Grandaj Relikvoj de Kristo , de la Sankta Lanco en la transversa brako kaj de la Kruceroj en la malsupra ŝafto. Temas pri la origina relikvujo de la imperiaj relikvoj. La imperia kruco troviĝas en la trezorejo de la Hofburg je Vieno .;joyas;481278;Imperia kruco;9509037;https://eo.wikipedia.org/w/index.php?oldid=9509037
La kruco havas kernon el kverkoligno kaj estas ekstere tegita per orlado samkiel interne subvestita per ruĝa ledo. Ĝi mezuras 77 oble 70 centimetrojn, la traboj estas 9 centimetrojn kaj la trabofinaĵoj 12 centimetrojn larĝaj.;joyas;481278;Imperia kruco;9509037;https://eo.wikipedia.org/w/index.php?oldid=9509037
La frontflanko de la kruco estas ornamita de perloj kaj gemoj (gemokruco). La dorsflanko montras realigite per nieloteĥniko la Dian Ŝafidon , ĉirkauatan de dek du apostoloj , same kiel la kvar evangeliisto -simbolojn sur la kvadrataj trabofinaĵoj.;joyas;481278;Imperia kruco;9509037;https://eo.wikipedia.org/w/index.php?oldid=9509037
Pro la produktiĝo kiel relikvujo pluraj partoj de la fronta flanko estas forleveblaj. En la tiam aperantaj malfermaĵoj la relikvoj estas meteblaj. La malfermaĵoj estas ene tegitaj per malhelruĝa ŝtofo kaj konformas precize kun la Sankta Lanco respektive la Kruceroj. El tio rezultas finfine la mezuroj de la tuta kruco. Ĉu pluaj pli malgrandaj kvadrataj malfermaĵoj estis destinitaj por dokumentoj aŭ pluaj relikvoj, ne plu estas eldireble. Sur la flankaj muroj staras la ĉirkaŭa enskribo:;joyas;481278;Imperia kruco;9509037;https://eo.wikipedia.org/w/index.php?oldid=9509037
La montro sur Chuonrad povus indiki al imperiestro Konrado la 2-a , en kies epoko ja la kruco estis pretigata. Ĉu tiu ĉi ankaŭ estis la origina komisiinto, ne estas konata. La interpreto de la kruco ne kiel torturilo, sed kiel venkosimbolo, estas karakteriza por la meza mezepoko kaj oni renkontas ĝin ankaŭ ofte en la romanika arto . Ĝi estas tial ankaŭ simbolo por la imperiestra pretendo de la mondoregeco.;joyas;481278;Imperia kruco;9509037;https://eo.wikipedia.org/w/index.php?oldid=9509037
La krucopiedo estas pli malfrua aldonaĵo kaj konsistas el orumita arĝento sur ligna kerno kaj estas ornamita per kvar emajlitaj blazonŝildetoj. La ŝildetoj prezentas sur fronta kaj dorsa flankoj la unukapan, dekstrenturnitan imperian aglon, dume tiuj surn la flankoj prezentas la arĝentan boheman leonon sur ruĝa fono. La piedo estas 17,3 centimetrojn alta kaj havas bazan areon da 29,2 oble 22 centimetroj.;joyas;481278;Imperia kruco;9509037;https://eo.wikipedia.org/w/index.php?oldid=9509037
Oni ĝenerale supozas, ke la imperia kruco ekestis en 1024/25 aŭ 1030 interalie en Loreno . Iomete pli frua produktado kiel mendaĵo de la antaŭulo de Konrado, Henriko la 2-a , tamen ankaŭ troviĝas en la literaturo.;joyas;481278;Imperia kruco;9509037;https://eo.wikipedia.org/w/index.php?oldid=9509037
La krucopiedo konsistas el orumita arĝento kaj estis pretigata anstataŭ pli malnova proksimume 1350 laŭ ordono de imperiestro Karolo la 4-a en Prago . Li ankaŭ igis produkti por la relikvoj, ĝis tiam gardataj en la imperia kruco, novajn relikvujojn.;joyas;481278;Imperia kruco;9509037;https://eo.wikipedia.org/w/index.php?oldid=9509037
La kruco kun la grandaj mallevaĵo-emajloj estas procesiokruco de la katedrala trezoro de Essen , kiu ekestis sub Abatino Matilda (mortinta en 1011) de Essen . La nomero mallevaĵo-emajloj indikas ĝian plej gravan ornamon: kvin grandaj emajloj faritaj laŭ emajlfarado en mallevaĵoj. La kruco konsideriĝas unu el la majstraĵoj de otida oraĵisto-arto.;joyas;484090;Kruco kun la grandaj mallevaĵo-emajloj;8373252;https://eo.wikipedia.org/w/index.php?oldid=8373252
La kruco estas 46 centimetrojn alta kaj 33,5 centimetrojn larĝa, ĝia kerno konsistas el kverkoligno . Temas pri latina kruco kun blokece plilarĝigitaj finaĵoj. Tiuj ĉi havas similecon kun la kubokapiteloj kun kolaj ringoj , kiuj populariĝis en la arĥitekturo proksimume la jaron 1000. La kruco estas sur la frontflanke tegita per orlado , dorsflanke per orumita kuprolado. Temas je la kruco pri pura gemokruco , je kiu la mallevaĵo-emajlo de la kruca kruciĝo tamen montras la krucumadon. La kruco kun la mallevaĵo-emajloj referencas al la plej malnova el la kvar procesiokrucoj de Essen, la Oto-Matilda-Kruco ankaŭ donacita de Matilda. Same kiel sur tiu ĉi interna kruco estas ĉirkaŭata de kadro. Dum kiam je la Oto-Matilda-Kruco tiu ĉi kadro konsistas el gemoj akompanataj de po du perloj , je la mallevaĵo-emajla kruco la kadro je la krucotraboj estas aldone pliampleksigita per origine 24 malgrandaj emajloj (21 ankoraŭ ekzistas), kiuj alternas kun gemoj akompanataj de po kvar perloj. La faco ene de la kadro estas je la mallevaĵo-emajla kruco ornamita per arte duoble tenata filigrano , gemoj, krucforme aranĝitaj perloj kaj antikva kameo . Je la kameo temas pri meduzokapo , tiel nomata gorgoneion . Estas sardonikso el la unua duono de la unua jarcento post Kristo. La kameo estas laborita en tri tavoloj (grizbruna, blanka kaj orbruna) kaj havas maksimuman diametron de 2,7 centimetroj. La kameo simbolas alude psalmon 91,13 la malbonon venkitan fare de la savanto.;joyas;484090;Kruco kun la grandaj mallevaĵo-emajloj;8373252;https://eo.wikipedia.org/w/index.php?oldid=8373252
Nomdonaj estas la kvin grandaj mallevaĵo-emajloj de la kruco. En la kruciĝejo de la kruco troviĝas iomete rektangula plato kun la prezentado de la krucumado: Kristo staras antaŭ la kruco, kun iomete klinita kapo kaj vaste malfermitaj okuloj. Ĉe la flankoj de sia kruco staras Maria kaj Johano , kiu estas identigebla pro sia meditema gesto. Super la horizontala krucotrabo la personigoj de suno kaj luno observas la okazaĵon. La emajlo kadrigita de malgrandegaj perloj elstaras pro sia grandeco malsupren iomete trans la kruciĝejo de la kruco. Ĉe la finaĵoj de la krucotraboj troviĝas ene de simpligita kadraĵo el gemoj, kiuj ekprenas la kadraĵon de krucotraboj, kvar aliaj grandaj, neregule formitaj mallevaĵo-emajloj kun la simboloj de la kvar evangeliistoj : Supre la aglo por Johano , malsupre la flugilhava homo por Mateo , dekstre la bovo por Luko kaj maldekstre la flugilhava leono por Marko . Tiuj ĉi grandaj mallevaĵo-emajloj montras altan teĥnikan kaj artan kvaliton, ekkonebla je la abunda koloreco kaj la delikata desegnaĵo de la flugiloj.;joyas;484090;Kruco kun la grandaj mallevaĵo-emajloj;8373252;https://eo.wikipedia.org/w/index.php?oldid=8373252
La dorsflanko, novigata en la 12-a jarcento, estas formita kiel vivarbo . En la plilarĝejoj de la krucofinaĵoj troviĝas kvar martelreliefigitaj medalionoj kun anĝeloj, en la kruca kruciĝejo medaliono kun la Agnus Dei .;joyas;484090;Kruco kun la grandaj mallevaĵo-emajloj;8373252;https://eo.wikipedia.org/w/index.php?oldid=8373252
La esploro kontinue datis la krucon proksimume la jaron 1000. Tiu ĉi datado baziĝas unuflanke sur la motivoj de la 21 malgrandaj emajloj de la kadro, kiuj ĉefe montras florecajn kaj tapiŝecajn skemojn, kiuj aperis je la turniĝo de la jarmiloj, aliaflanke sur la kadrigoj de la gemoj kaj kadroemajloj deduktitaj de la Oto-Matilda-Kruco datiĝanta de proksimume 982.;joyas;484090;Kruco kun la grandaj mallevaĵo-emajloj;8373252;https://eo.wikipedia.org/w/index.php?oldid=8373252
Okulfrapaj je la nomdonaj kvin grandaj mallevaĵo-emajloj estas la malegala formo de la evangeliisto-emajloj. Ankaŭ frapas la okulojn, ke la mallevaĵo-emajlo de la krucokruciĝejo ne estas samgranda kiel la kruciĝejo. Tiuj ĉi mallevaĵo-emajloj ekestis proksimume 1000. Pro la okulfrape malegala formo tiuj ĉi emajloj tamen verŝajne ekestus en alia interrilato kaj estus poste enigata en la krucon, kiu je tio estus konsiderinde alfirmata. Por ĉi tiu celo la emajloj kun la evangeliistoj, kies fariĝo ekkonigas bizancan influon, estis segataj konvenaj. La malegala formo je tio akcentis ilian karakteron de fremda transprenaĵo . Por enigi ĉi tiujn emajlojn, oni rezignis la originan formon de la krucofinaĵoj kaj de la kruca kruciĝejo, la simpligita kadrigo kaj ligis la emajlojn en la krucon kaj ankaŭ akcentis ĝian karakteron kiel posta enigaĵo.;joyas;484090;Kruco kun la grandaj mallevaĵo-emajloj;8373252;https://eo.wikipedia.org/w/index.php?oldid=8373252
Beuckers datas la transformadon en la regecon de la abatino Zofia en Essen. Je la transformado la kolringoj de la kapitelformaj krucofinaĵoj fariĝis sensencaj, la transformanta oraĵisto kaŝis tiujn per spiralforme tordita filigrandrato, tiel nomataj „abelkorboj“. Ĉi tiu ornamformo aperis aparte ofte en la medio de Henriko la 2-a , same kiel alia filigranformo, tiel nomataj florkronoj, kiu je la trezoraĵoj de Essen nure aperas ĉe la finaĵoj de ĉi tiu kruco. Zofia estis enpostenigita kiel abatino en Essen de Henriko la 2-a kaj estis intima al li, kio donis al ŝi la eblecon komisii la oraĵistojn el ties medio. Nek „abelkorboj“ nek florkronoj aperas en la trezoraĵoj, kiujn la posteulo de Zofia, Teofano , evidente produktigis en alia metiejo, tiel ke Beuckers forigas ŝin kiel komisiinto de la transformado de la mallevaĵo-emajla kruco. Pro kiu kaŭzo Zofia transformigis konsekritan objekton, kiun produktigis ŝia antaŭulo Matilda kaj je tio enmetigis la emajlojn el alian artverkon evidente gravan por Zofia, estas nekonata.;joyas;484090;Kruco kun la grandaj mallevaĵo-emajloj;8373252;https://eo.wikipedia.org/w/index.php?oldid=8373252
La kruco kun la grandaj mallevaĵo-emajloj ekestis pro la datado proksimume 1000 samtempe kun la kolekto relikvujo , kiu pli malfrue estis nomata relikvokesto de Sankta Marsuso . Tiu ĉi arverko rigardata kiel plej eminenta trezoro de la abatinejo estis mortmemora donaco de imperiestro Oto la 3-a por sia patro Oto la 2-a kaj estis laborita konvene altvalora. La relikvokesto perdiĝis en 1794, ilustraĵoj ne estas postlasitaj. Beuckers supozas, ke la kruco kaj la kesto, kiu estis ornamita per oremajloj, ekestis en metiejo, kiu lokis en Essen. Ĉar la kadraj emajloj de la kruco parte alprenas motivojn el Treviro, la kesto de Marsuso ekestis ekde 997 kaj la trevira metiejo de Egberto , la ununura konata otida emajlproduktejo, post 993 ne plu estas pruvebla, Beuckers supozas, ke abatino Matilda post la morto de Egberto en 993 irigis ties emajlejon al Essen. La kruco kun la grandaj mallevaĵo-emajloj ebligas tiel ankaŭ konkludojn pri la kvalito de alia perdiĝinta artverko.;joyas;484090;Kruco kun la grandaj mallevaĵo-emajloj;8373252;https://eo.wikipedia.org/w/index.php?oldid=8373252
La kruco troviĝas ekde sia estiĝo en Essen, senkonsidere milite kaj krize kaŭzitajn eksterurbajn kaŝadojn. La Inventarium reliquiarum Essendiensium de la 12-a de julio 1627, la plej frua listo de la abatinejo-trezoro, ne ebligas senriproĉan identigon, ĉar ĝi nur registras Zwei crucifixer fornhero mit vielen gesteinen und gold uberzogen, hinten aber kupfer uberguldet . Tiu ĉi priskribo estas ĝusta por ĉiuj kvar procesiokrucoj estantaj en la katedrala trezoro de Essen. Ankaŭ Liber ordinarius , kiu regulas la liturgian uzon de la abatinuja trezoro, mencias nur ĝenerale procesiokrucojn. Ĉar oni kutime ne fordonis donacitajn konsekritan objekton, estas tamen supozende, ke la kruco ekde sia donaciĝo ĝis la sekularigo de la abatinejo de Essen en 1802 apartenis al ĉi tiu seninterrompe. Dum la Tridekjara Milito la estrino de la abatinejo fuĝis kun la trezoro al Kolonjo, dum aliaj krizoj oni kaŝis la krucon supozeble ene de la abatineja teritorio. Dokumentita estas tio por 1794, kiam la francoj antaŭeniris al Essen, kaj la abatineja trezoro estis forportata al Steele (nun Essen-Steele ) en la orfejon fondintan de abatino Franciska Kristina de Palatinato-Sulzbach .;joyas;484090;Kruco kun la grandaj mallevaĵo-emajloj;8373252;https://eo.wikipedia.org/w/index.php?oldid=8373252
Je la sekularigo la katolika paroĥo Sankta Johano la Baptisto transprenis la abatinpreĝejon kune kun ties enaj havaĵoj kiel paroĥa preĝejo. Dum la Ruhr-Ribelo en 1920 oni portis la tutan abatinan trezoron je plej alta sekreteco al Hildesheim , de kie oni samsekrete reportis ĝin en 1925.;joyas;484090;Kruco kun la grandaj mallevaĵo-emajloj;8373252;https://eo.wikipedia.org/w/index.php?oldid=8373252
Dum la Dua Mondmilito la katedrala trezoro estis komence portata al Warstein , poste sur Kastelon Albrechtsburg en Meißen kaj de tie en bombŝirmejon je Siegen . Post la militfino tie trovita de usonaj trupoj, la kruco envenis kune kun la trezoro en la landan muzeon je Marburgo kaj poste en kolektejo por ekstermagazenitaj artverkoj al Kastelo Dyck ĉe Rheydt . De aprilo ĝis oktobro 1949 la trezoro de Essen estis ekspoziciata en Bruselo kaj Amsterdamo , por poste esti reportata al Essen.;joyas;484090;Kruco kun la grandaj mallevaĵo-emajloj;8373252;https://eo.wikipedia.org/w/index.php?oldid=8373252
Per la fondado de la episkopujo Essen en 1958 kaj la levado de la abatopreĝejo al katedralo la kruco alvenis al la episkopujo Essen.;joyas;484090;Kruco kun la grandaj mallevaĵo-emajloj;8373252;https://eo.wikipedia.org/w/index.php?oldid=8373252
Detaloj pri la liturgia uzo de la kruco ne estas konataj. Se la fontoj, ĉefe la Liber Ordinarius de Essen ekestinta proksimume 1400, preskribis por procesioj la uzadon de procesiokrucoj, tio okazis tute ĝenerale kaj sen menciado de apartaj krucoj. Procesiokrucojn oni ofte uzis pare, kaj pro la ekesto de la kruco kun la grandaj mallevaĵo-emajloj sub abatino Matilda oni supozas, ke la mallevaĵ-emajla kruco estis akirata kiel kompletigo de la Oto-Matilda-Kruco. La du krucoj estis uzataj kune ĝis en la epoko de kardinalo Hengsbach . Laste la mallevaĵ-emajla kruco estis antaŭportata en 1992 je la enoficigo de la dua episkopo de Essen, d-ro Hubert Luthe . Nuntempe oni ne plu uzas la mallevaĵ-emajlan krucon same kiel la ceterajn tri otidajn procesiokrucojn pro konservismaj konsideroj.;joyas;484090;Kruco kun la grandaj mallevaĵo-emajloj;8373252;https://eo.wikipedia.org/w/index.php?oldid=8373252
Gemokruco estas mezepoka kruco ekipita per gemoj , perlo aŭ filigrano . La kruciĝejo en la centro estas interpretata kiel loko de la Krista kapo kaj estas aparte valorege ekipita. Kutime ĉi tie troviĝas la krucorelikvo . La formoj de la kruco povas diferenci. Ankaŭ kronoj de regantoj kaj relikvo kestoj en la mezepoko estis ornamataj per gemoj.;joyas;484718;Gemokruco;8900549;https://eo.wikipedia.org/w/index.php?oldid=8900549
Gemokrucoj estas plejparte ekestintaj kiel relikvujoj kaj estis liturgie uzataj kiel procesiokrucoj . Ili prezentas la centran kristanisman simbolon, la krucon de Kristo , kiel venkosigno. Samtempe ili estas instrumentoj de regeca reprezentado kaj transportas la malnov-romian triumfoideon sur la kristanan simbolon.;joyas;484718;Gemokruco;8900549;https://eo.wikipedia.org/w/index.php?oldid=8900549
La plej malnovaj konserviĝintaj gemokrucoj devenas el la 6-a jarcento. La plimulto ekestis en la karolida kaj otida epokoj. La plej fama tiaspeca kruco estas la imperia kruco el la 11-a jarcento.;joyas;484718;Gemokruco;8900549;https://eo.wikipedia.org/w/index.php?oldid=8900549
La Teofano-Kruco estas unu el la kvar otidaj procesiokruco el la katedrala trezoro de Essen kaj apartenas al la plej gravaj oraĵistaj verkoj siatempaj. Ĝin donacis la abatino de Essen Teofano , kiu regis ekde 1039 ĝis 1058.;joyas;486300;Teofano-Kruco;8476087;https://eo.wikipedia.org/w/index.php?oldid=8476087
La baza formo estas latina kruco , muntita sur la kapeto de boteleto el egipta vitreca kvarco . La kruco estas 44,5 centimetrojn alta kaj 30 centimetrojn larĝa kaj havas kernon el cedroligno , ambaŭflanke kaj ĉe la randoj tegita. Ĉe la flankaj randoj iam troviĝis enskribo surigita sur orumita arĝento, nun plejparte perdiĝinta, kiu indikis Teofanon kiel donacinton. El la fragmentoj almenaŭ parto estas sencoplene kompletigebla jena teksto: EDITA REGALE GENERE NOBILIS ABBATISSA THEOPHANU HOC SIGNUM DEDIT („Ĉi tiun signon donacis la nobela abatino Teofano devenanta el reĝa gento“). La krucofinaĵojn, kies internaj randoj estas rondigitaj, finas transversaj rektangulaj trabetoj. Sur la kruciĝejo de la frontflanko troviĝas granda, ovala kvarco vitreca, malantaŭ kiu estas enfermitaj, envolvitaj en ruĝa veluro, ver-kruceroj kiel relikvoj . La kristalo estas surmetita sur ronda plato ornamita per perloj, gemoj kaj filigranoj , kiu superstaras la kruciĝejon kaj per tio formas plilarĝigon de la krucomezo. La kruco estas ornamita per du bendoj, kiuj laŭiras la randoj kaj estas ornamitaj alterne per emajloplatoj kaj enkadritaj gemoj sur filigrano de altkvalita farmaniero, mallarĝa mezostrio surmetita per filigranaĵoj estas libera de aliaj ornamaĵoj. Sur la transverstrabetoj estas aranĝitaj po du emajloplatetoj, ankaŭ ornamitaj per gemoj, perloj kaj filigranoj ambaŭflanke de gemo. La entute 18 emailoplatetoj estas enordigebla en tri grupojn: La unua grupo, konsistigata el la entute ok platetoj sur la transverstrabetoj bildigas stiligitajn bestojn kaj plantojn, kies naturaj modeloj nur povas deveni el okcidenta Azio. La ses tabuletoj muntitaj sur krucotrabo kaj trunko tamen montras plektobendan skemon. La tria grupo, kiun konsistigas grupon el la kvar bildofacoj faritaj alterne el ĉela kaj kava emajloj prezentas skvamostrukturon. La dorsflanko estas tegita per orumita, gravurita kuproplato, kiu montras en rondmedalionoj mezloke la brustobildon de Kristo Reganto de la Universo samkiel je la krucofinaĵoj la evangeliistaj simboloj .;joyas;486300;Teofano-Kruco;8476087;https://eo.wikipedia.org/w/index.php?oldid=8476087
Per la fragmente konserviĝinta enskribo Teofano estas identigebla kiel donacinto de la kruco. La donacado de la kruco estus pensebla kiel kontraŭa resp. kompletiga donacaĵo al la kruco kun la grandaj mallevaĵo-emajloj jam ekestinta sub la antaŭulo de Teofano, Matilda la 2-a . Preciza datigo de la donacado fare de Teofano ene de ŝia abateco preskaŭ ne eblas, tamen estas sencodone supozi rilaton al la konsekrado de la kripto en la abatopreĝejo de Essen el 1051 aŭ eble al la konsekrado de la altaltaro en 1054.;joyas;486300;Teofano-Kruco;8476087;https://eo.wikipedia.org/w/index.php?oldid=8476087
El la 18 emajlotabuletoj almenaŭ ok, verŝajne tamen eĉ 14 estas de bizanca deveno, la kvar tabuletoj el la tria grupo ŝajnas esti fabrikitaj en la Sankta Romia Imperio . Je la emajlaĵoj finfine temus pri pli malnovaj pecoj, kiujn la donacinto disponigis, simile al la gemoj, el sia persona posedaĵo trezora por la fabrikado de la kruco. En la bizancaj eroj de la Teofano-Kruco eble estas rekonebla deveno el la otida heredaĵo.;joyas;486300;Teofano-Kruco;8476087;https://eo.wikipedia.org/w/index.php?oldid=8476087
La Matilda-Kruco estas procesiokruco de la katedrala trezoro de Essen , kiu estis farita sub abatino Teofano de Essen (mortinta en 1058). Sian nomon ĝi ricevis laŭ abatino Matilda la 2-a , kiu estas bildigita sur emajlo ĉe la krucotrunko kiel donacinto. La kruco, ofte ankaŭ nomata la pli juna aŭ dua Matilda-Kruco, staras en arta rilato al la 60 jarojn pli malnova Oto-Matilda-Kruco , kiu ankaŭ estis nomata la „pli malnova“ aŭ unua Matilda-Kruco, kaj al la kruco kun la grandaj mallevaĵo-emajloj . Ĝi estas konservata en la katedrala trezorejo de Essen.;joyas;486430;Matilda-Kruco;8373216;https://eo.wikipedia.org/w/index.php?oldid=8373216
La tiel nomata Infankrono de Oto la 3-a estas ora , otida krono en la katedrala trezoro de Essen . Antaŭe oni supozis, ke eble temus pri la krono, per kiu trijara Oto la 3-a en 983 estis kronata kiel Reĝo de la Romanoj . Sed ĉi tiu opinio, verŝajne atribuenda al idealisma pensado de la historiistoj de Essen de la frua 20-a jarcento, intertempe validas kiel malnoviĝinta. Cetere ĝi tamen estas la plej malnova konserviĝinta liliokrono tutmonde.;joyas;487129;Infankrono de Oto la 3-a;9237636;https://eo.wikipedia.org/w/index.php?oldid=9237636
La formo konvenas laŭ sia formo al bizanca kapringo de 3,5 centimetroj da larĝeco. Ĝia diametro estas nun 12,5 centimetroj. La krono estas adaptita al la kapoperimetro de la Ora Madono . La gemosurmetaĵo de la krono estas regula kaj konformas kun la perimetro. La krono konsistas el ora ringo ege alojita per arĝento , sur kiu ekstere estas lutita den dua ringo el pli pura oro. En la interno de la krono estas videbla fera stabiliga ringo.;joyas;487129;Infankrono de Oto la 3-a;9237636;https://eo.wikipedia.org/w/index.php?oldid=9237636
La supra kaj malsupra randoj de la krono estas ornamitaj per perlo ŝnuroj, je kio la tiujn traiĝanta drato estas lutita per metalringoj sur la fundamenton. Sur la kvar liliopintoj kaj sur la ringo mem troviĝas multaj perloj kaj gemoj , je kio la gemoj rekte surmuntitaj sub la liliopintoj estas aparte altvalore enkadrigitaj. Inter la gemoj troviĝas malnovromia gemo, tranĉita el ruĝa almandino , kun meduzokapo same kiel safiro en triangula orkadro sur la frontflanko de la krono.;joyas;487129;Infankrono de Oto la 3-a;9237636;https://eo.wikipedia.org/w/index.php?oldid=9237636
Kompareblaj, tamen pli junaj kronoj en eklezia posedo troviĝas en la ekleziaj trezoroj de Hildesheim , Germanujo ( Germanio ) kaj en Conques, Francujo ( Francio ).;joyas;487129;Infankrono de Oto la 3-a;9237636;https://eo.wikipedia.org/w/index.php?oldid=9237636
La deveno de la krono estas ne klarigita. Longan tempon oni supozis, ke la krono estis farata en 983 por la kronado de Oto la 3-a, kiu ĝin siaflanke donacis al la abatinejo Essen . La abatinejo Essen sub estrado de abatino Matilda , kiu mem estis nepino de imperiestro Oto la 1-a , tenis aparte intimajn rilatojn kun la otida dinastio , kiu rezultis en gravaj donacadoj fare de la reĝoj kaj ankaŭ en tio, ke la fratino de Oto la 3-a, kiu nomiĝis same kiel la abatino Matilda , estis edukata en Essen. Je la komenco de februaro 993, do proksimume Kandelfeston , Oto la 3-a vizitis la abatinejon Essen, dum kiam li laŭ supozo de lokaj historiistoj li faris al la abatinejon Essen du elstarajn donacojn.;joyas;487129;Infankrono de Oto la 3-a;9237636;https://eo.wikipedia.org/w/index.php?oldid=9237636
La unua donaco estis glavo el damaska ŝtalo , kiu estis fabrikata supozeble proksimume 950 kaj estis uzata en bataloj, kiel sciigas uzopostsignoj ĉe la klingo. Ĉi tiu glavo, adorinda ja pro sia portinto, estis en Essen provizata per multvalora envolvaĵo el oro. En la sekvatempo la glavo estis rigardata kiel la ceremonia glavo de Kosmaso kaj Damiano , la sanktulaj patronoj de urbo kaj abatinejo Essen, kaj eniris tiamaniere en la blazonon de Essen . Por la nuntempa esploro la respondo de la demando, kiu estis la origina portanto de la glavo, estus interesa por pli bona kompreno de la interrilato de la abatinejo Essen kaj la familio de la Otidoj.;joyas;487129;Infankrono de Oto la 3-a;9237636;https://eo.wikipedia.org/w/index.php?oldid=9237636
La dua donaco de Oto la 3-a estis eble la ora krono. Mankas por tio ĉia skriba pruvo, tamen malgraŭ tio ekzistas argumentoj por ĉi tiu tezo. Unuflanke la krono estas datigata pro arthistoriaj komparoj je la finiĝanta 10-a jarcento. La fera stabiliga ringo konkludigas adaptadon al la kapa grando de la Ora Madono, tiel ke la krono origine estus fabrikita por alia celo. Tiu ĉi alia celo povus nur estis la kronado de infano, ĉar oni tion ja povus konkludi el la origina korpogrando. La kronado de Oto la en 983 je la katedralo de Akeno estas la ununura infankronado en la konvena tempospaco. Ankaŭ la fakto, ke la mezepoka kutimo kroni je Kandelfesto la 2-an de februaro statuojn de Maria, estas unuafoje dokumentebla en Essen, pledus por tio, ke Oto la 3-a transdonis je sia vizito en Essen je la komenco de februaro 993 la kronon al la abatinejo.;joyas;487129;Infankrono de Oto la 3-a;9237636;https://eo.wikipedia.org/w/index.php?oldid=9237636
Pli modernaj formelementoj (kies datigometodo en la esploro tamen ankaŭ estas pridubata) supozigas alifabrikadon de la krono en la mezo de la 11-a jarcento. En la sama tempospaco ekestis per la Teofano-Kruco kaj la kruconajo-relikvujo de abatino Teofano artverkoj, je kiuj oni reuzis emajlojn , kiuj devenis de la glorkrono de la Ora Madono, kiu je kronado de la statuo estis ĝena kaj pro tio estis forigita. Fabrikado de la krono antaŭ proksimume 1040/50 do estas verŝajna. En la pli nova esploro oni pripensas datigi la tutan kronon en la komenciĝantan 11-an jarcenton, pro certaj ornamformoj, tiel nomataj florkronoj kaj abelujoj, kiuj tre simile troveblas ĉe artverkoj alordigeblaj al la medio de Henriko la 2-a . Tiam la krono estus fabrikata origine por la kronado de la Ora Madono. En la katedrala trezorejo de Essen ĉi tiuj ornamformoj troviĝas sur la krucofinaĵoj de la kruco kun la grandaj mallevaĵo-emajloj , kiuj estis alifabrikataj supozeble sub Zofia (1011–1039). Por kreado de la krono kiel maria krono ankaŭ pledas, ke la regula gemosurmetaĵo igis neverŝajna plimalgrandigon de la krono.;joyas;487129;Infankrono de Oto la 3-a;9237636;https://eo.wikipedia.org/w/index.php?oldid=9237636
Kiel elstara ekzemplo por la otida oraĵista arto la krono estis bildigata en 1988 sur poŝtmarko de la Germana Federacia Poŝto.;joyas;487129;Infankrono de Oto la 3-a;9237636;https://eo.wikipedia.org/w/index.php?oldid=9237636
"Diademo estas speco de krono , specife ornama kaprubando portata de orientaj monarkoj kaj aliaj kiel insigno de reĝeco. La vorto venas de la greka διάδημα diádēma, ""bendo"" aŭ ""rubando"", de διαδέω diadéō, ""mi rondoligas"" aŭ ""mi bukas"". Laŭ Francisko Azorín Diademo estas Surkapa zono aŭ duonkrono ornamita ofte per juveloj. Li aldonas diademajn tipojn kiel asira, hinda, egipta, moderna, klarigitaj per bildoj.";joyas;508862;Diademo;9493496;https://eo.wikipedia.org/w/index.php?oldid=9493496
La esprimo originale rilatis al la brodita blanka silka rubando , finiĝanta en nodo kaj du franĝaj strioj ofte drapiritaj super la ŝultroj, kiuj ĉirkaŭis la kapon de la reĝo por indiki lian aŭtoritaton. Tiaj rubandoj ankaŭ kutimis kroni venkajn atletojn en gravaj sportludoj en antikvo. La termino estis poste aplikita al metala krono, ĝenerale en cirkla aŭ ĉirkaŭkapa formo. Ekzemple, la krono uzita de la reĝoj de anglosaksa Anglio estis diademo, kiel estis tiu de barono poste (en kelkaj landoj surigita per tri globoj). La antikvaj keltoj verŝajne uzis maldikan, duonovalan orpladon nomitan mint (en malnovirlanda ) kiel diademo. Kelkaj el la plej fruaj ekzemploj de tiuj specoj de kronoj povas esti trovitaj en antikva Egiptujo , de la simpla ŝtofa speco ĝis la pli kompleksa metala tipo kaj en la mondo de Egea Maro .;joyas;508862;Diademo;9493496;https://eo.wikipedia.org/w/index.php?oldid=9493496
Diademo estas ankaŭ juvelita ornamaĵo en la formo de duonkrono, uzata de virinoj kaj metata super la frunto (en tiu signifo, ankaŭ nomita tiaro ). En kelkaj socioj, ĝi povas esti girlando uzata ĉirkaŭ la kapo. La antikvaj persoj uzis altan kaj vertikalan reĝan tiaron ĉirkaŭitan per diademo.;joyas;508862;Diademo;9493496;https://eo.wikipedia.org/w/index.php?oldid=9493496
"Per etendaĵo, ""diademo"" povas esti uzata ĝenerale por emblemo de reĝa potenco aŭ digno. La kapregnaj simboloj portataj de romiaj imperiestroj, de la tempo de Diokleciano antaŭen, estas priskribitaj kiel diademoj en la originaj fontoj. Estis tiu objekto kiun la Foederatus generalo Odoakro revenigis al imperiestro Zenono (nome imperiestro de la Orienta Romia Imperio ) post lia elpelo far la uzurpanto Romulo Aŭgustulo for de Romo en 476 p.K.";joyas;508862;Diademo;9493496;https://eo.wikipedia.org/w/index.php?oldid=9493496
La Ora Ĉevaleto ( germane Goldenes Rössl ) estas la ĉefa juvelo de la trezorejo de Altötting kaj majstraĵo de la pariza oraĵista kaj emajla arto de la 15-a jarcento. Ĝi altas 62 centimetrojn, larĝas 45 centimetrojn kaj profundas 27 centimetrojn, ĝi havas surkonstruaĵon el orumita arĝento surmetitan per gemoj kaj orajn figurojn, kiuj estas tegitaj per emajlo . Ora Ĉevaleto apartenas al la plej altvaloraj artotrezoroj en okcidenta Eŭrazio kun altega asekura valoro.;joyas;514374;Ora Ĉevaleto;8897367;https://eo.wikipedia.org/w/index.php?oldid=8897367
En la jaro 1404 la franca reĝino Izabela de Bavarujo , Vitelsbaĥanino , disdonis la komision, produkti la Oran Ĉevaleton por sia edzo reĝo Karlo la 6-a de Francujo . Jam mallonge post sia ekesto ĝi iris kiel garantiaĵo por pagenda jara pensio en la posedon de la frato de Izabela, duko Ludoviko la 7-a de Bavarujo-Ingolstadt. Finfine la Ora Ĉevaleto fariĝis, post formorto de la dinastia linio de Ingolstadt, posedaĵo de la dukoj de Bavarujo- Landshut . Tiuj ĉi ĝin donis en 1506 al Altötting , por repagi pruntmonon por la milito pri la heredo de la duklando Landshut.;joyas;514374;Ora Ĉevaleto;8897367;https://eo.wikipedia.org/w/index.php?oldid=8897367
En 1801, dum la sekularigo , la Ora Ĉevaleto iris al Munkeno , kie oni volis fandi ĝin, sed tio ne okazis. 20 jarojn pli malfrue oni reportis ĝin al Altötting. En 1921 du viroj el Berlino provis ŝteli la artverkon. Ili perforte enpenetris en la sakristion kaj defajlis la ĉevaleton kaj la ĉevalejserviston. Kiam ili ekigis alarmon, la polico venis, je la postsekva interpafado la du rompŝtelistoj estis vundataj, kaj unu el ili mortis pro sia vundiĝo.;joyas;514374;Ora Ĉevaleto;8897367;https://eo.wikipedia.org/w/index.php?oldid=8897367
De 1992 ĝis 1995 la metiejo de la Bavara Nacia Muzeo en Munkeno elspeze kaj ĝisfunde riparis la Oran Ĉevaleton je spertula kunlaboro fare de arthistoriistoj de la pariza Luvro , de la Brita Muzeo en Londono same kiel el Germanujo kaj Aŭstrujo . La trezoraĵo revenis, post ekspozicio en aprilo de 1995 en Munkeno, sur sian lokon en la trezorejo de la kolegiatopreĝejo de Altötting.;joyas;514374;Ora Ĉevaleto;8897367;https://eo.wikipedia.org/w/index.php?oldid=8897367
"Antaŭ laŭbo abunde ornamita per grandaj rubenoj kaj safiroj same kiel perloj tronas sur soklo, dekoraciita per blazon iridoj , Maria kun fileto Jesuo. Tuj antaŭe genuas Johano la Baptisto kaj Johano la Evangeliisto , maldekstre apude Katarina de Aleksandrio , kiu de Jesuo ricevas ringon kiel indiko pri la ""mistika gefianĉiĝo"". Antau la soklo genuas sur kuseno preĝanta reĝo Karlo la 6-a en mantelo, ankaŭ riĉe dekoraciita per blazonirido, vidalvide ties kavaliro kun la kasko de la reganto enmane. La laŭbo baziĝas sur malalta volbo portata de kvar dikaj kolonoj, akompanata de du flankaj ŝtuparoj. Sub la volbo staras selita kaj orname bridita grizoĉevalo, kiun tenas rajdservisto. La laŭboinstalaĵo kaj ĉiuj figuroj konsistas el 24-karata (= pura) oro. La maldikege martelformitaj figuroj estas tegitaj per emajlo el blanko, ruĝo, verdo kaj bluo. La arĥitekturo de la sokloetaĝo estas farita el orumita arĝento.";joyas;514374;Ora Ĉevaleto;8897367;https://eo.wikipedia.org/w/index.php?oldid=8897367
La Verduna Retablo ( germane Verduner Altar ), ankaŭ nomata retablo de Klosterneuburg ( Klosterneuburger Altar ), nomiĝas laŭ la devenurbo de sia kreinto Nikolao de Verduno, kiu ĝin faris en 1181. La emajlaĵo, kiun konservas la monaĥejo de Klosterneuburg , prezentas supron de mezepoka oraĵistoarto , havas komplikan bildprogramon kaj estas de altega arta graveco.;joyas;514542;Verduna Retablo;8727168;https://eo.wikipedia.org/w/index.php?oldid=8727168
Je la origina verko de Nikolao de Verduno supozeble ne temas pri retablo, sed pri tegaĵo de predikejo en la romanika abatopreĝejo de Klosterneuburg. Tiucele oni produktis – verŝajne trans tempospaco de proksimume dek jaroj (1171-1181) – 45 emajltabulojn ( orumitajn kuproplatojn kun mallevaĵo-emajloj) kaj fiksis ilin sur lignan portilon. Ilin mendis preposto Wernher (1168-1185), kiel informas la omaĝenskribaĵo. Pri la enhava koncepto tamen verŝajne respondecis lia antaŭulo, preposto Rudiger (1167-1168).;joyas;514542;Verduna Retablo;8727168;https://eo.wikipedia.org/w/index.php?oldid=8727168
Al retablo oni kunmetis la tabulojn nur en 1330/1331. Post brulego en 1330, kiu estis detruinta grandajn partojn de la abatejo Klosterneuburg, la tiama preposto, Stefano de Sierndorf (1317-1335), transportigis la tabulojn al Vieno , por translaborigi ilin al retablo. Por ebligi la fermon de la aloj, oni kompletigis la verkon per ses novaj, stile adaptitaj tabuloj, tiel ke la retablo nun entenas entute 51 tabulojn. La dorsflankoj de la retablo ricevis tamen kvar grandajn temperopentraĵojn.;joyas;514542;Verduna Retablo;8727168;https://eo.wikipedia.org/w/index.php?oldid=8727168
La programo havas strukturon de tri bildovicoj kaj montras scenojn el la Malnova kaj Nova Testamentoj . Kiel klarigas la omaĝenskribaĵo, la verko klarigu la grandan signifon de la Malnova Testamento por la kompreno de la Nova Testamento. Je tio ĝi subdividas la savohistorion en tri epokojn:;joyas;514542;Verduna Retablo;8727168;https://eo.wikipedia.org/w/index.php?oldid=8727168
La plej frua epoko antaŭ la leĝo (t.e. scenoj el la Malnova Testamento ĝis Moseo), aperas en la plej supra vico. La tempon sub la leĝo prezentas la plej malsupra vico (t.e. scenoj el la Malnova Testamento post Moseo). Intere, en la meza vico, finfine troviĝas scenoj el la Nova Testamento, do el la subgraca epoko.;joyas;514542;Verduna Retablo;8727168;https://eo.wikipedia.org/w/index.php?oldid=8727168
Al ĉiu sceno el la Nova Testamento estas supre kaj malsupre kontraŭmetataj scenoj el la Malnova Testamento, kiuj certamaniere jam montras sur la agado de Kristo. La supozo, ke la scenoj el la vivo de Jesuo havas precizan paralelaĵon en la okazintaĵoj de la Malnova Testamento, ankaŭ nomiĝas antaŭfigurado. Pro tio ĉiam tri tabuloj apartenas unu al la alia, pro kio formiĝas kolumno.;joyas;514542;Verduna Retablo;8727168;https://eo.wikipedia.org/w/index.php?oldid=8727168
El la entute 17 kolumoj 15 sekvas ĉi tiun sistemaron – ankaŭ la tabuloj ekestintaj en la 14-a jarcento (8-a kaj 10-a kolumnoj). En la du lastaj kolumnoj de la dekstra alo (16-a kaj 17-a kolumnoj) oni jam en la 12-a jarcento malakceptis la programon. Anstataŭe oni prezentis la Lastan Juĝon . Maldekstra alo:;joyas;514542;Verduna Retablo;8727168;https://eo.wikipedia.org/w/index.php?oldid=8727168
En arta kaj teĥnika rilato la Verduna Retablo estas absoluta majstraĵo. Stile la tabuloj ja estas influitaj ankoraŭ de bizanca arto , sed je la korpeco de la figuroj estas sentebla forta influo de antikvaj modeloj. La figuroj estas en movo kaj anatomie ĝuste bildigitaj. Krom tio okazas aludo de emociaj sentoj, nekutima por la 12-a jarcento. Ankaŭ Nikolao de Verduno realigis je altega nivelo la tre postuleman procedon de malleviĝo-emajlo. La fonoj ja plejparte estas bluaj, tamen la artisto uzis ĉion eblecon, por krei ĉarmajn kolorajn transirejojn kun pluroblaj ombrigoj. Laŭ artisma kvalito la verko estas pli komparebla kun la Tri-Reĝo-Relikvujo en Kolonjo .;joyas;514542;Verduna Retablo;8727168;https://eo.wikipedia.org/w/index.php?oldid=8727168
"Iu ""Majstro de la Dorsflanko de la Verduna Retablo"" ( provizora nomo de gotika pentristo ) verkis en 1330/1331 la bildojn sur la dorsflanko de la retablo. Tiuj ĉi apartenas al la plej malnovaj konserviĝintaj ekzemploj de tabulpentrado norde de la Alpoj . Dum la transkonstruo al retablo oni kombinis ilin kun la emajltabuloj de Nikolao de Verduno. Nur en la 20-a jarcento oni reseparis ilin de tiuj ĉi pro konservokaŭzoj. La mezparto prezentas maldekstre la Morton de Maria kaj dekstre la Kronadon de Maria . Sur la maldekstra alo troviĝas la Krucumado de Kristo , sur kiun preposto Stefano de Sierndorf ankaŭ instaligis sian fondintoportreton. La dekstra alo montras la Reviviĝon de Kristo , je kio la prezentadoj de la Tri Mariaj ĉe la tombo same kiel la Ne-tuŝu-min -sceno estas kunigita en unu bildo. La stilo de la pentraĵoj montras kaj italajn (arĥitekturajn elementojn, sanktulbildajn detalojn) kaj ankaŭ francajn influojn (figurokomponadon, faltosvingon).";joyas;514542;Verduna Retablo;8727168;https://eo.wikipedia.org/w/index.php?oldid=8727168
Tiffany & Company (pli bone konata kiel Tiffany aŭ Tiffany's ) estas usona kompanio kiu vendas luksajn varojn. La ĉefvendejo estas en Novjorko .;joyas;560364;Tiffany & Co.;8983619;https://eo.wikipedia.org/w/index.php?oldid=8983619
Tiffany vendas juvelojn , arĝentaĵojn , porcelanon , kristalon , parfumon , horloĝojn kaj personajn akcesoraĵojn , kaj ankaŭ kelkajn ledaĵojn . Multaj de tiuj varoj estas vendataj per poŝto kaj ankaŭ en la vendejoj. Tiffany speciale konatiĝis pro sia diamanta juvelaro. Tiffany surmerkatas sin mem kiel juĝisto de gusto kaj stilo.;joyas;560364;Tiffany & Co.;8983619;https://eo.wikipedia.org/w/index.php?oldid=8983619
"Fondita en Novjorko en 1837, la vendejo vendis vastan varion de aĵoj kiel ""Tiffany, Young and Ellis"" en Manhattan . La nomo estis mallongigita kiel Tiffany & Company en 1853, kiam Charles Tiffany transprenis la gvidadon. Li establis la emfazon de la firmao je juveloj. Poste lia filo Louis Comfort Tiffany estis la unua dezajndirektoro de la familia entrepreno.";joyas;560364;Tiffany & Co.;8983619;https://eo.wikipedia.org/w/index.php?oldid=8983619
La torkoj de Leekfrith (angle: Leekfrith torcs ) estas kvar oraj keltaj torkoj el la Ferepoko , malkovritaj en la distrikto Leekfrith, Staffordshire, je decembro 2016 fare de du viroj per metal-detektiloj. La torkoj estas tri koltorkoj kaj unu braktorko, kiuj estis faritaj laŭ la stilo de la La-Tène-kulturo en la nuna Germanio aŭ Francio inter 400 kaj 250 a.K. Ĝi eble estas la trovo de plej antikvaj oraj juveloj en Britio .;joyas;582150;Torkoj de Leekfrith;8482955;https://eo.wikipedia.org/w/index.php?oldid=8482955
Egina kaŝtrezoro aŭ Egina trezoro estas grava minoa ora kaŝtrezoro, trovita sur la insulo de Egino , Grekio. Ekde 1892, ĝi estas parto de kolektaĵo de la Brita Muzeo .;joyas;586419;Egina kaŝtrezoro;8144336;https://eo.wikipedia.org/w/index.php?oldid=8144336
La trezoro estis onidire trovita en tombo sur insulo Egino en 1891, sed la ekzaktaj cirkonstancoj neniam estis klarigitaj. La Brita Muzeo aĉetis plejparton de la trezoro en 1892 de Cresswell Brothers, Londona firmao de spongo -komercistoj. Pliajn pecojn de la trezoro aĉetis la muzeo en 1914.;joyas;586419;Egina kaŝtrezoro;8144336;https://eo.wikipedia.org/w/index.php?oldid=8144336
La Egina kaŝtrezoro konsistas plejparte el oro- juvelaro , kiu estis datita - bazita je ties stilo kaj ikonografio-, je la greka bronza epoko inter 1850 kaj 1550 antaŭ Kristo. Ĝi inkludas du parojn de ornamitaj orelringoj, tri diademojn , unu brusto-pendaĵon, braceleton , oran tason, kvar ringojn, ornamitajn platojn kaj platajn striojn. Tie estas ankaŭ kvin ringegoj kaj multaj bidoj kaj ringoj faritaj de diversaj materialoj inkluzive de oro, lazurŝtono , ametisto, kvarco, karneolo kaj verda jaspo .;joyas;586419;Egina kaŝtrezoro;8144336;https://eo.wikipedia.org/w/index.php?oldid=8144336
"La plej ellaboritaj aĵoj en la trezoro estas brusto-pendaĵoj kaj paro de orelringoj. La pendaĵo ŝajnas reprezenti Kretan diaĵon akompanata de du anseroj sur la kampo. La diaĵo estis nomita la ""Majstro de la Bestoj"" fare de arkeologoj . La paro de orelringoj estas desegnita en la formo de du-kapa serpento , kiu ĉirkaŭas du leporhundojn , starantaj unu al la alia.";joyas;586419;Egina kaŝtrezoro;8144336;https://eo.wikipedia.org/w/index.php?oldid=8144336
Abeloj de Malia estas nomata tiu ora pendaĵo el la minoa periodo , kiu estis malkovrita en la Palaco de Malia sur Kreto kaj hodiaŭ estas ekspoziciita en la Arĥeologia Muzeo de Iraklio .;joyas;586422;Abeloj de Malia;8886801;https://eo.wikipedia.org/w/index.php?oldid=8886801
La abeloj de Malio estis malkovrita en 1930 fare de la francaj arĥeologoj Fernand Chapouthier kaj Pierre Demargne en la Nekropolo de la tomb-konstruaĵo. La pendaĵo havas larĝon de 4,9 cm kaj alton de 4,6 cm. Ĝi figurigas du abelojn aŭ vespojn, kiuj suĉas mielguton. Laŭ alia interpretado, ili metas la mielguton en la mielĉelaron , kiu troviĝas inter ili. La korpoj de la insektoj estas kurbitaj ĉirkaŭ la mielĉelaro, tiel ili formas kune ringon. Super la kapoj de la abeloj troviĝas ora globo en drata kaĝo. Ĉe ambaŭ flugiloj kaj tie, kie iliaj pikoj kuntuŝas unu la alian, pendas disketoj kun granulitaj randoj. La oraj granuletoj sur la mielĉelaro kaj la abdomenoj de la insektoj estas pli fajnaj kaj havas diametron de ĉ. 0,4 mm. La juvelo estis produktita en ĉ. 1800 ĝis 1700 a.K., ĉe fino de la minoa malnova palasta epoko kaj validas kiel majstra verko de la minoa arto.;joyas;586422;Abeloj de Malia;8886801;https://eo.wikipedia.org/w/index.php?oldid=8886801
Brilianto estas diamanto aŭ alia juvelo , tranĉita partikulare montrante multajn facetojn por havi eksterordinaran brilon. Ĝia formo similas al tiu de konuso kaj havigas maksimuman revenon de lumo en la supra parto de la diamanto.;joyas;597709;Brilianto;9432346;https://eo.wikipedia.org/w/index.php?oldid=9432346
Laŭ PIV Brilianto estas Diamanto kun aparta tajlo (du horizontalaj plataĵoj kunigitaj per facetoj) k brilpovo.;joyas;597709;Brilianto;9432346;https://eo.wikipedia.org/w/index.php?oldid=9432346
Laŭ Francisko Azorín Brilianto estas simple Facetita diamanto. Li indikas etimologion el latina brilleus el berilo (brilŝtono, berilo ).;joyas;597709;Brilianto;9432346;https://eo.wikipedia.org/w/index.php?oldid=9432346
La moderna ronda brilianto konsistas de 58 facetoj (aŭ 57 se la kuleto ekskludiĝas), 33 sur la krono * (la supra duono super la mezo aŭ talizono * de la ŝtono) kaj 25 sur la pavilono * (la malsupra duono sub la talizono). Dum lastatempaj jardekoj, plej multaj talizonoj estas facetitaj. Multaj talizonoj havas 32, 64, 80, aŭ 96 facetoj, tiuj ĉi facetoj nombriĝas en la tuto. Dum la faceta nombro estas norma, la efektivaj proporcioj (krona alteco kaj angulo, pavilona profundeco, k.t.p.) ne universale konsentiĝas. Oni povas paroli pri la Usona tranĉaĵo aŭ la Skandinavia Normo (Scan. D.N.) por doni nur du ekzemploj.;joyas;597709;Brilianto;9432346;https://eo.wikipedia.org/w/index.php?oldid=9432346
En socio kaj ĉefe pro propagando farita el holivudaj filmoj , donaci ringon kun brilianto estas plej taŭga maniero per kiu viro petu geedziĝon al virino. Eĉ necesas partikulara ceremonio proponi ĝin surgenue ene de skatolo malfermota de ŝi.;joyas;597709;Brilianto;9432346;https://eo.wikipedia.org/w/index.php?oldid=9432346
En ekonomio kaj politiko, la industrio tajli diamantojn estas grava sektoro de ekonomio ekzemple en kelkaj urboj de Flandrio kaj Nederlando , konektita kun importado de diamantoj ekzemple el afrikaj landoj.;joyas;597709;Brilianto;9432346;https://eo.wikipedia.org/w/index.php?oldid=9432346
Intajlo estas ĉiu malmola ŝtono gravurita en vakuo, speciale tiu uzita kiel sigelo . La fajnaj ŝtonoj uzataj tiucele estas ofte agatoj , karneoloj , smeraldoj , ametistoj , grenatoj kaj lazuritoj .;joyas;598829;Intajlo;9072979;https://eo.wikipedia.org/w/index.php?oldid=9072979
La arto produkti kameojn kaj aliajn similajn objektojn kiaj intajloj estas nomata gliptiko (de la greka glypho , gravuri) kaj estas speciala fako de la skulptarto . La produktaĵoj de la gliptiko utilis el pratempoj de la Historio por fabriki sigelojn , ornamaĵojn kaj religiajn objektojn aŭ proprajn de superstiĉo .;joyas;598829;Intajlo;9072979;https://eo.wikipedia.org/w/index.php?oldid=9072979
Laŭ Francisko Azorín intajlo estas Konkave gravurita gemo, uzata kiel sigelilo aŭ kiel ornamo. Li indikas etimologion el la latina intalio el en + talliare (tajli).;joyas;598829;Intajlo;9072979;https://eo.wikipedia.org/w/index.php?oldid=9072979
Medaliono estas rondoformaj ornamaĵoj ĝenerale metalaj, sed ankaŭ ceramikaj, kiuj estas uzataj penditaj el kolo por persona ornamado. Ofte ili pendas unuigitaj al kolieroj, kvankam ili havas propran signifon. Specifa tipo de medaliono estas tiuj kiuj fariĝas kiel artaj rememoraĵoj pri eventoj, omaĝoj, premioj ktp. La scienco kiu studas medalionojn estas la medaliko .;joyas;599889;Medaliono (juvelo);9087738;https://eo.wikipedia.org/w/index.php?oldid=9087738
Laŭ Francisko Azorín medaliono estas Bareliefo ĉirkaŭita per ronda aŭ ovala kadro. Ronda juvelo en kiu oni metas miniaturon, rememoraĵon. Li indikas etimologion el la latina medalia (medalo), pligrandiga formo.;joyas;599889;Medaliono (juvelo);9087738;https://eo.wikipedia.org/w/index.php?oldid=9087738
Gefianĉiĝa ringo estas tiu ringo kiu, laŭ kutimoj de Okcidento , estas portata por indiki fianĉiĝon, tio estas engaĝiĝo por geedziĝo .;joyas;624150;Gefianĉiĝa ringo;8199040;https://eo.wikipedia.org/w/index.php?oldid=8199040
La ringo estas portata ĉirkaŭ la ringofingro (kiu laŭ tradicio enhavas la vena amoris (amvejno)). La mano en kiu oni portas la ringon tradicie dependas el la zono. El jarcentoj oni diras, ke la vena amoris pasas en la maldekstra ringofingro kaj konektas rekte al la koro. Tradicie, ĝi estas proponita de la viro kiel donaco al lia fianĉino dum aŭ tuj kiam ŝi akceptas la proponon de geedziĝo. Ĝi reprezentas formalan akcepton de la engaĝiĝo por la futuro. Etnologoj kaj antropologoj sugestas ankaŭ la simbolan formon de la kuniĝo de la ringo al la fingro, ĉu kiel ligilo ĉu kiel aludo al amoro.;joyas;624150;Gefianĉiĝa ringo;8199040;https://eo.wikipedia.org/w/index.php?oldid=8199040
La fianĉino normale ricevas kiel gefianĉiĝa ringon el arĝento (kaj malpli ofte el oro , ĉar oro estas uzata ĝenerale por la geedziĝa ringo ), kiu ĝenerale montras juvelon, ĉu valorega ĉu duonvalorega (kiel diamanto ). Tradicie, en Hispanio oni proponis juvelon al la fianĉino en familia ceremonio de nuptopropono (plej ofte braceleto ) kaj nur ĵuse, kaj plej verŝajne pro influo de filmoj el Holivudo , disvastiĝas la ringo kiel preferato.;joyas;624150;Gefianĉiĝa ringo;8199040;https://eo.wikipedia.org/w/index.php?oldid=8199040
Louis Comfort TIFFANY (18a de Februaro, 1848 – 17a de Januaro, 1933) estis usona artisto kaj desegnisto kiu laboris en ornamaj artoj kaj estas plej bone konata pro sia laboro en vitraloj . Li estas la usona artisto plej ofte asocia kun la movadoj de Art Nouveau kaj estetikismo . Li kunlaboris kun prestiĝa grupo de desegnistoj konataj kiel Associated Artists, inter kiuj Lockwood de Forest, Candace Wheeler, kaj Samuel Colman. Tiffany desegnis vitralajn fenestrojn kaj lampojn , vitrajn mozaikojn, blovovitron, ceramikon , juvelojn , emajlojn kaj metalaĵojn. Li estis la unua dezajndirektoro de sia familia entrepreno, Tiffany & Co. , fondita de sia patro Charles Lewis Tiffany.;joyas;711184;Louis Comfort Tiffany;9396408;https://eo.wikipedia.org/w/index.php?oldid=9396408
Imperia Ŝtata Krono ( angle Imperial State Crown ) estas krono, kiu estas parto de la Reĝaj Juveloj de Britio.;joyas;732396;Imperia Ŝtata Krono;8239044;https://eo.wikipedia.org/w/index.php?oldid=8239044
Simila laŭ projektado al la krono de Krono de Sankta Eduardo , kun la bazo de la krono kvar krucoj (kruco Pattée) inkluzive de kvar Lilifloroj . Sur la bazo de la krono estas kvar duone arkaĵoj , kiuj kuniĝas en ligo ora Mondeo kiu suportas krucon . Sub la arkoj de velura kupolo kies randoj estas ornamitaj per armino. La krono estas ornamita per 2 868 diamantoj , 273 perloj , 17 safiroj , 11 smeraldoj kaj 5 rubenoj .;joyas;732396;Imperia Ŝtata Krono;8239044;https://eo.wikipedia.org/w/index.php?oldid=8239044
"La krono estas ferlita per konataj grandvaloraj ŝtonoj - en la kruco supre de la krono estas safiro nomita ""Sankta Eduardo Safiro"", kiu verŝajne estas prenita de la ringo de Eduardo la Konfesanto . Ankaŭ antaŭ la krono estas inkrustita la rubena Nigra Princo. La fronto de la krono estas metita per diamanto Cullinan II, ankaŭ konata kiel la ""Eta Stelo de Afriko"".";joyas;732396;Imperia Ŝtata Krono;8239044;https://eo.wikipedia.org/w/index.php?oldid=8239044
La krono estas eluzita ĉe la fino de la kronada ceremonio kiam la nova monarko forlasas Abatejon Westminster - kaj ne dum la kronada ceremonio mem. Reĝino Viktorio kaj Reĝo Eduardo la 7-a portis la kronon de la ŝtata imperio dum la kronada ceremonio mem pro la granda pezo de la krono de Sankta Eduardo .;joyas;732396;Imperia Ŝtata Krono;8239044;https://eo.wikipedia.org/w/index.php?oldid=8239044
La monarko portas la kronon de la ŝtata imperio ĉe la jara malferma ceremonio de la parlamento. Laŭ tradicio, la krono kaj aliaj kronjuveloj estas transportitaj al Palaco Buckingham en sia propra kaleŝo de tie ili estas transportitaj dum la reĝino alvenas en la kaleŝo al Palaco de Westminster , kaj alportita al la palaca robĉambro kie la reĝino portas la robon kaj kronon.;joyas;732396;Imperia Ŝtata Krono;8239044;https://eo.wikipedia.org/w/index.php?oldid=8239044
La nuna krono de la imperio estas krono kreita por la kronado de Georgo la 6-a en 1937 . La krono estas kreita de la diamanta kompanio Gerard & Co. kaj estas ekzakta kopio de la krono de la ŝtata imperio kreita por reĝino Viktorio - sed pli malpeza kaj pli komforta por porti.;joyas;732396;Imperia Ŝtata Krono;8239044;https://eo.wikipedia.org/w/index.php?oldid=8239044
Kaj la krono de la ŝtata imperio kaj la restaĵoj de la origina krono uzita de reĝino Viktorio konserviĝas en la Juvela Domo de la Turo de Londono .;joyas;732396;Imperia Ŝtata Krono;8239044;https://eo.wikipedia.org/w/index.php?oldid=8239044
"La Glavo de Kompato ( angle : Sword of Mercy ), aŭ ""glavo de Eduardo la Konfesanto "", estas glavo inter la Kronaj Juveloj de Britio.";joyas;732400;Glavo de Kompato;7223928;https://eo.wikipedia.org/w/index.php?oldid=7223928
La glavo estas amputita , kaj tial ĝia klingo estas mallonga kaj kvadrata. Ĝia formo simbolas la gradon de kompato de la suvereno . Laŭ legendo, anĝelo fortranĉis la glavon por malhelpi nepravigan mortigon .;joyas;732400;Glavo de Kompato;7223928;https://eo.wikipedia.org/w/index.php?oldid=7223928
"La Glavo de Kompato estas unu el kvin glavoj uzataj en la kronado de la reĝoj de Britio . Ŝi estas portita en la kronada procesio inter la ""portempa glavo de justeco"" kaj la ""glavo de spirita justeco"". Laŭ tradicio, ĉi tiuj tri glavoj estis kreitaj antaŭ la kronado de Karlo la 1-a, Reĝo de Anglio , kaj estas inter la malmultaj kronjuveloj ne elfanditaj de Oliver Cromwell .";joyas;732400;Glavo de Kompato;7223928;https://eo.wikipedia.org/w/index.php?oldid=7223928
La sfalerito aŭ blendo estas mineralo de sulfido de zinko (ZnS). Ĝi estas konata delonge, sed en Eŭropo oni ne sciis ekhavi metalon el tio.;joyas;736685;Sfalerito;8448584;https://eo.wikipedia.org/w/index.php?oldid=8448584
Tiu fakto, ties simileco foje al la galeno kaj ties ofta asocio faris, ke Georgius Agricola donis al ĝi, en sia verko De natura fossilium ( La naturo de mineraloj ), la nomon de galena inanis («neutila galeno»). Laŭ indiko en la enhavtabelo, la germanaj ministoj konis ĝin kiel blende (blendo). Probable la vorto blende estis uzita por komprenigi la falsecan aspekton de la mineralo, kiel interpretita de Agrikola kaj aliaj antikvaj fakuloj, sed oni supozis ankaŭ, ke la vorto devenas de blenden , 'blindigi'. Poste la termino -blende estis uzita en germana por formi la nomon de aliaj mineraloj de nigra koloro kaj duonmetala brilo ( Pechblende , 'peĉoblendo', Hornblende , 'kornoblendo').;joyas;736685;Sfalerito;8448584;https://eo.wikipedia.org/w/index.php?oldid=8448584
Aliflanke, inter 1735 kaj 1756, la kemiaj studoj ebligis determini la estadon de zinko kiel esenca konstituanto de tiu mineralo kaj eĉ dezajni metodojn por ties elpreno, kio faris, ke la iama Blende estu Zincblende , 'zinkoblendo'. La ŝanĝoj en la mineralogia nomenklaturo en la unua duono de la 19-a jarcento faris, ke tiu mineralo estu nomata, laŭ Haüy, zinc sulfuré (sulfurzinko), kaj laŭ Glocker Sphalerites , kiel genro (kun variaj specioj, kiuj fakte estas simple variaĵoj), ene de duuma nomenklaturo simila al tiu de Linneo por vegetaloj kaj animaloj. Tiu lasta nomo devenas de la greka sfaleros , 'trompema' (pro la konfuzo kun la galeno). La duuma sistemo de minerala nomenklaturo ne sukcesis, sed ja la nomo elektita por tiu mineralo, tiel ke la nomoj blendo kaj sfalerito (kun ekvivalentoj en la diversaj lingvoj) estis uzitaj ekde tiam sinonime.;joyas;736685;Sfalerito;8448584;https://eo.wikipedia.org/w/index.php?oldid=8448584
En la 1980-aj jaroj, la koncerna komisiono pri minerala nomenklaturo de la Internacia Asocio de Mineralogio reviziis la aferojn de la mineraloj por kiuj oni estis uzante variajn nomojn (kiel blendo/sfalerito, cianito/disteno, mispikelo/arsenopirito) kaj elektis unu kiel oficiala, sfalerito tiuokaze.;joyas;736685;Sfalerito;8448584;https://eo.wikipedia.org/w/index.php?oldid=8448584
La sfalerito kristaliĝas en la kuba sistemo (ĉu izometra ĉu regula), en la klaso -43m, heksatetraedra, spaca grupo F-43m. La malgrandaj kristaloj estas tre bone difinitaj, dum la pli grandaj, kiuj povas atingi grandojn de ĝis 30 cm, aperas distorditaj kaj kun kurbaj flankoj. La plej oftaj figuroj estas la kvaredro {111} kaj la romba dekduedro {110}, kaj la trilateropiramidigita kvaredro ĉu pozitiva {311} ĉu negativa (3-11).;joyas;736685;Sfalerito;8448584;https://eo.wikipedia.org/w/index.php?oldid=8448584
Ties teoria kompono estas SZn, sed en la realaj specimenoj la zinko povas esti anstataŭita de aliaj metaloj. La plej ofta anstataŭanto estas la fero , kiu povas atingi tre altajn proporciojn, ĝis rilato Zn/Fe de 6/5, kvankam ĝenerale tio estas multe pli malgrandaj. La estado de fero produktas la malheligon de la mineralo, ĝis alveni al la nigra koloro. La sfalerito kun tiuj karakteroj ricevas la nomon de variaĵo marmatito . Oftas ankaŭ la trovo en la sfalerito de kadmio , kaj en pli malgrandaj proporcioj, de indiumo . Alia elemento trovebla, sed ne ĉiam, estas la mangano , kiu escepte povas anstataŭi ĝis 36 % de la zinko. Sfalerito povas enhavi ankaŭ kupron, hidrargon, galiumon kaj germaniumon . ​ La koncentroj de unu kaj aliaj estas rilataj.;joyas;736685;Sfalerito;8448584;https://eo.wikipedia.org/w/index.php?oldid=8448584
La ekzisto de tiuj elementoj en la sfalerito havas ekonomiajn kaj mediajn konsekvencojn. La plej parto de la akirita germaniumo devenas de la subproduktoj de la procezado de la sfalerito por akiri zinkon. Oni akiras ankaŭ galiumon , kvankam tiuokaze la ĉefa fonto estas la baŭksito . La estado de hidrargo okazigas problemon de media poluado kaj risko por laboristoj en la industrioj de produktado de zinko, ĉar en la procezo de varmigo de la sfalerito, hidrargo vaporiĝas kaj pasas al la atmosfero aŭ restas sur la ekipaĵoj. Fine de 2012 okazis akcidento en fabriko de la entrepreno Asturiana de Zinc en San Juan de Nieva, Hispanio, en kiu kelkaj dekoj de laboristoj de alia entrepreno por bontenado suferis venenigon pro hidrargo stokita en tuboj laŭlonge de la tempopaso. ​;joyas;736685;Sfalerito;8448584;https://eo.wikipedia.org/w/index.php?oldid=8448584
En la 2000-aj jaroj , la plej multaj formoj de bidbrodigo estis uzataj por fari vestaĵojn (vestoj, monujoj kaj ŝuoj) aŭ juvelaĵojn , sed bidbrodigo estis uzata por krei arton , ornami konstruaĵojn kaj skulptaĵojn . La juvelo kreita el bidbrodigo nomiĝas koliero .;joyas;744702;Bidbrodigo (metio);8560826;https://eo.wikipedia.org/w/index.php?oldid=8560826
Bidbrodigo povas esti farita kombinante malsamajn materialojn kiel ekzemple: fimo, ceramiko , vitraj bidoj, metaloj kaj altvaloraj ŝtonoj .;joyas;744702;Bidbrodigo (metio);8560826;https://eo.wikipedia.org/w/index.php?oldid=8560826
Plej multaj kulturoj de la mondo uzis bidbrodigajn teknikojn por krei juvelaĵojn, vestaĵojn kaj artaĵojn . Arkeologia indico indikas la ekziston de bidbrodigteknikoj jam en prahistoria kulturo, proksimume 15 000 jarojn antaŭ nia tempo.;joyas;744702;Bidbrodigo (metio);8560826;https://eo.wikipedia.org/w/index.php?oldid=8560826
En 2021, La plej frua konata juvelaĵo de la mondo datrilatas al antaŭ proksimume 150 000 jaroj kaj estis malkovritaj en prahistoria kaverno de Maroko apud la marobordo . Tiu estas en formo de 33 bidbrodigitaj konkoj, estis trovita de arkeologoj, kiuj diras, ke la konkaj bidoj eble estis portitaj sur kolĉeno. Kelkaj el la konkoj estas de Gastropodoj de la specio Tritia gibbosula , kiu ankaŭ estis uzita por simila celo fare de homoj de Musterio en la orienta baseno de la Mediteraneo .;joyas;744702;Bidbrodigo (metio);8560826;https://eo.wikipedia.org/w/index.php?oldid=8560826
En antikvaj tempoj estis multe da uzo de bidbrodigoj kaj bidoj en la produktado de religiaj objektoj kaj amuletoj , kiujn ĝiaj uzantoj kredis alporti bonŝancon aŭ resanigon. Hodiaŭ kun la progreso de teknologio tre altkvalitaj bidoj kaj larĝa gamo de koloroj kaj kovraĵoj estis evoluigitaj. Ĉi tiuj bidoj estas uzataj de la bidbrodigaj majstroj en diversaj teknikoj evoluigitaj al metoda nivelo.;joyas;744702;Bidbrodigo (metio);8560826;https://eo.wikipedia.org/w/index.php?oldid=8560826
La Sveda Historia Muzeo ( SHM , svede Statens historiska museum aŭ Historiska museet ), en Stokholmo estas la sveda nacia historia kaj arkeologia muzeo. La muzeo havas pli ol 20 milionojn da objektoj de prahistoriaj kaj mezepokaj tempoj en Svedio , la plej granda parto de ili kiel esplorobjektoj. La Historia Muzeo estas subigita al la Ŝtata Historia Muzeo- Administracio.;joyas;854706;Sveda Historia Muzeo;9431915;https://eo.wikipedia.org/w/index.php?oldid=9431915
La SHM estas la ĉefa historia muzeo de Svedio. Ĝi estas respondeca muzeo, kun ĝenerala respondeco pri arkeologiaj trovaĵoj kaj preĝeja arto. Fondita en 1866 kiel publika institucio, la Sveda Historia Muzeo havas siajn originojn en la kolektoj de arto kaj historiaj objektoj kunvenitaj fare de la sveda monarkio frue en la 16-a jarcento. Havante la statuson de ŝtata agentejo, ĝia celo estas kolekti kaj konservi objektojn de loka kaj regiona historio, kaj certigi ke la scio akirita pri ili estu havebla al publiko per ĝiaj ekspozicioj kaj publikaj okazaĵoj .;joyas;854706;Sveda Historia Muzeo;9431915;https://eo.wikipedia.org/w/index.php?oldid=9431915
La ekspozicio de la muzeo kovras la kulturhistorion de Svedio de la Ŝtonepoko ĝis la 16-a jarcento (la posta tempoperiodo estas reprezentita en la Nordia Muzeo ). La ĉefkonstruaĵo de la muzeo estis dezajnita en 1935-1940. arkitektoj Bengt Romare kaj Georg Sherman. La muzeo situas ĉe la intersekciĝo Narvavägen / Linnégatan en la distrikto Östermalm de Stokholmo.;joyas;854706;Sveda Historia Muzeo;9431915;https://eo.wikipedia.org/w/index.php?oldid=9431915
"La SHM estas parto de centra muzea agentejo ( Statens historiska museer , SHM) (""Naciaj Historiaj Muzeoj""). Aliaj muzeoj sub la egido de ĉi tiu agentejo estas La Reĝaj Armilejoj, La Ekonomika Muzeo, Kastelo Skokloster, Muzeo Hallwyl, La Tumba Papermueleja Muzeo kaj La Sveda Holokaŭsta Muzeo.";joyas;854706;Sveda Historia Muzeo;9431915;https://eo.wikipedia.org/w/index.php?oldid=9431915
"La kulminaĵo de la muzeo estas la tiel nomata ""Ora Ĉambro"" ( Guldrummet ), kreita de la arkitekto Leif Blomberg, kie estas elmontrita granda kvanto da orjuvelaĵoj. La muzeo ankaŭ enhavas kolekton de artefaktoj reprezentantaj la vikingan periodon.";joyas;854706;Sveda Historia Muzeo;9431915;https://eo.wikipedia.org/w/index.php?oldid=9431915
La unua ampleksa ekspozicio en la muzeo malfermis je la 17an de aprilo 1943. Ĝi estis nomita Dek mil jaroj en Svedio kaj konsistita ekspoziciojn de la Ŝtona Aĝo al la mezepoko. La muzeo poste ŝanĝis kaj refari la konstantajn ekspoziciojn nombro de tempoj tiel kiel gastigita plurajn novajn specialajn ekspoziciojn reokazintaj ĉiujare. Ĉi tiuj estas ofte rilatitaj al iu temo kiu estas nuntempe sub debato aŭ alie aktuala. Kelkaj de la specialaj ekspozicioj estis la jenaj:;joyas;854706;Sveda Historia Muzeo;9431915;https://eo.wikipedia.org/w/index.php?oldid=9431915
La muzeo havas ampleksan retan datumbazon. En 2011, informoj pri 480,000 objektoj povus esti aliritaj rete, de kiuj 65,000 havis enskribojn ilustritajn per desegnaĵoj aŭ fotoj. Krome, proksimume 267,300 osttrovaĵoj estis katalogitaj, kun totala pezo de proksimume 111 tunoj. Informoj ankaŭ estas kolektitaj pri 55,900 arkeologiaj lokoj kaj aliaj ejoj en Svedio kie la trovaĵoj estis faritaj.;joyas;854706;Sveda Historia Muzeo;9431915;https://eo.wikipedia.org/w/index.php?oldid=9431915
Len'emaljer ( ruse «Ленэмальер» , ruse «Ленинградский - Эмальер» - kooperativo de emajlistoj, poste uzino, funkciita en Leningrado ek de 1929 ĝis 31-an de aŭgusto 2010 jaro. Dum sia vivo «Len'emaljer» evoluis de kooperativo, kies partoprenantoj laboris en siaj hejmoj, ĝis uzino-avano en produktado de galanterio en USSR. La entrepreno estis tipa sovetia organizo, kies historio kongruis al historio de Soveta Unio kaj posta Rusio. Kvankam artaj atingoj de «Len'emaljer» ne estis elstaraj, produktaĵoj de la entrepreno estas celo de diverstemaj kolektoj kaj muzeoj en postSovetiaj landoj.;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
La kooperativo estis establigita en Leningrado en novembron 1929 . Tri malgrandaj kooperativoj («Mundirovŝĉik» ruse «Мундировщик» , «Emajlmetalo» ruse «Эмальметалл» kaj «Progress» ruse «Прогресс» ) kuniĝis por fondi kooperativo «Leningrada-Emaljer» ( esperante Leningrada emajlisto ) (plena titolo ruse Кооперативная промыслово-производственная артель «Ленинградский-Эмальер» ). La nova interpreno estis subigita al depertamento pri loka industrio de Leningrada urba plenumkomitato ( ruse Управлению местной промышленности Ленгорисполкома ). La 20-an de februaro 1930 Lenurbplenkom akceptis Regularon de la kooperativo .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Ek de komenco kooperativanoj plej parte laboris ne en komunaj produktejoj, sed en siaj loĝlokoj. Ekzemple, por galvanizo estis uzataj personaj elektroakumuliloj. Tamen po iom la kooperativo aĉetis ekipaĵon kaj aparataron. En 1930 jaro la kooperativo jam estis ekipita per 6 frotadaj premiloj, 5 ekscentrikaj premiloj, kelkaj permanaj premiloj, premcilindroj kaj tornilo .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
La adreso de la administrejo de «Len'emaljer» estis str. Konnaja, domo 2/5. Ties produktejoj situis laŭ du adresoj: str. Mitninskaja, domo 12 kaj str. Dzerĵinskaja, domo 46. Poste la kooperativo konsistis en Leningrada Galanteria-industria-kooperativa Unio «Lengalpromsojuz» ( ruse Ленинградский Галантерейно-промыслово-кооперативный Союз «Ленгалпромсоюз» ) En 1930 jaro la kooperativo kunigis 200 laborantojn kaj ĝi estis unu el la plej grandaj kooperativoj de Leningrado . En 1933 jaro la personaro de la kooperativo atingis 1100 homojn .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Produktado de kooperativo «Len'emaljer» aktive kreskis dum la 2-a sovetia kvinjaro. Dum 1932 jaro la kooperativo produktis varojn kostaj 2 788 000 rubloj. Dum 1937 jaro la kosto atingis 24 522 000 rubloj, en 1938 jaro - 32 130 000 kio superis jaran planon 30 500 000 rubloj .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
En 1939 jaro kooperativo «Len'emaljer» estis plej granda en USSR inter interprenoj produktitaj emajlitajn galanteriaĵojn. La kooperativo kunigis pli ol 1600 personoj, kiuj produktis varojn kostaj pli ol 40 milionoj rubloj . Laborantoj de la kooperativo aktive subtenis Staĥanov-movadon . Ekzemple, majstro-staĥanovisto Knjazjkov inventis teknologian plibonigon kaj kolorigis ne unu broĉon, sed partion, divigite operaciojn por la tuta partio. Li kolorigis aparte konkretan elementon ĉe ĉiuj broĉoj de la partio, kaj poste komencis kolorigi la alian elementon de la partio. Rezulte li elspezis 36 min 53 s por unu broĉo kompare kun 41 min 42 s laŭ tradicia teknologio .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
La kooperativo estis sufiĉe fama interpreno en sia sfero kaj ĝin ofte vizitis delegitaroj el diversaj regionoj de Soveta Unio por ricevi sperton de novaj teknologioj kaj organizaj eventoj. Ekzemple en 1939 jaro kooperativon «Len'emaljer» vizitis delegitoj de juvelira kooperativaro el Krasnoje-na-Volge (ru) ( Kostroma provinco ). Post la vizito kooperativoj en Krasnoje-na-Volge sukcese akiris sperton de produktado de emajlitaj varoj per novaj teknologioj .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Dum la Granda Patriota milito (1941-1945), inkluzive Sieĝon de Leningrado , la kooperativo produktis metalajn rangsignojn kaj municiojn . En januaro de 1943 jaro en la Ruĝa armeo kaj floto okazis ŝanĝo de rangsignoj: kolumsignoj anstataŭiĝis per epoletoj. La ŝanĝo postulis produkti grandan kvanton de rangsignaj elementoj. Kaj kooperativo «Len'emaljer» ricevis grandan mendon. La kooperativo devis produkti pli ol 3 milionojn rangsognajn elementojn :;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Tamen en kondiĉoj de sieĝo provizii la produktadon per materialoj estis granda problemo. Por solvi la problemon 22-an de novembro 1942 la buroo de Leningrada urba komitato de Komunista partio aparte decidis malkonservi el la ŝtata rezerva fondo kaj transdoni al la kooperativo stokon da materialoj :;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Ankaŭ estis decidite desponigi ĉiudiurne elektroenergion: por produktejo №1 (str. Mitninskaja, domo 12) - 70 kilovathoroj kaj por Administrejo (str. Konnaja, domo 2/5) - 40 kilovathoroj.;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Tuj post fino de la Milito kooperativo «Len'emaljer» aktive lanĉis produktadon de pacecaj varoj. 22-an de majo 1945 «Len'emaljer» faris kontrakton por produkti cigaredujojn, pudrujojn, broĉoj laŭ skizoj de la ŝtata mendanto. Jam 25-an de julio 1945 la kooperativo prezentis cigaredujon kun stampita monumento de Petro la Granda .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
En la sama jaro la kooperativo faris kontrakton por produkti varojn el aminoplasto (ru) - kio bezonis akiri sperton de nova teknologio. La kontrakto detale priskribis dizajnon de broĉo «Sep elefantoj» . Samtempe la kooperativo lanĉis produktadon de tradiciaj metalaj varoj: cigaredujo kun vidindaĵoj de Leningrado, pudrujo kun gravuraĵo laŭ motivoj de popolaj fabeloj, porvirina garnituro «Kameo» (broĉo, braceleto, orelornamaĵo), kolumbutono kaj kvar specoj de manumbutonoj .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Dum la postmilitaj jaroj kooperativo «Len'emaljer» sukcesis novan teknologion «Malvarma emajlo», kiun uzos por pli altigi kvanton de produktitaj varoj. La teknologion «Len'emaljer» evoluigos ĝis fino de sia funkciado. Ĝenerale teknika kaj teknologia parto de produktado okupis grandan parton en plua evoluo. Tamen, ankaŭ arteca parto de varoj estas grava kaj laŭ tiama modo ili dekoraciitaj per vidindaĵoj de Leningrado. Samtempe sur la varoj aperas modernaj sovetiaj bildoj .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Sortimento de varoj konsistis ne nur el varoj kun vidindaĵoj: krom pudrojoj, cigaredujoj kaj simalaj la kooperativo produktis vestajn prembutonojn, virajn ŝtrumpetoŝelkojn kaj tiel plu .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
10-an de aŭgusto 1956 per rezolucio de Ministraro de RSFSR № 554 la kooperativo estis reorganizita al uzino «Len'emaljer». La uzino produktis galanterion, gasajn fajrilojn, gasajn kartoĉojn por la fajriloj, diversajn insignojn (inter kiaj pormilitistaj), glasingojn kaj tiel plu. Inter la produktaĵoj estis kaj ĉiutagecaj aĵoj kaj artobjektoj . Dokumentoj atestas, ke al la nova uzino estis dungitaj 769 personoj. Inter ili estis 660 laborantoj . En 1959 jaro, post fino de arta lernejo, en la uzino komencis sian artan biografion fama skulptisto-medalisto kaj pedagogo Nikolaj Aleksndroviĉ Nosov .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Ĉefa parto de produktado de la uzino estis matala galanterio, kaj inter la varoj gravan lokon okupis platetoj en formo de telero. En 1958 jaro la uzino produktis pli ol dudek miloj da la teleroj, en 1960 - 24 003. Kaj ĝuste la platetoj estas ekzemplo de malalta arta gusto de ties varoj. En 1961 jaro Tiĥomirova M. A. atakis la platetojn de uzino «Len'emaljer»:;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
—Тихомирова М. А., Улучшить художественное качество бытовых вещей// Художник-оформитель. Сборник статей. Ленинград: Художник РСФСР. 1962. С. 43-47.;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
La kritiko montris arkaikecon de artaj modeloj, kiujn uzis la uzino kaj provo anstataŭigi altekvalitajn materialojn per malmultekostaj. «Len'emaljer» ne preteriris la kritikon kaj ŝanĝis aliron al dekoraciado de varoj .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Fino de 1960-aj - komenco de 1970-aj jaroj estis periodo de aktiva kresko de la uzino. Dum la jaroj estis konstruitaj novaj produktejoj kio permesis vastigi sortimenton kaj pligrandigi dimension de produktataj varoj. Por konstruado de novaj produktejoj estis uzitaj novaj teknologioj de konstruado . Pro la kresko uzino «Len'emaljer» ĝis 1977 jaro sukcesis duobligi produktadon de varoj .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Dum 1974-1980 jaroj uzino «Len'emaljer» estis parto de Leningrada scienc-produkta unuiĝo «Lenvarmoaparato» ( ruse Ленинградское научно-производственное объединение «Лентеплоприбор» ). La scienc-produkta unuiĝo estis parto de Ĉefa departamento «Glavjuvelirtorg» ( ruse Главное управление «Главювелирторг» de Minesterio pri aparatproduktado, iloj por aŭtomatigi kaj administri de USSR ( ruse Министерство приборостроения, средств автоматизации и систем управления СССР ) .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Ek de 28-an de majo 1980 la scienc-produkta unuiĝo «Lenvarmoaparato» (kaj uzino «Len'emaljer») eniris en scienc-produkta unuiĝo «Burevestnik» ( Буревестник (научно-производственное предприятие) (ru) ). Ĉefa produkto de la scienc-produkta unuiĝo estis rentgenografia aparataro por medicino kaj industrio . Malgraŭ aliĝo al la unuiĝo ĉefa produktaĵo restis distingita al amasaj uzantoj:fajriloj kaj gasboteloj por ili, porvira kaj porvirina galanterio, cigaredujoj kaj insignoj, suveniroj kaj broĉoj. Sed sortimento plivastiĝis per elektroteknika aparataro: aparatoj por kontroli teknologiajn procezojn, elektromezuriloj, optika aparataro por civitana kaj militista uzo, sciencesplora ilaro .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
En 1982 jaro uzino «Len'emaljer» produktis pli ol 2/3 gasaj fajriloj en Soveta Unio, preskaŭ 50% de ungilaraj kompletoj en USSR .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Ek de 25-an de januaro 1991 uzino «Len'emaljer» estis eligita el scienc-produkta unuiĝo «Burevestnik» kaj iĝis ŝtata interpreno «Uzino «Len'emaljer» .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Dum 1990-aj jaroj en Rusio ŝtataj interprenoj amase iĝis privataj. Laŭ Plano de malŝtatigo kaj per decido de registra administrejo de Sankt-Peterburgo № 3194 ŝtata interpreno uzino «Len'emaljer» estis reorganizita al akcia kompanio ( ruse Акционерное общество открытого типа ) «Len'emaljer». 14-an de junio 1996 pro decido de akciuloj la kompanio estis reorganizita al nova formo de interpreno kaj alinomigis ĝin al Malfermita akcia kompanio ruse Открытое акционерное общество «Ленэмальер» «Len'emaljer». Dum postsovetaj jaroj uzino «Len'emaljer» produktis galanterion kaj suvenirajn varojn .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
En 2001 jaro ruiniĝita domo en kia estis deponitaj arkivoj de la interpreno ek de 1929 ĝis 1970 jaroj kio kaŭzis perdon de la arkivoj .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
31-an de oktobro 2008 la kunsido de akciuloj de la interpreno decidis ĉasigi funkciadon de kompanio «Len'emaljer» .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Arta stilo de la produktaĵo de la kooperativo kaj poste uzino «Len'emaljer» aparteniĝas al la «Leningrada stilo». Specifeco de la artaj artifikoj, temoj, dekoracioj, ornamoj, koloraj gamoj estis celita al amasa produktado kaj neneigis rolon de majstro. Arta specifeco de produktoj de «Len'emaljer» estas interkruciĝo de la alta aristokrateca stilo de antaŭrevoluciaj majstroj, la kamparana etnografieco kaj la amasa urba dizajno. La amaseco de produktado diktis simplajn dizajnajn solvojn kaj samtempe montras fortajn ligojn kun lokaj tradicioj, evoluantaj dum vivo de lando sinkrone al amasaj ŝatoj kaj gustoj .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Tamen «Len'emaljer» estis infano de sia tempo kaj ensorbis sovetiajn produktajn traduciojn. Ĝi dependis de mendoj de superaj institucioj kaj ne penis anticipi evoluon de amasa mendo .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
En postUSSR-a Rusio varoj produktitaj de «Len'emaljer» iĝis temo de kolektado kaj ofte eniras ekspoziciojn de diverstemaj muzeoj .;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Dum tuta sia vivo kooperativo kaj poste uzino «Len'emaljer» estis evoluinta entrepreno, kies porproduktiga spaco pligrandiĝis de jaro al jaro kaj la produktejoj situis en diversaj partoj de Leningrado (ek de 1991 jaro renomiĝis Sankt-Peterburgo).;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
La unua adreso de kooperativo «Len'emaljer» estis str. nome Gercen , domo 24 - la eksa sidejo de mondfama juvela entrepreno de Peter Karl Faberĵe . La sidejo de kooperativo funkciis dum la unua periodo, dum kia la kooperativanoj laboris en propraj hejmoj.;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
Post oficiiĝo la kooperativo kreskis, aĉetis porproduktadan ekipaĵon kaj aparataron kaj jam en 1930 jaro kooperativo «Len'emaljer» konsistis el tri fakoj :;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
En 1956 jaro, kiam la kooperativo transformiĝis al uzino, ĝi estis sufiĉe granda entrepreno, inkluzivinta multajn produktejojn :;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
En 1980 jaro uzino «Len'emaljer» atingis la pinton de sia evoluo kaj ĝiaj produktejoj kaj fakoj situis ĉe diversaj adresoj de Leningrado :;joyas;855329;Len'emaljer;9455291;https://eo.wikipedia.org/w/index.php?oldid=9455291
La Monda Ora Konsilio ( angle World Gold Council ) estas tutmonda lobiada organizaĵo por la orminada industrio . Ĝia celo estas stimuli kaj subteni la postulon je oro . La organizaĵo havas sian ĉefsidejon en Londono kaj oficejojn en Barato , Ĉinio , Singapuro , UAE kaj Usono . La organizaĵo estis fondita en 1987 de orminadaj kompanioj.;joyas;936522;Monda Ora Konsilio;9313441;https://eo.wikipedia.org/w/index.php?oldid=9313441
Laŭ ĝia ĉarto, ĉiu kompanio kiuj produktas almenaŭ 100 000 trojuncojn da oro ĉiukvaronjare povas fariĝi membro.;joyas;936522;Monda Ora Konsilio;9313441;https://eo.wikipedia.org/w/index.php?oldid=9313441
Pro la kutimo montri kronitan reĝkapon sur moneroj, la valuta ĉefunuo de pluraj landoj nomiĝas krono .;ropa;6305;Krono (kapoornamo);9403293;https://eo.wikipedia.org/w/index.php?oldid=9403293
Felo estas detirita haŭto de hararhava mamulo , eventuale de birdo kun plumaro (aŭ nur kun lanugo ). Ĝi estas uzata en peltindustrio por produktado de pelt kolumoj , pelt manteloj kaj aliaj produktoj. Per senharigo kaj tanado estiĝas ledo .;ropa;8321;Felo;9343136;https://eo.wikipedia.org/w/index.php?oldid=9343136
Krude detirita felo ( krudaĵo ) venas en komercon kiel aĉet-sortimenta felo , kiu ankoraŭ ne estas enspecigita inter felojn tanistajn , peltistajn aŭ felt-produktajn . Tanindustrio uzas senharigaĵon , ellesivitan felon, senigitan je haroj kaj korniĝintaj partoj de la epidermo .;ropa;8321;Felo;9343136;https://eo.wikipedia.org/w/index.php?oldid=9343136
La industrio utiligas felojn el ĉiuj korpopartoj de bestoj (kapo, muzelo, lipo, vango, frunto, nuko, ĉenuko, gorĝo, lobo, dorso, ventro, ingveno, ŝultro, kruro, piedo, plando, gropo, eĉ vosto kaj skroto). Felkerno estas kompakta parto de felo el dorso kaj gluteoj de besto. Esprimo felkerno kun flankoj ampleksas la tutan felon sen nuko, dum ezoko male felkernon kun nuko, sed sen flankoj. - Ĉe maljunaj virporkoj estiĝas hipertrofia rigidiĝinta felparto sur la dorso, ŝultroj kaj korpflankoj nomata ŝildfelo . - Ĉevalfela spegulloko estas loko en la ĉevalfela gropo karakterizata per ekstreme kompakta strukturo de la dermo . - Pri felrimeno oni parolas ĉe felparto konsistanta el epidermo kaj dermo, averse regule surkreskita per hararo, eventuale per plumaro.;ropa;8321;Felo;9343136;https://eo.wikipedia.org/w/index.php?oldid=9343136
Laŭ deveno felo povas esti enlanda (de enlande vivanta besto), eksterlanda, importita, ĉas-besta, hejm-besta kaj farm-besta .;ropa;8321;Felo;9343136;https://eo.wikipedia.org/w/index.php?oldid=9343136
Laŭ la kvalito oni distingas felojn: vintra (vintre mortigita besto), transira (mortigo dum sezono de regula hararŝanĝo), somera, kompakta (kun dense traplektitaj fibroj en la dermo), grasoza (kun troa kvanto da graso sur la reverso), karnoza (kun iom granda kvanto da muskolhisto sur la reverso), dik-rimena (kun tro dika felrimeno kompare al aliaj feloj de la sama speco), malkompakt-flanka (kun pli granda areo de maldensaj flankoj), plata (kun malgrandigita diko de felkerno kaj kun pli granda areo de flankoj), fetfelo (el ido frunaskita, abortita, mortnaskita aŭ elprenita el korpo de morta patrino), falfelo (el mortinta besto, en la dermo kaj hipodermo ĝi regule havas koaguliĝintan sangon), nemezdistranĉita (detirita el la besto sen laŭlonga tranĉo), mezdistranĉita (mezdistranĉita per ventra aŭ dorsa tranĉo antaŭ la detiro aŭ post la detiro), nesenostigita (dum la detiro lasitaj ostoj en la kruroj aŭ vosto), konfiskita (akirita en veterinara asanacia instituto), senvaloraĵo (kiu ne konvenas al postuloj ordonitaj en la normo, oni ne povas ĝin enlasi en kiun ajn kvalitklason).;ropa;8321;Felo;9343136;https://eo.wikipedia.org/w/index.php?oldid=9343136
Verajn felojn oni akiras el ĉinĉilo (kies felo 30-oble pli molas ol homa hararo), ornitorinĥo , didelfo , dasiuro , triĥosuro , talpo , ermeno , neĝmustelo , kolonoko , lutreolo , vizono , putoro , furo , marteso , foino , zibelo , mustelo , gulo , melo , taksideo , melogalo , mefito , lutro , mar-lutro , lav-urso , urso , vivero , hundo , lupo , vulpo , izatiso , korsako , mustel-hundo , kato , linko , oceloto , leopardo , jaguaro , tigro , uncio , gepardo , mar-urso , foko , sciuro , marmoto , burunduko , kastoro , hamstro , ondatro , rato , kobajo , ĉevalo , azeno , mulo , hino , leporo , kuniklo , porko , apro , pekario , boaco , cervo , damao , ĉamo , kapreolo , kapro , kaprido , muflono , ŝafo , ŝafido , bubalo , gruntbovo , bovo , molanaso , grebo , mevo , cigno , ansero , nikteo .;ropa;8321;Felo;9343136;https://eo.wikipedia.org/w/index.php?oldid=9343136
Froko estas larĝa longa kapuĉa survesto kun larĝaj maniko . Pliofte ĝi estas ĝispiede longa. Ĝi estis moda en frumezepoka Eŭropo kaj iĝis poste tipa monaĥa robo .;ropa;17419;Froko;9091492;https://eo.wikipedia.org/w/index.php?oldid=9091492
Kvasto estas aro de pendantaj mallongaj fadenoj en formo de tufo aŭ peniko fine de ŝnuro . Kvastoj celas ornami vestojn aŭ meblojn , kusenojn aŭ kurtenojn . Supre ili kutime limiĝas de arte plektita nodo .;ropa;17606;Kvasto;9043864;https://eo.wikipedia.org/w/index.php?oldid=9043864
"Ĉe bestoj oni ankaŭ nomas la haran finon de vosto ""kvasto"". La samo validas por la vostofinoj de fantaziaj bestoj kaj mitologiaj kreaĵoj kiel ekzemple drakoj aŭ diabloj .";ropa;17606;Kvasto;9043864;https://eo.wikipedia.org/w/index.php?oldid=9043864
Kitono aŭ ĥitono (malnovgreke χίτων chiton ) estas helena tuniko (subvesto) ligita per agrafo . Ĝi estis portata de liberuloj tiel, ke ĝi tuŝis ambaŭ ŝultron , ĉe la sklavoj la dekstra ŝultro restis nekovrita.;ropa;17608;Kitono;9074817;https://eo.wikipedia.org/w/index.php?oldid=9074817
Laŭ la greka verkisto Palaifato , la amazonoj estis viroj kiuj surhavis kitonon tiel, kiel trakiaj virinoj.;ropa;17608;Kitono;9074817;https://eo.wikipedia.org/w/index.php?oldid=9074817
Laĉo estas ŝnureto el kotono , silko , ledo ktp., kiun oni trapasigas tra specialaj truetoj por streĉi kaj kunligi la du partojn de la vestoj, korseto , ŝuoj .;ropa;17618;Laĉo;9086343;https://eo.wikipedia.org/w/index.php?oldid=9086343
Laĉoj krome havas ornaman funkcion , aparte en korsetoj de virina tradicia vesto . La la vasta uzado de laĉoŝuoj, ŝuo fiksita per laĉo, finiĝis kun la inventado de la lapfermilo .;ropa;17618;Laĉo;9086343;https://eo.wikipedia.org/w/index.php?oldid=9086343
Supozeble ili estis konataj kaj uzataj jam en la ŝtonepoko, sed pro la delikateco de la materialo, ni nur tre rare vidos arkeologian laĉon.;ropa;17618;Laĉo;9086343;https://eo.wikipedia.org/w/index.php?oldid=9086343
Basko estas ornamenta pendanta parto de striktaj vestoj , kiu libere pendas sub la talio . Oni povas trovi baskojn en ambaŭ vira kaj virinaj vestoj. Ekzemploj, oni povas trovi baskojn ĉe smokingo aŭ formala longa robo .;ropa;17619;Basko;9348350;https://eo.wikipedia.org/w/index.php?oldid=9348350
Kitelo estas vasta manikhava vira vesto el vulgara ŝtofo, kovrante de la ŝultro ĝis la genuoj (flegista, buĉista, vendista ktp. kitelo);ropa;17622;Kitelo;9087159;https://eo.wikipedia.org/w/index.php?oldid=9087159
La friga ĉapo estis unue abunde uzata dum la Franca Revolucio de 1789 . Estis periodo kiam ĉiuj devis nepre havi tion surkape, tiuj, kiuj ne faris tiel, estis suspektataj pro malbona revoluciismo. Poste ĝi estis uzata de ĉiuj respublikaj reĝimoj de Francio . La virino, kiu simbolas la francan respublikon , t.e. Marianne , surhavas frigan ĉapon.;ropa;18484;Friga ĉapo;8183419;https://eo.wikipedia.org/w/index.php?oldid=8183419
Poste, la friga ĉapo estis ankaŭ uzata sur la naciaj blazonoj de Argentino , Bolivio , Haitio , Kubo , Kolombio , Nikaragvo kaj Salvadoro . Ankaŭ ĝi troviĝas sur la blazonoj de la usona subŝtato Nov-Ĵerzejo , de la brazila subŝtato Sankta Katarino , kaj de la argentinaj provincoj Bonaero , Katamarkio , Corrientes , Jujuy , Mendoza , San-Juan kaj Tukumano , Jujuy kaj Mendoza havas ĝin ankaŭ sur la flagoj.;ropa;18484;Friga ĉapo;8183419;https://eo.wikipedia.org/w/index.php?oldid=8183419
"Legendoj miksiĝis al veraj historioj pri Lampião en brazila folkloro kaj li iĝis populara heroo. Estas multaj artaĵoj faritaj de popularaj artistoj, skulptaĵoj, pentraĵoj, ktp. pri li. Ankaŭ estas sennombro de popularaj poemoj pri Lampião, eĉ lia edzino estis komparita al Heleno aŭ Roksano en popolmuziko ""Mulher nova bonita e carinhosa faz o homem gemer sem sentir dor"" ( Juna, bela kaj karesa virino iĝis ke viro ĝemu sen doloro ).";ropa;24032;Lampião;9257996;https://eo.wikipedia.org/w/index.php?oldid=9257996
"En grupo de Lampião, estis multaj poetoj, kaj ofte li estis omaĝita per poemoj kaj promovis konkursojn de poezio. Lampião mem ankaŭ estis poeto kaj verkis kanzonon ""Mulé Rendeira"" (Puntistino):";ropa;24032;Lampião;9257996;https://eo.wikipedia.org/w/index.php?oldid=9257996
"""Ho, puntistino Ho, puntvestita virino Vi instruos al mi puntfaradon Kaj mi instruos al vi amidumon """;ropa;24032;Lampião;9257996;https://eo.wikipedia.org/w/index.php?oldid=9257996
"""Lampião malsupreniris montaron Li venis por dancado La knabinoj de Pesqueira Venis por amidumi lin""";ropa;24032;Lampião;9257996;https://eo.wikipedia.org/w/index.php?oldid=9257996
Lampião ankaŭ estis kudristo kaj kudris tre artajn vestaĵojn kaj ĉapelojn, kiel ni povas vidi en fotoj. Krom poezio, muziko kaj kudristarto, la grupo de Lampião kreis la baleton xaxado kiu reprezentas la terlaboron. Aliaj artoj tre ŝatataj de Lampião kaj siaj kunuloj estis fotografion kaj ksilogravuron.;ropa;24032;Lampião;9257996;https://eo.wikipedia.org/w/index.php?oldid=9257996
En Aŭstralio estas leĝo, ke ĉiuj biciklantoj devas porti kaskon. Sed tia regulo ne ekzistas en multaj aliaj landoj.;ropa;30319;Kasko;9184750;https://eo.wikipedia.org/w/index.php?oldid=9184750
La kaskoj pretiĝis origine el ledo , kupro , bronzo , fero aŭ ŝtalo , sed ankaŭ el forigitaj teksaĵoj aŭ eĉ oro . Hodiaŭ la kaskoj produktiĝas el rezistokapabla plastomaterialo , ekz. kevlaro .;ropa;30319;Kasko;9184750;https://eo.wikipedia.org/w/index.php?oldid=9184750
Ganto (aŭ manvesto ) estas vestaĵo , kies ĉefa funkcio estas la protekto de la mano de la portanto.;ropa;31211;Ganto;9031210;https://eo.wikipedia.org/w/index.php?oldid=9031210
Plua diferenco ekzistas ĉe la formo: pugnaj gantoj (aŭ dufingraj gantoj) tenas 4 fingrojn kune kaj la dikfingron aparte, la fingrogantoj tenas ĉiun fingron aparte, la biciklaj gantoj (duonganto) tenas la fingojn apartaj, sed la granda parto de fingroj restas libere.;ropa;31211;Ganto;9031210;https://eo.wikipedia.org/w/index.php?oldid=9031210
La trikitaj aŭ subŝtofitaj gantoj tenas varme la manojn de la portanto en malvarmo. Origine oni pretigis varmigajn gantojn el ledo, kiujn oni subŝtofis per lano, nuntempe oni produktas la gantojn tute el sintezaj fibroj. La pugnogantoj (dufingraj gantoj) havas pli bonan efikon kontraŭ malvarmego, ĉar ili havas pli malgrandan surfacon ol fingrogantoj.;ropa;31211;Ganto;9031210;https://eo.wikipedia.org/w/index.php?oldid=9031210
Tiu ĉi grupo de la gantoj devas formi netrapenetreblan obstaklon por kemiaĵoj (ekz. acidoj kaj lesivoj ) kaj mikroorganismoj, sed ili devas doni la eble plej grandan movkapablon kaj ankaŭ (sur kirurgia kampo) la maksimuman palposenton. Ili konsistas pro tio plej ofte el eble plej maldika gumo (sinteza kaŭĉuko) aŭ simila materialo (ekz. latekso ).;ropa;31211;Ganto;9031210;https://eo.wikipedia.org/w/index.php?oldid=9031210
En kelkaj sporto-specioj ekz. kiel ĉe golfludo aŭ teniso oni portas gantojn por la pli bona preno. Ĉe aliaj sportoj (ekz. rajdado) gravas la mekanika protekto, ĉe aliaj la ( skisporto ) la termoprotekto. La boksa ganto obtuzigas la batojn. La golulo havas kaptajn gantojn. Ĉe skermado oni uzas skermogantojn, sabrogantojn.;ropa;31211;Ganto;9031210;https://eo.wikipedia.org/w/index.php?oldid=9031210
Ĝis komenco de la 18-a jarcento ofte aperis sur ilustraĵoj la gantoj, sed ĝis ĉ. 1800 nur apenaŭ. Poste, kun apero de kurt-manika modo disvastiĝas la longmanikaj gantoj, kiuj kovras eĉ ĝis la supra parto de la brakoj de damoj. Tiuj gravis ĝis mezo de la 20-a jc. kaj estis preskaŭ devigaj ĉe baloj. Por la viroj la gantoj gravis ekde 1700 kiel termo-, kaj mekanikaj protektiloj. Esceptoj estis la profesiaj uniformoj (fiakristo, ŝoforo, ĉambristo (angle butler) ktp.).;ropa;31211;Ganto;9031210;https://eo.wikipedia.org/w/index.php?oldid=9031210
"Socidece en la altaj klasoj, kaj poste al ĉiuj klasoj pro ""alburĝeco"" de la deconormoj, kaj ankaŭ ĉe servistoj en altrangaj domoj, surhavo de ganto iĝis dumtempe (pr. 1850-1950) deviga, kaj detalaj ĝentilo-normoj ekzistis pri kiam oni devas surmeti ilin aŭ devas demeti ilin laŭ la cirkonstancoj.";ropa;31211;Ganto;9031210;https://eo.wikipedia.org/w/index.php?oldid=9031210
Basbala ĉapo estas tipo de mola ĉapo kun longa, rigidita, kurba viziero . Ĝi estas uzata de ĉiu jaraĝo, sed ĉefe ĉe junuloj, sportistoj , sportoŝatantoj. La antaŭa parto estas el teksaĵo , la malantaŭa parto estas ofte el truriĉa plastaĵo kun adapta ligilo.;ropa;37775;Basbala ĉapo;7946406;https://eo.wikipedia.org/w/index.php?oldid=7946406
Biciklista ŝorto aŭ haŭtŝorto estas tre strikta ŝorto , ofte preskaŭ ĝisgenua, el elastana ŝtofo. La ŝortoj por viroj kaj virinoj estas forte malsimilaj.;ropa;40266;Biciklista ŝorto;7950161;https://eo.wikipedia.org/w/index.php?oldid=7950161
Kalsono estas vestaĵo , kiun ambaŭ seksoj portas kvazaŭ pantalonon de la subvesto . Ĝi kovras la malsupran parton de la torso ekde la talio aŭ koksoj , kaj foje etendiĝas plu ĝis la femuroj , genuoj aŭ eĉ maleoloj .;ropa;40269;Kalsono;9201207;https://eo.wikipedia.org/w/index.php?oldid=9201207
Virinaj kalsonetoj estas nuntempe tre ofte dekoksaj : ili komenciĝas ĉe la koksoj, ne ĉe la talio. Se dekoksa kalsoneto estas tajlita tiel, ke ĝi arkiĝas alte ĉe la flankoj, oni nomas ĝin bikineca , ĉar ĝi similas al la malsupra parto de la originaj bikinoj .;ropa;40269;Kalsono;9201207;https://eo.wikipedia.org/w/index.php?oldid=9201207
Subŝorto estas malstrikta ŝorto uzata kiel vira kalsono. Ĝi havas elastan zonon, ofte plisitan, kaj ĝenerale antaŭklapon, foje butoneblan, foje ne.;ropa;40269;Kalsono;9201207;https://eo.wikipedia.org/w/index.php?oldid=9201207
La subŝorto unue aperis en la 1930-aj jaroj, sed la samtempe inventita vira kalsoneto fariĝis pli populara. Subŝortoj komencis bone vendiĝi nur post la Dua mondmilito .;ropa;40269;Kalsono;9201207;https://eo.wikipedia.org/w/index.php?oldid=9201207
Kuloto estas historia speco de pantalono , ĝenerale malstrikta super la genuoj , sed striktigita tuj sub la genuoj, kie ĝi efektive finiĝas. Nuntempe oni portas ĝin nur en kelkaj tre formalaj ceremonioj.;ropa;40271;Kuloto;9201209;https://eo.wikipedia.org/w/index.php?oldid=9201209
Oni nomas kuloton ankaŭ tiun tute malstriktan modernan ŝorton , kiu etendiĝas ĝis preskaŭ la genuoj.;ropa;40271;Kuloto;9201209;https://eo.wikipedia.org/w/index.php?oldid=9201209
Bikinoj estas nenio nova. Oni vidas atletinojn en senŝelkaj bikinoj eĉ en pentraĵoj sur helenaj urnoj de 1400 a.K.;ropa;50325;Bikino;9415622;https://eo.wikipedia.org/w/index.php?oldid=9415622
"Tamen, la modernan bikinon inventis du francoj, Jacques Heim kaj Louis Réard , en 1946. Heim, modisto el la urbo Cannes , fasonis tre malgrandan bankostumon, kiun li nomis ""Atome"" ( atomo ), laŭ la plej malgranda materiero. Li reklamis ĝin per fumaviadilo, kiu skribis sur la ĉielo anoncon pri ""la plej eta bankostumo en la mondo"".";ropa;50325;Bikino;9415622;https://eo.wikipedia.org/w/index.php?oldid=9415622
La bikino de Réard estis tiel malgranda, ke li ne povis trovi modelinon pretan porti ĝin. Li fine dungis tiucele Micheline Bernardini (Q153182) , nuddancistinon el la Pariza Kazino (Q1187828) .;ropa;50325;Bikino;9415622;https://eo.wikipedia.org/w/index.php?oldid=9415622
Kelkaj fabrikistoj de virinaj subvestaĵoj uzas la vorton bikino ankaŭ por nomi (ŝajne arbitre) diversajn specojn de malgrandaj kalsonetoj . La tipan subveston de moderna virino — mamzonon kaj kalsoneton — oni ja povas tamen nomi subbikino .;ropa;50325;Bikino;9415622;https://eo.wikipedia.org/w/index.php?oldid=9415622
Sportaj bikinoj gajnis popularecon jam de la 1990-aj jaroj en libertempa kaj ripoza sportado. Tamen naĝistinoj ne uzas bikinojn ofte en naĝaj konkuroj. La Internacia Federacio de Naĝado (FINA) voĉdonis malpermesi naĝistinojn konkuri en bikinoj en oficialaj internaciaj konkursoj je sia kunveno de la somera olimpiko 1960 en Romo .;ropa;50325;Bikino;9415622;https://eo.wikipedia.org/w/index.php?oldid=9415622
La supra parto de bikino ankaŭ nomiĝas mamzono, sed la socia diferenco estas ke ĝi estas parto de banvesto kaj ne de subvesto .;ropa;51522;Mamzono;9456477;https://eo.wikipedia.org/w/index.php?oldid=9456477
Jako estas ekstera vesto en multaj kulturoj, malfermebla ĉe la antaŭa parto, manikhava , kovranta la korpon ekde la ŝultroj ĝis proksimume la koksoj. La malfermejo povas pendi nefermite aŭ oni povas fermi ĝin diversmetode: per butonoj , hokoj aŭ ruband bentoj . Se tia vestaĵo estas klare pli longa ol ĝis la koksoj, oni prefere parolas pri mantelo aŭ palto .;ropa;51524;Jako;9087921;https://eo.wikipedia.org/w/index.php?oldid=9087921
Jupo estas vesto , kutime porvirina , kovranta la malsupran parton de la korpo, ekde la zono . Malsimile de la pantalonoj , la jupoj ne fermas la interkruran spacon .;ropa;51525;Jupo;9088601;https://eo.wikipedia.org/w/index.php?oldid=9088601
Laŭkulture, la jupo estas portita de viroj ( kilto , sapeta , sarongo ...) aŭ virinoj. Post longa malaperado de la jupo ĉe la viroj de okcidenteŭropaj landoj (escepte de la kilto de la skotoj ), kelkaj viroj reekis la kutimon uzi porvirajn jupojn. Estas interese rimarki, ke virinoj tutrapide ekis la uzadon de pantalonoj. Oni povas teoriumi plurajn kaŭzojn por la rapida enpantalonigado de virinoj - Unue, virinoj post feminisma periodo deziris akiri pli profesian aspekton portantaj pantalonojn (do ne la tradicia bildo de virino je jupoj, ligita al antaŭjuĝoj kiel malforteco , ktp). Male, uzi jupojn ĉe viroj povus doni la impreson de malforta, malkapabla viro. Due, homoj je jupoj havas siajn genitalojn pli facile atingeblaj, dum homoj je pantalonoj havas genitalojn kaŝitajn, tio estas, protektitaj de pantalonoj. Laŭ konservativa rolo pri viroj kaj virinoj, la viro montras la interesiĝon pri sekso kaj la virino akceptas aŭ rifuzas - tio estas tute kongrua kun viroj je pantalonoj nur, kaj virinoj aŭ je pantalonoj aŭ je jupoj. Kompreneble ĉi lasta hipotezo estas laŭ psikologia vidpunkto , do subjektiva.;ropa;51525;Jupo;9088601;https://eo.wikipedia.org/w/index.php?oldid=9088601
Laŭ Francisko Azorín masko estas Artefarita vizaĝo kun pli malpli karikaturaj trajtoj. Oni uzas ĝin diverscele, ĉar la antikvaj aktoroj uzis ĝin ĉe teatraj sceno, ĝi estas alegorio de la teatra arto. Li indikas etimologion el la araba maskara (maskulo, bufono). Kaj li aldonas la terminon maski , por kaŝi per masko.;ropa;52483;Masko;9032157;https://eo.wikipedia.org/w/index.php?oldid=9032157
"En oktobro 2019 la ĉefadministrantino de Hongkongo , Carrie Lam anoncis malpermeson kontraŭ maskoj, aplikota al homoj ĉeestontaj ""kontraŭleĝajn"" kunvenojn, kiuj uzas perforton, kaj ĝi ekskluzivas tiujn kiuj maskas sin pro ""legitima bezono"". Tio okazis kadre de politikaj protestoj en la urbo.";ropa;52483;Masko;9032157;https://eo.wikipedia.org/w/index.php?oldid=9032157
Hiĝabo (alilingve konata sub nomoj hejab , hijaab , hijāb , arabe حِجَاب , IFA : aŭ , translit. DMG : ḥiǧāb ) estas specifa formo de « kaptuko », pli precize, kaptuko kiu kovras la hararon kaj la kolon de islamana virino, dum restas senkaŝe la tuta vizaĝo, diference de la burko kaj la nikabo , kiuj kaŝas respektive la tuton aŭ suban parton de la vizaĝo kaj diference de ŝajlaoj , kiuj estas similaj al hiĝaboj, sed kovras ankaŭ la ŝultrojn.;ropa;80188;Hiĝabo;9485213;https://eo.wikipedia.org/w/index.php?oldid=9485213
Hiĝabo devenas de la araba radiko haĝaba , «ŝteli de la rigardo, kaŝi», kaj signifas «vualon metitan antaŭ persono aŭ objekto por kaŝi aŭ izoli ĝin». Do en la araba lingvo la vorto signifas ankaŭ «kurtenon» aŭ «ekranon»;ropa;80188;Hiĝabo;9485213;https://eo.wikipedia.org/w/index.php?oldid=9485213
Hiĝabo estas grava laŭ la kunteksto socikultura en kiu ĝi aperas, sed ne estas fundamenta principo de islamo . La vorto «hiĝabo» troviĝas sepfoje en la Korano . Neniam temas pri la virina vestaĵo, por kiu aliaj terminoj estas uzataj.;ropa;80188;Hiĝabo;9485213;https://eo.wikipedia.org/w/index.php?oldid=9485213
"La vorto hiĝab havas la signifon de ""kurteno kiu kreas privatecon"" por indiki la izolaĵon truditan al la edzinoj de la profeto Mohamedo , suraoj 33, 53:";ropa;80188;Hiĝabo;9485213;https://eo.wikipedia.org/w/index.php?oldid=9485213
Hiĝabo tamen evoluis kies signifo estas nun plejofte uzata por indiki la ideon de kaptuko de islamana virino .;ropa;80188;Hiĝabo;9485213;https://eo.wikipedia.org/w/index.php?oldid=9485213
En la islama respubliko uzado de hiĝabo estas deviga por la virinoj kaj ĝia manko estas punata laŭleĝe. La morto de Mahsa Amini, arestita de polico pro malhavo de hiĝabo, kaj laŭdire batita en policejo, kaŭzis amasajn protestojn en 2022, severe subpremitajn .;ropa;80188;Hiĝabo;9485213;https://eo.wikipedia.org/w/index.php?oldid=9485213
En aŭgusto de 2019 modelo en hiĝabo aperis en reklamo de granda rusia vestomarko — unuafoje en historio de la lando.;ropa;80188;Hiĝabo;9485213;https://eo.wikipedia.org/w/index.php?oldid=9485213
Kapuĉo estas kapkovrilo fiksita al vestaĵo kiel ekzemple mantelo , anorako aŭ pluvjako , kiu celas protekton kontraŭ malvarmo , vento kaj pluvo . Ĝi povas havi bendojn aŭ ŝnureton por alĝustigi ĝin ĉirkaŭ la kapo . Ekde la fino de la 20-a jarcento ankaŭ puloveroj aŭ sveteroj kelkfoje havas kapuĉojn.;ropa;126684;Kapuĉo;9448446;https://eo.wikipedia.org/w/index.php?oldid=9448446
Anorako estas jako kun kapuĉo por protekti sin kontraŭ malbona malvarma vetero. Se ĝi protektas nur kontraŭ ventan veteron, ne kontraŭ malvarmon, tiam oni nomas ĝin ventojako. Antaŭe la anorako havas nur malfermaĵon por la kapo , kiun oni povas fermi per butonoj aŭ zipo . La suba parto de la anorako, kontraŭe al aliaj jakoj, ne malfermeblas. Nuntempe oni tamen ankaŭ uzas la vorton anorako por kapuĉojakoj tute malfermeblaj per zipo.;ropa;126685;Anorako;9046835;https://eo.wikipedia.org/w/index.php?oldid=9046835
La vorto anorako devenas de inuita lingvo , la gronlanda lingvo . En tiu lingvo annoraaq signifas io kontraŭ la vento . La unuaj anorakoj estis faritaj el fokofeloj .;ropa;126685;Anorako;9046835;https://eo.wikipedia.org/w/index.php?oldid=9046835
Kilto estas el lano teksita , malantaŭe faldita vind - jupo kiun ĉefe viroj portas en Skotio . La longo estas tia, ke la kilto tuŝas la genuojn , sed la portanto genuiĝanta, la kilto ne devas tuŝi la plankon .;ropa;126709;Kilto;9485019;https://eo.wikipedia.org/w/index.php?oldid=9485019
Pri certeco de la plejdo kiel vestaĵo oni certas nur ekde la 16-a jarcento . Ĝi kreiĝis, kiel indikas la nomo, el kovrilo, kiujn komence de la 17-a jarcento, skotoj portis sur la ŝultro super siaj kiteloj, nomitaj cotte . Iom poste oni trovas la vorton por nomi la ŝtofon el kiu estis farita la vestaĵoj de la skota armeo dum la popolmilito de 1693 : ŝtrumpoj , pantalonoj kaj kamizoloj . El tekstoj el tiu epoko tamen evidentiĝas ke ili estis unue unukolora kaj ne kvadratitaj.;ropa;126709;Kilto;9485019;https://eo.wikipedia.org/w/index.php?oldid=9485019
"En iuj cirkonstancoj, subvestoj povas esti necesaj. Ekzemple kiam oni dancas aŭ sportas, kiel dum la ""Highland Games"" ( Altlandaj Ludoj ). Estas ofte malfacile al iu, kiu ne kutimas la porton de kilto, konservi sian dececon, kiam li portas ĝin ""regimente"", ĉefe dum ventego. Unu el la unuaj kilto- fabrikistoj , kaj ankaŭ la plej malnova perpoŝtvendejo de Skotlando , proponas subvestojn specifajn por kiltoj, sed la plejmultaj kiltoportantoj preferas uzi tradiciajn kalsonojn aŭ subvestojn.";ropa;126709;Kilto;9485019;https://eo.wikipedia.org/w/index.php?oldid=9485019
Bankostumo estas vestaĵo , kiun oni portas dum banado en la naturo ( maro , rivero , lago ) aŭ en publikaj lokoj ( naĝejo ).;ropa;129669;Bankostumo;9395677;https://eo.wikipedia.org/w/index.php?oldid=9395677
Ĝi kovras preskaŭ la tutan trunkon kaj havas malfermaĵojn por la kolo , brakoj kaj kruroj . Duera virina bankostumo nomiĝas bikino . Viroj plejofte portas bankalsonojn aŭ banŝortojn .;ropa;129669;Bankostumo;9395677;https://eo.wikipedia.org/w/index.php?oldid=9395677
"Ŝajnas, ke la franca vorto cravate devenas de la kroata . La kroataj militistoj dungitaj de Ludoviko la 13-a en la 17-a jarcento kutimis porti koltukojn noditajn ĉirkaŭ la kolo. Fakte oni trovas menciojn de la vorto cravate jam el la 14-a jarcento , kaj en Francio kaj en Italio , ĉefe en la verkoj de Eustache Deschamps . ""Kroata"" en la kroata lingvo skribiĝas Hrvat kaj ties prononco similas multe al la franca cravate . Do eble la vesto de tiuj soldatoj modiĝis tiam denove la kravaton.";ropa;129701;Kravato;9043142;https://eo.wikipedia.org/w/index.php?oldid=9043142
Burko ( ruse бу́рkа búrka, kartvele ნაბადი nabadi) estas felta sonorilforma mantelo ĉe multaj etnoj de Kaŭkazio kaj apudaj regionoj. Supozeble de tiuj etnoj la vestaĵon transprenis ankaŭ slavaj etnaj subgrupoj ( kazakoj ).;ropa;130805;Burko (Kaŭkazo);9117387;https://eo.wikipedia.org/w/index.php?oldid=9117387
Burko ne havas manikojn , ĝi estas vasta, malpeza kaj varma. Oni facile rulas ĝin kaj transportas ĝin fiksita malantaŭ la selo .;ropa;130805;Burko (Kaŭkazo);9117387;https://eo.wikipedia.org/w/index.php?oldid=9117387
Burko estas vira vestaĵo. Ofte ĝi estas paŝtista vesto (kiel en Turkio ). Oni uzas ĝin interalie kiel portebla matraco , sur kiu eblas dormi.;ropa;130805;Burko (Kaŭkazo);9117387;https://eo.wikipedia.org/w/index.php?oldid=9117387
Cilindra ĉapelo aŭ cilindro estas speco de alta, supre plata, rande larĝa ĉapelo portata far viroj tra la 19-a kaj fru-20-a jarcentoj, nun kutime portata nur kun matena vesto aŭ vespera vesto . La unuan cilindron faris John Hetherington en 1797 , tiaj ĉapeloj fariĝis popularaj dum la 1820-oj . Viroj ordinare portis cilindrojn por komerco (negocoj), sociaj eventoj, kaj plezuro.;ropa;140017;Cilindro (ĉapelo);9155784;https://eo.wikipedia.org/w/index.php?oldid=9155784
"Populara speco, aparte en 19-jarcenta Usono , estis la angle stovepipe hat (""fornotuba ĉapelo""), kiun popularigis Abraham Lincoln dum sia prezidenteco. Malkiel multaj cilindroj, tiu speco estis rekta, kvazaŭ tuba, ne pli larĝa ĉe la supro, nek ĉe la malsupro. Ili ofte pli altis ol la tipa cilindra ĉapelo.";ropa;140017;Cilindro (ĉapelo);9155784;https://eo.wikipedia.org/w/index.php?oldid=9155784
Gasmasko (ankaŭ ŝirma masko ) servas por ŝirmado de vizaĝo , okuloj kaj spirorganoj de portanto ĝenerale antaŭ toksaj materioj. Tiuj povas esti en stato aŭ firma ( polvaj partikloj), fluida ( aerosoloj ) aŭ gasa (diversaj batalgasoj ).;ropa;192347;Gasmasko;9446629;https://eo.wikipedia.org/w/index.php?oldid=9446629
Vangoparto - ĝi estas kutime fabrikita el gumizita teksaĵo, la armeaj masko tipe el forta tavolo de speciala porkemie rezista gumo. Ties funkcio estas ŝirmi la vizaĝon de la portanto antaŭ kontakto kun danĝeraj partikloj kaj samtempe estas en ĝi lokigitaj travidejoj, travideblaj vitraj aŭ plastaj aperturoj servantaj por observi la ĉirkaŭaĵon, ĉe la armeaj maskoj kutime duobligita kiel ŝirmo kontraŭ trabato, ekz. per ŝtono ks. Plu estas en la vangoparto enspira kaj elspira filtrilo aŭ kunigilo kun la filtrilo en formo de tubo.;ropa;192347;Gasmasko;9446629;https://eo.wikipedia.org/w/index.php?oldid=9446629
Kapta sistemo - ĝi estas alfiksita al la vangoparto kaj ĝi servas al firma kapto de la masko al la vizaĝo.;ropa;192347;Gasmasko;9446629;https://eo.wikipedia.org/w/index.php?oldid=9446629
A - bruna - organikaj gasoj kaj vaporoj , solviloj, ekz. butanolo , tetraklormetano , kloroformo , tolueno , diklormetano , cikloheksanolo , propanolo , benzeno , vinilacetato ktp.;ropa;192347;Gasmasko;9446629;https://eo.wikipedia.org/w/index.php?oldid=9446629
Makramo (el la turka makrama ) estas manlabora tekniko ligita al punto . Ĝi konsistas el ornamaj nodoj de fadenoj , kiujn oni surmetas, teksas aŭ plektas por ornami sakojn , ŝtofojn , tapiŝojn , kurtenojn kaj tiel plu.;ropa;201875;Makramo;8781154;https://eo.wikipedia.org/w/index.php?oldid=8781154
La plej bazaj teknikoj baziĝas sur du malsamaj nodoj, tamen ekzistas teknikoj (ekz. la ĉina ), kiuj uzas almenaŭ dudek malsamajn nodojn.;ropa;201875;Makramo;8781154;https://eo.wikipedia.org/w/index.php?oldid=8781154
Kamizolo estas jakforma vira survesto aŭ alisence - pli konate - speco de subkorsaĵo, iam portita de virinoj super la korseto .;ropa;202232;Kamizolo;9037876;https://eo.wikipedia.org/w/index.php?oldid=9037876
Lapfermilo estas fermilo aŭ ligo, kiu povas esti ripete facile fiksata kaj malfiksata, uzata interalie en vestaĵoj. Estas du kuniĝantaj partoj: kroĉiloj unuflanke kaj harsimilaj bukloj aliflanke.;ropa;203074;Lapfermilo;9121181;https://eo.wikipedia.org/w/index.php?oldid=9121181
Gaino estas virina subvesto kiu strikte alĝustiĝas al la korpo. Ĝi estas farita el elasta ŝtofo kaj ĉefe kovras la koksojn. Oni surmetas la gainon ŝovante ĝin laŭlonge de la korpo. ( PIV );ropa;203558;Gaino;8198202;https://eo.wikipedia.org/w/index.php?oldid=8198202
"Gaino estas ankaŭ ""Tubo metala aŭ plasta, ofte fleksebla, ŝirme entenanta telefondratojn, elektrokablojn ks, enigotajn en muron, betonon, teron k.a."" Tiukadre vidu artikolon Foldomaŝino .";ropa;203558;Gaino;8198202;https://eo.wikipedia.org/w/index.php?oldid=8198202
Kuglorezista veŝto estas protekta vesto, kiu ŝirmas la supran korpoparton kontraŭ pafaĵoj. Ĝin uzas soldatoj, policanoj kaj kontraŭteroristaj agad-grupoj.;ropa;205578;Kuglorezista veŝto;8463225;https://eo.wikipedia.org/w/index.php?oldid=8463225
Oni ellaboris en la 1960-aj jaroj novtipajn veŝtojn, kies lamenoj konsistis el ŝtalo kaj bor-karbido. La pli modernaj tipoj iĝis pli efikaj per aldono de pluraj tavoloj de nilono, kiu amortizas energion de la pafaĵo.;ropa;205578;Kuglorezista veŝto;8463225;https://eo.wikipedia.org/w/index.php?oldid=8463225
Tasko de la ŝtalkiraso aŭ durplasto estas la malebligo de la pafaĵa enpenetro. La ceramika kiraso abrupte malrapidigas la pafaĵon kaj transprenas ties energion, kiam la kuglo trafas la kirason. Tiel la lamenojn aŭ kaĥelojn de la kuglorezista veŝto oni devas ŝanĝi post trafo de kuglo.;ropa;205578;Kuglorezista veŝto;8463225;https://eo.wikipedia.org/w/index.php?oldid=8463225
Fundoshi estas japana tradicia vira subvesto . Kutima ĝis la komenco de la 20-a jarcento , la fundoshi malaperis post la dua mondmilito , kiel malaperis ankaŭ la publikaj ambaŭseksaj banoj. Fakte Usono alportis sian prudecon, kaj la fundoshi cedis sian lokon al okcidentaj subvestoj. Tamen ankoraŭ nuntempe ĝi estas tre kutima dum tradiciaj japanaj festoj, portitaj sola aŭ kun jako .;ropa;206186;Fundoshi;8195612;https://eo.wikipedia.org/w/index.php?oldid=8195612
La longeco estas tiu de la enverguro . Ĝi estas T-forma, unu granda ŝtofpeco (simila al rokushaku ) kun ĉiuflanke rimeno aŭ bendo kiun oni ligas ĉe la talio . Ĝi estas tre facile portebla, sed ankaŭ malmultekosta. Ĝi estas la plej disvastiĝinta.;ropa;206186;Fundoshi;8195612;https://eo.wikipedia.org/w/index.php?oldid=8195612
Ĝi montras la formon de tiu ĉi sed estas metita kiel kalsono kun nur bendeto ligota flanke.;ropa;206186;Fundoshi;8195612;https://eo.wikipedia.org/w/index.php?oldid=8195612
Lerneja uniformo estas deviga en la plej multaj lernejoj en Japanio . Kaj knabaj kaj knabinaj uniformoj havas eŭropo|eŭropajn modelojn;ropa;206277;Lernejaj uniformoj en Japanio;8323747;https://eo.wikipedia.org/w/index.php?oldid=8323747
"La nomo gakuran estas kunigo de la mallongigoj gakkō (学校 = lernejo ) kaj randa (らんだ). Tiu lasta vorto estas el la Edo-epoko kaj verŝajne mallongigo de oranda (和蘭, portugale : olanda = Nederlando ) kaj signifas ""eŭropa vesto"".";ropa;206277;Lernejaj uniformoj en Japanio;8323747;https://eo.wikipedia.org/w/index.php?oldid=8323747
Unue serafuku estis enkondukita en 1921 en la knabinan akademion Fukuoka (福岡女学院 Fukuoka jogakuin ) de la estrino Elisabeth Lee laŭ modelo de sportvestoj portitaj en Britio . Tuj poste maristaj vestoj disvastiĝis kiel lerneja uniformo por knabinoj en la tuta lando, dum la modelo por knabaj uniformoj estis tiuj de la prusa armeo.;ropa;206277;Lernejaj uniformoj en Japanio;8323747;https://eo.wikipedia.org/w/index.php?oldid=8323747
Tradicie la marista ĉemmizo kaj jupo estas malhelbluaj, escepte la blankaj strioj. Nuntempe aldoniĝis aliaj koloroj, sed plej ofte oni vidas malhelbluajn, blankajn aŭ grizajn uniformojn.;ropa;206277;Lernejaj uniformoj en Japanio;8323747;https://eo.wikipedia.org/w/index.php?oldid=8323747
Maristaj uniformoj konatiĝis eksterlande pro mangaoj kaj animeoj . Oni trovas ilin ĉefe stiligitaj ( Sailor Moon ) aŭ kiel lerneja uniformo, ekz. en Inu Yasha (Kagome), en Detektivo Conan (Kazuha), en Magic Kaito (Nakamori Aoko).;ropa;206277;Lernejaj uniformoj en Japanio;8323747;https://eo.wikipedia.org/w/index.php?oldid=8323747
Ĉar iuj japanaj viroj fiksiĝis je junaj virinoj kiel sekssimbolo kaŭzis ke dancistinoj kaj servistinoj en multaj trinkejoj portas lernejajn uniformojn, eĉ se la portanto jam estas pli ol 30-jara. Multaj seksaktorinoj fakiĝis pri rolado de lernejaninoj.;ropa;206277;Lernejaj uniformoj en Japanio;8323747;https://eo.wikipedia.org/w/index.php?oldid=8323747
Kebajo ( indonezie/malajzie : kebaya ) estas tradicia bluzo portata de indoneziaj kaj malajziaj virinoj. Plejofte ĝi estas bele ornamita ekzemple per brokato aŭ punto . Ĝi kutime estas portata kun sarongo .;ropa;206872;Kebajo;8266442;https://eo.wikipedia.org/w/index.php?oldid=8266442
Versio de la kebajo kun sarongo estas la uniformo por stevardinoj de la flugkompanioj Singapuraj Aerlinioj kaj Malajziaj Aerlinioj.;ropa;206872;Kebajo;8266442;https://eo.wikipedia.org/w/index.php?oldid=8266442
Bivakĉapo estas kapvestaĵo por kovri la tutajn kapon kaj kolon , lasante ne kovrita sole malgrandan parton de la vizaĝo , la okulojn , buŝon .;ropa;208854;Bivakĉapo;7609815;https://eo.wikipedia.org/w/index.php?oldid=7609815
Ankaŭ fajrobrigadistoj uzas la ĉapon kiel protekton kontraŭ fajro kaj varmego, ĝi estas parto de la spirprotekta ekipaĵo .;ropa;208854;Bivakĉapo;7609815;https://eo.wikipedia.org/w/index.php?oldid=7609815
Mantelo (el la germana : Mantel devenanta el la latina : mantulum , kovrilo, tegilo) estas nuntempe supervesto en formo de longa jako aŭ kaftano.;ropa;211610;Mantelo;9082989;https://eo.wikipedia.org/w/index.php?oldid=9082989
Aparta, karakteriza kolumo en la eŭropa modo de la 16-a kaj 17-a jarcentoj estis la krispo .;ropa;211612;Kolumo;9429868;https://eo.wikipedia.org/w/index.php?oldid=9429868
Agrafoj estis antaŭe multe uzataj por fermi vestaĵojn aŭ por malhelpi hazardan malfermiĝon de zipoj ĉe jupoj aŭ pantalonoj . Nuntempe ilin ofte anstataŭas butonoj, zipoj kaj lapfermiloj . Agrafaj fermiloj estas ankoraŭ kutimaj precipe en mamzonoj.;ropa;213083;Agrafo (vestero);7288756;https://eo.wikipedia.org/w/index.php?oldid=7288756
"Kudrilo estas speciala stanget-forma ilo kun akra pinto je unu ekstremo kaj trueto ĉe la alia. Ĝi servas por kunligi materialojn plej ofte el teksaĵoj per fadenoj . Tiun procezon oni nomas kudrado , dum kudrado per maŝino nomiĝas ""stebado"" .";ropa;213142;Kudrilo;8293300;https://eo.wikipedia.org/w/index.php?oldid=8293300
Ekzistas pluraj proverboj pri kudrilo en la Proverbaro Esperanta de L. L. Zamenhof , inter ili :;ropa;213142;Kudrilo;8293300;https://eo.wikipedia.org/w/index.php?oldid=8293300
Fibolo (el la latina : fibula , fiksilo) estas praa sendanĝera pinglo , ilo por kunfiksi partojn de vestaĵoj aŭ ŝtofojn .;ropa;213174;Fibolo;9283203;https://eo.wikipedia.org/w/index.php?oldid=9283203
Fiboloj utilis ĉefe por kunfiksi vestaĵojn, sed kelkaj havis nuran ornaman funkcion. Ĝi estas plibonigo de la nodo aŭ de la simpla pinglo , kiu perdiĝis pli facile.;ropa;213174;Fibolo;9283203;https://eo.wikipedia.org/w/index.php?oldid=9283203
Kvankam la kapo de la fibolo estis ofte ornamita, la arko , aŭ korpo ofte surhavis pli belajn ornamojn, kiuj en certaj kulturoj havis simbolan signifon aŭ estis ligitaj al rango , profesio , edziĝstato, ktp. Kelkaj romiaj fiboloj prezentis armerangon .;ropa;213174;Fibolo;9283203;https://eo.wikipedia.org/w/index.php?oldid=9283203
Fiboloj foje estis amuletoj por protekti kontraŭ malsano aŭ malbona sorto. Iuj estis ofertitaj en sanktejoj aŭ sur altaroj .;ropa;213174;Fibolo;9283203;https://eo.wikipedia.org/w/index.php?oldid=9283203
"En centra Eŭropo , komence de la Tene-epoko , fiboloj estis tre ofte el bronzo , produktitaj laŭ tekniko de ""fandiĝonta vakso "" aŭ de kaldrono per batado de bronza metalpeco laŭvice rekuirita por pligrandigi la elaston de la movebla parto, la risorto .";ropa;213174;Fibolo;9283203;https://eo.wikipedia.org/w/index.php?oldid=9283203
Korpo de fibolo estas nomata aŭ arko aŭ plato laŭ ĝia baza formo. Arko estas kutime longa kaj mallarĝa, ofte kurba. Plato estas plata kaj larĝa. Platoj povas esti densaj aŭ malfermaj verkoj. La korpo estis ofte ornamita. La kapo estas la ekstremo de la fibolo, kiu entenas la fiksilon. La piedo estas la ekstremo, kie la pinglo fermiĝas. Laŭ fiboltipo kaj kulturo la fibolo estis portata kapon supre, flanke aŭ malsupre.;ropa;213174;Fibolo;9283203;https://eo.wikipedia.org/w/index.php?oldid=9283203
La pinglo uzita por fiksi la vestaĵon estas aŭ kontinuo de la fibolkorpo aŭ aparta peco fiksita al la korpo. La fibolo estas fermita, fiksante la ekstremon de la pinglo al ingo.;ropa;213174;Fibolo;9283203;https://eo.wikipedia.org/w/index.php?oldid=9283203
Korpo kaj pinglo kunigas ĉe risorto aŭ ĉarniro . La plej fruaj fiboloj havis risorton, kiu donis tension al la pinglo. La risorto estas spiralo unuflanka aŭ ambaŭflanka. Unuflanka risorto vindiĝas nur unudirekten. Unuflankaj risortoj estas la plej fruaj, aperante dum la 14-a jarcento a.K. . Ambaŭflankaj risortoj, kiuj vindiĝas ambaŭflanke de la fibola korpo, estiĝis dum la 6-a jarcento a.K. . Ambaŭflankaj risortoj povas esti tre malgrandaj kun nur unu aŭ du volvaĵoj ĉiuflanke, aŭ ĝis 10 centimetrojn longaj. La plejmultaj ambaŭflankaj risortoj estis faritaj el unu metalpeco kaj pro tio havas risortkordon, draton, kiu iras de unu ekstremo de la risorto al la alia. La risortkordo povas esti antaŭ aŭ malantaŭ la fibolkorpo. Ambaŭflankaj risortoj vindiĝas ĉirkaŭ pinglo aŭ akso . Ili kutime estas faritaj el fero, eĉ se la cetero de la fibolo kaj risorto estas el kupra alojo . Dum la 1-a jarcento ekzistis fiboloj kun risortoj fiksitaj sub metala kovrilo, kiu estis vastigaĵo de la fibolkorpo. Tiajn risortojn oni nomas kovritajn aŭ kaŝitajn risortojn.;ropa;213174;Fibolo;9283203;https://eo.wikipedia.org/w/index.php?oldid=9283203
Necesas aldoni, ke pluraj konstrutipoj estis uzitaj samtempe. Kvankam ĉarniraj fiboloj aperis post la risortaj fiboloj oni daŭre produktis tiujn. Tial ĉarnira fibolo ne estas ĉiam pli juna ol risorta fibolo.;ropa;213174;Fibolo;9283203;https://eo.wikipedia.org/w/index.php?oldid=9283203
La rapida disvastiĝo de la romia imperio dum la 1-a jarcento kaŭzis pligrandiĝon de la nombro kaj dezajnoj de fiboloj tra la tuta Eŭropo kaj proksima Oriento . La arĉoj estis pli kompleksaj, ĉarniroj aperis samtempe kiel ambaŭflankaj risortoj kaj grandega vario de platdesegnaĵoj ekestis.;ropa;213174;Fibolo;9283203;https://eo.wikipedia.org/w/index.php?oldid=9283203
La ringa fibolo estas tre malfacile datigebla, ĉar la dezajno preskaŭ ne ŝanĝis inter la 2-a 2-a kaj la 14-a jarcento . Foje ia unika formo aŭ ornamaĵo povas indiki, ĉu ringa fibolo estas romia aŭ mezepoka .;ropa;213174;Fibolo;9283203;https://eo.wikipedia.org/w/index.php?oldid=9283203
La klasika fibolo de la malfrua romia epoko ( 3-a kaj 4-a jarcentoj ), kaj fakte la plej bone konata fiboltipo, estas la arbalesta fibolo. Ĝi konsistas el tre kurbita, duoncirkla arĉo, kutime el kvadrata sekcio kaj longa plata piedo. La fibolo havas larĝan transversan stangeton (aŭ brakojn) ĉe la kapo, kiu entenas pingloĉarniron. Arbalestaj fiboloj havas kutime tri globajn cepformajn ansojn : unu ĉe la kapo kaj la aliaj ĉe la ekstremoj de la transversa stangeto.;ropa;213174;Fibolo;9283203;https://eo.wikipedia.org/w/index.php?oldid=9283203
Ekzistas multaj tipoj de postromiaj fiboloj. La gotaj arĉfiboloj havas rondan aŭ triangulan platan kapplaton, ofte kun 3, 5 aŭ 7 ansetoj, malgrandan kurban arkon kaj longan platan, diamantforman piedon. Ili estis tre uzitaj de ĝermanaj visigotoj , ostrogotoj kaj gepidoj , kaj neĝermanaj slavoj kaj avaroj . Tiaj fiboloj estas trovitaj en vastaj partoj de suda kaj okcidenta Eŭropo dum la 5-a kaj 6-a jarcentoj . La plej multaj vikingaj fiboloj estas variaĵoj de la ringa aŭ ronda modelo.;ropa;213174;Fibolo;9283203;https://eo.wikipedia.org/w/index.php?oldid=9283203
"Ekzistas grandega nombro de diversaj mezepokaj broĉtipoj. Oni plej ofte uzas la vorton "" broĉo "" por mezepokaj fiboloj. La plej komunaj estas ringaj broĉoj, inkluzive kvadrataj, rondaj aŭ floraj dezajnoj, kaj plataj broĉoj en formo de homoj aŭ bestoj (tiujn oni nomas pilgrimulaj broĉoj .";ropa;213174;Fibolo;9283203;https://eo.wikipedia.org/w/index.php?oldid=9283203
La kudrado estas uzata por produkti vestaĵojn , ŝuojn , teksaĵojn por la domo ( littukoj , kusenkovriloj, mantukoj , tablotukoj , kurtenoj , ktp), kaj ankaŭ ledaĵojn ( valizoj , poŝoj , sakoj , monujoj , ktp), ktp.;ropa;222046;Kudrado;9262027;https://eo.wikipedia.org/w/index.php?oldid=9262027
La historio de kudrado datiĝas de la Paleolitiko . Kudrado kutime estis uzata por ligi je punkto por punkto haŭtojn de bestoj por produkti veston aŭ ŝirmejon. Ĉe la glacimumio Ötzi , kiu vivis antaŭ pli ol 5000 jaroj, estis trovitaj profesie kudritaj vestaĵoj el kaprofelo kaj bovoledo. La indiĝenaj popoloj de la amerikaj ebenaĵoj uzis kompleksajn metodojn de kudrado por kunigi la pecojn de la tipioj . La kudrado estas asociita al teksado de folioj en Afriko por krei korbojn . La produktado de vesto farita el naturaj fibroj devenas de la Meza Oriento ĉirkaŭ 4000 AC.;ropa;222046;Kudrado;9262027;https://eo.wikipedia.org/w/index.php?oldid=9262027
Bluzo estas virina ĉemizo , t.e., ĝi estas ŝirma manikhava vesto por la supra parto de l' korpo, ĝis la talio aŭ la genuo , aŭ mallonga kitelo uzita de virinoj . La distingo inter bluzo respektive ĉemizo kaj jako baziĝas precipe sur tio, ke bluzoj kaj ĉemizoj ordinare konsistas el pli malpeza ŝtofo kaj ke oni portas ilin senpere sur la haŭto (kaj eventuale sub jako), dum jakoj tendencas esti el pli peza kaj pli varmiga ŝtofo kaj eventuale estas subŝtofitaj.;ropa;226043;Bluzo;9032175;https://eo.wikipedia.org/w/index.php?oldid=9032175
Kiel virina vesto, ĝenerale ĝi estas pli ornamita ol ĉemizo. Ĝenerale, oni portas bluzon kun jupo aŭ pantalono .;ropa;226043;Bluzo;9032175;https://eo.wikipedia.org/w/index.php?oldid=9032175
En antikva Grekio laŭrokronoj ornamis elstarajn sportistojn kaj poetojn. Aparte la venkintoj en la sportaj konkursoj de Olimpio honoriĝis per ili.;ropa;228732;Laŭrokrono;9338045;https://eo.wikipedia.org/w/index.php?oldid=9338045
En antikva Romio la laŭrokrono estis signo de milita venko dum la triumfa reveno de armeestro al Romo . Poste la romiaj imperiestroj konstante portis laŭrokronoj kiel signo de sia gloro.;ropa;228732;Laŭrokrono;9338045;https://eo.wikipedia.org/w/index.php?oldid=9338045
"Franck Sorbier naskita la 4-an de januaro 1961 de Georgette Suraud kaj Raoul Sorbier, nomoj de arboj, resp. sambuko kaj sorbuso, kiuj antaŭdestinis lin al arta metio, estas unu el la dek kudristoj oficiale agnoskataj de ""Fédération française de la haute couture"" (Franca Federacio de alta kudrarto). Naskita en Var , li pasigis infanecon en Biarritz antaŭ ol instali sin en Parizo en 1980. Provoka lernanto de lernejo ESMOD, li prezentis sian unuan kolekton de konfekciaj vestoj en 1987, tamen li plej bone disvolvis sian arton en kudrarto al kiu li plene dediĉiĝis ekde 1999.";ropa;228896;Franck Sorbier;6434496;https://eo.wikipedia.org/w/index.php?oldid=6434496
"En 2005, li kreis 280 malsamajn scenejajn kostumojn por la spektaklo ""Avant que l'ombre... à Bercy"" de Mylène Farmer .";ropa;228896;Franck Sorbier;6434496;https://eo.wikipedia.org/w/index.php?oldid=6434496
Kostumo estas vesto propra al difinita popolo , klaso, tempo, ofico aŭ cirkonstanco. Ĝi konsistas el pluraj vestaĵoj kaj akcesoraĵoj .;ropa;241422;Kostumo;9175034;https://eo.wikipedia.org/w/index.php?oldid=9175034
Indikoj pri portado de harretoj troviĝas jam en la Antikva Grekio . En nordokcidenta Eŭropo harretoj en sprango tekniko estis trovitaj en la virintombejo de Skrydstrup el la 15-a aŭ 14-a jarcento a.K. . Pli novaj trovaĵoj (8-a ĝis 5-a jarcento a.K. ) estas tiuj de la marĉokadavroj de la virino de Haraldskær kaj de la virino de Arden el Danio . Aliaj trovaĵoj ekzistas el rubejo de la romia militkampo Vindonissa en Svislando kaj de la saksa tombejo de Liebenau el la 6-a jarcento . En Germanio kaj Anglio ekzistis fajnaj harreto el silko ekde la 13-a jarcento .;ropa;249144;Harreto;9370129;https://eo.wikipedia.org/w/index.php?oldid=9370129
Bankalsono estas tipo de banvesto por viroj (foje ankaŭ por virinoj). Temas pri kalsono farita el speciala elastano por uzo en naĝejo aŭ marakvo . En multaj publikaj naĝejoj la porto de bankalsono estas deviga.;ropa;250352;Bankalsono;9104600;https://eo.wikipedia.org/w/index.php?oldid=9104600
Kvankam ĝia uzo estas malpermesita en iuj naĝejoj (ekz. francaj publikaj naĝejoj ), viroj, kaj ĉefe junuloj multe ŝatas porti ĝin, ĉefe sur la strando , sed ankaŭ dum akvaj sportoj aŭ plonĝado .;ropa;250379;Banŝorto;7945821;https://eo.wikipedia.org/w/index.php?oldid=7945821
Nimfeto povas ankaŭ esti knabino kiu atentigas pedofilon , laŭ ties vidpunkto, ĉu ŝia sinteno estas seksalloga aŭ ne. La termino ankaŭ signas pornografiajn kaj malleĝajn figurojn kaj fotografiojn de neplenaĝulinoj, aŭ de plenaĝulinoj kies eksteraĵo estas vole infaneca.;ropa;253305;Nimfeto;6086447;https://eo.wikipedia.org/w/index.php?oldid=6086447
Nimfeto estas eta nimfo (el la greka νύμφη / númphê = junulino), voluptema duondiino de naturo en helena mitologio , fama pro siaj rilatoj kun satirusoj . La esprimo nimfomanio (seksa urĝo ĉe inoj) havas la saman originon.;ropa;253305;Nimfeto;6086447;https://eo.wikipedia.org/w/index.php?oldid=6086447
La vorto estis notinde uzita fare de franca verkisto Pierre de Ronsard kaj precipe popularigita fare de rusdevena anglalingva verkisto Vladimir Nabokov en sia 1955-a romano Lolita , kiu temas pri rilatoj inter 12-jara knabino kaj matura viro, la t.n. nimfolepto .;ropa;253305;Nimfeto;6086447;https://eo.wikipedia.org/w/index.php?oldid=6086447
Laŭ la protagonisto de tiu romano, la eldiro de la vorto lolita , karesa mallongigo de la persona nomo Dolores , estas kvazaŭ volupta invito en si mem, karakterizara de tri frapoj de la palato fare de la lango .;ropa;253305;Nimfeto;6086447;https://eo.wikipedia.org/w/index.php?oldid=6086447
Hanfuo ( ĉina lingvo : 漢服/汉服, pinyin : hànfú, minnana : aŭ hanhok , kantona : aŭ honfuk ) estas tradicia han-nacieca vestaĵo. La nomo devenas de la libro Han Shu . Post la enmarŝo de la manĉua popolo en 1644 la Dinastio Qing malpermesis surmeti hanfu al hanoj kaj aliaj nacietoj, do ili eksurmetis qipao (manĉua tradicia vestaĵo).;ropa;253841;Hanfu;7989218;https://eo.wikipedia.org/w/index.php?oldid=7989218
Manumbutono estas nefiksita butono, kiu estas uzata por ornami ĉemizon kiu ne havas fiksitan butonon sur la manumo sed nur butontruojn ( meza bildo ). La manumbutonoj principe ĉiam estas videblaj, ĉar la manumoj estas videblaj sub jako aŭ trikotvesto .;ropa;259705;Manumbutono;9068570;https://eo.wikipedia.org/w/index.php?oldid=9068570
Kudrista pupo aŭ kudrista torso aŭ manekeno estas homgranda kaj homsimila pupo aŭ formo, kiu ebligas al kudristoj krei vestaĵojn laŭ la mezuroj de la kliento. Temas kutime pri torso , sen brakoj nek kruroj, kiun oni povas facile vesti.;ropa;260600;Kudrista pupo;7542515;https://eo.wikipedia.org/w/index.php?oldid=7542515
Nemodifebla kudrista pupo estas aŭ kreita laŭ specifaj mezuroj de unu kliento aŭ laŭ standardaj vestaĵgrandecoj.;ropa;260600;Kudrista pupo;7542515;https://eo.wikipedia.org/w/index.php?oldid=7542515
Laŭ la vestaĵoj kudrotaj, ekzistas ankaŭ kudristaj pupoj kun brakoj aŭ kruroj. Aĉeteblas kudristaj pupoj por virinoj, viroj kaj infanoj en diversaj grandecoj kaj diverse modifeblaj.;ropa;260600;Kudrista pupo;7542515;https://eo.wikipedia.org/w/index.php?oldid=7542515
Temas pri malgranda cirkla peco de ŝtofo aŭ foje ledo , kiu povas esti riĉe ornamita kaj kovras la malantaŭan parton de la kapo. Ofte kipaoj estas fiksitaj al la hararo per metala fiksigilo. Portado de la kipao estas kutima dum ĉiu speco de preĝoj, sed ankaŭ krome dum ĉiaspecaj vizitoj de judaj religiaj lokoj kiel sinagogoj aŭ judaj tombejoj . Multaj ortodoksaj judoj ankaŭ portas kipaon dum la ĉiutaga vivo, tamen ne dum dormado.;ropa;264582;Kipao;9421541;https://eo.wikipedia.org/w/index.php?oldid=9421541
La mitro , el la greka μίτρα, estas religia ĉapelo kaj parto de la tradicia, ceremonia vestaro de episkopoj kaj certaj pastroj de la latina eklezio . Ĝi ankaŭ estas populara simbolo de la papo .;ropa;264948;Mitro;9049342;https://eo.wikipedia.org/w/index.php?oldid=9049342
En okcidentaj kulturoj, mortmasko estas modluro el vakso aŭ gipso de la vizaĝo de ĵus mortinta persono. Mortmasko estas uzebla kiel memoraĵo de la mortinto, aŭ kiel modelo por la pentrado de portreto . Kelkfoje eblas identigi la portretojn, kiuj estis pentritaj surbaze de mortmasko, pro tipaj misformadoj de la vizaĝaj trajtoj. En aliaj kulturoj, mortmasko povas ekzemple esti masko el argilo , fabrikita por kovri la vizaĝon de la mortinto antaŭ la enteriga soleno. La plej famaj tiaj mortmaskoj estis tiuj uzitaj en Antikva Egiptio kiel parto de la mumiiga procezo , ekzemple la enteriga masko de Tutanĥamono .;ropa;267031;Mortmasko;8370727;https://eo.wikipedia.org/w/index.php?oldid=8370727
En la 17-a jarcento en kelkaj eŭropaj landoj, oni kutimis uzadi la mortmaskojn kiel partoj de statuoj de la mortintoj. Dum la 18-a kaj 19-a jarcentoj , oni cetere uzis mortmaskojn por registri la trajtojn de nekonataj kadavroj : tiun funkcion poste plenumis fotografio .;ropa;267031;Mortmasko;8370727;https://eo.wikipedia.org/w/index.php?oldid=8370727
Koltuko estas ĝenerala vorto por tuko , kiun oni portas ĉirkaŭ la kolo por ŝirmi ĝin kontraŭ malvarmo aŭ frosto , ĉefe en vintro , aŭ dekorocele. Aparte rekonigan signifon koltukoj havas ĉe la skoltoj .;ropa;273145;Koltuko;9114672;https://eo.wikipedia.org/w/index.php?oldid=9114672
Kiel ĉiuj maŝinoj ankaŭ por la kudromaŝino ekzistas longa prahistorio. En pratempo la homoj uzis fiŝostojn por kudri. Tiam sekvis kudriloj el pintaj ostoj aŭ korno kun kudriltrueto. Nur en la 14-a jarcento oni sukcesis fabriki kudrilon el ŝtalo . Dum jarcentoj la ŝtalkudrilo estis la plej grava ilo por fabriki vestaĵojn.;ropa;273590;Kudromaŝino;9170899;https://eo.wikipedia.org/w/index.php?oldid=9170899
En la jaro 1846 la usona inventisto Elias Howe inventis la duoblostebpik-kudromaŝinon. Li tamen ne trovis fabrikiston, kiu pretis surmerkatigi lian inventon, li tial forlasis Usonon kaj vojaĝis al Anglio sed ankaŭ tie ne sukcesis. Reveninte al Usono li konstatis, ke intertempe Isaac Merritt Singer inventis similan maŝinon. Howe plendis kontraŭ la patento de Singer kaj sukcesis, ke Singer devis dividi la profiton de la patento kun Howe.;ropa;273590;Kudromaŝino;9170899;https://eo.wikipedia.org/w/index.php?oldid=9170899
"En la 14-a jarcento, ""livreo"" temis pro indulgo de ajna speco (por okazo la urbo Exeter en Devon , Anglio havas straton nomatan kiel ""Livery Dole"" ( Livreo Dole ) pro la Livery Dole Almozulejo kaj Kapelo , fondita en marto de 1591 ), sed precipe ŝtofojn oni liveris al servantoj kaj membroj de la familio. Ĉi tiaj aĵoj povus esti konservitaj en livrea ŝranko .";ropa;279452;Livreo;9494687;https://eo.wikipedia.org/w/index.php?oldid=9494687
"La sento poste malpligrandiĝis al proviantoj de servantoj kaj apartaj nomaj grupoj, ofte en koloro-plano aparta al la familio, kiel la manteloj portitaj de lakeoj en grandaj domoj ĝis la Unua mondmilito , kaj al furaĝo por ĉevaloj , de kiu estas heredita "" livrea stabileco "" (1705) .";ropa;279452;Livreo;9494687;https://eo.wikipedia.org/w/index.php?oldid=9494687
Naztuko aŭ poŝtuko estas peco de ŝtofo aŭ celulozo , kiu precipe uzatas por la purigo de la nazo, pli konkrete por la forigo de elblovata naza muko de la nazo. Ŝtofaj naztukoj estas laveblaj kaj tial kutime post lavado uzataj plurfoje, dum celulozaj naztukoj post la uzo forĵetindas. Ne en ĉiu homa kulturo de la tero por la purigo de la nazo uzatas tia tuko, kaj ankaŭ en Eŭropo tiu ago ĝis la moderna epoko ankoraŭ ne estis kutimo.;ropa;279981;Naztuko;9114679;https://eo.wikipedia.org/w/index.php?oldid=9114679
Speciala formo de la naztuko estas dekora tuko, kiun sinjoroj por festaj okazoj portas en brusta poŝo de eleganta jako: tiu tuko vere nur estas dekora kaj ne havas praktikan funkcion.;ropa;279981;Naztuko;9114679;https://eo.wikipedia.org/w/index.php?oldid=9114679
"En la Romia Imperio de la antikva epoko ekzistis tukoj por forviŝi ŝviton kaj por purigi la buŝon kaj la manojn ekzemple dum manĝado. Historiistoj nomas ilin ""etikedaj tukoj"", kaj substrekas ke ilia uzo dum reprezentaj festoj kaj festenoj estis signo de eleganteco kaj nobleco. Por la purigo de nazo tiuj tukoj ne uzatis. Ŝvitotukojn unuafoje skribe mencias la poeto Katulo - li nomas ilin sudarium ( latine sudor = ŝvito). Tiuj tukoj estis el egiptia lino kaj estis metitaj en faldon de la togo . Jarcenton pli poste aperis la kutimo de buŝtuko nomata orarium (latine oris = buŝo). Krome en la antikva Romia Imperio jam ekzistis apartaj buŝtukoj dum festaj manĝoj, kiuj latine nomatis mappa , kaj estis pli dekoraj ol praktike uzataj. Sub la imperiestro Aŭreliano ekestis la kutimo, en la amfiteatro bonvenigi sociajn gravulojn kaj ŝatatajn aktorojn per la svingo de koloraj tukoj. Mappa kaj orarium kiel liturgiaj simboloj restis en la katolika diserva ceremonio.";ropa;279981;Naztuko;9114679;https://eo.wikipedia.org/w/index.php?oldid=9114679
Ekde la 11-a jarcento teksitaj tukoj kiel sekreta simbolo de amo havis gravan rolon en la ritoj de korteza amo . Kiel simbolo pri fideleco, kavaliroj kunprenis tiajn tukojn en la batalojn, kaj redonis ĝin poste - ofte makulite per ŝvito kaj sango - al la amatino. Foje tiaj tukoj ankaŭ estis malferme fiksitaj ĉe la lanco , se la posedantino ne estis edzinigita al alia kavaliro.;ropa;279981;Naztuko;9114679;https://eo.wikipedia.org/w/index.php?oldid=9114679
"La teksisto Baptiste el Kambrezo , tiutempe nederlande Kamerijk en Flandrio , nuntempe Cambrai en norda Francio , ĉirkaŭ la jaro 1300 laŭdire produktis la unuajn naztukojn el ŝtofo. Sub la nomo drapesello panetto di naso ( itale ""simplaj tukoj ŝtofaj por la nazo"") ili tamen nur uziĝis malofte. La tukoj konserviĝis en poŝo ĉe la zono .";ropa;279981;Naztuko;9114679;https://eo.wikipedia.org/w/index.php?oldid=9114679
"Katerino de Mediĉo dum la 16-a jarcento enkondukis la ""tualetan tukon"" ĉe la reĝa kortego de Francio. Ĝi nomiĝis mouchoir kaj precipe havis reprezentan celon. Por la forviŝo de naza muko ankaŭ la nobeloj dum tiu epoko ankoraŭ uzis la fingrojn. Erasmo de Roterdamo , kiu laŭ inventario posedis 39 naztukojn, konsideriĝas escepto. La dekoraj tukoj de la damoj estis sorbigitaj per parfumo kaj kiel ama simbolo donacitaj al sinjoroj. La pola-litova kaj poste franca reĝo Henriko la 3-a donacis tiajn tukojn al siaj favoratoj . Ankaŭ en la aliaj kortegoj de Eŭropo tiaj tukoj ekde la komenco de la 16-a jarcento modiĝis.";ropa;279981;Naztuko;9114679;https://eo.wikipedia.org/w/index.php?oldid=9114679
Per la inventoj de pli kaj pli profesiecaj teksiloj meze de la 18-a jarcento la produktado de ŝtofo pli kaj pli malmultekostiĝis. Tial ankaŭ naztukoj povis esti produktataj pli kaj pli malmultekoste kaj iom post iom de luksa akcesoraĵo iĝis ĉiutaga objekto. Dum la dua duono de la 19-a jarcento la damoj kutimis meti sian parfumigitan naztukon en la dekoltaĵon aŭ en jupan manikon, por rapide havi ĝin ĉemane .;ropa;279981;Naztuko;9114679;https://eo.wikipedia.org/w/index.php?oldid=9114679
Baleta jupo estas jupo vestata kiel kostumo en baleta spektaklo , ofte kun alfiksita korsaĵo . Ĝi povas esti sola tavolo pendanta suben, aŭ multaj amelizitaj tavoloj kaj apogiloj. Ĝia konstruo strebas aldoni aspekton de flosanta moviĝo de baletistino dancanta en la jupo.;ropa;290930;Baleta jupo;9442385;https://eo.wikipedia.org/w/index.php?oldid=9442385
Baleta jupo estas ofte volvita ĉirkaŭ jupo, almuntita en la talia zono per rubandoj alkudritaj en ĝia talia zono. Ĝi povas ankaŭ ligiĝi al supraj trikaĵoj, kutime kun longaj brakingoj.;ropa;290930;Baleta jupo;9442385;https://eo.wikipedia.org/w/index.php?oldid=9442385
Dum kiam romantikaj baletaj jupoj estas kutime portataj en kutimaj vestaj sakoj, la klasikaj diskoformaj baletaj jupoj postulas specialajn rondajn sakojn por protekto kaj transportado, faldado de ĝi devus ruinigi ĝian formon kaj aspekton. Trejnadaj baletaj jupoj (du/tri tavoloj de teksaĵo) estas malofte protektataj en sako.;ropa;290930;Baleta jupo;9442385;https://eo.wikipedia.org/w/index.php?oldid=9442385
Gluvesto , aŭ Moldvesto estas elasta , strikte alĝustigita vesto, kiu kvazaŭe «gluas» ĉe la haŭto de tiu, kiu surhavas ĝin. Tiaj versioj aperis en la 1970-aj jaroj , kiam estis trovataj novaj elastaĵoj, kiel elastano .;ropa;292145;Gluvesto;9033120;https://eo.wikipedia.org/w/index.php?oldid=9033120
Ĉar ĝi bone elmontrigas la muskolaron , kolora gluvesto estas la favorigita vesto de la superherooj de bildliteraturo .;ropa;292145;Gluvesto;9033120;https://eo.wikipedia.org/w/index.php?oldid=9033120
Fornoganto aŭ izolganto estas remburita ganto por teni varmajn kuirilojn sen brulvundi la manojn. Ĝi povas enteni malgrandan magneton por facile ĝin fiksi ie sur stovo , por ke la ganto estu je dispono de la kuiranto.;ropa;293512;Fornoganto;8762491;https://eo.wikipedia.org/w/index.php?oldid=8762491
Ĉar tiaj gantoj estas uzindaj rapide plej ofte ne havas apartajn ingojn por ĉiu fingro, sed nur por la dikfingro , ĉar la aliaj devos funkcii kiel tutaĵo.;ropa;293512;Fornoganto;8762491;https://eo.wikipedia.org/w/index.php?oldid=8762491
La kruringo en la antikvo estis parto de la armaĵo , kiu kovris la krurojn (PIV). En tiukaze, ĝi estis el forta ledo aŭ el metalo kaj protektis kontraŭ tranĉoj, vundiĝoj.;ropa;303043;Kruringo;8418215;https://eo.wikipedia.org/w/index.php?oldid=8418215
La nikabo ( arabe ‏نقاب‎ , translit. DMG : niqāb , turke peçe ) estas vualo por la vizaĝo portata de kelkaj islamaj virinoj. Ili portas ĝin kun ĉadoro , ĉarŝafo aŭ kun alia, plej ofte nigra vesto.;ropa;305945;Nikabo;9492121;https://eo.wikipedia.org/w/index.php?oldid=9492121
Pli precize, la nikabo kaŝas preskaŭ la tutan vizaĝon, kiel la burko , sed lasas almenaŭ videblaj la okulojn. Tiu okulareo povas esti ĉu iom pli malferma ĉu iom pli fermita. Plej ofte, la nikabo estas tute nigra.;ropa;305945;Nikabo;9492121;https://eo.wikipedia.org/w/index.php?oldid=9492121
Fundamentismaj islamanoj pravigas la vestpreskribojn per citaĵoj el la Korano kaj ties klarigoj el pli „fruaj fontoj de la islamo“ kiel tiuj de Ibn Kathir kaj At-Tabarī. Tie ili vidas la pruvojn por la portado de la nikabo.;ropa;305945;Nikabo;9492121;https://eo.wikipedia.org/w/index.php?oldid=9492121
"En la korano, dulingva araba-esperanto, eld. TK Kopenhago 1970, estas skribita en ĉapitro ('surao') 24 An-Nur-32 : ""Kaj diru al la kredantinoj, ke ili mallevu sian rigardon, kaj gardu sian ĉastecon, kaj ke ili ne montru siajn ĉarmojn, krom tio, kio devas esti videbla, kaj ke ili kovru per siaj vestaĵoj la bruston."" Sekvas listigo, por sciigi, kiuj membroj de la familio rajtas...";ropa;305945;Nikabo;9492121;https://eo.wikipedia.org/w/index.php?oldid=9492121
En Tunizio , la nikabo estas malpermesata. En Egiptio , la nikabo en universitatoj estas pridiskutata. En Turkio , la nikabo en lernejoj kaj universitatoj estas malpermesata, sed la malpermeso de la islama kaptuko valida dum jardekoj estis aboliciita en 2008, kaj de tiam la jura situacio ne klaras. En Francujo , ekzistas malpermeso por lernantinoj kaj studentinoj porti iun ajn islaman kaptukon (do evidente supervestan vualon, kiel burkon aŭ nikabon) en klasĉambroj kaj sur la lerneja areo, pli ĝenerale, leĝo, eldonita je la 11-a de aprilo 2011, malpermesas al iu ajn kaŝi tielmaniere sian tutan vizaĝon sur iu ajn publika areo.;ropa;305945;Nikabo;9492121;https://eo.wikipedia.org/w/index.php?oldid=9492121
Bermuda pantaloneto estas ŝorto kies krurumoj atingas ĝis iom super la genuoj kaj ofte estas ornamita per helkoloraj desegnaĵoj. Tiuj desegnaĵoj plejofte estas bildoj de floroj aŭ aliaj someraj scenoj.;ropa;307687;Bermuda pantaloneto;7949447;https://eo.wikipedia.org/w/index.php?oldid=7949447
Baŝliko estas rusa ĉapo simila al kapuĉo , kiu havas du longajn pintojn, kiujn oni volvas ĉirkaŭ la kolo. Ĝi estas tradicia vestaĵo de iranaj popoloj, turkoj kaj kozakoj kaj estis farita el ledo , lano aŭ felo . Ekzistas variaĵoj, laŭ kiuj baŝlikoj estas vintraj aŭ someraj, popola vestaĵo, militista vestaĵo, riĉe ornamitaj ktp.;ropa;317692;Baŝliko;9350221;https://eo.wikipedia.org/w/index.php?oldid=9350221
Ekzistas pluraj maskoj , ĉiuj prezentitaj kiel la aŭtentika printo de la vizaĝo de Napoleono, ĉiuj nomitaj laŭ la nomoj de unu el la kuracistoj de Sankta Heleno kiuj faris la printon : Archibald Arnott , Francis Burton kaj François Antommarchi . La oficiala masko ekspoziciita en la Muzeo pri Armeo de la Hotelo de la Invalidoj ( Parizo ) estas la « masko Antommarchi ».;ropa;326172;Mortmasko de Napoleono;9079671;https://eo.wikipedia.org/w/index.php?oldid=9079671
La 5-an de majo 1821 je la 17a kaj 49 minutoj mortis Napoleono Bonaparto, kaptito de la britoj en la insulo Sankta Heleno .;ropa;326172;Mortmasko de Napoleono;9079671;https://eo.wikipedia.org/w/index.php?oldid=9079671
Inter la membroj de tiu malgranda kolonio, kiu vivis apud la Imperiestro, Grafino Bertrand, edzino de la Granda Marŝalo Henri Gatien Bertrand , decidis muldi la kapon de la fama malliberulo, antaŭ la nekropsio . La decidon aprobis la Sankta-Helena guberniestro, Sir Hudson Lowe .;ropa;326172;Mortmasko de Napoleono;9079671;https://eo.wikipedia.org/w/index.php?oldid=9079671
Sed ĉar la gipso mankis en Jamestown , ĉefurbo de la insulo, Grafino Bertrand, proponis pisti kelkajn statuojn por ricevi la necesan materialon. Ŝi ne sciis, ke tio ne estas la maniero por akiri gipson por muldado. Neeblis fari muldadon la 6-an de majo matene. La nekropsio okazis, kaj la enbalzamigo fare de Doktoro Antommarchi.;ropa;326172;Mortmasko de Napoleono;9079671;https://eo.wikipedia.org/w/index.php?oldid=9079671
Dumtempe, Doktoro Francis Burton, iris kolekti gipsoŝtonon , kies minon li konis en la insuleto de George-Island, en la sudoriento de la Sankta Heleno. La gipso ricevita post malbona gipsfarado ne estis bonkvalita. La 7-an de majo , Antommarchi malkonsentis fari muldon. Burton do prezidis tiun delikatan operacion, Sed li iom tro ambiciis kaj volis muldi la tutan kapon (oni tiam kredis je la frenologio , kaj oni povis opinii, ke la tuberoj de la Napoleona kranio estis ekzamenindaj).;ropa;326172;Mortmasko de Napoleono;9079671;https://eo.wikipedia.org/w/index.php?oldid=9079671
Pro la malbona gipso, la muldilo rompiĝis je tri partoj kiam oni demetis ĝin. Sed malfruis, oni metis la korpon en la ĉerkon. En la posta tago, la 8-an de majo matene, Antommarchi kaj Burton kunvenis kaj kune procedis al la elmuldo el la negativoj antaŭtage faritaj. Ankoraŭ okazis malbonaĵo: kredante povi ricevi po plurajn kopiojn el unu muldilo, ili faris elmuldaĵon el ĉiu peco unuope. Sed estis neeble liberigi la muldaĵon sen detrui la muldilon.;ropa;326172;Mortmasko de Napoleono;9079671;https://eo.wikipedia.org/w/index.php?oldid=9079671
La kapo de la Imperiestro estis en tri partoj malfacile kunigeblaj. La plej granda kongruis al la vizaĝo kun la komenco de la maldekstra orelo, la dua parto, al la kranio, la dekstra flanko de la kapo kaj la orelo, fine la lasta al la kolo, la mentono kaj la malsupra maldekstra flanko de la kapo.;ropa;326172;Mortmasko de Napoleono;9079671;https://eo.wikipedia.org/w/index.php?oldid=9079671
Koncerne la aliajn kopiojn, asertitaj esti tiuj de la Imperiestro, ili estas tiom malsimilaj, ke iliaj pretendoj pri vereco estas malfacile kredindaj.;ropa;326172;Mortmasko de Napoleono;9079671;https://eo.wikipedia.org/w/index.php?oldid=9079671
L'avenir des statues (la estonto de la statuoj) de René Magritte figuras la maskon de Napoleono, farbita ĉiela bluo kun nuboj .;ropa;326172;Mortmasko de Napoleono;9079671;https://eo.wikipedia.org/w/index.php?oldid=9079671
Mini-jupoj estis inventitaj de Mary Quant , kiu ankaŭ inventis kaj promociis kurtkalsonojn (hot pants). Estas malkonsento pri kiu havis la ideon unue, kelkaj parolas pri André Courrèges aŭ John Bates . La kurta kaj ĉiam pli kurta jupo jam evoluis inter modemaj junaj virinoj: la dezajnistoj kiuj adaptis ĝin nur helpis disvastigi la stilon. Rilate al Quant, ŝi donis al ĝi nomon. Mary Quant nomis la mini-jupon laŭ la nomo de sia preferata aŭto, la Mini . Ŝi tiel ŝatis la aŭton, ke ŝi havis iun speciale dezajnitan por ŝi.;ropa;352015;Minijupo;9291622;https://eo.wikipedia.org/w/index.php?oldid=9291622
Ĝi konsistis el blanka ledo , kiun la tamburistoj portis sur la maldekstra femuro por protekti la tukan vestaĵon kontraŭ frotado de la tamburo . Escepte la 25-a divizio ĝin uzis ĉiuj prusaj armepartoj.;ropa;355822;Genuofelo;8373155;https://eo.wikipedia.org/w/index.php?oldid=8373155
Gho estas tradicia vestaĵo uzata de la viroj de Butano . Ĝi konsistas el longa robo ĝis la genuoj kun zono (nomata kerao ) faldita tiamaniere ke ĝi iĝas kvazaŭ poŝo ĉe la stomako. Ĝi estis enkondukita en la 17-a jarcento de Shabdrung Ngawang Namgyal por doni al la butananoj pli distingigan identecon.;ropa;371059;Gho;8204310;https://eo.wikipedia.org/w/index.php?oldid=8204310
La Kirao estas la tradicia vestaĵo kiun portas la virinoj de Butano . Kirao konsistas el ampleksa kaj longa rektangula ĝismaleola vestaĵo kaj kiu estas portata sur longmanika bluzo ĉiam ruĝkolora. Ĝi estas tena ĉe la ŝultro per pinglo kaj ligata ĉe la talio. Sur la kirao estas portata silka mallonga jupo nomata toego . La ĉiutage uzata kirao estas el kotono aŭ lano , la silka kirao nur estas uzata dum specialaj okazoj.;ropa;371064;Kirao;5818645;https://eo.wikipedia.org/w/index.php?oldid=5818645
Kuirista ĉapo plejofte estas blanka ĉapo de kuiristo . La celo de la ĉapo estas la plibonigado de higieno en la kuirejo . Per kuirista ĉapo oni preventas, ke ŝvito aŭ haroj falu en la manĝaĵon .;ropa;381472;Kuirista ĉapo;8293328;https://eo.wikipedia.org/w/index.php?oldid=8293328
La franca kuiristo Marie-Antoine Carême (1784 - 1833), kiu kuiris por la partoprenantoj de la Kongreso de Vieno , verŝajne estas la inventisto de la rigida kuirista ĉapo. Li muntis rigidan paperan ringon en la interno de la ĉapo pro kio ĝi pli bone restas rekte staranta.;ropa;381472;Kuirista ĉapo;8293328;https://eo.wikipedia.org/w/index.php?oldid=8293328
La bereto, aŭ vasko ĉapo , estas la nacia kapvesto de Francio . En multaj armeoj la bereto estas uzata kiel kapvesto.;ropa;382309;Bereto;9067674;https://eo.wikipedia.org/w/index.php?oldid=9067674
Bireto ( latine biretum ) estas kapvesto de i.a. kristanaj ekleziuloj . Ĝi havas tri aŭ kvar pintojn kaj kelkfoje pomponon sur la supro. Tradicie la tripinta bireto estas portata de rom-katolikaj kaj kelkaj anglikanaj kaj luteranaj ekleziuloj. La kvarpinta bireto estas portata kiel universitata vestero dum doktoriĝo en pontifika fakultato aŭ pontifika universitato . Kelkfoje la bireto estas portata de advokatoj en kortumoj , ekzemple la advokatoj de la Manikinsuloj .;ropa;382780;Bireto;9049343;https://eo.wikipedia.org/w/index.php?oldid=9049343
Kaskedo estas ĉapo kun viziero el (artefarita) ŝtofo , ledo aŭ alia materialo . Ekde la komenco de la 19-a jarcento ĝi estis populara precipe inter junuloj kaj sportuloj pro ĝia praktika uzvaloro. La viziero povas ŝirmi la okulojn kontraŭ la sunlumo. Ĝi estas ofte uzata kapvesto en armeo , polico , fajrobrigado k.t.p., kutime kiel parto de ilia uniformo .;ropa;383037;Kaskedo;9047879;https://eo.wikipedia.org/w/index.php?oldid=9047879
La Gandhi-ĉapo ( hindie : गांधी टोपी) estas blanka ĉapo kun pintoj je la fronto kaj malfronto. Ĝi estas farita el khadi . La nomo venas de la hinda gvidanto Mahatma Gandhi , kiu unue popularigis ĝin dum la lukto por hindia sendependeco. Ĉefe portita de aktivuloj por la sendependeco de Hindio , ĝi fariĝis simbola tradicio por politikistoj kaj politikaj aktivuloj por porti ĝin en sendependa Hindio.;ropa;383049;Ghandi-ĉapo;8204269;https://eo.wikipedia.org/w/index.php?oldid=8204269
Kufo estas simpla kapvesto por virinoj. Plejofte ĝi estas farita el ŝtofo sed ankaŭ el aliaj materialoj. Kelkfoje ankaŭ viroj portas kufojn.;ropa;383224;Kufo;9037399;https://eo.wikipedia.org/w/index.php?oldid=9037399
Noktoĉemizo estas longa ĉemizo por knabinoj kaj virinoj . En antaŭaj jaroj ankaŭ viroj portis noktoĉemizon, sed nuntempe tio ne plu estas tre kutima.;ropa;400002;Noktoĉemizo;9104667;https://eo.wikipedia.org/w/index.php?oldid=9104667
Noktoĉapo estis uzata dum la mezepokoj por preventi la disvastiĝon de kapopedikoj kaj por varmteni la kapon dum la nokto. Dum la perukperiodo la noktoĉapo estis tre populara, ĉar oni sub la peruko ne havis harojn aŭ nur malmultajn kaj la peruko estis kovejo por pedikoj.;ropa;400177;Noktoĉapo;8389703;https://eo.wikipedia.org/w/index.php?oldid=8389703
"La pinta noktoĉapo plejofte estis rezervata por viroj: virinoj portis rondan ""dormĉapelon"", de kiu la ŝtofo estis faldita ĉirkaŭ la kapo.";ropa;400177;Noktoĉapo;8389703;https://eo.wikipedia.org/w/index.php?oldid=8389703
Mantilo ( hispane : mantilla ) estas hispana kapvualo , farita el punto . Ĝi estas parto de la hispana tradicia vestaro de la virino. Oni metas ĝin sur la kombilo ( hispane : peineta) kiu estas fiksita en la hartubero ( france : chignon).;ropa;436343;Mantilo;9088750;https://eo.wikipedia.org/w/index.php?oldid=9088750
En la vivo de Sankta Patricio el Muirĉu estas atribuata al la Druidoj la profetaĵo kiu volus ke venus iu “en sia domo surkape tarborita” klara referenco al la kazublo.;ropa;445700;Kazublo;9276203;https://eo.wikipedia.org/w/index.php?oldid=9276203
Per la termino “kazublo” oni difinis, krome, ĉelon loĝatan de monaĥo kaj malgrandan kamparan kapelon .;ropa;445700;Kazublo;9276203;https://eo.wikipedia.org/w/index.php?oldid=9276203
Laŭlonge de la tempo, la vesto, kiu origine atingis la kalkanojn el ĉiu partoj de la korpo, trafis ŝanĝojn en la longo sur la brakoj kaj kruroj. Ĝi fariĝis vesto duonrigida, subŝtofita, kiu ne kovris iujn perojn de la brakoj kaj bezonis esti vestita kun kunligoj ĉe la flankoj.;ropa;445700;Kazublo;9276203;https://eo.wikipedia.org/w/index.php?oldid=9276203
Ĉiukaze kazublo ricevis ŝanĝojn rilate mezurojn kaj formojn ankaŭ el la landaj kaj regionaj stiloj kaj foje – se resti en la katolikaj medioj – iuj novaĵoj estis riproĉataj kaj malpermesitaj de la romaj kongregacioj.;ropa;445700;Kazublo;9276203;https://eo.wikipedia.org/w/index.php?oldid=9276203
Krispo estas kolumo , simpla aŭ duobla, el plisita aŭ rondfaldetita amelita tolo aŭ punto , surmetita ĉefe de viroj, sed ne nur, en la 16-a kaj 17-a jarcentoj , kaj de virinoj en pli modernaj tempoj. Tiu ornamo de la kutima vesto, propra de altaj sociaj klasoj, videblas ankoraŭ en multnombraj pentraĵoj de monarkoj kaj gravuloj de la menciitaj jarcentoj. Ĉe rifinitaj skoloj pentrartaj , tiu detalo de la vestaro povas iĝi grava ero de la pentraĵo.;ropa;446285;Krispo;9031994;https://eo.wikipedia.org/w/index.php?oldid=9031994
Moderna aŭzajo havas du partojn: Alt- koluma longa robo, kaj pantalonoj, kiuj alvenas ĉe la maleoloj. Kostumo kaj vespera robo, aŭzajo, povas havi buntajn dezajnojn kun floroj, birdoj, kaj bildoj.;ropa;449502;Aŭzajo;9257818;https://eo.wikipedia.org/w/index.php?oldid=9257818
Burkino (= BURko + biKINO) estas banvesto speciale fasonita por islamaninoj . Ĝi lasas senkovrita nur la vizaĝon, la manojn kaj la piedojn.;ropa;457235;Burkino (vesto);9485251;https://eo.wikipedia.org/w/index.php?oldid=9485251
Kiel notis la instruisto de Dress Studies and Fashion Design (Studoj kaj modofasonado pri kostumoj), Heather Marie Akou, la argumentoj pri la burkino (kaj aliaj formoj de virina vesto) ne estas nur vesto, sed ankaŭ rilatas al la simbolismo, la supozoj kaj la politikaj implicaĵoj projektitaj sur ĝi. Diskutoj pri feminismo , enmigrado kaj sekularismo prezentas vaste malsamajn vidpunktojn pri la burkino kaj aliaj pudoraj bankostumoj . La burkino povas esti vidita kiel persona elekto, indikilo de virina subjugado, kapabligilo de pliiĝinta fizika libereco de virinoj aŭ religia simbolo kaj fundamentisma religia simbolo, kiu defias la sekularan ŝtaton.;ropa;457235;Burkino (vesto);9485251;https://eo.wikipedia.org/w/index.php?oldid=9485251
"La dekreto de la Valona registaro, la 13an de junio 2013 pri publikaj naĝejoj, deklaras ke ""dum publikaj malfermaj horoj, naĝantoj portas bankostumojn en naĝejoj, ekskluzive rezervitajn por tiu uzo, kongrua kun higieno"" ..";ropa;457235;Burkino (vesto);9485251;https://eo.wikipedia.org/w/index.php?oldid=9485251
La burkino ne maskas la vizaĝon, do ĝi ne malpermesatas laŭ la franca leĝo kontraŭ la integrala vualo ( nikabo ) ekvalidigita de post 2011 .;ropa;457235;Burkino (vesto);9485251;https://eo.wikipedia.org/w/index.php?oldid=9485251
Iuj starigis paralelojn inter la burkino-malpermeso kaj la franca socia malakcepto de la katolika sutano post la franca leĝo de 1905 pri la leĝo malkuniganta eklezion kaj ŝtaton .;ropa;457235;Burkino (vesto);9485251;https://eo.wikipedia.org/w/index.php?oldid=9485251
La Kolekto Philippi riĉe arigas ĉapelojn de diversaj kredoj , religioj kaj spiritecoj kaj aliajn objektojn kompletigantajn la vestaron de religianoj.;ropa;458197;Kolekto Philippi;9421536;https://eo.wikipedia.org/w/index.php?oldid=9421536
La Kolekto Philippi estas privata muzeaĵaro kolektita de Dieter Philippi el Saarbrücken . Inter la grave interesaj pecoj troviĝas pli ol kvincento da vertoĉapoj uzitaj aŭ uzataj en la kristana , islama juda , kaoda , budhana , ŝintoisma medioj, kaj de la Libera Eklezio , de Sufismo , Anabaptismo kaj de aliaj religiaj komunumoj.;ropa;458197;Kolekto Philippi;9421536;https://eo.wikipedia.org/w/index.php?oldid=9421536
Al la kolekto apartenas, krome, 52 ŝnuregoj por episkopa kruco, parte produktitaj en luksega pasamentaro manlaborita. En la Katolika Eklezio papoj, kardinaloj kaj episkopoj alkroĉas sian brustan krucon al tiaj ŝnuregoj.;ropa;458197;Kolekto Philippi;9421536;https://eo.wikipedia.org/w/index.php?oldid=9421536
"Frako estas vesto por viroj. La frako estas tradicie portata dum formalaj festoj kaj ceremoniaj okazintaĵoj. Kutime oni mencias sur la invitilo, kie oni atendas ke la sinjoro aperas en frako, la vortojn ""cravate blanche/white tie"" (blanka kravato) aŭ ""décorations"" (dekoracioj). Sinjorinoj tiam aperas en longa robo . La frako estas la plej ŝika vesto kiun viro povas porti. Ĝi konsistas el jako kun du akraj pintoj kaj longaj baskoj , amelita ĉemizo kun staranta kolumo , blanka veŝto , blanka bantokravato kaj nigra pantalono kun unuopa aŭ duobla galono je ambaŭ flankoj.";ropa;459464;Frako;9092913;https://eo.wikipedia.org/w/index.php?oldid=9092913
Bantokravato , bantkravato , papilia kravato aŭ papilikravato estas tipo de kravato kiu estas portata ĉefe de viroj sur la kolumo de ĉemizo . La nomo venas de la formo, kiu similas al banto aŭ papilio (kiel ŝunodo ).;ropa;459553;Bantokravato;9120433;https://eo.wikipedia.org/w/index.php?oldid=9120433
"La bantokravato konsistas el rubando (ofte farita el silko , sed ankaŭ el katono aŭ poliestero ) kiu estas bantigita ĉirkaŭ la kolo. La rubando finas je ambaŭ ekstremaĵoj en banto, pro kio la kravato post la bantigo ekhavas papiliecan formon. Nuntempe oni portas ĝin ĉefe dum paradaj festoj : nigra ĉe smokingo (""nigra kravato"") kaj blanka ĉe frako (""blanka kravato""). Unuopuloj portas bantokravaton ĉe kompletoj ĉe kiu la plejmulto de la sinjoroj portas normalan kravaton .";ropa;459553;Bantokravato;9120433;https://eo.wikipedia.org/w/index.php?oldid=9120433
Konataj portantoj kaj portintoj de bantokravato estas Albert Schweitzer , Winston Churchill kaj Elio Di Rupo;ropa;459553;Bantokravato;9120433;https://eo.wikipedia.org/w/index.php?oldid=9120433
Banto estas ornamcela nodo en rubando aŭ alia aĵo. Ĝi havas du buklojn. Fakte ĝi estas sama kiel ŝunodo . Ĝi estas uzata interalie en bantokravato kaj hararanĝoj . Knabinoj havis dum la 1950-aj jaroj ofte banton en la hararo (harbanto ).;ropa;459936;Banto;9459966;https://eo.wikipedia.org/w/index.php?oldid=9459966
La ŝtofo estas nomita laŭ sia triobla fadeno. La nomo venas el la malnova altgermana lingvo kaj retroiras ĝis la latina trilix ‚triobla‘ kaj licium ‚fadeno‘.;ropa;460682;Dreliko;9201196;https://eo.wikipedia.org/w/index.php?oldid=9201196
La Carrera jeans estis fondita en 1965 en Provinco de Verono (Italio) de Imerio Tacchella kaj du liaj fratoj: Tito kaj Dominico . Por pliproksimigi produktejojn al la krudaĵ-bazejoj oni transportis ilin al la ĉefaj landoj de kotonproduktado. Tamen loĝistiko kaj fina kontrolo okazas en Italio, por ke la kompanio povu garantii sufiĉan kvaliton de varoj.;ropa;475462;Carrera Jeans;8983466;https://eo.wikipedia.org/w/index.php?oldid=8983466
La ĉefa uzino situas en Taĝikio — unu el la plej gravaj landoj de kotonproduktado, ĝin posedas Carrera 93-procente kaj la Monda Banko — 7-procente .;ropa;475462;Carrera Jeans;8983466;https://eo.wikipedia.org/w/index.php?oldid=8983466
En la kristanaj eklezioj , la klerikaj vestoj estas tiuj vestoj ĝenerale surmetitaj de ekleziuloj, kiam ili ne celebras la liturgiajn ritojn .;ropa;478975;Klerikaj vestoj;9255449;https://eo.wikipedia.org/w/index.php?oldid=9255449
La nereligiaj vestoj de luteranoj kutime imitas tiujn katolikajn: sutano aŭ klerika ĉemizo kaj kolringo malfiksebla kaj episkopo, laŭvole, rajtas surmeti brustan krucon .;ropa;478975;Klerikaj vestoj;9255449;https://eo.wikipedia.org/w/index.php?oldid=9255449
Per la termino Galera ĉapelo oni intencas, en la Katolika Eklezio , grandan prelatan ĉapelon (nekonfuzendan, tamen ,kun la ĉapelo saturno ). Ekzistas du formoj de galera ĉapelo: unu versio, provizita per ŝnuro funkcianta kiel gorĝobendo kaj per certa nombro de kvastoj kiuj malsupreniras flanken, surmetebla de episkopoj , ĉefepiskopoj , patriarkoj kaj kardinaloj (ĉiu kategorio montras apartan strukturon), kaj unu versio ekskluzive kardinala, nesurmetebla, dotita per tre larĝa randaĵo, kiel akcesoraĵo, sendependa kaj deigita de grupo de kvastoj.;ropa;479230;Galera ĉapelo;8198304;https://eo.wikipedia.org/w/index.php?oldid=8198304
Ĝis la epoko de papo Johano la 23-a , la kardinaloj ricevis la galeran ĉapelon de la manoj de la papo mem la tagon de publika surmeto . Hodiaŭ sur la kapo de la ĵus proklamita kardinalo estas surmetitaj nur la Vertoĉapo kaj la bireto n.;ropa;479230;Galera ĉapelo;8198304;https://eo.wikipedia.org/w/index.php?oldid=8198304
Korsaĵo historie, estas virina vestaĵo , kovranta la korpon inter la kolo kaj la talio , kaj portata supre de la bluzo .;ropa;490290;Korsaĵo;9164383;https://eo.wikipedia.org/w/index.php?oldid=9164383
En moderna uzokutimo, korsaĵo tipe rilatas al supra vesto kiu havas forpreneblajn manikojn aŭ neniujn manikojn, ofte dekoltita, portita en Eŭropo de la 16-a jarcento ĝis la 18-a jarcento, aŭ super korseto aŭ en loko de ĝi. Por atingi modan formon kaj apogi la buston , la korsaĵo estis ofte rigidigita kun bastonetoj el speco de kano , aŭ balenosto . La korsaĵo estis diferenca de la tiutempa korseto ĉar ĝi estis portata super la alia vesto. En la fruaj periodoj, korsaĵoj kaj korsetoj estis ligitaj en helikforma maniero, kun unu kontinua laĉo. En pli postaj periodoj, ambaŭ estis ligitaj kiel la moderna tenisa ŝuo, kun hokingoj alfrontantaj unu la alian. Tio estis pli oportuna por virinoj kiuj povis vestiĝi sen helpo.;ropa;490290;Korsaĵo;9164383;https://eo.wikipedia.org/w/index.php?oldid=9164383
Dekoltaĵo estas maniero, kiel vesto ĉirkaŭ kaj malsupre de la kolo estas tajlita aŭ dekoltita . La formo de dekoltaĵo dependas de la sekso de la vestanto kaj de la modo .;ropa;490396;Dekoltaĵo;9122531;https://eo.wikipedia.org/w/index.php?oldid=9122531
La dekoltaĵo varias laŭ la modo, la sezono kaj la loko. Por virinaj vestoj ĝi estas tre varia, ĝi povas esti ronda, v-forma, kvadrata aŭ boatforma.;ropa;490396;Dekoltaĵo;9122531;https://eo.wikipedia.org/w/index.php?oldid=9122531
La lernejo prosperis kaj transvivis la abolon de la monaĥejoj far imperiestro Jozefo la 2-a , ĉar ĝi havis socian celon. En 1860 estis 400 lernantinoj kiuj lernis la faradon de pluraj specoj de punto.;ropa;513093;Kantcentrum;9057561;https://eo.wikipedia.org/w/index.php?oldid=9057561
Dum la Unua Mondmilito la lernejo estis fermita, ĝi remalfermiĝis nur en 1922. Estis tuja sukceso kaj estis konstruita nova lernejo. Ekde 1930 estis malprogreso pro la apero de maŝinfarita punto. La dumtaga instruado reduktiĝis al unu tago semajne kaj en 1961 eĉ tute ĉesis. La vesperaj kursoj restis. La Kantcentrum fondiĝis en 1970 por transpreni la taskojn de la Apostolinoj.;ropa;513093;Kantcentrum;9057561;https://eo.wikipedia.org/w/index.php?oldid=9057561
Origine ĝi troviĝis apud la Jerusalempre ĝejo en dometoj en kiuj antaŭ duona jarcento ankoraŭ loĝis puntofaristinoj. Je la 19-a de septembro 2014 nova punto-centro malfermiĝis je Balstraat 16, en la renovigita kantlernejo de la Apostolinoj. Dum la tuta jaro okazas kursoj kaj estas daŭra ekspozicio. Aldone estas dumtempaj ekspozicioj kun punto el en- kaj eksterlandaj kolektoj.;ropa;513093;Kantcentrum;9057561;https://eo.wikipedia.org/w/index.php?oldid=9057561
Laborvesto estas speciala vesto kiu plifaciligas aŭ plisekurigas la laboron, aŭ estas devigata de la aŭtoritato, aŭ estas portata de dungitoj de kompanio por havi dignan kaj karakterizan aspekton.;ropa;515442;Laborvesto;9118306;https://eo.wikipedia.org/w/index.php?oldid=9118306
Ofte higieno kaj sekureco estas la plej gravaj kaŭzoj por surhavi laborveston. Aliaj kialoj povas esti, leĝdonado kaj kompania imago. Uniformo ankaŭ povas esti formo de laborvesto.;ropa;515442;Laborvesto;9118306;https://eo.wikipedia.org/w/index.php?oldid=9118306
Kunkudro estas la junto kie du aŭ pli da tavoloj de teksaĵo , ledo , aŭ aliaj materialoj estas kuntenataj per kudreroj . Antaŭ la invento de la kudromaŝino , ĉiu kudrado estis farita permane. En modernaj amasproduktitaj teksaĵoj kunkudroj estas farataj per komputilregataj maŝinoj .;ropa;515748;Kunkudro;8293555;https://eo.wikipedia.org/w/index.php?oldid=8293555
Antaŭe nilonaj ŝtrumpoj havis, je la dorsa flanko, kunkudron kiu post longtempa foresto nuntempe (2015) estas laŭmodaj.;ropa;515748;Kunkudro;8293555;https://eo.wikipedia.org/w/index.php?oldid=8293555
La naskiĝdomo de Levi Strauss konstruiĝis ĉ. la jaro 1786. Iam temis pri unuetaĝa framskeleta domo . Ne antaŭ la 19-a jarcento ĝi akiris sian unuan aspekton. Strauss loĝis tie kun la gepatroj kaj ses gefratoj teretaĝe. Ekde la jaro 2000 ĝi gastigas la intertempe plurfoje premiitan Strauss-muzeon. En la 2011-a jaro povis esti finita la pligrandigo de la flanka domo (Marktstraße nr 31). De tiam la muzeo havas ankaŭ salonoj por nedaŭraj ekspozicioj kaj specialaj kunvenoj, kafejon kaj butiketon.;ropa;526064;Muzeo Levi Strauss;9389302;https://eo.wikipedia.org/w/index.php?oldid=9389302
La tajloristo Jacob Davis el Reno (Nevado) unuafoje plisolidigis per nitoj la de li kudritajn pantalonojn ĉe lokoj tre uzitaj. Por tiu ĉi inventitaĵo li akiris, kune kun Strauss, patenton. Je la komenco ili estis simpla laborvestaĵo por la homoj de la usona okcidento, sed baldaŭ la ĝinzoj iĝis la plej ofte portataj pantalonoj en la tuta mondo.;ropa;526064;Muzeo Levi Strauss;9389302;https://eo.wikipedia.org/w/index.php?oldid=9389302
"En la muzea butiko situanta en la nova apuda konstruaĵero troviĝas ĉiam la aktualaj kolektoj da specimenoj de ĝinzoj . Specialaĵo estas ""Levi's Vintage Clothing"", reproduktitaj ĝinzoj laŭ la originalaj modeloj. Ili alie en Germanlando nur en malmultaj vendejoj haveblas. Okazas krome diversaj nedaŭraj ekspozicioj pri la temo de vakeraj pantalonoj. Por grupoj (lernantoj, aliaj) ekzistas aparta pedagogia seminariejo.";ropa;526064;Muzeo Levi Strauss;9389302;https://eo.wikipedia.org/w/index.php?oldid=9389302
En tiu ĉi artikolo estas uzita traduko de teksto el vikipedia artikolo Museum Levi Strauss latine .;ropa;526064;Muzeo Levi Strauss;9389302;https://eo.wikipedia.org/w/index.php?oldid=9389302
Kutimo estas fliki truon en vesto per surkudrado aŭ surgluado de peco da ŝtofo sur la ŝirita loko. Ankaŭ eblas fliki biciklan tubon, tegmenton aŭ fliki poton aŭ kaldronon per lutado aŭ nitado de peco da metalo sur la truo.;ropa;531544;Flikaĵo;9114497;https://eo.wikipedia.org/w/index.php?oldid=9114497
André Courrèges (naskiĝis la 9-an de marto 1923 (nun 1923-03-09 ) en Pau , mortis la 7-an de januaro 2016 en Neuilly-sur-Seine ) estis franca modkreanto, konata pro sia futurisma dezajno. 25-jaraĝa, post studado de civila inĝenierado, li iris al Parizo por labori en la mod-entrepreno de Jeanne Lafaurie . Kelkajn monatojn poste, li iris al Balenciaga .;ropa;546734;André Courrèges;9324767;https://eo.wikipedia.org/w/index.php?oldid=9324767
Courrèges estis influita de moderna arĥitekturo, teĥnologio, novaj ŝtofoj, modernismo kaj futurismo en Arto kaj Dezajno. Pluraj dezajnistoj faris similan aserton, inkluzive Coco Chanel , kiu kunlaboris kaj konis multajn modernajn artistojn, kaj Mary Quant , kies kariero iamaniere paralelis tiun de Courrèges (ambaŭ Courrèges kaj Mary Quant faris postulon pri la invento de la minijupo ). Courrèges puŝis siajn ideojn ĝis la ekstremo, produktante iujn el la plej distingaj dezajnoj de la 20-a jarcento.;ropa;546734;André Courrèges;9324767;https://eo.wikipedia.org/w/index.php?oldid=9324767
Ĝi estas tradicia azia kapvesto, kiu speciale en Vjetnamio estas tre populara. La vjetnama Nón lá estas vera varomarko por Vjetnamio. Ĝia nomo estas satkat koree.;ropa;547771;Azia konusa ĉapelo;7944226;https://eo.wikipedia.org/w/index.php?oldid=7944226
La ĉapelo plejofte estas farita el rizo pajlo aŭ palmfolioj kaj per el ŝtofo farita mentonzono fiksata al la kapo. Antaŭ ĉio ĝi servas kiel protekto kontraŭ pluvo kaj forta sunradiado.;ropa;547771;Azia konusa ĉapelo;7944226;https://eo.wikipedia.org/w/index.php?oldid=7944226
Jupotago ( nederlande rokjesdag ), estas la unua tago de la printempo kiam ĉiuj virinoj kaj knabinoj subite surhavas jupon kun sub ĝi nudaj gamboj . Grandioza kaj belega tago laŭ Martin Bril . kiu estis nederlanda verkisto kaj kolumnisto, kiu popularigis la terminon “rokjesdag” en Nederlando . Pro Martin Bril la difino “rokjesdag” nuntempe estas sufiĉe konata en Nederlando.;ropa;555326;Jupotago;8258280;https://eo.wikipedia.org/w/index.php?oldid=8258280
Krinolino estas vestero kiun virinoj portis ekde la 1840-aj jaroj por doni al la formo de la vestaro daŭre pli vastan (' virinecan ') sonorilformon .;ropa;555546;Krinolino;9022008;https://eo.wikipedia.org/w/index.php?oldid=9022008
Dum la kuro de la 19-a jarcento la modo por la virino daŭre fariĝis pli mallarĝa kaj streĉa en la talio kaj pli vasta je la suba parto. Por realigi tion pli kaj pli da subjupoj estis portataj, sed post longa tempo tio fariĝis tre peza. La solvo estis la krinolino, kiu estis portata sub la jupoj. Krinolino estis kaĝokonstrua ĵo el suplaj ŝtalaj ringegoj kiuj de la talio pendis suben. Tio estis pli malpeza ol la antaŭaj subjupoj, kiuj havis la saman funkcion. Tial ili povis havi enormajn proporciojn. Grandaj kvantoj da ŝtofo, plenaj de rubandoj , bantoj , strioj kaj floroj necesis por aspekti laŭmoda. En tia jupo kelkfoje oni uzis 40 metrojn da ŝtofo.;ropa;555546;Krinolino;9022008;https://eo.wikipedia.org/w/index.php?oldid=9022008
Brilaĵeto estas disketforma ornamaĵo sur vestoj . Nuntempe ili precipe estas faritaj el plasto , sed en pasinteco ili konsistis el maldika disketo de perlamoto , arĝento aux briloraĵo . Ankaŭ etaj butonoj , konkoj , speguloj kaj moneroj povas utili kiel brilaĵeto.;ropa;560632;Brilaĵeto;7954144;https://eo.wikipedia.org/w/index.php?oldid=7954144
Brilaĵetoj havas trueton en la mezo, sed ankaŭ estas rondaj ornamaĵoj kun trueto je unu flanko. Tiuj povas esti fiksataj tiel, ke ili facile povas moviĝi, tial la brilanta efiko pligrandiĝas. Pro la sama kaŭzo kelkaj brilaĵetoj estas faritaj kun facetoj .;ropa;560632;Brilaĵeto;7954144;https://eo.wikipedia.org/w/index.php?oldid=7954144
Estas indikoj ke oraj brilaĵetoj jam estis uzataj en 2500 a.K por ornami vestojn kaj aliaj objektoj en la Indusa valo .;ropa;560632;Brilaĵeto;7954144;https://eo.wikipedia.org/w/index.php?oldid=7954144
"Dirndelo , Dirndlo aŭ Dirndl ( germane Dirndl , bavare Dirndel, Diandl, Dirndl, Deandl aŭ plenforme Diandlgwand ) estas speco de tradicia robo portata en Germanio , aparte en Bavario , Aŭstrio kaj Sudtirolo , surbaze de la tradicia vestaĵo de Alpaj kamparanoj. Robo, kiu estas loze bazita sur la dirndelo estas konata kiel Landhausmode (""land-inspira modo"").";ropa;564062;Dirndelo;9153604;https://eo.wikipedia.org/w/index.php?oldid=9153604
Dirndelojupo ĝenerale signifas malpezan cirkle tonditan robon, kolektitan ĉe la talio, kaj kiu kovras ĝis la genuoj .;ropa;564062;Dirndelo;9153604;https://eo.wikipedia.org/w/index.php?oldid=9153604
Gravas distingi inter dirndelo kaj tipa regiona robo. Regionaj kostumoj havas karakterizaĵojn de kio oni povas distingi el kiu regiono ĝi venas kaj en kelkaj kazoj eĉ la socian pozicion de la persono kiu uzas ĝin. Dirndelo hodiaŭ havas regionajn karakterizojn, sed estas evidente influita de la bavaraj kostumoj. Sekve, eĉ ekzistas diferenco inter tipa dirndelo (vestita kun antaŭtuko kaj, kutime, tradiciaj broditaj ŝtofoj) kaj kampa dirndelo, kiu estas lina robo.;ropa;564062;Dirndelo;9153604;https://eo.wikipedia.org/w/index.php?oldid=9153604
La vintrostila dirndelo havas pezajn, varmajn jupojn kaj antaŭtukojn faritajn el dika kotono , linaĵo , veluro aŭ lano , kaj longaj manikoj . La koloroj estas kutime riĉaj kaj malhelaj. La somera stilo estas pli malpeza kaj pli malkaŝanta, havas mallongajn manikojn, kaj ofte estas farita el malpeza kotono .;ropa;564062;Dirndelo;9153604;https://eo.wikipedia.org/w/index.php?oldid=9153604
Akcesoraĵoj povas inkludi longan antaŭtukon ligitan je la talio, veŝton aŭ lanan ŝalon. En multaj regionoj, aparte la pejzaĝo Ausseerland ĉirkaŭ la urbo Bad Aussee , vigla-koloraj, man-presitaj silkaj koltukoj kaj silkantaŭtukoj estas uzataj. Kiel dekoraĵojn virinoj ofte elmontras kolĉenojn, orelringojn kaj broĉojn faritajn el arĝento, la kornaro de cervoj aŭ eĉ de bestaj dentoj. Por pli malvarma vetero ekzistas pezaj dirndelomanteloj en la sama tranĉo kiel la roboj, kun alta kolo kaj frontaj butonoj, dikaj mufgantoj kaj lanĉapeloj.;ropa;564062;Dirndelo;9153604;https://eo.wikipedia.org/w/index.php?oldid=9153604
"Moderna dirndelo venis al la urbo Munkeno , ĉefurbo de Bavario , tra la vendejo "" Wallach "" de judaj posedantoj. Tie vestitaj per buntaj ŝtofoj kaj ĉeftemoj estis venditaj, kion lin valoris kiel granda rekono al liaj mastroj, Julius kaj Moritz Wallach. Krom manipuli la vendon, la Wallach-fratoj ankaŭ organizis defiladojn en Oktobrofesto kaj operojn en la teatro "" Im Weißen Rössl "", kiu helpis popularigi robon inter la altaj klasoj de la urbo. Nomo Wallach iĝis sinonimo de la moderna dirndelo en Munkeno kaj Supra Bavario . En 1939 ili estis devigitaj vendi sian komercon pro la politiko de Hitler . Tamen, post la milito ili sukcesis rekuperi.";ropa;564062;Dirndelo;9153604;https://eo.wikipedia.org/w/index.php?oldid=9153604
Ekde la Germana reunuiĝo , dirndelo denove populariĝis kaj povas esti vidita ĉie en Bavario kaj Aŭstrio . Hodiaŭ, portado de dirndelo estas rigardata kiel simbolo de la regiona fiero, kvankam multaj virinoj uzas ĝin ĉar ili trovas ĝin erotika, ĉar ĝi akcentas inajn kurbojn kaj kaŝas ĉian neperfektaĵon.;ropa;564062;Dirndelo;9153604;https://eo.wikipedia.org/w/index.php?oldid=9153604
En Aŭstrio kaj Bavario, dirndelo ofte povas esti vidita sur virinoj laborantaj en turismo-rilataj entreprenoj, kaj foje servistinoj en tradici-stilaj restoracioj aŭ bierĝardenoj . Ĝi ankaŭ vidiĝas en tiuj regionoj sur virinoj en la folklormuzika komerco.;ropa;564062;Dirndelo;9153604;https://eo.wikipedia.org/w/index.php?oldid=9153604
La Couturière Parisienne , aliflanke, asertas ke Lederhosen estis origine ne ekskluzive bavara vestaĵo sed estis uzataj ĉie en Eŭropo, precipe fare de rajdantoj , ĉasistoj , kaj aliaj homoj subĉiele aktivaj. La fronta, malsupren falda klapo eble estis unika bavara invento. La frontklapa stilo iĝis tiel populara en la 18-a jarcento ke ĝi estis konata en Francio kiel à la-bavaroise (en la bavara stilo) .;ropa;564188;Lederhosen;9201210;https://eo.wikipedia.org/w/index.php?oldid=9201210
Lederhosen restis regione popularaj kaj estas populare asociitaj kun vireco kaj galantino. Kelkaj viroj uzas ilin ĝardenume, migrante, laborante ekstere, aŭ partoprenante folklorfestivalojn aŭ bierĝardenojn . Ili estas simbolo de regiona fiero pri Bavario kaj la aliaj areoj kie ili estas uzataj, sed malofte vidiĝas aliloke. La rolo de Lederhosen en Bavario estas tiel komparebla al tiu de la kilto en Skotlando kaj la vakera ĉapelo en Usono .;ropa;564188;Lederhosen;9201210;https://eo.wikipedia.org/w/index.php?oldid=9201210
Germanaj knaboj kutimis uzi Lederhosen ĝis la aĝo de proksimume dek ses jaroj. Tiuj Lederhosen ne estis ornamitaj kun brodaĵo sed havis la tipajn atributojn de ŝelkoj kaj antaŭa klapo. Eĉ hodiaŭ, kelkaj germanaj kaj francaj skoltoj eluzas diversajn formojn de Lederhosen , kvankam en la plej multaj kazoj ili ne estas parto de ilia oficiala uniformo.;ropa;564188;Lederhosen;9201210;https://eo.wikipedia.org/w/index.php?oldid=9201210
Lederhosen ankaŭ estis uzataj fare de aŭstraj knaboj de la 1930-aj jaroj ĝis la 1970-aj jaroj. Hodiaŭ ili estas uzataj dum specialaj okazoj, kiel ekzemple bierĝardenoj aŭ biertendaj festoj. Knabinoj portas dirndlon , kio estas parto de la aŭstra Tracht .;ropa;564188;Lederhosen;9201210;https://eo.wikipedia.org/w/index.php?oldid=9201210
Gamsbart ( germane Gamsbart , laŭlitere ĉamobarbo, plurale Gamsbärte ) estas hartufo tradicie eluzita kiel ornamado sur Trachten -ĉapeloj en la alpaj regionoj de Aŭstrio kaj Bavario .;ropa;564280;Gamsbart;8198489;https://eo.wikipedia.org/w/index.php?oldid=8198489
Origine eluzita kiel ĉastrofeo kaj farita ekskluzive el hararo de la malsupra kolo de la ĉamo , Gamsbärte hodiaŭ estas produktitaj el la grandskalo de la hararo de diversaj bestoj kaj estas ofte venditaj fare de specialiĝintaj komercistoj kaj ankaŭ en suvenirobutikoj .;ropa;564280;Gamsbart;8198489;https://eo.wikipedia.org/w/index.php?oldid=8198489
Tradicie, Gamsbärte estas ekskluzive poziciigitaj sur ĉapelojn eluzitajn fare de viroj, tamen, lastatempaj evoluoj en dirndlo -modo vidis Gamsbärte aldonitan al diversaj lokoj sur virinaj vestoj.;ropa;564280;Gamsbart;8198489;https://eo.wikipedia.org/w/index.php?oldid=8198489
La juda ĉapelo aŭ flava ĉapelo (en germana lingvo : Judenhut , aŭ en latino pilleus cornutus ) estis ĉapelo konusforma , ofte blanka aŭ flava, uzata de la judoj dum la Mezepoko kaj en Eŭropo kaj en kelkaj partoj de la islama mondo . Komence ĝi estis volonte surportita. Ĝia uzo plifortiĝis en kelkaj regionoj de Eŭropo pro la 4-a Laterana Koncilio post 1215 por la judaj viroj kaj adoltuloj por distingiĝi unu de la aliaj. Samtempe ekaperis getoj en la urboj por ne miksiĝi kun la kristana loĝantaro. Ĝi similas al la friga ĉapo kaj verŝajne ĝi originas el la persa antaŭislama kulturo, kie la judoj de Babilono devis porti ĉapelon. La ĉapo estis uzata de la zoroastraj sacerdotoj en pentraĵoj de Dura Europos (sed ne en bildoj de tiutempaj sinagogoj ) kaj tre similas al kelkaj fruaj eŭropaj ekzempleroj, kiaj tiuj uzitaj de la roluloj de la Malnova Testamento en la reliefoj de la pordo San Zeno de Verono , ĉ. 1100.;ropa;573692;Juda ĉapelo;8257668;https://eo.wikipedia.org/w/index.php?oldid=8257668
Cycle Chic (ŝika biciklado) estas moda tendenco en popularaj grandaj urbaj centroj de Eŭropo kaj Norda Ameriko por stile bicikli en belaj, elegantaj vestaĵoj. Cycle chic estas asociata kun la urba bicikla kulturo en multaj urboj kie la loĝantoj altgrade biciklas kiel en Amsterdamo , Antverpeno , Berlino , Berno , Kopenhago . Kun la sistemo de pruntbicikloj urbanoj pli kaj pli malkovras ke ili povas vojaĝi en kaj ĉirkaŭ la urbo facile kaj eĉ en la urbocentro kie pli kaj pli ne plu eblas veturi per aŭtoj.;ropa;584467;Cycle chic;9151927;https://eo.wikipedia.org/w/index.php?oldid=9151927
"Originale ĉi tiu termino estis uzata de la dana kinoreĝisoro kaj fotisto Mikael Colville-Andrsena en la 2006 jaro kiu uzis ĝin por sia blogo Kopenhago Cycle Chic. La Brita ĵurnalo "" The Guardian "" publikigis pri ĝi sub la titolo Two wheels (d u radoj) . Ĉirkaŭ la tempo ke Mikael Colville-Andersen stampis la frazon Cycle Chic , la biciklo estis plejparte malaperinta el la publika konscio kiel transport-eblon, almenaŭ en multaj nordamerikaj urboj, kvankam tio ne estis la situacio en norda Eŭropo.";ropa;584467;Cycle chic;9151927;https://eo.wikipedia.org/w/index.php?oldid=9151927
En lastaj jardekoj (21a jarcento) la modo de dorsdekoltaĵo ŝanĝis ankaŭ la kutimojn esti fotita , kaj tiele ĉefe aktorinoj, modelulinoj kaj aliaj famulinoj ĉe eventoj montriĝas ne fronte kiel deĉiame sed flanke aŭ eĉ dorse por montri la belon aŭ kuraĝon de la dekoltita dorso. Plej aŭdacaj atingas ĝis la postaĵan fendeton.;ropa;585743;Dorsdekoltaĵo;8238895;https://eo.wikipedia.org/w/index.php?oldid=8238895
Edziniĝa robo plej ofte estas longa blanka (en multaj landoj) robo de edziniĝonta fianĉino , speciale aĉetita aŭ almezurita por la nupto . Multaj edziniĝontinoj preferas longan robon, ofte kombinata kun blanka ĝartero . Por kompletigi la tuton ŝi ofte portas vualon aŭ arte en la hararo plektitan ornamaĵon en la formo de floroj aŭ bantoj .;ropa;585749;Edziniĝa robo;9204753;https://eo.wikipedia.org/w/index.php?oldid=9204753
En multaj okazoj tiu vesto estas la plej kosta kaj plej ornama vesto kiun virino aĉetas, mendas aŭ preparas en la vivo. Ĝi estas celo de dum semajnoj (aŭ eĉ monatoj) la planoj de la paro, aŭ almenaŭ de la edziniĝonta. En kelkaj etosoj estas kutimo konsideri ke la edziĝonto ne devas vidi la edziniĝonton jam vestita nupte, ĉar tio alportos malbonan sorton. Ankaŭ en kelkaj etosoj, ĉar plej ofte tiu vesto estas uzata nur unu tagon en la vivo, ĝi povas esti heredata kaj aranĝita por venonta okazo. En okazoj de famulinoj, la informo pri la edziniĝa robo, kaj ĉefe se ĝi estas mendata al fama tajloro , estas celo de sekreto kaj de intereso por specifaj gazetoj .;ropa;585749;Edziniĝa robo;9204753;https://eo.wikipedia.org/w/index.php?oldid=9204753
"Duonbikino estas termino uzata por diversaj specoj de unupecaj sinjorinaj banvestoj kun kiel simileco ke ili ĉiuj estis inspiritaj je la bikino . En iuj naciaj lingvoj aperis alternativa vorto monokini - vorto internacia, sed ne esperanta. Ĝi traktas la parton bi- de bikini kvazaŭ prefikson kun la senco ""du"". Estas vera ke bi en ""bikino"" ne rilatas al la vorto du , sed tamen ĝi pensigas pri tio. Pro tio evidentas nomi la unupartan veston "" mono kini"". Sed en neniuj esperantaj vortaroj la imagebla vorto ""monokino"" dokumentiĝis, do en Esperanto la vorto estas 'duonbikino'.";ropa;585777;Duonbikino;9414921;https://eo.wikipedia.org/w/index.php?oldid=9414921
Miniĉemizo estas ofte facilportebla vestero , plej ofte por virinoj. Ĝi aperas en diversaj variantoj: kiel T-ĉemizo , subĉemizo , 'senŝelka' kaj 'senmanika' ĉemizo, kaj kelkfoje ĝi ankaŭ ne kovras la umbilikzonon .;ropa;586017;Miniĉemizo;6499649;https://eo.wikipedia.org/w/index.php?oldid=6499649
Dum la 1990-aj jaroj , sub influo de tendencinstigaj publikuloj kiel filmstelulinoj kaj popmuzikistoj (ekzemple Madonna ), ekestis la modotendenco por uzi vesterojn kiujn oni antaŭe konsideris kiel subvestoj ankaŭ porti kiel supervestoj. Miniĉemizoj el punto ne plu estis kovrataj per supervesto.;ropa;586017;Miniĉemizo;6499649;https://eo.wikipedia.org/w/index.php?oldid=6499649
Karbona tetra-klorido estas kemia kombinaĵo, senkolora volatila likvaĵo kun dolĉeca odoro, tre uzata kiel solvanto, en fajrestingiloj kaj kiel antaŭanto de glaciigagentoj kaj kiel purigagento. Karbona tetra-klorido estas venena kaj toksa substanco kiam absorbita tra haŭto aŭ inhalita.;ropa;590120;Karbona tetraklorido;9433576;https://eo.wikipedia.org/w/index.php?oldid=9433576
En 1907, Dow Chemical Company ekis larĝskalan komercan produktadon. Iom poste, la produkto ekuzis kiel domaj purigagentoj kaj en fajrestingiloj. Ĉirkaŭ 1914, la jara produktado de karbona kvarklorido estis proksimume 4500 metraj tunoj. Dum la 1-a Mondmilito la produktado de karbona kvarklorido forte ekspansiiĝis, ĉefe pro ĝia uzo en grajno-fumigado kaj en la kaŭĉuka industrio.;ropa;590120;Karbona tetraklorido;9433576;https://eo.wikipedia.org/w/index.php?oldid=9433576
Masonistdekoltaĵo estas eŭfemisma termino por videbla supra parto de la interglutea fendo super la pantalonzono.;ropa;591554;Masonistdekoltaĵo;8356703;https://eo.wikipedia.org/w/index.php?oldid=8356703
Magreca ĝinzo estas ĝinzo kiu estas tre strikta ĉirkaŭ la korpo kaj speciale la gamboj . Tiu modelo ankaŭ ekzistas el aliaj materialoj ol denimo . Plejofte ĝi konsistas el miksaĵo de denimo kaj kelkaj percentoj elastano , tiel eblas por ŝovi la malvastajn krurumojn trans la piedoj. Alia ebleco estas zipoj en la krurumoj.;ropa;594652;Magreca ĝinzo;9201215;https://eo.wikipedia.org/w/index.php?oldid=9201215
Dezipa pantalono estas pantalono , ĉe kiu la krurumoj (plejofte en la mezo) estas ligataj per zipoj . Tiel oni povas varii la longon. Ekzistas ankaŭ tipoj kun po krurumo du zipoj, tiel oni povas elekti la longon inter 1/1, 3/4 kaj 1/2. Aldone la dezipeblaj krurumoj povas havi zipon en laŭlonga direkto por plifacila sur- kaj demeto.;ropa;595327;Dezipa pantalono;9201216;https://eo.wikipedia.org/w/index.php?oldid=9201216
Bracelo de armaĵo estis la parto de lamenoj kiu utilis por ŝirmi la suprajn membrojn, el la ŝultro ĝis la mano .;ropa;597705;Bracelo;7953223;https://eo.wikipedia.org/w/index.php?oldid=7953223
Laŭ Francisko Azorín bracelo estas Peco de milita armuro, por ŝirmi la brakon. Kaj li aldonas kiel sinonimojn la terminojn brakobendo, brakostrio, kaj brakoŝirmilo. Kaj li indikas etimologion el latina brachiale el brachium (brako).;ropa;597705;Bracelo;7953223;https://eo.wikipedia.org/w/index.php?oldid=7953223
Mufo estas cilindroforma moda akcesoraĵo el pelto aŭ teksaĵo , kun du aperturoj por varmteni la manojn . La mufo aperis en la 16-a jarcento en la sinjorina modo kaj populariĝis kiel akcesoraĵo por virinoj kaj viroj dum la 17-a kaj 18-a jarcento . En la 19-a jarcento la mufo grandparte eksmodiĝis. Kiam ĝi dum la 1940-aj kaj 1950-aj jaroj mallonge reviviĝis tio estis nur en la sinjorina modo. La vorto deriviĝis de la malnovfranca mouffle (dika ganto), kio laŭvice deriviĝis de la mezepoka latina muffula .;ropa;610709;Mufo;9498080;https://eo.wikipedia.org/w/index.php?oldid=9498080
Mufo (mof) ankaŭ estas insultvorto por germano, kio antaŭ ĉio uzatas en Nederlando , kaj Flandrio .;ropa;610709;Mufo;9498080;https://eo.wikipedia.org/w/index.php?oldid=9498080
Noktoĉemizeto ( angle : Babydoll ) estas mallonga, kelkfoje senmanika, facile portebla noktoĉemizo aŭ negliĝo , celata kiel nokto vesto por virinoj. Ĝi kelkfoje havas antaŭformitajn mamingojn kun alkudrita, loze konvena jupo plej ofte kun longo ĝis la supra parto de la femuro .;ropa;613122;Noktoĉemizeto;8389704;https://eo.wikipedia.org/w/index.php?oldid=8389704
La kaftanoj kiuj estis portataj de la Otomanaj sultanoj apartenas al la plej valoraj kolektoj de la Topkapi-palaco en Istanbulo .;ropa;615854;Kaftano;9022331;https://eo.wikipedia.org/w/index.php?oldid=9022331
En la 6-a jarcento pro elmigrado de araboj al Nord-Afriko la kaftano ankaŭ disvastiĝis en landoj kiel Alĝerio , Maroko kaj Tunizio . Aliaj Nord-Afrikaj triboj transprenis la kaftanon kaj ĝi nun estas parto de la Nord-Afrika tradicia vestaro .;ropa;615854;Kaftano;9022331;https://eo.wikipedia.org/w/index.php?oldid=9022331
Nuntempe la kaftano ankaŭ estas portata de virinoj, en gajaj koloroj kun bidoj kaj brodaĵoj . Ĉiujare en diversaj arabaj landoj estas kaftanmodparadoj en kiuj konataj modistoj montras siajn kaftanojn.;ropa;615854;Kaftano;9022331;https://eo.wikipedia.org/w/index.php?oldid=9022331
Hunkemöller estas nederlanda subvesta butikĉeno . La ĉefoficejo estas en Hilversumo kaj la kompanio aktivas en Eŭropo, la Mezoriento, Nord-Afriko kaj la Nederlandaj Antiloj.;ropa;624693;Hunkemöller;9413399;https://eo.wikipedia.org/w/index.php?oldid=9413399
La kompanio fondiĝis en 1886 de la germana enmigranto Wilhelm Hunkemöller (1858-1927) kaj lia el Roermond devena edzino Josephina Lexis. En la komenco Hunkemöller produktis korsetojn en Amsterdamo. Dum la sekvaj jaroj, la familio Hunkemöller malfermis butikojn en Amsterdamo, Roterdamo kaj Utreĥto.;ropa;624693;Hunkemöller;9413399;https://eo.wikipedia.org/w/index.php?oldid=9413399
En 1974 la kompanio vendiĝis al Vendex -filia kompanio Confendex B.V. kaj ricevis la hodiaŭan nomon. Post la transpreno la plej grava celo fariĝis en- kaj eksterlanda ekspansio.;ropa;624693;Hunkemöller;9413399;https://eo.wikipedia.org/w/index.php?oldid=9413399
Hunkemöller pliposte vendiĝis al Maxeda . Maxeda siavice vendis la ĉenon en 2010 al la investgrupo PAI Partners . Per pli ol 400 butikoj la jara debito estis 200 milionoj da eŭroj. PAI pagis proksimume 265 milionoj da eŭroj. Post la transpreno la kompanio plivastiĝis al 700 establoj kun 2700 dungitoj. En decembro 2015 PAI vendis la subvestĉenon al la usona investfonduso Carlyle .;ropa;624693;Hunkemöller;9413399;https://eo.wikipedia.org/w/index.php?oldid=9413399
Je la fino de 2017 konatiĝis ke Hunkemöller volas transpreni ĉiujn establojn de sia nederlandaj franĉizoprenantoj.;ropa;624693;Hunkemöller;9413399;https://eo.wikipedia.org/w/index.php?oldid=9413399
"Ekde aŭgusto 2017 modelo Doutzen Kroes estas ""Marka Ambasadorino"". Sub la nomo Doutzen's Stories diversaj kolektoj estas lanĉitaj.";ropa;624693;Hunkemöller;9413399;https://eo.wikipedia.org/w/index.php?oldid=9413399
Levi Strauss & Co. (/ˈliːvaɪ ˈstraʊs/ LEE-vy STROWSS, prononcata en la angla: Levi Strauss ) estas usona teksaĵa firmao kiu specialiĝas pri produktado de vestaĵoj, ĉefe tiuj faritaj el ĝinzoŝtofo (denimo), sub la varomarko Levi's ˈliːvaɪzde (/ Levi / LEE-vyze) .;ropa;632691;Levi Strauss & Co.;9201217;https://eo.wikipedia.org/w/index.php?oldid=9201217
La firmao estis fondita en Usono en 1853 de german-juda enmigrinto nomita Levi Strauss , kiu enmigris al la Okcidenta marbordo de Usono por malfermi teksaĵbutikon kune kun sia frato.;ropa;632691;Levi Strauss & Co.;9201217;https://eo.wikipedia.org/w/index.php?oldid=9201217
La 26-an de septembro 1902 Strauss mortis en la aĝo de 73 jaroj. La nevoj de Strauss heredis la firmaon post lia morto.;ropa;632691;Levi Strauss & Co.;9201217;https://eo.wikipedia.org/w/index.php?oldid=9201217
Armani estas itala moddomo fondita fare de la dezajnisto Giorgio Armani . Armani estas varmako kiu specialiĝas pri altmoda moda dezajno kaj krom vestaĵoj ankaŭ produktas modajn akcesoraĵojn , juvelaĵojn , horloĝojn , ŝuojn , sunokulvitrojn , kosmetikaĵojn kaj meblojn .;ropa;633030;Armani;8983774;https://eo.wikipedia.org/w/index.php?oldid=8983774
En 1975 , Giorgio Armani, kune kun sia vivpartnero, Sergio Galotti, fondis la modfirmaon Giorgio Armani Sp. A en Milano , kun baza ĉefurbo de 10,000 dolaroj. En la unua jaro de sia fondo, la firmao okupiĝis pri la dezajno kaj produktado de viraj vestaĵoj nur, kaj poste komencis okupiĝi pri virina modo ankaŭ. La vestaĵoj de Armani por virinoj estis karakterizitaj per la fakto ke la inspiro por ilia dezajno estis bazita sur la viraj vestaĵoj kiujn li dizajnis kaj li eĉ uzis la samajn ŝtofojn kiuj estis uzitaj por la vestaĵoj de la viroj. La kostumoj, kiujn li desegnis por virinoj, estis en modesta eleganta stilo kaj elstaris pro sia preciza tranĉo kaj zorgema tajloro. En 1985 , Galotti mortis pro aidoso kaj Armani daŭre gvidis la firmaon sole.;ropa;633030;Armani;8983774;https://eo.wikipedia.org/w/index.php?oldid=8983774
Marko, kiu specialiĝas pri alta modo por viroj kaj virinoj kaj inkluzivas modajn akcesoraĵojn kaj parfumojn. La produktoj estas nur haveblaj en Armani-butikoj kaj elektitaj grandmagazenoj.;ropa;633030;Armani;8983774;https://eo.wikipedia.org/w/index.php?oldid=8983774
"Marko, kiu specialiĝas pri nuna kaj moderna modo je pli malmultekosta prezo. Ĝi estas la nura modlinio desegnita rekte de Giorgio Armani kaj la sola kiu ricevas atenton ĉe Milan Fashion Week dum la aliaj modlinioj de la varmako ne. La dezajnoj de la varmako estas venditaj en butikoj de Emporio Armani, elektitaj grandmagazenoj kaj en la retejo de la varmako. En januaro 2010 , la futbalisto Cristiano Ronaldo estis elektita por esti la vizaĝo de la varmako kune kun Megan Fox . En 2011 Fox estis anstataŭigita per Rihanna kaj komencante en 2017 la prezentisto de la firmao estas Shawn Mendes . Emporio Armani kunlaboris kun Reebok por krei modajn ŝuojn sub la nomo ""EA7"".";ropa;633030;Armani;8983774;https://eo.wikipedia.org/w/index.php?oldid=8983774
"Antaŭe nomita "" Giorgio Armani Le Collezioni "". Liaj dezajnoj estas pli multekostaj ol la varmako Emporio kaj pli malmultekostaj ol la varmako Giorgio Armani. Liaj dezajnoj temigas luksajn kostumojn. Estas nur du vendejoj en Milano kaj Parizo , kiuj nur vendas ĉi tiun varmakon. Krome, la varmako estas vendita en prestiĝaj grandmagazenoj kiel Holt Renfrew , Bloomingdales , David Jones, Harvey Nichols, Harrods , Neiman Marcus, Nordstrom kaj Saks Fifth Avenue. En 2012 , sporta modlinio nomita ""Armani Collezioni Active"" estis lanĉita.";ropa;633030;Armani;8983774;https://eo.wikipedia.org/w/index.php?oldid=8983774
Modlinio lanĉita en 1991 en Usono . Ĉi tiu linio inspiras en danca muziko kaj estas konata pro siaj provokaj reklamaj kampanjoj. Ĉi tiu estas la plej alirebla modlinio inter Armani-modlinioj. La produktoj de ĉi tiu linio estas haveblaj en 264 butikoj tra 31 landoj kaj en la retejo de la varmako.;ropa;633030;Armani;8983774;https://eo.wikipedia.org/w/index.php?oldid=8983774
Marko de la kompanio Armani havebla en 15 vendejoj tutmonde kaj en individuaj vendejoj de Neiman Marcus kaj Saks Fifth Avenue .;ropa;633030;Armani;8983774;https://eo.wikipedia.org/w/index.php?oldid=8983774
La produktserio de Armani por la hejmo, kiu inkluzivas meblojn, lampojn, litaĵojn kaj manĝilaron.Havebla en 40 vendejoj tutmonde kaj en la grandmagazeno Neiman Marcus.;ropa;633030;Armani;8983774;https://eo.wikipedia.org/w/index.php?oldid=8983774
"La belecmarko de Giorgio Armani inkluzive de kosmetikaĵoj, haŭtzorgado, kolonio kaj parfumo. Ĝi estas produktita kaj distribuita kiel la luksa modlinio de la varmako L'Oréal . Havebla en multaj grandmagazenoj tra la mondo. Armani ankaŭ estas engaĝita en la manĝkomerco kun 14 kafejoj sub la nomo Emporio Armani kaj Armani Jeans. Ekzistas ankaŭ drinkejo nomita laŭ armeno en Honkongo kaj du restoracioj. Armani ankaŭ havas librovendejon nomitan ""Armani Libri"" kaj florbutikon, ""Armani Fiori"".";ropa;633030;Armani;8983774;https://eo.wikipedia.org/w/index.php?oldid=8983774
"Giorgio Armani subskribis interkonsenton kun la firmao ""Emaar Properties"" en 2004 por konstrui almenaŭ sep luksajn hotelojn kaj 3 feriejojn portantajn la Armani-nomon. La kompanio Armani respondecos pri la dezajno. Unu el la hoteloj lokiĝas en Dubajo . La unuaj 39 etaĝoj de la Burj Khalifa turo estas dediĉitaj al Armani-hotelo enhavanta 160 ĉambrojn kaj seriojn kaj 144 loĝejojn. La hotelo malfermiĝis la 27-an de aprilo 2010 .";ropa;633030;Armani;8983774;https://eo.wikipedia.org/w/index.php?oldid=8983774
Startinte per puntaĵoj, la desegnoj de Schiaparelli estis tre forte influitaj de superrealistoj kiel siaj kunlaborantoj Salvador Dalí kaj Jean Cocteau . Ŝiaj klientoj estis ekzemple la heredantino Daisy Fellowes kaj la aktorino Mae West . Schiaparelli ne adaptiĝis al ŝanĝoj en modo post la Dua mondmilito kaj ŝia modentrepreno fermiĝis en 1954.;ropa;633460;Elsa Schiaparelli;9490142;https://eo.wikipedia.org/w/index.php?oldid=9490142
La abajo estas tradicia virina vesto , kiu estas portata super la aliaj vestaĵoj en kelkaj mezorientaj , islamaj landoj kaj areoj, ĉefe en Sauda Arabio , en la landoj de Persa Golfo kaj de Magrebo . La tradicia abajo estas nigra kaj povas konsisti el granda kvadrato el tolo, kiu kovras ankaŭ la ŝultrojn sed ne la kapon, kiel longa kaftano . Abajo kovras la tutan korpon escepte la kapon, la piedojn kaj la manojn. Ĝi povas esti uzata kun nikabo , nome vualo por la vizaĝo, kiu kovras ĉion escepte la okulojn. Kelkaj virinoj uzas ankaŭ nigrajn longajn gantojn , kaj tiel ankaŭ iliaj manoj estas kovritaj.;ropa;635258;Abajo;9485202;https://eo.wikipedia.org/w/index.php?oldid=9485202
Litamo (en araba: لِثَام‎, translit. DMG lithām , foje prononcita lifam), ankaŭ alilingve litham aŭ tagelmust ( Tamasheq , la homoj de la vualo) sed ankaŭ konata kiel cheich (ŝeŝo) aŭ en la berbera : cheche aŭ éghéwed ) estas lin-vestaĵo el tinkturita kotono ofte en indigo sed nuntempe ankaŭ en aliaj koloroj, iel inter turbano kaj vualo . Ĝin portas tuaregaj kaj beduenaj viroj en diversaj ŝtatoj de okcidentafriko kaj povas esti ĝis 15 metroj longa .;ropa;638554;Litamo;9240244;https://eo.wikipedia.org/w/index.php?oldid=9240244
"Rilate aktivismon, ""kudrorondo"" (en la angla sewing circle ) estas esprimo uzita de Marlene Dietrich , ekzemple por priskribi la submondan, fermitan etoson de lesbanaj kaj duseksemaj filmaktorinoj kaj de iliaj rilatoj en Holivudo , precipe dum la Orepoko de Holivudo de la 1910-aj ĝis la 1950-aj jaroj. Tiu uzo estis stampita de aktorino Alla Nazimova . Dum la unua talibana reĝimo en Afganujo ekzistis sekreta edukejo por virinoj kaŝita sub la nomo Kudrolernejo de la Ora Kudrilo .";ropa;646658;Kudrorondo;8293302;https://eo.wikipedia.org/w/index.php?oldid=8293302
Buko estas fermilo en la formo de metala ringo kun stifto , nomata la dorno, per kio oni povas fermi vesteron , zonon , ŝuon aŭ valizon . La dorno povas ĉarniri ĉirkaŭ akso kio estas muntata en la mezo de la ringo. La ringo povas havi multajn formojn, rektangula, ronda aŭ ovala plej ofte aperas.;ropa;648149;Buko;9071772;https://eo.wikipedia.org/w/index.php?oldid=9071772
Normale ĝi estas farita el plektita pajlo kaj havas rigidan platan kronon kaj randon, tipe kun solida aŭ striita rubando ĉirkaŭ la krono. Kanotĉapeloj estis popularaj kiel somera kapvesto, speciale por kanotado aŭ velado , tial la nomo. Ile estis verŝajne portitaj de FBI agentoj kiel speco da neoficiala uniformo dum la jaroj antaŭ la Unua Mondmilito . Ĝi ankaŭ estis portita de virinoj, ofte kun ĉapel pingloj por teni ĝin sur la ĝusta loko. Nuntempe ili estas rare vidata escepte dum veladaj aŭ remadaj eventoj, periodaj teatraj kaj muzikalaj spektakloj (ekz. Barberŝopa muziko ) aŭ kiel parto de malnovmodaj lernejaj uniformoj . Ekde 1952, La kanotĉapelo estis parto de la uniformo de la Princeton Universitato Muzikgrupo .;ropa;653325;Kanotĉapelo;6686274;https://eo.wikipedia.org/w/index.php?oldid=6686274
Hugo Boss AG , ofte markita simple kiel Boss , estas germana moddomo kaj marko kun ĉefsidejo en Metzingen , Baden-Württemberg , Germanio , kaj specialiĝanta pri alta modo por viroj. La firmao vendas vestaĵojn, akcesoraĵojn, piedvestojn kaj parfumoj. Hugo Boss estas unu el la grandaj germanaj vestaĵaj firmaoj, kun tutmondaj vendoj de 2.9 miliardoj da eŭroj en 2019 . Ties akcio estas ero de la MDAX .;ropa;659309;Hugo Boss;9254337;https://eo.wikipedia.org/w/index.php?oldid=9254337
La marko estas nomita laŭ Hugo Ferdinand Boss (1885-1948) kiu estis membro de la NSDAP kaj la dezajnisto de la uniformoj de la SS . Aktuale, la markoj de la moddomo estas distribuitaj en 103 landoj en la mondo kaj estas venditaj en pli ol 5,000 vendejoj . La produktoj, kiujn la kompanio produktas kaj distribuas, inkluzivas: vestojn , ŝuojn , parfumojn kaj modajn akcesoraĵojn .;ropa;659309;Hugo Boss;9254337;https://eo.wikipedia.org/w/index.php?oldid=9254337
La moddomo estas nomita laŭ Hugo Ferdinand Boss (1885-1948) kiu fondis la firmaon en 1923 en Metzingen, Germanio, kie ĝi daŭre funkcias . En 1924 li fondis fabrikon kune kun du partneroj. La firmao produktis ĉemizojn , jakojn , laborvestojn , sportajn vestaĵojn kaj pluvmantelojn . Pro la ekonomia klimato de Germanio tiutempe, Boss estis devigita en bankroton . En 1931 li atingis interkonsenton kun siaj kreditoroj kaj lasis ses kudromaŝinojn en siaj manoj por rekomenci la negoco .;ropa;659309;Hugo Boss;9254337;https://eo.wikipedia.org/w/index.php?oldid=9254337
Hugo Boss estis membro de la NSDAP ekde 1931 . Inter 1932-1945, profesoro Carl Dibitsch kaj Walter Heck dezajnis la nigrajn uniformojn por la SS kaj ili estis produktitaj fare de la Hugo Boss firmao. Trudlaboristoj ankaŭ estis dungitaj en la firmao (simila al multaj firmaoj kiuj kunlaboris kun la nazia reĝimo dum la milito) . Hugo Boss ankaŭ dezajnis kaj produktis la brunajn ĉemizojn por la SS , la naziaj ŝtormsoldatoj, kaj kelkaj el la SS-uniformoj. Krome, tiutempe la firmao okupiĝis pri la produktado de uniformoj por la poŝta servo kaj la polico en Germanio. En 1997, la firmao agnoskis la servojn kiujn ĝi disponigis al la nazia reĝimo kaj asertis ke ĝi estis unu el miloj da firmaoj kiuj disponigis similajn servojn dum Nazia Germanio .;ropa;659309;Hugo Boss;9254337;https://eo.wikipedia.org/w/index.php?oldid=9254337
"Post la milito, Hugo Ferdinand Boss estis malpermesita posedi komercon kiel rezulto de sia implikiĝo en la NSDAP. Lia bofilo Eugen Holley transprenis la proprieton kaj administradon de la firmao kaj produktis laborvestojn. En 1950, la firmao ricevis sian unuan mendon por viraj kostumoj, kaj kiel rezulto disetendiĝis al 150 dungitoj antaŭ la fino de la jaro. Ĝis 1960, la firmao produktis pretajn vestokompletojn laŭmendajn. En 1969, Eugen Holley demisiis kaj lasis la firmaon al siaj filoj, kiuj komencis evoluigi internaciajn agadojn. En 1970, la unuaj vestokompletoj estis produktitaj sub la ""Boss"" marko, kiam la marko iĝis registrita varmarko en 1977 . En tiuj jaroj, la firmao komencis kunlabori kun motorsportoj , sponsorante la Formulo 1 -ŝoforon, Niki Lauda , kaj poste la McLaren Groupn .";ropa;659309;Hugo Boss;9254337;https://eo.wikipedia.org/w/index.php?oldid=9254337
En 1989 , Boss lanĉis ŝian linion de sunokulvitroj . Poste en la jaro la firmao estis akirita fare de japana grupo .;ropa;659309;Hugo Boss;9254337;https://eo.wikipedia.org/w/index.php?oldid=9254337
En 1991 , la tekstila grupo Marazzoto aĉetis 77.5% de ĝiaj akcioj por 165 milionoj USD, la markoj Hugo kaj Baldessarini estis lanĉitaj en 1993. En 1995, la firmao lanĉis sian ŝuolinion.;ropa;659309;Hugo Boss;9254337;https://eo.wikipedia.org/w/index.php?oldid=9254337
En 1997 , la tegmentofirmao de Hugo Boss, la Marazzotto Group, kunfandis kun alia tekstila grupo en Italio por krei la plej grandan tekstilan korporacion en Eŭropo .;ropa;659309;Hugo Boss;9254337;https://eo.wikipedia.org/w/index.php?oldid=9254337
Komence de la 2000-aj jaroj, Procter & Gamble ricevis la licencon por produkti kaj surmerkatigi parfumojn sub la marko Hugo Boss .;ropa;659309;Hugo Boss;9254337;https://eo.wikipedia.org/w/index.php?oldid=9254337
La produktoj de la firmao estas surmerkatigitaj sub tri markoj, Boss Black por formala vestaĵo, Boss Orange por pli neformalaj kaj sportaj produktoj kaj Boss Green por sportvesto. Krome, la grupo inkluzivas la filion Baldessarini por pli ekskluzivaj produktoj.;ropa;659309;Hugo Boss;9254337;https://eo.wikipedia.org/w/index.php?oldid=9254337
Cicitoj estas judaj ceremoniaj ligiloj. Literumitaj ankaŭ kiel tzitzit en hebrea צִיצִית , plurale tsitsijot ) estas speciale ligitaj ritaj franĝoj , aŭ kvastoj , vestitaj en antikveco de Izraelidoj kaj nuntempe de praktikantaj Judoj kaj Samarianoj . Cicitoj estas ligitaj al la kvar anguloj de la tallit gadol , (preĝoŝalo) kutime referencata simple kiel tallit aŭ tallis , kaj tallit katan (ĉiutaga subvestaĵaro).;ropa;663441;Cicito;9407698;https://eo.wikipedia.org/w/index.php?oldid=9407698
Kimeŝeko ( kazaĥa : кимешек, karakalpako: кимешек) aŭ eleĉeko ( kirgiza : элечек) estas tradicia kapvestaĵo de pliaĝaj edzinitaj virinoj (avinoj) en Kazaĥio , Karakalpakstano kaj Kirgizio .;ropa;667627;Kimeŝeko;7025441;https://eo.wikipedia.org/w/index.php?oldid=7025441
Akademia vesto aŭ talaro estas tradicia formo de vesto uzita por akademiaj ceremonioj , ĉefe en triaranga edukado (kaj foje en kelkaj landoj ankaŭ en mezlernejoj ), vestita ĉefe fare de tiuj kiuj atingis universitatan gradigon (aŭ simile), aŭ tenas statuson kiu rajtigas ilin atribuiĝi per ĝi (ekz., subgradaj studentoj en kelkaj malnovaj universitatoj ).;ropa;675149;Akademia vesto;9055470;https://eo.wikipedia.org/w/index.php?oldid=9055470
Norma ĝinzo havas kvin poŝojn , du malantaŭe kaj du je la fronto. Je la dekstra frontflanko situas poŝeto, kie oni antaŭe konservis poŝhorloĝon . Ĉar la horloĝo tro suferis en normala poŝo la poŝeto speciale estis farata por tiu celo.;ropa;686767;Horloĝopoŝeto;9249999;https://eo.wikipedia.org/w/index.php?oldid=9249999
Plej ofte oni celas sporteman jakon el leĝera ŝtofo, kvankam la origina signifo estas (mararmeblua) jako kun oraj butonoj, inspirita je la 19-a jarcenta uniformojako de la Brita mararmeo (vidu bildon). La blazeroj tradicie havis duoblan vicon da butonoj sur la brusto, kvankam nuntempe unuopa vico da butonoj estas pli kutima.;ropa;686876;Blazero;8561968;https://eo.wikipedia.org/w/index.php?oldid=8561968
Lancetado (angle: bobbin lace, germane: Klöppeln ) estas maniero de kunigado de pli granda nombro (3-800) fadenoj per reciproka rekrucigado kaj tordigado. La lancetado estas farata kiel permana laboro aŭ en specialaj maŝinoj, kies celo estas produkto de plata travidebla tekstilo - punto .;ropa;690174;Lancetado;9318015;https://eo.wikipedia.org/w/index.php?oldid=9318015
La dhoti , konata ankaŭ kiel panĉe , vesti , dhuti , mardani , ĉaadra , dhotar aŭ panĉej , estas tipo de sarongo kiu deekstere similas al pantalono . Ĝi estas malalta vesto kiu formas parton de la nacia aŭ etna vesto de viroj en la Hinda subkontinento . La dhoti estas farita el ortangula peco de nekudrita tolo , kutime ĉirkaŭ 4.5 metrojn longa, volvita ĉirkaŭ la talio kaj la kruroj kaj nodita ĉu antaŭe ĉu malantaŭe. Dhoti estas de simplaj aŭ solidaj koloroj, silka dhoti kun borditaj bordoj estas konsiderata formala vesto, estas vestita en religiaj ceremonioj kaj en naci/etnaj festivaloj. Kontraste al dhoti, mult-kolora kaj jupeca sarongo nomita longji (lungi) estas vestita hejme kaj por hazarda ekstera promenado. Tamen estas esceptoj al tiu regulo en kelkaj sudbarataj komunumoj kie tutblanka longji (mundu) kun brodaĵo estas la etna formala vesto por viroj.;ropa;690242;Dhoti;7922058;https://eo.wikipedia.org/w/index.php?oldid=7922058
La origino de la Sapeo ne estas bone konata. La sapeistoj asertas, ke ilin inspiras la eŭropaj dandoj de la 19-a jarcento . Temis pri imitado de la kolonianojn per akaparo de ilia vesta stilo kaj iliaj kondutoj, unue por esti akceptitaj en iliaj medioj, sed ankaŭ por esti altestimataj de sia propra komunumo. La Sapeo vere populariĝis dum la 1960-aj jaroj en Brazavilo kaj Kinŝaso, kaj disvolviĝis en la kongaj diasporoj en Francio kaj Belgio .;ropa;691943;La Sapeo;9510850;https://eo.wikipedia.org/w/index.php?oldid=9510850
Ekzistas du specoj de sapeistoj. Unue la anoj de Christian Loubaki, kun la kodoj de burĝa dandismo de la 19-a kaj frua 20-a jarcentoj. La anoj de la dua kategorio estas membroj de la Socio de Etosistoj kaj Elegantaj Homoj, kiel Ley Mamadou priskribis en la jaroj 1983-84. La unuaj simple vestiĝas laŭ okcidentuloj, la duaj serĉas elmontri, eĉ kun akratune koloraj vestaĵoj.;ropa;691943;La Sapeo;9510850;https://eo.wikipedia.org/w/index.php?oldid=9510850
La konga komunumo estas perturbata, kaj tre kritikas la sapeistojn, konsideratajn kiel vivante ekster la reala vivo. La kulto de vesto praktikita de la sapeistoj do ŝajnas nekonvena, konsiderante ilian ofte malfortikan situacion. Aliaj akuzas ilin, spektakliĝi kaj elspezi grandajn monsumojn en grupoj vivantaj kun materiaj kaj manĝaj prizorgoj.;ropa;691943;La Sapeo;9510850;https://eo.wikipedia.org/w/index.php?oldid=9510850
"Hector Mediavilla fotis dum ok jaroj ĉefajn rolulojn de la konga Sapeo en sia serio ""Sapeo: Socio de Etosistoj kaj Elegantaj Homoj "". Alain Mabanckou , konga verkisto, ĉerpas la sekvan analizon el la verkoj de la fotisto :";ropa;691943;La Sapeo;9510850;https://eo.wikipedia.org/w/index.php?oldid=9510850
La filozofio de La Sapeo akompanas kun dek fundamentaj ordonoj, kiuj regas la konduton de sapeologoj kaj resumas iliajn valorojn:;ropa;691943;La Sapeo;9510850;https://eo.wikipedia.org/w/index.php?oldid=9510850
Aĵuro estas ornama tekniko uzata sur teksaĵoj, ofte ĉe la randoj. En aĵuroj, oni fortiras aŭ fortondas iom da fadenoj en teksaĵo, tiel kreante travideblan truon aŭ fadenmankon. Tio estas farata regule. Ofte, en la truo, oni faras plian ornamentadon, ekzemple per kunfaskigo aŭ kunnodado de kelkaj fadenoj, farante geometriajn formojn klare videblajn en la truo.;ropa;693467;Aĵuro;9440357;https://eo.wikipedia.org/w/index.php?oldid=9440357
La termino devenas de la franca termino ajour , kiu estas substantivigo de la sintagmo à jour “al tago”, ĉar aĵuroj en ekz. kurtenoj enlasas la tagan lumon.;ropa;693467;Aĵuro;9440357;https://eo.wikipedia.org/w/index.php?oldid=9440357
Jam fine de la 20-jarcento ili populariĝis en kelkaj orientaziaj landoj ĉefe kontraŭ poluado , kaj tutmonde pro evito de malsufero pro alergioj kaj evito de kontaĝo de diversaj malsanoj inter kiuj COVID-19 .;ropa;699794;Buŝnazmasko;7955943;https://eo.wikipedia.org/w/index.php?oldid=7955943
La kardiganoj povas esti desegnitaj por ajna tipo de uzantoj: viroj, virinoj, infanoj, beboj... Ili estas konsiderataj sportecaj vestoj, malpli formalaj ol plej el la ceteraj puloveroj kaj certe malpli ol konvenciaj jakoj . La trikitaj kardiganoj jam aperas en la vestaro de la tenisistoj, ambaŭsekse, en la 1910-aj jaroj . Kiel ĉe aliaj trikitaj vestoj, ili estas je la servo de la modo kaj ties populareco varias laŭlonge de la tempo.;ropa;700297;Kardigano;9327769;https://eo.wikipedia.org/w/index.php?oldid=9327769
Balenvosto ( angle : Whale tail) estas la Y-forma parto de tangao aŭ ŝnurkalsoneto videbla super la talirando de malalta ĝinzo , pantaloneto aŭ jupo, kiu similas al vosto de baleno. Popularigita de kelkaj inaj famuloj inkluzive de Christina Aguilera , Victoria Beckham , Mariah Carey , Paris Hilton kaj Britney Spears , montri balenajn vostojn populariĝis komence de la 2000-aj jaroj, kune kun la pliiĝo de malaltaj ĝinzoj kaj minimumecaj kalsonoj, sed malkreskis ene de la unua jardeko.;ropa;732518;Balenvosto;9135064;https://eo.wikipedia.org/w/index.php?oldid=9135064
"Lanĉapelo kaj foje ŝtrumpeta ĉapelo estas kapvesto trikita de lano kaj vestita super la kapo . Hodiaŭ la ĉapelo estas farita el lano aŭ fliso en rondeta ŝablono aŭ ""rabista"" bildigo, en kiu la randoj de la ĉapelo estas longaj kaj ĉirkaŭas la kapon de la supro ĝis la kolo kaj meze de unu flanko estas ronda truo por la vizaĝo .";ropa;739887;Lanĉapelo;8432988;https://eo.wikipedia.org/w/index.php?oldid=8432988
Jean-Jacques Picart helpis lin eklabori ĉe Hermès en 1978. Gradiĝinta en la École du Louvre , en 1981, li eklaboris por Jean Patou. Laŭlonge de la 1980-aj jaroj kun helpo de Jean-Jacques Picart, ili sukcesis en la kreado de propraj kolektoj de alta fasonado, junularecaj kaj ornamemaj. En 1989, Lacroix publikigis kolektojn de juveloj , dosierujoj, ŝuoj, okulvitroj, skarpoj kaj kravatoj. Samjare, li malfermis vendejojn en Parizo , Arles , Aix-en-Provence , Tuluzo , Londono , Ĝenevo kaj Japanio. La varomarko havis ĝis 60 vendejojn en Francio kaj totalon de 1 000 tra la tuta mondo. Vintre de 2007, li asociiĝis kun Avon Cosmetics por enkonduki novan ekskluzivecan parfumon , nome Christian Lacroix Rouge (ruĝa, por virinoj) kaj Christian Lacroix Noir (nigra, por viroj).;ropa;751009;Christian Lacroix;9458277;https://eo.wikipedia.org/w/index.php?oldid=9458277
Spite la famon kiun Christian Lacroix ĝuis dum la 1980-aj kaj 1990-aj jaroj, kaj la fakto, ke li estis konsiderata unu de la mododezajnistoj plej influaj kaj admiritaj, ĉio tio ne sufiĉis por superi la ekonomiajn problemojn de la firmao, la firmao perdis pli ol 44 milionoj da eŭroj. Lia lasta kolekto de alta vestindustrio, prezentita en julio 2009, markis la finon de la hegemonio en la moda etoso. Li devis fermi vendejojn kaj maldungi laboristojn.;ropa;751009;Christian Lacroix;9458277;https://eo.wikipedia.org/w/index.php?oldid=9458277
La D-ro No Bikino aŭ la blanka bikino de Ursula Andress kiu rolis kiel Honey Ryder en la filmo pri James Bond el 1962. La D-ro No bikino estas konata kiel la plej fama bikino de ĉiuj tempoj kaj ikoneca momento en kinematografia kaj modhistorio.;ropa;764247;D-ro No Bikino;7501154;https://eo.wikipedia.org/w/index.php?oldid=7501154
La blanka bikino de Ursula Andress estas rigardata kiel monumenta en la historio de la bikino, kaj vendo de la dupeca banvesto eksplodis post la apero de Andress en D-ro No. La malsupra parto de la bikino havas larĝan blankan britan armean zonon kun latuna buko kaj fiksaĵoj. Sur la maldekstra flanko estas glavingo por teni grandan tranĉilon.;ropa;764247;D-ro No Bikino;7501154;https://eo.wikipedia.org/w/index.php?oldid=7501154
"Brustligado aŭ mamligado estas platigo de mamoj pere de brustplatigiloj ( subvestoj kreitaj por tiu celo), sportaj mamzonoj aŭ volvita ĉirkaŭ brusto teksaĵo, elastano aŭ alia streĉebla materialo. Ofte oni ankaŭ portas plurajn ĉemizojn, unu super la alia, por atingi deziratan aspekton de plata brusto. La vorto mamoj povas kaŭzi genran disforion por pluraj personoj, tial preferindas diri brustligado kaj brustplatigilo aŭ simple ""platigilo""";ropa;764870;Brustligado;9455611;https://eo.wikipedia.org/w/index.php?oldid=9455611
En tiu ĉi artikolo estas uzita traduko de teksto el vikipedia artikolo Breast binding angle .;ropa;764870;Brustligado;9455611;https://eo.wikipedia.org/w/index.php?oldid=9455611
La stilo Hedera Ligo (angle: Ivy League ) estas fasonmoda stilo en Nordorienta Usono ekde la 1950-aj jaroj ĝis la mezo de 1960-aj jaroj , kaj oni diras ke ĝi devenis de kolegiaj kampusoj , precipe ĉe membroj de la Ivy League . Ekde la komenco de 1920-aj jaroj , la superaj sociaj klasanoj en Britio kaj Usono forigis surmeti tradician kostumon por praktiki la modajn sportojn en sia rondo (teniso, golfo, ktp.).;ropa;786322;Ivy League (vesto);9272122;https://eo.wikipedia.org/w/index.php?oldid=9272122
"Novaj vestaĵoj evoluigitaj per simpla teksado : la poloo per ĵerzo , golfaj pantalonoj per desegnaĵo "" Princo de Kimrujo "" aŭ bluzoj per peza kotono kaj ledojakoj . La vestaĵoj ne taŭgis al Eŭropo , dum usonanoj pli kaj pli uzis ĉi tiujn vestaĵojn, ĉar la Dua Mondmilito helpis modernigon de usona pretportaĵo .";ropa;786322;Ivy League (vesto);9272122;https://eo.wikipedia.org/w/index.php?oldid=9272122
En la 1950-aj jaroj, studentoj en la ok gravaj orientmarbordaj universitatoj en Usono, la Ivy League — inkluzivante Harvard kaj Universitato Yale — komencis uzi ĝin amase, forlasante la kostumojn ke ĝi estas sporta vestaro ( velura jako , oksfordaj ĉemizoj , mokasenoj , ktp.).;ropa;786322;Ivy League (vesto);9272122;https://eo.wikipedia.org/w/index.php?oldid=9272122
Gabardino estas teksaĵo inventita en 1879 de Thomas Burberry, kiu patentis ĝin en 1888 . La ŝtofo estas akvorezista kaj estas uzata por krei pluvmantelojn kaj aliajn vestaĵojn.;ropa;819119;Gabardino;9435601;https://eo.wikipedia.org/w/index.php?oldid=9435601
Burberry (antaŭe Burberry de Londono) estas brita altmoda moddomo kiu produktas vestaĵojn kaj modakcesoraĵojn . Burberry estas publika kompanio, kies akcioj estas komercitaj ĉe la London Stock Exchange , simbolo BRBY, kaj estas parto de la FTSE 100 Index .;ropa;819124;Burberry;9514704;https://eo.wikipedia.org/w/index.php?oldid=9514704
Burberry estis fondita en 1856 fare de Thomas Burberry (1835-1926), 21-jaraĝa juna viro kiu laboris kiel metilernanto por ŝtofkomercisto, malfermis sian propran butikon en Basingstoke , grandurbo en la nordoriento de Hampŝiro , Anglio .;ropa;819124;Burberry;9514704;https://eo.wikipedia.org/w/index.php?oldid=9514704
Ĝis la 1870-aj jaroj , la vendejo faris nomon por si kiel specialisto en la produktado de vestaĵoj destinitaj por subĉielaj agadoj. En 1880, Burberry inventis la gabardinŝtofon , fortan kaj veterrezistan ŝtofon, krom esti aera kaj komforta. En 1888 Burberry ricevis patenton por la gabardinŝtofo.;ropa;819124;Burberry;9514704;https://eo.wikipedia.org/w/index.php?oldid=9514704
En 1891 Burberry malfermis butikon sur Haymarket Street en Londono , Anglio, kiu funkcias ĝis hodiaŭ kaj ĝia konstruaĵo ankaŭ funkciis kiel la ĉefsidejo de la firmao.;ropa;819124;Burberry;9514704;https://eo.wikipedia.org/w/index.php?oldid=9514704
"En 1901, la firmaemblemo estis establita kaj registrita kiel varmarko : galopanta kavaliro sur ĉevalo, kaj la latina vorto Prorsum , kiu signifas ""antaŭen"". En 1911, la firmao disponigis ekipaĵon por Roald Amundsen sur lia vojaĝo al la suda poluso kaj en 1914, ĝi disponigis ekipaĵon por Ernest Shackleton kiu gvidis sian ekspedicion en krucado al Antarkto .";ropa;819124;Burberry;9514704;https://eo.wikipedia.org/w/index.php?oldid=9514704
Burberry estis sendependa firmao ĝis 1955 kiam ĝi estis aĉetita fare de Great Universal Stores. La firmao iĝis publika kaj ĝiaj akcioj komencis komerci sur la Londono-Borso en 2002. En 2005, Great Universal Stores vendis la reston de ĝiaj akcioj en la firmao.;ropa;819124;Burberry;9514704;https://eo.wikipedia.org/w/index.php?oldid=9514704
Jam en 1947, modaj aĵoj aĉetitaj de famuloj kiel ekzemple Jacqueline Kennedy kaj Grace Kelly estis lanĉitaj.;ropa;819125;Gucci;9243323;https://eo.wikipedia.org/w/index.php?oldid=9243323
En 1981 , la marko komencis okupiĝi pri vestaĵo kaj havis sukceson kun siaj novigaj dezajnoj. En 1994, kiam la valoro de la marko malpliiĝis, la juna usona dezajnisto Tom Ford eniris la artan administradon, kiu poziciigis la vestaĵojn en moderna kaj juna spirito kaj la kompanio denove prosperis. La seksaj kaj specialaj malĝojoj estis centro de altiro, kaj inter liaj klientoj estis la kantisto Madono .;ropa;819125;Gucci;9243323;https://eo.wikipedia.org/w/index.php?oldid=9243323
En 2003, la marko sub la administrado de Tom Ford lanĉis provokacian kampanjon enkonstruante la logoon en la plej modesta areo de la korpo de la modelo.;ropa;819125;Gucci;9243323;https://eo.wikipedia.org/w/index.php?oldid=9243323
Laŭ la revuo Time , la jara vendospezo de Gucci por 2007 estis proksimume 7,7 miliardoj da dolaroj kaj ĝi estis en la 46-a loko en la listo de la 100 plej grandaj markoj en la mondo. Laŭ la venda indico, Gucci estas la dua plej granda modmarko en la mondo post Giorgio Armani , kaj la plej granda itala modmarko en la mondo. Gucci havas pli ol 4021 butikojn tutmonde kaj ĝiaj produktoj eĉ estas venditaj pogrande al franĉizposedantoj kaj altaj modbutikoj.;ropa;819125;Gucci;9243323;https://eo.wikipedia.org/w/index.php?oldid=9243323
Calvin Klein Inc. estas usona modfirmao. La firmao estis konata en la 80-aj jaroj de la 20-a jarcento pro la subvestoj kaj ĝinzo kiujn ĝi produktis. Aldone al vestaĵoj por homoj de ĉiuj aĝoj, la firmao produktas ledvarojn , modakcesoraĵojn, ŝuojn , hejmajn meblojn , parfumojn , kosmetikaĵojn , juvelaĵojn kaj horloĝojn .;ropa;819128;Calvin Klein (firmao);9201669;https://eo.wikipedia.org/w/index.php?oldid=9201669
La firmao estis fondita en 1968 fare de Calvin Klein kaj Barry Schwartz, kaj en 2003 estis aĉetita de PVH por proksimume 700 milionoj da dolaroj. La ĉefsidejo de la firmao situas en Meza Manhatano , Novjorko .;ropa;819128;Calvin Klein (firmao);9201669;https://eo.wikipedia.org/w/index.php?oldid=9201669
"La firmao vastigis sian komercon preter Usono, kaj hodiaŭ la firmao ankaŭ havas filiojn en Eŭropo , Azio kaj Japanio . La firmao inkludas tri branĉojn: la Calvin Klein-kolekto por viroj, virinoj kaj hejmo, cK por viroj kaj virinoj, kaj ""cK ĝinzo"" por viroj, virinoj kaj infanoj.";ropa;819128;Calvin Klein (firmao);9201669;https://eo.wikipedia.org/w/index.php?oldid=9201669
Kelkaj el la novigaj reklamkampanjoj de la firmao, kiel ekzemple reklamoj de 1993 en kiuj Kate Moss prezentiĝis nuda en la Heroin Chic stilo, provokis kritikon, inkluzive de kritiko de la Prezidanto de Usono , Bill Clinton , por la uzo de ĉi tiu stilo, same kiel reklamoj de 1995 kiuj inkludis adoleskantojn en diversaj seksaj scenoj.;ropa;819128;Calvin Klein (firmao);9201669;https://eo.wikipedia.org/w/index.php?oldid=9201669
En 2003 , la firmao estis aĉetita fare de la firmao PVH por proksimume 700 milionoj da dolaroj.;ropa;819128;Calvin Klein (firmao);9201669;https://eo.wikipedia.org/w/index.php?oldid=9201669
Cartier estis fondita en Parizo en 1847 post kiam Louis-Fansois Cartier komencis administri la oraĵistlaborrenkontiĝon en kiu li specialiĝis. En 1874 , lia filo Alfred Cartier transprenis la administradon de la firmao kaj post li la tri filoj de Alfred komencis administri la firmaon: Ludoviko, Pierre kaj Jacques. Tiuj knaboj respondecas pri la florado de la firmao kaj akirado de ĝia pozicio kiel eminentaj oraĵistoj en la mondo.;ropa;819129;Cartier;8984170;https://eo.wikipedia.org/w/index.php?oldid=8984170
"En 1904, la brazila piloto Alberto Santos-Dumont plendis al sia amiko Louis Cartier pri la nefidindeco kaj efikeco de poŝhorloĝoj dum flugado. Cartier provis renkonti la defion kaj evoluigis platan brakhorloĝon kun kvadrata horciferdisko. Ĉi tiu horloĝo estis ŝatita de Santos-Dumont kaj eĉ de multaj aliaj klientoj de la kompanio. La modelo portas la nomon de la piloto ""Santos"" ĝis hodiaŭ. Ĝi estis la unua brakhorloĝo por viroj ( Patek Philippe kreis la unuan brakhorloĝon, ĝi estis ununura horloĝo farita speciale por virino).";ropa;819129;Cartier;8984170;https://eo.wikipedia.org/w/index.php?oldid=8984170
Ludoviko restis en pagendaĵo de la branĉo de la firmao en Parizo kaj estis translokigita en 1899 al Rue de la Paix, unu el la plej prestiĝaj stratoj en la grandurbo ĝis hodiaŭ. Ludoviko respondecis pri la dezajno de kelkaj el la famaj juvelaĵoj de la firmao. En 1907 , la firmao Cartier subskribis interkonsenton kun Edmond Jaeger, kiu jesis provizi la firmaon per la horloĝmekanismoj. En ĉi tiu etapo la firmao jam malfermis filiojn en Londono, Novjorko kaj Sankt-Peterburgo kaj iĝis unu el la plej sukcesaj horloĝfirmaoj en la mondo. En 1912 , la firmao prezentis la Baignoire kaj Tortue-horloĝmodelojn kiuj daŭre estas en produktado hodiaŭ. En 1917 , ĝi prezentis la Tank-modelon inspiritan de la nova militarmilo en la Unua Mondmilito : la tanko . Eĉ ĉi tiu modelo daŭre estas produktita en malsamaj versioj ĝis hodiaŭ.;ropa;819129;Cartier;8984170;https://eo.wikipedia.org/w/index.php?oldid=8984170
Jacques Cartier administris la branĉon de la firmao en Londono kaj movis ĝin al ĝia nuna hejmo sur Bond Street, la elstara butikstrato en Londono. En 1909 Pierre Cartier malfermis la filion de la firmao en New York kaj en 1917 ĝi moviĝis al Kvina Avenuo , al granda kaj luksa konstruaĵo kiun li aĉetis por 100 USD en kontantmono kaj duoblan ŝnuron de naturaj perloj kies valoro tiutempe estis taksita je 1 miliono USD .;ropa;819129;Cartier;8984170;https://eo.wikipedia.org/w/index.php?oldid=8984170
En la mezo de la 20-a jarcento, la kvara generacio de la Cartier-familio daŭre administris la firmaon. Post la morto de Pierre Cartier en 1964 , Jean-Jacques Cartier (filo de Jacques), Claude Cartier (filo de Ludoviko) kaj Marion Claudel (filino de Pierre), kiuj estris la filiojn de la firmao en Londono, Novjorko kaj Parizo, decidis vendi la kompanion;ropa;819129;Cartier;8984170;https://eo.wikipedia.org/w/index.php?oldid=8984170
Moschino estas itala luksa moda domo fondita en 1983 fare de Franco Moschino en Milano . Ĝi estas famkonata pro ĝia super-la-supra, kampa desegnaĵoj . La firmao specialigas en preta-al-vestiĝ-aĵoj, mansakoj kaj modaj akcesoraĵoj . Ĝi produktas kaj disvolvas vestaĵojn de viroj, virinoj kaj infanoj. Ĝi evoluigis la vestaĵojn uzitajn fare de Plínio Machado (la estro de brazila kantisto Anitta ), Kylie Minogue kaj Madonna en iliaj muzikaj koncertoj.;ropa;819148;Moschino;8984171;https://eo.wikipedia.org/w/index.php?oldid=8984171
La moderna dezajno estas unika trajto de la produktoj de la marko, kiuj ankoraŭ konserviĝas post la morto de Franco Moschino, sed la marko plifortiĝas ĉiutage. Lia ĉeesto estas memorita de grandaj nomoj kiel John Galliano kaj Giorgio Armani . La marko Moschino estis lanĉita en 1983 kaj la sekvan jaron la unua kolekto estis lanĉita. Dum dek jaroj Moschino establis sin sur la internacia modo sceno kiel unu el la plej aŭdacaj kaj novigaj nomoj en modo. La stilo de Moschino desegnas inspiron de ĝia pasio por teatro.;ropa;819148;Moschino;8984171;https://eo.wikipedia.org/w/index.php?oldid=8984171
Jean-Paul Gaultier (naskita la 24-an de aprilo 1952 , en Arquil, Francio ) estas franca modokreanto , kaj iama televida programestro.;ropa;819162;Jean Paul Gaultier;9192744;https://eo.wikipedia.org/w/index.php?oldid=9192744
Ĉe la fino de la 1980-aj jaroj, Gauthier perdis plurajn da siaj personaj parencoj, inkluzive de sia persona kaj komerca partnero Francis Menuge, kiu mortis pro komplikaĵoj rezultiĝantaj pro la aidosmalsano .;ropa;819162;Jean Paul Gaultier;9192744;https://eo.wikipedia.org/w/index.php?oldid=9192744
En la 1990-aj jaroj, Gauthier dezajnis la faman konusforman mamzonkostumon de Madonna , kaj multajn el ŝiaj turneokostumoj. Li ankaŭ dezajnis kelkajn el la kostumoj de rokkantisto Marilyn Manson por ŝiaj albumoj kaj koncertoj. Gauthier dezajnis la faman plumarrobon de la kantisto Dana International por ŝia prezento en la Eŭrovido-Kantokonkurso de 1998 .;ropa;819162;Jean Paul Gaultier;9192744;https://eo.wikipedia.org/w/index.php?oldid=9192744
Nuntempe li laboras samtempe pri siaj popularaj kaj elitaj modkolektoj, kaj pri la kolekto de la franca modokreanto Hermès , konata pro siaj ledaj vestaĵoj kaj luksaj mansakoj . En la lastaj jaroj, Gaultier ankaŭ publikigis kolektojn de sunokulvitroj .;ropa;819162;Jean Paul Gaultier;9192744;https://eo.wikipedia.org/w/index.php?oldid=9192744
"Aldone al moddezajno, Gaultier ankaŭ estas konata pro siaj popularaj parfumoj . Lia unua parfumo por virinoj, ""Classic"", estis lanĉita en 1993, kaj ĝia dezajno estis en la formo de virina korpo kun korseto . Du jarojn poste, la unua parfumo por viroj estis lanĉita, ""La Maille"", kies dezajno estis en la formo de supra korpo de viro kun striita marista ĉemizo. Ambaŭ estis tre sukcesaj, ekzemple ""La Maile"" estas nuntempe en unua loko en vendo de viraj parfumoj en Eŭropa Unio , kaj ankaŭ sukcesas en Aŭstralio kaj Usono . Lia tria parfumo, Fragile por virinoj, estis lanĉita en 2000, sed nun estas en limigita cirkulado pro relative malalta vendo. En 2005 la unuseksa parfumo ""Fragrance for Humanity"" estis publikigita, kaj en 2007 la vira parfumo ""Flour du Mile"". Ĉiuj parfumoj de Gaultier estas produktitaj laŭ longdaŭra licenco de la pariza fako de la japana kompanio Shiseido , kiu respondecas pri la produktado de parfumoj por aliaj dezajnistoj en la mondo.";ropa;819162;Jean Paul Gaultier;9192744;https://eo.wikipedia.org/w/index.php?oldid=9192744
Max Mara estas konata itala moddomo . La moddomo estis fondita en 1951 en la urbo Reggio Emilia fare de Achille Maramotti.;ropa;819216;Max Mara;9407937;https://eo.wikipedia.org/w/index.php?oldid=9407937
"La termino ""metroseksa"" estis origine kombinaĵo de la vortoj "" metropolano "" kaj "" aliseksemulo "", kaj celis priskribi certan tipon de aliseksemulo , kiu adoptis kondutojn kiuj devus esti ŝajne identigitaj laŭ la sociaj antaŭjuĝoj kun gejoj - ĉefe temas pri aliĝo al modo kaj konscio pri ekstera aspekto . Tamen, la termino ne asertas ĉu la metroseksema estas malsamseksema, geja aŭ ambaŭseksema viro kaj la sekso aŭ seksa orientiĝo de metroseksulo estas ĝenerale ambigua kaj malsamas en ĉiu viro al alia.";ropa;834369;Metroseksemo;8742264;https://eo.wikipedia.org/w/index.php?oldid=8742264
La termino estas konstruita sur la akceptita ŝablono por priskribi la seksan orientiĝon de homo kiel elpensite en la 19-a jarcento parte de la aŭstro-hungara ĵurnalisto Karl-Maria Kertbeny , kvankam en la nuna kazo la koncepto ne rilatas al seksa orientiĝo. La ideo sur kiu la koncepto baziĝas estas ke normoj de konduto kiel ekzemple sentemo, investo en vestaĵo kaj korpo kaj haŭtoprizorgo, kiuj estis parto de la samseksema stereotipo (kiu rezultiĝis el la antaŭjuĝoj pri inoj) fariĝis pli kaj pli akcepteblaj normoj de konduto por viroj sendepende de ilia seksa orientiĝo.;ropa;834369;Metroseksemo;8742264;https://eo.wikipedia.org/w/index.php?oldid=8742264
"Brita ĵurnalisto Mark Simpson elpensis la terminon kiel neologismo en 1994, en artikolo kiun li publikigis en The Independent . La termino akiris impulson post artikolo en la retejo Salon , kiu traktis la rolulon de la futbalisto David Beckham , kiu estas konsiderita la prototipo de la metroseksa viro. Alia artikolo en The New York Times , kiu estis publikigita en 2003 sub la titolo ""Metroseksuloj eliras el la ŝranko"" ankaŭ aldonis al la establado de la koncepto. Ekde tiam la termino havis sennombrajn aludojn kaj diskutojn en televidspektakloj kaj revuoj traktantaj libertempajn kaj distrajn aferojn. Interalie, la televida programo South Park dediĉis epizodon al la fenomeno.";ropa;834369;Metroseksemo;8742264;https://eo.wikipedia.org/w/index.php?oldid=8742264
Komence, la termino celis priskribi ĝeneralan ŝanĝon en la percepto de vireco de aliseksemaj urbaj viroj, sed poste ĝia signifo vastiĝis kaj nuntempe ĝi rilatas al viroj ĝenerale, sendepende de seksa orientiĝo . La primoda amaskomunikilaro kultivis la konsumantflankon de la termino, kaj kreis la bildugon de la aliseksema viro bezonanta varmarkojn kaj ĝustigproduktojn. Tiel la termino iĝis ilo por antaŭenigi la konsumon de modaj markoj kaj prizorgaj produktoj por kaj fare de aliseksemaj viroj.;ropa;834369;Metroseksemo;8742264;https://eo.wikipedia.org/w/index.php?oldid=8742264
La butontruoj estas fendoj farataj en la parto mala al la loko de la butono tiel ke pasigante tiun tra la butontruo kaj restigante ĝin je tiu kontraŭ flanko, la vestaĵo restas fermita kaj fiksita. Por ke la butontruoj ne disiĝu, oni kompletigas ilin per plifortigoj ĉu el tolo aŭ plej ofte el fadeno , ĉefe dika fadeno kaj maŝinkudrite por eviti fadendisiĝon kaj helpi pli elegantan rezulton.;ropa;855818;Butontruo;9107799;https://eo.wikipedia.org/w/index.php?oldid=9107799
Foje butontruo havas nur aŭ plie ornaman celon ĉu en brodado ĉu kiel la butontruo de roverso en jakoj kaj kostumoj, kiu laŭmode utilas por bori tra ĝi insignon, floron ktp.;ropa;855818;Butontruo;9107799;https://eo.wikipedia.org/w/index.php?oldid=9107799
Fermosistemo per butonoj laŭlonge de la 20-a jarcento estis iom post iom anstataŭita per aliaj pli modernaj sistemoj kiel zipo , lapfermilo ktp.;ropa;855818;Butontruo;9107799;https://eo.wikipedia.org/w/index.php?oldid=9107799
Fendi kunlaboris kelkfoje en kino dezajnante la vestaĵaron de filmoj de kulto kiel Once Upon a Time in America , Evita kaj The Royal Tenenbaums . Nombraj reĝisoroj de la 1970-aj jaroj elektis felojn de Fendi por siaj roluloj, por ekzemplo: Luchino Visconti , Federico Fellini , Franco Zeffirelli , kaj Mauro Bolognini . Fendi vestis ankaŭ kinajn stelulilojn kaj famulinojn kiel Sophia Loren , Diana Ross , Jacqueline Kennedy Onassis , Soraya , kaj Liza Minnelli .;ropa;856219;Fendi;9065390;https://eo.wikipedia.org/w/index.php?oldid=9065390
El riĉa familio, en 1969 edziniĝis (por dua fojo) kun la venezuala riĉulo Reinaldo Herrera, eldonisto de la revuo Vanity Fair . Ekde la 1960-aj jaroj ŝi ligiĝis al la usona alta klaso, ĉefe en Novjorko , kun tieaj diboĉuloj kiel tiuj de Studio 54, Andy Warhol , Diana Vreeland, Halston kaj Bianca Jagger . En la 1980-aj jaroj ŝi dezajnis vestojn por la plej altnivelaj kaj riĉaj virinoj de la tuta mondo, kiel Nicole Kidman , Anna Wintour, la reina Letizia , Meghan Markle , Renée Zellweger , Amy Adams , Jessica Alba , Oprah Winfrey , Jessica Simpson , Blake Lively , Angelina Jolie , Cynthia Nixon , Katie Holmes , Tina Fey , Dianna Agron , Julie Bowen , Michelle Pfeiffer , Robert Pattinson , Kim Kardashian , Katy Perry , Amber Heard , Kristen Stewart , Meryl Streep , Halle Berry , Beyonce , Salma Hayek , Shakira , Celine Dion , Karlie Kloss, Taylor Swift , Tyra Banks , Sofía Vergara , Lucy Liu , Lady Gaga , inter aliaj. Ŝi ankaŭ vendas produktojn de ŝminkado kaj parfumoj.;ropa;856229;Carolina Herrera;9153486;https://eo.wikipedia.org/w/index.php?oldid=9153486
Ekde 2008 ŝi eksportas al la tuta mondo pere de 50 butikoj kaj distribulinioj en 280 komerccentroj en 104 landoj.;ropa;856229;Carolina Herrera;9153486;https://eo.wikipedia.org/w/index.php?oldid=9153486
Motorcikla kasko aŭ motokasko , oficiale nomata protekta kasko , familiare nomata kraŝkasko , estas peco de protekta vestaĵo por motorciklantoj kaj protektas iliajn kapon kontraŭ vivminacaj vundoj en kazo de trafikakcidento . La portado de protekta kasko estas deviga por motorciklantoj en publika trafiko en la plej multaj landoj.;ropa;916242;Motorcikla kasko;9159863;https://eo.wikipedia.org/w/index.php?oldid=9159863
Kiel rimedo de pasiva protekto de la kapo de la ŝoforo aŭ ilia akompananto dum veturado per motorciklo aŭ skotero . Ĝi estas deviga por uzo de motorciklantoj kaj ŝoforoj de mopedoj , skoteroj, motorsledoj en la plej multaj landoj de la mondo, kaj estas uzata ankaŭ por potencaj duradaj elektraj ekipaĵoj: elektraj motorcikloj, elektraj skoteroj, elektraj mopedoj (eĉ elektraj bicikloj kaj elektraj skoteroj kun potenco pli granda ol 250 W). Krom protekti la kapon en kazo de akcidento, la kasko protektas kontraŭ alvenanta vento , akvo , ŝtonetoj , insektoj kaj polvo . La kasko ankaŭ devas esti fiksita.;ropa;916242;Motorcikla kasko;9159863;https://eo.wikipedia.org/w/index.php?oldid=9159863
Bela kasko povas krei certan bildigon, same kiel vestaĵoj kaj modaj akcesoraĵoj . Ekzistas pluraj variaĵoj de kaskoj. La uzo de unu aŭ alia tipo estas determinita de la personaj preferoj de la motorciklanto.;ropa;916242;Motorcikla kasko;9159863;https://eo.wikipedia.org/w/index.php?oldid=9159863
La protektaj ecoj de motorciklaj kaskoj estas reguligitaj de internaciaj kaj ŝtataj normoj, same kiel de regularoj de neregistaraj testlaboratorioj, kiuj provizas sendependan fakulan taksadon de la kvalito de kaskoj.;ropa;916242;Motorcikla kasko;9159863;https://eo.wikipedia.org/w/index.php?oldid=9159863
Bicikla kasko estas formo de kapprotekto por biciklantoj , kiu reduktas la fortojn agantajn sur la kranio n dum akcidento , tiel malhelpante aŭ mildigante vundojn. Ĝi estas pasiva trafiksekureca aparato.;ropa;916245;Biciklista kasko;9227039;https://eo.wikipedia.org/w/index.php?oldid=9227039
Biciklantoj kaj distraj biciklantoj origine biciklis nudkape aŭ nur kun ombroprovizanta ĉapo . Ekde ĉirkaŭ 1920, malmolŝelaj kaskoj, similaj al la malnovaj duonŝelaj kaskoj portataj de motorciklantoj, estis foje uzataj en vetkuroj.;ropa;916245;Biciklista kasko;9227039;https://eo.wikipedia.org/w/index.php?oldid=9227039
De la 1960-aj jaroj ĝis la 1990-aj jaroj, amatoraj biciklantoj portis nur kraŝĉapon (ankaŭ konatan kiel kraŝĉapo ). La remburitaj, led-kovritaj tuboj, kiuj kuris de antaŭe al malantaŭe, ripozis rekte sur la kapo. Male al kaskoj, la fleksebla konstruo ofte povus esti fleksita aŭ faldita al pli malgranda grandeco. Pro sia fleksebla dezajno, la kraŝrando estas konsiderata oferti nur limigitan protekton.;ropa;916245;Biciklista kasko;9227039;https://eo.wikipedia.org/w/index.php?oldid=9227039
La potencialo por aerdinamikaj plibonigoj en kapa aerfluo estis agnoskita pli frue, kaj la dezajno de biciklaj kaskoj, precipe por tempotestoj kaj trakvetkuro, estis akompanata de ventotunelaj testoj. La kreskanta populareco de biciklaj kaskoj ekster sportoj komenciĝis fine de la 1980-aj jaroj. La unuaj devigaj kaskoleĝoj estis enkondukitaj komence de la 1990-aj jaroj.;ropa;916245;Biciklista kasko;9227039;https://eo.wikipedia.org/w/index.php?oldid=9227039
Ĉirkaŭ 2012, la sistemo MIPS (Multi-Directional Impact Protection System - Multdirekta Impakta Protektosistemo) estis enkondukita sur la merkaton. Interna enigaĵo en la kasko povas moviĝi ĉirkaŭ 1 cm por absorbi rotaciajn movojn dum falo.;ropa;916245;Biciklista kasko;9227039;https://eo.wikipedia.org/w/index.php?oldid=9227039
Post la Unua Hindoĉina Milito en 1946, la kolonia kasko, ankaŭ konata sub la nomo pith helmet en la angla lingvo , tamen fariĝis simbolo de la Popola Armeo de Vjetnamio . Antaŭe simbolo de la eŭropa imperiismo, la kasko ŝanĝis signifon por la vjetnamanoj, fariĝante simbolo de libereco kaj sendependeco rilate al franca regado .;ropa;919334;Kolonia kasko;9418476;https://eo.wikipedia.org/w/index.php?oldid=9418476
Simile, post kiam Kenjo akiris sian sendependecon en 1963 , la provincaj administrantoj adoptis la uniformojn kaj reguligitajn ĉapelojn de la iamaj koloniistoj, tiel simbolante la reziston kontraŭ eŭropa imperiismo kaj la revenon al la kulturo kaj identeco de libera kaj aŭtonoma Kenjo .;ropa;919334;Kolonia kasko;9418476;https://eo.wikipedia.org/w/index.php?oldid=9418476
Bottega Veneta estas itala luksa primoda entrepreno. Ili havas sian sidejon en Milano , Italio . Ili produktas vestaĵojn, mansakojn, ŝuojn, akcesoraĵojn, juvelojn kaj parfumojn.;ropa;920102;Bottega Veneta;9185237;https://eo.wikipedia.org/w/index.php?oldid=9185237
"La ŝveljupoj de Balenciaga kaj liaj strangaj, virinecaj, tamen ""modernismaj"" siluetoj fariĝis la varmarko de la moddomo.";ropa;920198;Balenciaga;9185247;https://eo.wikipedia.org/w/index.php?oldid=9185247
La moddomo estis fondita de Alexander McQueen en 1992 . Lia debuta kolekto, Taxi Driver , referencis la samnoman filmon de Martin Scorsese . Ĉiu posta spektaklo karakteriziĝis per tre teatra aliro, kiu, kombinita kun la dezajno, kaŭzis grandan polemikon. Ĉi tio fariĝis la markostampo de la marko.;ropa;920421;Alexander McQueen;9441010;https://eo.wikipedia.org/w/index.php?oldid=9441010
En 2001, la moddomo aliĝis al la Gucci Grupo (nun Kering Grupo). McQueen poste vendis 51%-an parton en la kompanio.;ropa;920421;Alexander McQueen;9441010;https://eo.wikipedia.org/w/index.php?oldid=9441010
En februaro 2010, Alexander McQueen memmortigis sin. Lia delonga kunlaborantino, Sarah Burton , estis nomumita kreiva direktoro.;ropa;920421;Alexander McQueen;9441010;https://eo.wikipedia.org/w/index.php?oldid=9441010
'''

# Fase 2 – Librería de NLP para Esperanto

## 2.1. Reglas lingüísticas utilizadas

En el tema de Lingüística se explica que las palabras se componen de lexemas y morfemas. Los morfemas flexivos aportan información gramatical.

Para el Esperanto se aprovechan las terminaciones regulares recogidas en la guía:

| Terminación | Categoría |
|---|---|
| -o | Sustantivo |
| -a | Adjetivo |
| -e | Adverbio |
| -i | Verbo en infinitivo |
| -as, -is, -os, -us, -u | Formas verbales |
| -j | Plural |
| -n | Acusativo |

Para obtener la forma base, primero se retiran las marcas flexivas de plural y acusativo y después se identifica la terminación de la categoría. En los verbos conjugados se recupera el infinitivo sustituyendo la terminación verbal por -i.

Esto es una lematización basada en reglas morfológicas regulares, no un diccionario completo. No resuelve todas las ambigüedades ni analiza palabras derivadas complejas.





## 2.2. Tokenizador

El tema de NLP define la tokenización como la división del texto en unidades más pequeñas (tokens). Se utiliza una expresión regular para separar palabras, números y signos de puntuación.

Se incluyen las letras propias del Esperanto (ĉ, ĝ, ĥ, ĵ, ŝ, ŭ) para no dividir incorrectamente sus palabras.

In [ ]:
import re
PATRON_TOKEN = re.compile(
    r"[A-Za-zĈĉĜĝĤĥĴĵŜŝŬŭ]+|[0-9]+|[^\w\s]",
    flags=re.UNICODE
)

def tokenizar(texto):
    """Devuelve las palabras, números y signos encontrados en el texto."""
    if not isinstance(texto, str):
        raise TypeError("El texto debe ser una cadena de caracteres.")
    return PATRON_TOKEN.findall(texto)

ejemplo = "La bela robo estas nova. Ŝi portas ŝuojn!"
print(tokenizar(ejemplo))

['La', 'bela', 'robo', 'estas', 'nova', '.', 'Ŝi', 'portas', 'ŝuojn', '!']


## 2.3. Gestión de stopwords

El tema de NLP define las stopwords como palabras que no aportan información relevante para el análisis y que pueden eliminarse según el objetivo.

Se nos proporciona un recurso específico para Esperanto: vortoj.json. La función siguiente intenta descargarlo. Si no hay conexión o cambia el formato del recurso, utiliza una lista básica de respaldo para que el cuaderno pueda seguir ejecutándose. La lista de respaldo queda identificada como tal y no sustituye al recurso dado.

In [ ]:
URL_STOPWORDS = (
    "https://raw.githubusercontent.com/"
    "jparisu/nlp-esperantilo/main/resources/esperanto/vortoj.json"
)

STOPWORDS_RESPALDO = {
    "la", "kaj", "de", "en", "al", "por", "kun", "pri", "pro",
    "ĉe", "el", "per", "sen", "sur", "sub", "inter", "ĝis",
    "mi", "vi", "li", "ŝi", "ĝi", "ni", "ili", "oni", "si",
    "estas", "estis", "estos", "estu", "ke", "sed", "aŭ",
    "se", "ĉar", "kiel", "kiu", "tio", "ĉi", "ne"
}

def extraer_palabras_json(objeto):
    """Recoge cadenas de un JSON, tanto si contiene listas como diccionarios."""
    palabras = set()

    if isinstance(objeto, str):
        palabras.add(objeto.casefold())
    elif isinstance(objeto, list):
        for elemento in objeto:
            palabras.update(extraer_palabras_json(elemento))
    elif isinstance(objeto, dict):
        for valor in objeto.values():
            palabras.update(extraer_palabras_json(valor))

    return palabras

def cargar_stopwords():
    try:
        respuesta = requests.get(URL_STOPWORDS, timeout=10)
        respuesta.raise_for_status()
        contenido = respuesta.json()
        palabras = extraer_palabras_json(contenido)
        if palabras:
            print("Stopwords cargadas desde vortoj.json del profesor.")
            return palabras
    except (requests.RequestException, ValueError, json.JSONDecodeError):
        pass

    print("No se pudo descargar vortoj.json. Se utiliza la lista básica de respaldo.")
    return STOPWORDS_RESPALDO.copy()

STOPWORDS = cargar_stopwords()

def es_stopword(palabra):
    """Indica si la palabra está incluida en el conjunto de stopwords."""
    return palabra.casefold() in STOPWORDS

print("Número de stopwords:", len(STOPWORDS))
print("Ejemplos:", sorted(STOPWORDS)[:20])

Stopwords cargadas desde vortoj.json del profesor.
Número de stopwords: 352
Ejemplos: ['*supren* (upwards) is this word plus the directional *-n*.', '-ever', '-self', 'a content adjective, listed only as the root of the greeting form *bonan* (*bonan tagon*).', 'a content adjective, listed only as the root of the greeting form *feliĉan* (*feliĉan naskiĝtagon*).', 'a noun, like `miliono`. 10^9.', 'a noun, not a numeral: `du milionoj da homoj`.', 'about', 'above', 'accompaniment, as opposed to *per* (instrument).', 'according to', 'across', 'adiaŭ', 'adverb', 'adverbo', 'after', 'against', 'ah', 'ajn', 'al']


## 2.4. Lematizador

El tema de NLP diferencia el stemming de la lematización. El stemming elimina morfemas para obtener una raíz, la lematización busca la forma base teniendo en cuenta la categoría gramatical.

Por ello, no se devuelve simplemente una raíz cortada: en los verbos conjugados se reconstruye el infinitivo terminado en -i. Para sustantivos y adjetivos se eliminan las marcas de plural y acusativo, conservando la terminación propia de su categoría.

In [ ]:
TERMINACIONES_VERBALES = ("as", "is", "os", "us", "u")

def quitar_flexion_numero_caso(palabra):
    """Elimina -n (acusativo) y -j (plural), en ese orden."""
    resultado = palabra.casefold()

    if resultado.endswith("n") and len(resultado) > 1:
        resultado = resultado[:-1]

    if resultado.endswith("j") and len(resultado) > 1:
        resultado = resultado[:-1]

    return resultado

def lematizar_palabra(palabra):
    """Obtiene una forma base aproximada mediante reglas del Esperanto."""
    palabra = palabra.casefold()

    # Primero se retiran las marcas flexivas -n y -j.
    base = quitar_flexion_numero_caso(palabra)

    # Las formas conjugadas se convierten al infinitivo -i.
    for terminacion in TERMINACIONES_VERBALES:
        if base.endswith(terminacion) and len(base) > len(terminacion):
            return base[:-len(terminacion)] + "i"

    # El infinitivo ya está en su forma base.
    if base.endswith("i"):
        return base

    # Sustantivos, adjetivos y adverbios conservan su terminación de categoría.
    return base

ejemplos_lemas = ["robojn", "belajn", "portas", "portis", "portos", "porti"]
for palabra in ejemplos_lemas:
    print(palabra, "->", lematizar_palabra(palabra))

robojn -> robo
belajn -> bela
portas -> porti
portis -> porti
portos -> porti
porti -> porti


## 2.5. Etiquetador morfológico / gramatical (POS-tagging)

El POS-tagging consiste en asignar a cada palabra una categoría gramatical. El tema de Lingüística presenta, entre otras, las categorías sustantivo, adjetivo, verbo, adverbio, determinante, pronombre, preposición y conjunción.

Se usan dos tipos de reglas:
1. Palabras funcionales conocidas, como el artículo la o preposiciones frecuentes.
2. Terminaciones morfológicas del Esperanto para sustantivos, adjetivos, adverbios y verbos.

**Limitación:** los apuntes advierten que una palabra puede cambiar de categoría según el contexto. Este etiquetador utiliza reglas sobre la palabra aislada y, por tanto, es una aproximación, no realiza análisis sintáctico ni desambiguación contextual.

In [ ]:
ARTICULOS = {"la"}
PRONOMBRES = {
    "mi", "vi", "li", "ŝi", "ĝi", "ni", "ili", "oni", "si",
    "ĉi", "tiu", "kio", "kiu", "ĉio", "neniu"
}
PREPOSICIONES = {
    "al", "anstataŭ", "antaŭ", "apud", "ĉe", "ĉirkaŭ", "da",
    "de", "dum", "ekster", "el", "en", "ĝis", "inter", "je",
    "kontraŭ", "krom", "kun", "laŭ", "malantaŭ", "malgraŭ",
    "per", "po", "por", "post", "preter", "pri", "pro", "sen",
    "sub", "super", "sur", "tra", "trans"
}
CONJUNCIONES = {"kaj", "aŭ", "sed", "nek", "do"}
ADVERBIOS_COMUNES = {"ne", "jes", "tre", "ankoraŭ", "jam", "nur", "ankaŭ"}

def etiquetar_pos(palabra):
    """Asigna una etiqueta gramatical mediante reglas y listas de palabras."""
    palabra = palabra.casefold()

    # Primero se comprueban palabras funcionales conocidas.
    if palabra in ARTICULOS:
        return "DET"
    if palabra in PRONOMBRES:
        return "PRON"
    if palabra in PREPOSICIONES:
        return "ADP"
    if palabra in CONJUNCIONES:
        return "CCONJ"
    if palabra in ADVERBIOS_COMUNES:
        return "ADV"

    # Se retiran las marcas de plural y acusativo antes de mirar la categoría.
    base = quitar_flexion_numero_caso(palabra)

    if base.endswith("o"):
        return "NOUN"
    if base.endswith("a"):
        return "ADJ"
    if base.endswith("e"):
        return "ADV"
    if base.endswith(("i", "as", "is", "os", "us", "u")):
        return "VERB"

    if palabra.isdigit():
        return "NUM"
    if re.fullmatch(r"[^\w\s]", palabra, flags=re.UNICODE):
        return "PUNCT"

    return "X"

ejemplos_pos = ["la", "robo", "ŝuoj", "bela", "belajn", "rapide", "porti", "portas", "kun"]
for palabra in ejemplos_pos:
    print(palabra, "->", etiquetar_pos(palabra))

la -> DET
robo -> NOUN
ŝuoj -> NOUN
bela -> ADJ
belajn -> ADJ
rapide -> ADV
porti -> VERB
portas -> VERB
kun -> ADP


## 2.6. Objetos Token y Doc

Se crean dos clases propias:

- Token: representa una unidad del texto y guarda su texto original, lema, etiqueta gramatical y si es stopword.
- Doc: representa el documento completo y contiene una lista de objetos Token.



In [ ]:
class Token:
    """Representa un token con sus atributos lingüísticos."""

    def __init__(self, texto):
        self.text = texto
        self.lemma_ = lematizar_palabra(texto)
        self.pos_ = etiquetar_pos(texto)
        self.is_stop = es_stopword(texto)

    def __repr__(self):
        return (
            f"Token(text={self.text!r}, lemma_={self.lemma_!r}, "
            f"pos_={self.pos_!r}, is_stop={self.is_stop})"
        )


class Doc:
    """Representa un documento como una secuencia de tokens."""

    def __init__(self, texto):
        self.text = texto
        self.tokens = [Token(palabra) for palabra in tokenizar(texto)]

    def __iter__(self):
        return iter(self.tokens)

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, posicion):
        return self.tokens[posicion]


def nlp(texto):
    """Procesa un texto y devuelve un objeto Doc."""
    return Doc(texto)

doc = nlp("La bela robo estas nova. Ŝi portas ŝuojn!")
for token in doc:
    print(token)

Token(text='La', lemma_='la', pos_='DET', is_stop=True)
Token(text='bela', lemma_='bela', pos_='ADJ', is_stop=False)
Token(text='robo', lemma_='robo', pos_='NOUN', is_stop=False)
Token(text='estas', lemma_='esti', pos_='VERB', is_stop=False)
Token(text='nova', lemma_='nova', pos_='ADJ', is_stop=False)
Token(text='.', lemma_='.', pos_='PUNCT', is_stop=False)
Token(text='Ŝi', lemma_='ŝi', pos_='PRON', is_stop=True)
Token(text='portas', lemma_='porti', pos_='VERB', is_stop=False)
Token(text='ŝuojn', lemma_='ŝuo', pos_='NOUN', is_stop=False)
Token(text='!', lemma_='!', pos_='PUNCT', is_stop=False)


## 2.7. Funciones de consulta y filtrado

Se añade una función para obtener los tokens que no son stopwords. Esta operación permite preparar posteriormente el texto para otras tareas de NLP, como la representación del corpus que se desarrollará en la Fase 3.

In [ ]:
def palabras_relevantes(doc):
    """Devuelve los tokens que no están marcados como stopwords ni puntuación."""
    return [
        token.lemma_
        for token in doc
        if not token.is_stop and token.pos_ != "PUNCT"
    ]

texto_prueba = "La bela robo estas nova kaj ŝi portas ŝuojn."
doc_prueba = nlp(texto_prueba)

print("Texto original:", doc_prueba.text)
print("Palabras relevantes:", palabras_relevantes(doc_prueba))

Texto original: La bela robo estas nova kaj ŝi portas ŝuojn.
Palabras relevantes: ['bela', 'robo', 'esti', 'nova', 'porti', 'ŝuo']


## 2.8. Pruebas unitarias

El enunciado exige que la librería esté acompañada de pruebas. Se utiliza unittest, incluido en Python. Las pruebas comprueban el comportamiento esperado de la tokenización, las reglas morfológicas, el POS-tagging, las stopwords y los objetos Doc/Token.

In [ ]:
import unittest

class TestEsperantoNLP(unittest.TestCase):

    def test_tokenizador_palabras_y_puntuacion(self):
        self.assertEqual(
            tokenizar("La robo!"),
            ["La", "robo", "!"]
        )

    def test_tokenizador_letras_esperanto(self):
        self.assertEqual(
            tokenizar("ŝuo ĉapelo"),
            ["ŝuo", "ĉapelo"]
        )

    def test_lematizar_sustantivo_plural_acusativo(self):
        self.assertEqual(lematizar_palabra("robojn"), "robo")

    def test_lematizar_adjetivo_plural_acusativo(self):
        self.assertEqual(lematizar_palabra("belajn"), "bela")

    def test_lematizar_verbo_presente(self):
        self.assertEqual(lematizar_palabra("portas"), "porti")

    def test_lematizar_verbo_pasado(self):
        self.assertEqual(lematizar_palabra("portis"), "porti")

    def test_pos_sustantivo(self):
        self.assertEqual(etiquetar_pos("robo"), "NOUN")

    def test_pos_adjetivo(self):
        self.assertEqual(etiquetar_pos("bela"), "ADJ")

    def test_pos_verbo(self):
        self.assertEqual(etiquetar_pos("portas"), "VERB")

    def test_pos_preposicion(self):
        self.assertEqual(etiquetar_pos("kun"), "ADP")

    def test_doc_contiene_tokens(self):
        documento = nlp("La robo.")
        self.assertEqual(len(documento), 3)
        self.assertEqual(documento[1].text, "robo")

    def test_token_guarda_atributos(self):
        token = Token("robo")
        self.assertEqual(token.lemma_, "robo")
        self.assertEqual(token.pos_, "NOUN")
        self.assertIsInstance(token.is_stop, bool)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestEsperantoNLP)
resultado = unittest.TextTestRunner(verbosity=2).run(suite)

assert resultado.wasSuccessful(), "Hay pruebas que no se han superado."

test_doc_contiene_tokens (__main__.TestEsperantoNLP.test_doc_contiene_tokens) ... ok
test_lematizar_adjetivo_plural_acusativo (__main__.TestEsperantoNLP.test_lematizar_adjetivo_plural_acusativo) ... ok
test_lematizar_sustantivo_plural_acusativo (__main__.TestEsperantoNLP.test_lematizar_sustantivo_plural_acusativo) ... ok
test_lematizar_verbo_pasado (__main__.TestEsperantoNLP.test_lematizar_verbo_pasado) ... ok
test_lematizar_verbo_presente (__main__.TestEsperantoNLP.test_lematizar_verbo_presente) ... ok
test_pos_adjetivo (__main__.TestEsperantoNLP.test_pos_adjetivo) ... ok
test_pos_preposicion (__main__.TestEsperantoNLP.test_pos_preposicion) ... ok
test_pos_sustantivo (__main__.TestEsperantoNLP.test_pos_sustantivo) ... ok
test_pos_verbo (__main__.TestEsperantoNLP.test_pos_verbo) ... ok
test_token_guarda_atributos (__main__.TestEsperantoNLP.test_token_guarda_atributos) ... ok
test_tokenizador_letras_esperanto (__main__.TestEsperantoNLP.test_tokenizador_letras_esperanto) ... ok
test_toke

## 2.9. Crear los archivos de la librería

Para que la librería pueda instalarse desde GitHub, se genera una estructura mínima de paquete Python. Estos archivos se deben incorporar al repositorio del equipo.

Estructura generada:

```text
esperanto_nlp/
    __init__.py
    nlp.py
setup.py
README.md
```

La librería queda separada del dataset: sus funciones no dependen de las categorías ropa, calzado y joyas, por lo que puede procesar textos de otras temáticas.

In [ ]:
from pathlib import Path

CARPETA_LIBRERIA = Path("esperanto_nlp")
CARPETA_LIBRERIA.mkdir(exist_ok=True)

CODIGO_LIBRERIA = r'''
import re
import json
import requests

PATRON_TOKEN = re.compile(
    r"[A-Za-zĈĉĜĝĤĥĴĵŜŝŬŭ]+|[0-9]+|[^\w\s]",
    flags=re.UNICODE
)

URL_STOPWORDS = (
    "https://raw.githubusercontent.com/"
    "jparisu/nlp-esperantilo/main/resources/esperanto/vortoj.json"
)

STOPWORDS_RESPALDO = {
    "la", "kaj", "de", "en", "al", "por", "kun", "pri", "pro",
    "ĉe", "el", "per", "sen", "sur", "sub", "inter", "ĝis",
    "mi", "vi", "li", "ŝi", "ĝi", "ni", "ili", "oni", "si",
    "estas", "estis", "estos", "estu", "ke", "sed", "aŭ",
    "se", "ĉar", "kiel", "kiu", "tio", "ĉi", "ne"
}

def _extraer_palabras(objeto):
    palabras = set()
    if isinstance(objeto, str):
        palabras.add(objeto.casefold())
    elif isinstance(objeto, list):
        for elemento in objeto:
            palabras.update(_extraer_palabras(elemento))
    elif isinstance(objeto, dict):
        for valor in objeto.values():
            palabras.update(_extraer_palabras(valor))
    return palabras

def _cargar_stopwords():
    try:
        respuesta = requests.get(URL_STOPWORDS, timeout=10)
        respuesta.raise_for_status()
        palabras = _extraer_palabras(respuesta.json())
        if palabras:
            return palabras
    except (requests.RequestException, ValueError):
        pass
    return STOPWORDS_RESPALDO.copy()

STOPWORDS = _cargar_stopwords()

def tokenizar(texto):
    if not isinstance(texto, str):
        raise TypeError("El texto debe ser una cadena.")
    return PATRON_TOKEN.findall(texto)

def es_stopword(palabra):
    return palabra.casefold() in STOPWORDS

def _quitar_flexion(palabra):
    resultado = palabra.casefold()
    if resultado.endswith("n") and len(resultado) > 1:
        resultado = resultado[:-1]
    if resultado.endswith("j") and len(resultado) > 1:
        resultado = resultado[:-1]
    return resultado

def lematizar_palabra(palabra):
    base = _quitar_flexion(palabra)
    for terminacion in ("as", "is", "os", "us", "u"):
        if base.endswith(terminacion) and len(base) > len(terminacion):
            return base[:-len(terminacion)] + "i"
    return base

ARTICULOS = {"la"}
PRONOMBRES = {
    "mi", "vi", "li", "ŝi", "ĝi", "ni", "ili", "oni", "si",
    "ĉi", "tiu", "kio", "kiu", "ĉio", "neniu"
}
PREPOSICIONES = {
    "al", "anstataŭ", "antaŭ", "apud", "ĉe", "ĉirkaŭ", "da",
    "de", "dum", "ekster", "el", "en", "ĝis", "inter", "je",
    "kontraŭ", "krom", "kun", "laŭ", "malantaŭ", "malgraŭ",
    "per", "po", "por", "post", "preter", "pri", "pro", "sen",
    "sub", "super", "sur", "tra", "trans"
}
CONJUNCIONES = {"kaj", "aŭ", "sed", "nek", "do"}
ADVERBIOS_COMUNES = {"ne", "jes", "tre", "ankoraŭ", "jam", "nur", "ankaŭ"}

def etiquetar_pos(palabra):
    palabra = palabra.casefold()
    if palabra in ARTICULOS:
        return "DET"
    if palabra in PRONOMBRES:
        return "PRON"
    if palabra in PREPOSICIONES:
        return "ADP"
    if palabra in CONJUNCIONES:
        return "CCONJ"
    if palabra in ADVERBIOS_COMUNES:
        return "ADV"
    base = _quitar_flexion(palabra)
    if base.endswith("o"):
        return "NOUN"
    if base.endswith("a"):
        return "ADJ"
    if base.endswith("e"):
        return "ADV"
    if base.endswith(("i", "as", "is", "os", "us", "u")):
        return "VERB"
    if palabra.isdigit():
        return "NUM"
    if re.fullmatch(r"[^\w\s]", palabra, flags=re.UNICODE):
        return "PUNCT"
    return "X"

class Token:
    def __init__(self, texto):
        self.text = texto
        self.lemma_ = lematizar_palabra(texto)
        self.pos_ = etiquetar_pos(texto)
        self.is_stop = es_stopword(texto)

    def __repr__(self):
        return (
            f"Token(text={self.text!r}, lemma_={self.lemma_!r}, "
            f"pos_={self.pos_!r}, is_stop={self.is_stop})"
        )

class Doc:
    def __init__(self, texto):
        self.text = texto
        self.tokens = [Token(p) for p in tokenizar(texto)]

    def __iter__(self):
        return iter(self.tokens)

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, posicion):
        return self.tokens[posicion]

def nlp(texto):
    return Doc(texto)

def palabras_relevantes(doc):
    return [
        token.lemma_ for token in doc
        if not token.is_stop and token.pos_ != "PUNCT"
    ]
'''

(CARPETA_LIBRERIA / "nlp.py").write_text(CODIGO_LIBRERIA, encoding="utf-8")
(CARPETA_LIBRERIA / "__init__.py").write_text(
    "from .nlp import Doc, Token, nlp, tokenizar, lematizar_palabra, "
    "etiquetar_pos, es_stopword, palabras_relevantes\n",
    encoding="utf-8"
)

Path("setup.py").write_text(
    "from setuptools import setup, find_packages\n\n"
    "setup(name='esperanto-nlp-equipo', version='0.1.0', "
    "packages=find_packages(), install_requires=['requests'])\n",
    encoding="utf-8"
)

Path("README.md").write_text(
    "# Esperanto NLP\n\n"
    "Librería propia de NLP basada en reglas para Esperanto.\n\n"
    "Incluye tokenización, lematización aproximada, POS-tagging, "
    "stopwords y objetos Doc/Token.\n\n"
    "No utiliza spaCy, NLTK ni Stanza.\n\n"
    "## Uso\n\n"
    "```python\n"
    "from esperanto_nlp import nlp\n"
    "doc = nlp('La bela robo estas nova.')\n"
    "for token in doc:\n"
    "    print(token.text, token.lemma_, token.pos_, token.is_stop)\n"
    "```\n",
    encoding="utf-8"
)

print("Archivos creados:")
for archivo in [CARPETA_LIBRERIA / "nlp.py",
                CARPETA_LIBRERIA / "__init__.py",
                Path("setup.py"), Path("README.md")]:
    print("-", archivo)

Archivos creados:
- esperanto_nlp/nlp.py
- esperanto_nlp/__init__.py
- setup.py
- README.md


In [2]:

import zipfile
from google.colab import files

# Crear un ZIP con los archivos de nuestra librería
with zipfile.ZipFile("libreria_esperanto.zip", "w") as zipf:
    zipf.write("esperanto_nlp/nlp.py")
    zipf.write("esperanto_nlp/__init__.py")
    zipf.write("setup.py")
    zipf.write("README.md")

# Descargar el ZIP al ordenador
files.download("libreria_esperanto.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 2.10. Prueba de instalación local

Se instala el paquete generado en modo editable y se comprueba que puede importarse. En el repositorio público, el equipo deberá subir la carpeta esperanto_nlp, setup.py y README.md. Una vez publicado, podrá instalarse desde GitHub siguiendo el formato indicado en el enunciado.

In [ ]:
%pip install -e . --quiet

from esperanto_nlp import nlp as nlp_instalada

doc_instalado = nlp_instalada("La bela robo.")
[(t.text, t.lemma_, t.pos_, t.is_stop) for t in doc_instalado]

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for esperanto-nlp-equipo (pyproject.toml) ... done


[('La', 'la', 'DET', True),
 ('bela', 'bela', 'ADJ', False),
 ('robo', 'robo', 'NOUN', False),
 ('.', '.', 'PUNCT', False)]

## Referencias utilizadas

- Apuntes de clase **MT_02_01 – Lingüística**, apartado 2: morfología, lexemas, morfemas y categorización.
- Apuntes de clase **MT_02_02 – Natural Language Processing**, apartados 2.1–2.7: división, tokenización, stopwords, stemming, lematización y POS-tagging.
- Apuntes de clase **MT_01_03 – Strings**, apartado 6: expresiones regulares.
- Enunciado de la práctica, apartado 4.2: requisitos mínimos de la librería.
- Guía: [NLP Esperantilo](https://jparisu.github.io/nlp-esperantilo/).
- Recurso de stopwords: [vortoj.json](https://github.com/jparisu/nlp-esperantilo/blob/main/resources/esperanto/vortoj.json).